# Notebook 01 · Build the canonical corpus and verify API families

Fresh-run release. Follow the input and review gates in RUN_ORDER.md.

In [1]:
from pathlib import Path
import json
import os
import platform
import sys


CORPUS_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/"
        "anonuser01/apis-dataset/APIs"
    ),
    Path(
        "/kaggle/input/"
        "apis-dataset/APIs"
    ),
]



STAGE_00_CANDIDATES = [
    Path("/kaggle/input/datasets/anonuser01/stage-00"),
    Path("/kaggle/input/stage-00"),
    Path(
        "/kaggle/input/datasets/"
        "anonuser01/"
        "prose-operation-stage-00-runtime"
    ),
    Path(
        "/kaggle/input/"
        "prose-operation-stage-00-runtime"
    ),
]


CORPUS_ROOT = next(
    (
        path
        for path in CORPUS_CANDIDATES
        if path.is_dir()
    ),
    None,
)

STAGE_00_DATASET_ROOT = next(
    (
        path
        for path in STAGE_00_CANDIDATES
        if path.is_dir()
    ),
    None,
)


if CORPUS_ROOT is None:
    raise FileNotFoundError(
        "The original APIs dataset was not "
        "found. Checked:\n"
        + "\n".join(
            f"- {path}"
            for path in CORPUS_CANDIDATES
        )
    )

if STAGE_00_DATASET_ROOT is None:
    raise FileNotFoundError(
        "The Stage 00 dataset was not found. "
        "Checked:\n"
        + "\n".join(
            f"- {path}"
            for path
            in STAGE_00_CANDIDATES
        )
    )


external_manifest_candidates = sorted(
    STAGE_00_DATASET_ROOT.rglob(
        "stage_00_manifest.json"
    ),
    key=lambda path: len(path.parts),
)

internal_manifest_candidates = sorted(
    STAGE_00_DATASET_ROOT.rglob(
        "00_manifest.json"
    ),
    key=lambda path: len(path.parts),
)


if not external_manifest_candidates:
    raise FileNotFoundError(
        "Stage 00 external manifest "
        "was not found."
    )

if not internal_manifest_candidates:
    raise FileNotFoundError(
        "Stage 00 internal manifest "
        "was not found."
    )


STAGE_00_EXTERNAL_MANIFEST_PATH = (
    external_manifest_candidates[0]
)

STAGE_00_INTERNAL_MANIFEST_PATH = (
    internal_manifest_candidates[0]
)

STAGE_00_PROJECT_ROOT = (
    STAGE_00_INTERNAL_MANIFEST_PATH
    .parent
    .parent
)


stage_00_external_manifest = json.loads(
    STAGE_00_EXTERNAL_MANIFEST_PATH
    .read_text(encoding="utf-8")
)

stage_00_internal_manifest = json.loads(
    STAGE_00_INTERNAL_MANIFEST_PATH
    .read_text(encoding="utf-8")
)


external_status = (
    stage_00_external_manifest.get(
        "status"
    )
)

internal_status = (
    stage_00_internal_manifest.get(
        "stage_status"
    )
)


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

STAGE_01_OUTPUT_ROOT = Path(
    "/kaggle/working/stage_01_fresh_dataset"
)


provider_directory_count = sum(
    1
    for path in CORPUS_ROOT.iterdir()
    if path.is_dir()
)


cell_1_report = {
    "python_version": (
        sys.version.split()[0]
    ),

    "platform": platform.platform(),

    "kaggle_kernel_run_type": (
        os.environ.get(
            "KAGGLE_KERNEL_RUN_TYPE"
        )
    ),

    "corpus_root": str(
        CORPUS_ROOT
    ),

    "provider_directory_count": (
        provider_directory_count
    ),

    "stage_00_dataset_root": str(
        STAGE_00_DATASET_ROOT
    ),

    "stage_00_project_root": str(
        STAGE_00_PROJECT_ROOT
    ),

    "external_manifest": str(
        STAGE_00_EXTERNAL_MANIFEST_PATH
    ),

    "external_status": (
        external_status
    ),

    "internal_manifest": str(
        STAGE_00_INTERNAL_MANIFEST_PATH
    ),

    "internal_status": (
        internal_status
    ),

    "project_root": str(
        PROJECT_ROOT
    ),

    "stage_01_output_root": str(
        STAGE_01_OUTPUT_ROOT
    ),

    "status": (
        "PASS"
        if (
            external_status == "PASS"
            and internal_status == "PASS"
        )
        else "BLOCKED"
    ),
}


print(
    json.dumps(
        cell_1_report,
        indent=2,
    )
)


if external_status != "PASS":
    raise RuntimeError(
        "Stage 00 external status "
        "is not PASS."
    )

if internal_status != "PASS":
    raise RuntimeError(
        "Stage 00 internal status "
        "is not PASS."
    )


print("\nNOTEBOOK 01 — CELL 1: PASS")

{
  "python_version": "3.13.16",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "kaggle_kernel_run_type": "Interactive",
  "corpus_root": "/kaggle/input/datasets/anonuser01/apis-dataset/APIs",
  "provider_directory_count": 701,
  "stage_00_dataset_root": "/kaggle/input/datasets/anonuser01/stage-00",
  "stage_00_project_root": "/kaggle/input/datasets/anonuser01/stage-00/stage_00_protocol_runtime_v1/prose_operation_experiment",
  "external_manifest": "/kaggle/input/datasets/anonuser01/stage-00/stage_00_manifest.json",
  "external_status": "PASS",
  "internal_manifest": "/kaggle/input/datasets/anonuser01/stage-00/stage_00_protocol_runtime_v1/prose_operation_experiment/registries/00_manifest.json",
  "internal_status": "PASS",
  "project_root": "/kaggle/working/fresh/stage_01/prose_operation_experiment",
  "stage_01_output_root": "/kaggle/working/stage_01_fresh_dataset",
  "status": "PASS"
}

NOTEBOOK 01 — CELL 1: PASS


In [2]:
import hashlib


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(
        data
    ).hexdigest()


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while chunk := handle.read(
            chunk_size
        ):
            digest.update(chunk)

    return digest.hexdigest()


stage_00_verification_problems = []

stage_00_artifacts = (
    stage_00_internal_manifest.get(
        "artifacts",
        [],
    )
)


for artifact in stage_00_artifacts:
    relative_path = artifact[
        "artifact_path"
    ]

    artifact_path = (
        STAGE_00_PROJECT_ROOT
        / relative_path
    )

    if not artifact_path.is_file():
        stage_00_verification_problems.append(
            f"missing:{relative_path}"
        )
        continue

    actual_size = (
        artifact_path.stat().st_size
    )

    if actual_size != artifact["bytes"]:
        stage_00_verification_problems.append(
            f"size_mismatch:{relative_path}"
        )

    actual_hash = sha256_file(
        artifact_path
    )

    if actual_hash != artifact["sha256"]:
        stage_00_verification_problems.append(
            f"hash_mismatch:{relative_path}"
        )


actual_internal_manifest_hash = (
    sha256_file(
        STAGE_00_INTERNAL_MANIFEST_PATH
    )
)

expected_internal_manifest_hash = (
    stage_00_external_manifest.get(
        "internal_manifest_sha256"
    )
)

if (
    expected_internal_manifest_hash
    and actual_internal_manifest_hash
    != expected_internal_manifest_hash
):
    stage_00_verification_problems.append(
        "internal_manifest_hash_mismatch"
    )


stage_00_config_paths = sorted(
    (
        path
        for path in (
            STAGE_00_PROJECT_ROOT
            / "config"
        ).rglob("*")
        if path.is_file()
    ),
    key=lambda path: (
        path.relative_to(
            STAGE_00_PROJECT_ROOT
        ).as_posix()
    ),
)


configuration_hash_payload = b"".join(
    (
        path.relative_to(
            STAGE_00_PROJECT_ROOT
        ).as_posix().encode("utf-8")
        + b"\0"
        + path.read_bytes()
    )
    for path in stage_00_config_paths
)

actual_configuration_hash = (
    sha256_bytes(
        configuration_hash_payload
    )
)

expected_configuration_hash = (
    stage_00_internal_manifest.get(
        "configuration_hash"
    )
)

if (
    actual_configuration_hash
    != expected_configuration_hash
):
    stage_00_verification_problems.append(
        "configuration_hash_mismatch"
    )


archive_name = (
    stage_00_external_manifest.get(
        "archive"
    )
)

archive_file_path = (
    STAGE_00_DATASET_ROOT
    / archive_name
    if archive_name
    else None
)

archive_file_present = bool(
    archive_file_path
    and archive_file_path.is_file()
)

verification_mode = (
    "archive_and_internal_manifest"
    if archive_file_present
    else "expanded_internal_manifest"
)


cell_2_report = {
    "verification_mode": (
        verification_mode
    ),

    "archive_file_present": (
        archive_file_present
    ),

    "internal_manifest_sha256": (
        actual_internal_manifest_hash
    ),

    "configuration_hash": (
        actual_configuration_hash
    ),

    "expected_artifact_count": (
        stage_00_internal_manifest.get(
            "artifact_count"
        )
    ),

    "verified_artifact_count": len(
        stage_00_artifacts
    ),

    "verification_problems": (
        stage_00_verification_problems
    ),

    "status": (
        "PASS"
        if not stage_00_verification_problems
        else "BLOCKED"
    ),
}


print(
    json.dumps(
        cell_2_report,
        indent=2,
    )
)


if stage_00_verification_problems:
    raise RuntimeError(
        "Stage 00 verification failed:\n"
        + "\n".join(
            stage_00_verification_problems
        )
    )


print("\nNOTEBOOK 01 — CELL 2: PASS")

{
  "verification_mode": "expanded_internal_manifest",
  "archive_file_present": false,
  "internal_manifest_sha256": "d9a89cf537e942c42d397aa7d993ed467a8a2fcb456fbe6594e6418c3236fd22",
  "configuration_hash": "45314c4c308f11651ab551b717dba1c7cc09808d0c2d4c8761b854a2769d5063",
  "expected_artifact_count": 32,
  "verified_artifact_count": 32,
  "verification_problems": [],
  "status": "PASS"
}

NOTEBOOK 01 — CELL 2: PASS


In [3]:
import importlib
import importlib.metadata


REQUIRED_PACKAGES = {
    "yaml": "PyYAML",
    "pandas": "pandas",
    "numpy": "numpy",
    "jsonschema": "jsonschema",
    "referencing": "referencing",
    "openapi_spec_validator": (
        "openapi-spec-validator"
    ),
}


dependency_audit = []

for (
    import_name,
    distribution_name,
) in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(
            import_name
        )

        installed_version = (
            importlib.metadata.version(
                distribution_name
            )
        )

        dependency_audit.append({
            "import_name": import_name,
            "distribution": (
                distribution_name
            ),
            "available": True,
            "version": str(
                installed_version
            ),
            "diagnostic": None,
        })

    except Exception as error:
        dependency_audit.append({
            "import_name": import_name,
            "distribution": (
                distribution_name
            ),
            "available": False,
            "version": None,
            "diagnostic": repr(error),
        })


missing_packages = [
    record["distribution"]
    for record in dependency_audit
    if not record["available"]
]


cell_3_report = {
    "python_version": (
        sys.version
    ),

    "packages": (
        dependency_audit
    ),

    "missing_packages": (
        missing_packages
    ),

    "status": (
        "PASS"
        if not missing_packages
        else "NEEDS_INSTALL"
    ),
}


print(
    json.dumps(
        cell_3_report,
        indent=2,
    )
)

if missing_packages:
    print(
        "\nPackages requiring installation:\n"
        + "\n".join(
            f"- {package}"
            for package in missing_packages
        )
    )
else:
    print(
        "\nNOTEBOOK 01 — CELL 3: PASS"
    )

{
  "python_version": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "packages": [
    {
      "import_name": "yaml",
      "distribution": "PyYAML",
      "available": true,
      "version": "6.0.3",
      "diagnostic": null
    },
    {
      "import_name": "pandas",
      "distribution": "pandas",
      "available": true,
      "version": "2.3.3",
      "diagnostic": null
    },
    {
      "import_name": "numpy",
      "distribution": "numpy",
      "available": true,
      "version": "2.1.3",
      "diagnostic": null
    },
    {
      "import_name": "jsonschema",
      "distribution": "jsonschema",
      "available": true,
      "version": "4.26.0",
      "diagnostic": null
    },
    {
      "import_name": "referencing",
      "distribution": "referencing",
      "available": true,
      "version": "0.37.0",
      "diagnostic": null
    },
    {
      "import_name": "openapi_spec_validator",
      "distribution": "openapi-spec-validator",
      "available": false,
     

In [4]:
%pip install --disable-pip-version-check --no-input openapi-spec-validator==0.9.0

importlib.invalidate_caches()

import openapi_spec_validator

validator_version = (
    importlib.metadata.version(
        "openapi-spec-validator"
    )
)

cell_4_report = {
    "package": (
        "openapi-spec-validator"
    ),

    "required_version": "0.9.0",

    "installed_version": (
        validator_version
    ),

    "import_path": (
        openapi_spec_validator.__file__
    ),

    "status": (
        "PASS"
        if validator_version == "0.9.0"
        else "BLOCKED"
    ),
}


print(
    json.dumps(
        cell_4_report,
        indent=2,
    )
)


if validator_version != "0.9.0":
    raise RuntimeError(
        "Unexpected validator version."
    )


print("\nNOTEBOOK 01 — CELL 4: PASS")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [openapi-spec-validator]tic-settings]
Note: you may need to restart the kernel to use updated packages.
{
  "package": "openapi-spec-validator",
  "required_version": "0.9.0",
  "installed_version": "0.9.0",
  "import_path": "/usr/local/lib/python3.13/dist-packages/openapi_spec_validator/__init__.py",
  "status": "PASS"
}

NOTEBOOK 01 — CELL 4: PASS


In [5]:
import os
import tempfile
from datetime import datetime, timezone

import yaml


preexisting_project_entries = (
    sorted(
        path.name
        for path in PROJECT_ROOT.iterdir()
    )
    if PROJECT_ROOT.exists()
    else []
)

if preexisting_project_entries:
    raise RuntimeError(
        "Notebook 01 requires a clean working "
        "project directory, but these entries "
        "already exist:\n"
        + "\n".join(
            f"- {name}"
            for name
            in preexisting_project_entries
        )
    )


STAGE_01_DIRECTORIES = [
    "config",
    "src",
    "notebooks",

    "data/inventory",
    "data/canonical",
    "data/audits",

    "reports/audit",

    "registries",
    "exports",
]


PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

for relative_path in (
    STAGE_01_DIRECTORIES
):
    (
        PROJECT_ROOT
        / relative_path
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

STAGE_01_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def atomic_write_bytes(
    path: Path,
    data: bytes,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    descriptor, temporary_name = (
        tempfile.mkstemp(
            dir=path.parent,
            prefix=f".{path.name}.",
            suffix=".tmp",
        )
    )

    try:
        with os.fdopen(
            descriptor,
            "wb",
        ) as handle:
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())

        os.replace(
            temporary_name,
            path,
        )

    except Exception:
        try:
            os.unlink(
                temporary_name
            )
        except FileNotFoundError:
            pass

        raise


def write_text(
    path: Path,
    text: str,
) -> None:
    atomic_write_bytes(
        path,
        text.encode("utf-8"),
    )


def write_json(
    path: Path,
    value,
) -> None:
    serialized = json.dumps(
        value,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ) + "\n"

    write_text(
        path,
        serialized,
    )


def write_yaml(
    path: Path,
    value,
) -> None:
    serialized = yaml.safe_dump(
        value,
        sort_keys=False,
        allow_unicode=True,
    )

    write_text(
        path,
        serialized,
    )


io_qualification = {
    "status": "PASS",

    "captured_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "encoding": "UTF-8",
    "atomic_replacement": True,
    "hash_algorithm": "SHA-256",
}


io_qualification_path = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "io_qualification.json"
)

write_json(
    io_qualification_path,
    io_qualification,
)


missing_directories = [
    relative_path
    for relative_path
    in STAGE_01_DIRECTORIES
    if not (
        PROJECT_ROOT
        / relative_path
    ).is_dir()
]


cell_5_report = {
    "project_root": str(
        PROJECT_ROOT
    ),

    "stage_01_output_root": str(
        STAGE_01_OUTPUT_ROOT
    ),

    "created_directory_count": len(
        STAGE_01_DIRECTORIES
    ),

    "missing_directories": (
        missing_directories
    ),

    "io_qualification_path": str(
        io_qualification_path
    ),

    "io_qualification_sha256": (
        sha256_file(
            io_qualification_path
        )
    ),

    "status": (
        "PASS"
        if not missing_directories
        else "BLOCKED"
    ),
}


print(
    json.dumps(
        cell_5_report,
        indent=2,
    )
)


if missing_directories:
    raise RuntimeError(
        "Stage 01 workspace creation failed."
    )


print("\nNOTEBOOK 01 — CELL 5: PASS")

{
  "project_root": "/kaggle/working/fresh/stage_01/prose_operation_experiment",
  "stage_01_output_root": "/kaggle/working/stage_01_fresh_dataset",
  "created_directory_count": 9,
  "missing_directories": [],
  "io_qualification_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/reports/audit/io_qualification.json",
  "io_qualification_sha256": "20941e66b635c5537894ab3c06c6a1ae0a42301604c8234d72e9c6af196dc183",
  "status": "PASS"
}

NOTEBOOK 01 — CELL 5: PASS


In [6]:
import subprocess


stage_01_policy = {
    "schema_version": "1.0",

    "stage": "01",

    "input_corpus_root": str(
        CORPUS_ROOT
    ),

    "upstream_stage_00": {
        "dataset_root": str(
            STAGE_00_DATASET_ROOT
        ),
        "internal_manifest_sha256": (
            actual_internal_manifest_hash
        ),
        "configuration_hash": (
            actual_configuration_hash
        ),
    },

    "candidate_file_extensions": [
        ".json",
        ".yaml",
        ".yml",
    ],

    "parsing": {
        "json": "json.loads",
        "yaml": "yaml.safe_load",
        "yaml_multiple_documents": (
            "reject"
        ),
        "required_top_level_type": (
            "object"
        ),
        "text_encoding_primary": (
            "UTF-8"
        ),
        "text_encoding_error_policy": (
            "record_failure"
        ),
    },

    "dialect_policy": {
        "openapi_3_0": "supported",
        "openapi_3_1": "supported",
        "swagger_2_0": (
            "unsupported_without_"
            "qualified_converter"
        ),
        "unknown": "unsupported",
    },

    "swagger_conversion": {
        "enabled": False,
        "reason": (
            "No pinned and qualified "
            "semantics-preserving converter "
            "has been registered."
        ),
    },

    "reference_policy": {
        "local_references": (
            "audit_and_validate"
        ),
        "remote_references": (
            "record_without_downloading"
        ),
        "recursive_references": (
            "preserve_with_visited_tracking"
        ),
        "unresolved_references": (
            "exclude_with_reason"
        ),
    },

    "canonicalization": {
        "format": "UTF-8 JSON",
        "sort_keys": True,
        "compact_separators": True,
        "ensure_ascii": False,
        "newline_at_end": True,
    },

    "duplicate_policy": {
        "raw_exact": (
            "SHA-256 of source bytes"
        ),
        "canonical_semantic": (
            "SHA-256 of canonical JSON "
            "without trailing newline"
        ),
        "representative_rule": (
            "lexicographically smallest "
            "stable source-relative path"
        ),
        "retain_all_provenance": True,
    },

    "family_policy": {
        "preferred_evidence": [
            "corpus provider directory",
            "provider hostname",
            "API title",
            "explicit version relationship",
        ],
        "generic_titles_not_merged": [
            "API",
            "OpenAPI",
            "Swagger",
        ],
        "uncertain_assignments": (
            "export_for_review"
        ),
    },

    "development_exposure": {
        "default": (
            "no real corpus family used "
            "for software development"
        ),
        "synthetic_fixtures_preferred": True,
    },
}


stage_01_policy_path = (
    PROJECT_ROOT
    / "config"
    / "stage_01.yaml"
)

write_yaml(
    stage_01_policy_path,
    stage_01_policy,
)


pip_freeze_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze",
        "--all",
    ],
    check=True,
    capture_output=True,
    text=True,
)

requirements_lock_path = (
    PROJECT_ROOT
    / "config"
    / "requirements.lock.txt"
)

write_text(
    requirements_lock_path,
    pip_freeze_result.stdout,
)


environment_record = {
    "captured_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": {
        "version": sys.version,
        "executable": sys.executable,
        "implementation": (
            platform.python_implementation()
        ),
    },

    "platform": {
        "platform": platform.platform(),
        "system": platform.system(),
        "release": platform.release(),
        "machine": platform.machine(),
    },

    "kaggle": {
        "kernel_run_type": os.environ.get(
            "KAGGLE_KERNEL_RUN_TYPE"
        ),
        "container_name": os.environ.get(
            "KAGGLE_CONTAINER_NAME"
        ),
        "accelerator_required": False,
    },

    "packages": {
        record["distribution"]: (
            record["version"]
        )
        for record in dependency_audit
        if record["available"]
    },

    "installed_validator_version": (
        validator_version
    ),
}


environment_path = (
    PROJECT_ROOT
    / "config"
    / "environment.json"
)

write_json(
    environment_path,
    environment_record,
)


upstream_verification_record = {
    "status": "PASS",

    "verification_mode": (
        verification_mode
    ),

    "stage_00_dataset_root": str(
        STAGE_00_DATASET_ROOT
    ),

    "external_manifest_path": str(
        STAGE_00_EXTERNAL_MANIFEST_PATH
    ),

    "internal_manifest_path": str(
        STAGE_00_INTERNAL_MANIFEST_PATH
    ),

    "internal_manifest_sha256": (
        actual_internal_manifest_hash
    ),

    "configuration_hash": (
        actual_configuration_hash
    ),

    "verified_artifact_count": len(
        stage_00_artifacts
    ),

    "verification_problems": (
        stage_00_verification_problems
    ),
}


upstream_verification_path = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "upstream_stage_00_verification.json"
)

write_json(
    upstream_verification_path,
    upstream_verification_record,
)


cell_6_report = {
    "stage_01_policy": {
        "path": str(
            stage_01_policy_path
        ),
        "sha256": sha256_file(
            stage_01_policy_path
        ),
    },

    "environment": {
        "path": str(
            environment_path
        ),
        "sha256": sha256_file(
            environment_path
        ),
    },

    "requirements_lock": {
        "path": str(
            requirements_lock_path
        ),
        "sha256": sha256_file(
            requirements_lock_path
        ),
        "bytes": (
            requirements_lock_path
            .stat()
            .st_size
        ),
    },

    "upstream_verification": {
        "path": str(
            upstream_verification_path
        ),
        "sha256": sha256_file(
            upstream_verification_path
        ),
    },

    "status": "PASS",
}


print(
    json.dumps(
        cell_6_report,
        indent=2,
    )
)

print("\nNOTEBOOK 01 — CELL 6: PASS")

{
  "stage_01_policy": {
    "path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/config/stage_01.yaml",
    "sha256": "2ba3b33be35fd0fbe2b822b94db827bceae979b1e11b481a2bd2157f9e1ff3a7"
  },
  "environment": {
    "path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/config/environment.json",
    "sha256": "6e59b886b3fa089e3c2533b1228770b9de9744c26acdb257881aca4bf5f751da"
  },
  "requirements_lock": {
    "path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/config/requirements.lock.txt",
    "sha256": "56916ac607e806171d075c5606f8282a44aafe0d567e51d1ab6e00097c429845",
    "bytes": 18278
  },
  "upstream_verification": {
    "path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/reports/audit/upstream_stage_00_verification.json",
    "sha256": "e33ab344bcb306160be5c3f8e750284d34724f1a9ab677ea757d53295472bbc2"
  },
  "status": "PASS"
}

NOTEBOOK 01 — CELL 6: PASS


In [7]:
import time

import pandas as pd


CANDIDATE_EXTENSIONS = {
    ".json",
    ".yaml",
    ".yml",
}


inventory_started = (
    time.perf_counter()
)


source_paths = sorted(
    (
        path
        for path in CORPUS_ROOT.rglob("*")
        if path.is_file()
    ),
    key=lambda path: (
        path.relative_to(
            CORPUS_ROOT
        ).as_posix()
    ),
)


inventory_records = []

for source_path in source_paths:
    relative_path = (
        source_path
        .relative_to(CORPUS_ROOT)
        .as_posix()
    )

    relative_parts = Path(
        relative_path
    ).parts

    provider_directory = (
        relative_parts[0]
        if len(relative_parts) > 1
        else None
    )

    extension = (
        source_path.suffix.lower()
    )

    raw_id = (
        "raw_"
        + hashlib.sha256(
            relative_path.encode("utf-8")
        ).hexdigest()[:20]
    )

    try:
        source_size = (
            source_path.stat().st_size
        )

        source_hash = sha256_file(
            source_path
        )

        inventory_status = "PASS"
        diagnostic = None

    except Exception as error:
        source_size = None
        source_hash = None
        inventory_status = "FAILED"
        diagnostic = repr(error)

    inventory_records.append({
        "raw_id": raw_id,
        "relative_path": relative_path,
        "provider_directory": (
            provider_directory
        ),
        "filename": source_path.name,
        "extension": extension,
        "bytes": source_size,
        "sha256": source_hash,
        "candidate_extension": (
            extension
            in CANDIDATE_EXTENSIONS
        ),
        "inventory_status": (
            inventory_status
        ),
        "diagnostic": diagnostic,
    })


inventory_dataframe = pd.DataFrame(
    inventory_records,
    columns=[
        "raw_id",
        "relative_path",
        "provider_directory",
        "filename",
        "extension",
        "bytes",
        "sha256",
        "candidate_extension",
        "inventory_status",
        "diagnostic",
    ],
)


if (
    inventory_dataframe[
        "relative_path"
    ].duplicated().any()
):
    raise RuntimeError(
        "Duplicate relative paths were "
        "found in the inventory."
    )

if (
    inventory_dataframe[
        "raw_id"
    ].duplicated().any()
):
    raise RuntimeError(
        "A raw_id collision was detected."
    )


inventory_path = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "raw_files.csv"
)

write_text(
    inventory_path,
    inventory_dataframe.to_csv(
        index=False
    ),
)


inventory_duration_seconds = (
    time.perf_counter()
    - inventory_started
)


candidate_dataframe = (
    inventory_dataframe[
        inventory_dataframe[
            "candidate_extension"
        ]
    ]
    .copy()
)


inventory_summary = {
    "status": "PASS",

    "source_file_count": int(
        len(inventory_dataframe)
    ),

    "candidate_yaml_json_count": int(
        len(candidate_dataframe)
    ),

    "noncandidate_file_count": int(
        len(inventory_dataframe)
        - len(candidate_dataframe)
    ),

    "inventory_failure_count": int(
        (
            inventory_dataframe[
                "inventory_status"
            ]
            != "PASS"
        ).sum()
    ),

    "provider_directory_count": int(
        inventory_dataframe[
            "provider_directory"
        ].dropna().nunique()
    ),

    "total_source_bytes": int(
        inventory_dataframe[
            "bytes"
        ].fillna(0).sum()
    ),

    "duration_seconds": round(
        inventory_duration_seconds,
        3,
    ),

    "inventory_path": str(
        inventory_path
    ),

    "inventory_sha256": sha256_file(
        inventory_path
    ),
}


inventory_summary_path = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "inventory_summary.json"
)

write_json(
    inventory_summary_path,
    inventory_summary,
)


print(
    json.dumps(
        inventory_summary,
        indent=2,
    )
)

print("\nFirst five inventory rows:")

display(
    inventory_dataframe.head(5)
)


if (
    inventory_summary[
        "inventory_failure_count"
    ]
    > 0
):
    print(
        "\nInventory completed with recorded "
        "file-read failures. These files will "
        "remain in the exclusion accounting."
    )


print("\nNOTEBOOK 01 — CELL 7: PASS")

{
  "status": "PASS",
  "source_file_count": 4138,
  "candidate_yaml_json_count": 4138,
  "noncandidate_file_count": 0,
  "inventory_failure_count": 0,
  "provider_directory_count": 701,
  "total_source_bytes": 749615240,
  "duration_seconds": 384.966,
  "inventory_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/inventory/raw_files.csv",
  "inventory_sha256": "733f3522c020d75f53e154b749d4141d96b187df7d56056a75a49b29b27a0132"
}

First five inventory rows:


,raw_id,relative_path,provider_directory,filename,extension,bytes,sha256,candidate_extension,inventory_status,diagnostic
0,raw_036188082e3a8eed83fe,1forge.com/0.0.1/swagger.yaml,1forge.com,swagger.yaml,.yaml,1446,66f7b3337053011fa1cf39af503ccef595dc78501eecac...,True,PASS,None
1,raw_6b85b3c54951f0a75ad0,1password.com/events/1.2.0/openapi.yaml,1password.com,openapi.yaml,.yaml,17339,567a912a79601940095aa6100f29b5cf9c9f12a10b9409...,True,PASS,None
2,raw_6291eafb314775009532,1password.local/connect/1.5.7/openapi.yaml,1password.local,openapi.yaml,.yaml,38611,0e14a654c095fe0763108359c8369a6e35e3ca2e8309d1...,True,PASS,None
3,raw_1431dda1595300f49e10,6-dot-authentiqio.appspot.com/6/openapi.yaml,6-dot-authentiqio.appspot.com,openapi.yaml,.yaml,18985,848cd48a639d89d7a1ec58d821ea3c19220c9efbaa88eb...,True,PASS,None
4,raw_1310fd0b7989354fa91c,ably.io/platform/1.1.0/openapi.yaml,ably.io,openapi.yaml,.yaml,44030,87a9a24552041b3a7cd6f175be52ecd36f95e357ccef44...,True,PASS,None



NOTEBOOK 01 — CELL 7: PASS


In [8]:
import json
import time
from collections import Counter

from yaml.constructor import (
    ConstructorError,
)
from yaml.resolver import BaseResolver

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


HTTP_METHODS = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}


class OpenAPISafeLoader(
    yaml.SafeLoader
):
    """
    Safe YAML loader that:

    1. Rejects duplicate mapping keys.
    2. Preserves timestamp-like values as
       strings instead of datetime objects.
    """


OpenAPISafeLoader.yaml_implicit_resolvers = {
    key: [
        resolver
        for resolver in resolvers
        if (
            resolver[0]
            != "tag:yaml.org,2002:timestamp"
        )
    ]
    for key, resolvers
    in yaml.SafeLoader
    .yaml_implicit_resolvers.items()
}


def construct_unique_mapping(
    loader,
    node,
    deep=False,
):
    mapping = {}

    for key_node, value_node in node.value:
        key = loader.construct_object(
            key_node,
            deep=deep,
        )

        try:
            duplicate = key in mapping
        except TypeError as error:
            raise ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                "found an unhashable key",
                key_node.start_mark,
            ) from error

        if duplicate:
            raise ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                f"found duplicate key: {key!r}",
                key_node.start_mark,
            )

        value = loader.construct_object(
            value_node,
            deep=deep,
        )

        mapping[key] = value

    return mapping


OpenAPISafeLoader.add_constructor(
    BaseResolver.DEFAULT_MAPPING_TAG,
    construct_unique_mapping,
)


def detect_dialect(document):
    openapi_value = document.get(
        "openapi"
    )

    swagger_value = document.get(
        "swagger"
    )

    if isinstance(openapi_value, str):
        if openapi_value.startswith("3.0"):
            return (
                "openapi_3_0",
                openapi_value,
                True,
            )

        if openapi_value.startswith("3.1"):
            return (
                "openapi_3_1",
                openapi_value,
                True,
            )

        return (
            "openapi_other",
            openapi_value,
            False,
        )

    if str(swagger_value) == "2.0":
        return (
            "swagger_2_0",
            "2.0",
            False,
        )

    return (
        "unknown",
        None,
        False,
    )


def count_paths_and_operations(
    document,
):
    paths_object = document.get(
        "paths"
    )

    if not isinstance(paths_object, dict):
        return 0, 0

    path_count = 0
    operation_count = 0

    for path_name, path_item in (
        paths_object.items()
    ):
        if not isinstance(path_name, str):
            continue

        if not path_name.startswith("/"):
            continue

        path_count += 1

        if not isinstance(path_item, dict):
            continue

        operation_count += sum(
            1
            for key in path_item
            if (
                isinstance(key, str)
                and key.lower()
                in HTTP_METHODS
            )
        )

    return (
        path_count,
        operation_count,
    )


# Record the exact YAML behavior used.
stage_01_policy[
    "parsing"
][
    "yaml"
] = (
    "OpenAPISafeLoader based on "
    "yaml.SafeLoader; duplicate keys rejected; "
    "timestamp-like scalars preserved as strings"
)

write_yaml(
    stage_01_policy_path,
    stage_01_policy,
)


parse_started = time.perf_counter()

specification_records = []


for row in tqdm(
    candidate_dataframe.itertuples(
        index=False
    ),
    total=len(candidate_dataframe),
    desc="Parsing OpenAPI files",
):
    source_path = (
        CORPUS_ROOT
        / row.relative_path
    )

    spec_id = None
    parse_status = "FAILED"
    document_status = "PARSE_FAILED"
    diagnostic = None

    dialect = None
    declared_version = None
    supported = False

    api_title = None
    api_version = None

    path_count = None
    operation_count = None

    canonical_sha256 = None
    canonical_relative_path = None
    canonical_bytes = None

    try:
        raw_bytes = source_path.read_bytes()

        source_text = raw_bytes.decode(
            "utf-8-sig"
        )

        if row.extension == ".json":
            document = json.loads(
                source_text
            )
            parser = "json.loads"

        else:
            document = yaml.load(
                source_text,
                Loader=OpenAPISafeLoader,
            )
            parser = "OpenAPISafeLoader"

        parse_status = "PASS"

        if not isinstance(document, dict):
            document_status = (
                "INVALID_TOP_LEVEL"
            )

            diagnostic = (
                "Top-level document is "
                f"{type(document).__name__}; "
                "an object is required."
            )

        else:
            (
                dialect,
                declared_version,
                supported,
            ) = detect_dialect(document)

            info_object = document.get(
                "info"
            )

            if isinstance(info_object, dict):
                title_value = info_object.get(
                    "title"
                )

                version_value = (
                    info_object.get(
                        "version"
                    )
                )

                api_title = (
                    str(title_value)
                    if title_value is not None
                    else None
                )

                api_version = (
                    str(version_value)
                    if version_value is not None
                    else None
                )

            (
                path_count,
                operation_count,
            ) = count_paths_and_operations(
                document
            )

            canonical_text = json.dumps(
                document,
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
            )

            canonical_bytes = (
                canonical_text.encode(
                    "utf-8"
                )
            )

            canonical_sha256 = (
                sha256_bytes(
                    canonical_bytes
                )
            )

            identity_payload = (
                f"{row.raw_id}|"
                f"{row.sha256}|"
                f"{canonical_sha256}"
            ).encode("utf-8")

            spec_id = (
                "spec_"
                + hashlib.sha256(
                    identity_payload
                ).hexdigest()[:24]
            )

            if supported:
                document_status = (
                    "SUPPORTED_PENDING_VALIDATION"
                )

                canonical_relative_path = (
                    "data/canonical/"
                    f"{spec_id}.json"
                )

                canonical_output_path = (
                    PROJECT_ROOT
                    / canonical_relative_path
                )

                atomic_write_bytes(
                    canonical_output_path,
                    canonical_bytes + b"\n",
                )

            else:
                document_status = (
                    "UNSUPPORTED_DIALECT"
                )

                if dialect == "swagger_2_0":
                    diagnostic = (
                        "Swagger 2.0 is unsupported "
                        "because no qualified "
                        "converter is registered."
                    )

                elif dialect == "unknown":
                    diagnostic = (
                        "No supported openapi or "
                        "swagger version field "
                        "was found."
                    )

                else:
                    diagnostic = (
                        "The declared OpenAPI "
                        f"version {declared_version!r} "
                        "is unsupported."
                    )

    except UnicodeDecodeError as error:
        parser = None
        document_status = "DECODE_FAILED"
        diagnostic = repr(error)[:2000]

    except json.JSONDecodeError as error:
        parser = "json.loads"
        document_status = "PARSE_FAILED"
        diagnostic = repr(error)[:2000]

    except yaml.YAMLError as error:
        parser = "OpenAPISafeLoader"
        document_status = "PARSE_FAILED"
        diagnostic = repr(error)[:2000]

    except (TypeError, ValueError) as error:
        parser = locals().get(
            "parser"
        )
        document_status = (
            "CANONICALIZATION_FAILED"
        )
        diagnostic = repr(error)[:2000]

    except Exception as error:
        parser = locals().get(
            "parser"
        )
        document_status = (
            "UNEXPECTED_FAILURE"
        )
        diagnostic = repr(error)[:2000]

    specification_records.append({
        "raw_id": row.raw_id,
        "spec_id": spec_id,

        "relative_path": (
            row.relative_path
        ),

        "provider_directory": (
            row.provider_directory
        ),

        "source_sha256": row.sha256,
        "source_bytes": row.bytes,

        "parser": parser,
        "parse_status": parse_status,
        "document_status": (
            document_status
        ),

        "dialect": dialect,
        "declared_version": (
            declared_version
        ),

        "supported": supported,

        "api_title": api_title,
        "api_version": api_version,

        "path_count": path_count,
        "operation_count": (
            operation_count
        ),

        "canonical_sha256": (
            canonical_sha256
        ),

        "canonical_relative_path": (
            canonical_relative_path
        ),

        "validation_status": (
            "PENDING"
            if supported
            and canonical_relative_path
            else "NOT_APPLICABLE"
        ),

        "diagnostic": diagnostic,
    })


specifications_dataframe = pd.DataFrame(
    specification_records
)


if (
    len(specifications_dataframe)
    != len(candidate_dataframe)
):
    raise RuntimeError(
        "Specification row accounting failed."
    )

if (
    specifications_dataframe[
        "raw_id"
    ].duplicated().any()
):
    raise RuntimeError(
        "Duplicate raw_id values appeared "
        "in the specification table."
    )


specifications_path = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "specifications.csv"
)

write_text(
    specifications_path,
    specifications_dataframe.to_csv(
        index=False
    ),
)


parse_duration_seconds = (
    time.perf_counter()
    - parse_started
)


dialect_counts = {
    str(key): int(value)
    for key, value
    in specifications_dataframe[
        "dialect"
    ]
    .fillna("unavailable")
    .value_counts()
    .to_dict()
    .items()
}


parse_summary = {
    "status": "PASS",

    "candidate_file_count": int(
        len(candidate_dataframe)
    ),

    "parse_pass_count": int(
        (
            specifications_dataframe[
                "parse_status"
            ]
            == "PASS"
        ).sum()
    ),

    "parse_failure_count": int(
        (
            specifications_dataframe[
                "parse_status"
            ]
            != "PASS"
        ).sum()
    ),

    "supported_pending_validation_count": int(
        (
            specifications_dataframe[
                "document_status"
            ]
            == "SUPPORTED_PENDING_VALIDATION"
        ).sum()
    ),

    "unsupported_or_failed_count": int(
        (
            specifications_dataframe[
                "document_status"
            ]
            != "SUPPORTED_PENDING_VALIDATION"
        ).sum()
    ),

    "canonical_file_count": int(
        sum(
            1
            for path in (
                PROJECT_ROOT
                / "data"
                / "canonical"
            ).glob("*.json")
            if path.is_file()
        )
    ),

    "dialect_counts": dialect_counts,

    "supported_operation_count": int(
        specifications_dataframe.loc[
            (
                specifications_dataframe[
                    "document_status"
                ]
                == "SUPPORTED_PENDING_VALIDATION"
            ),
            "operation_count",
        ]
        .fillna(0)
        .sum()
    ),

    "duration_seconds": round(
        parse_duration_seconds,
        3,
    ),

    "specifications_path": str(
        specifications_path
    ),

    "specifications_sha256": (
        sha256_file(
            specifications_path
        )
    ),
}


parse_summary_path = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "parse_and_dialect_summary.json"
)

write_json(
    parse_summary_path,
    parse_summary,
)


print(
    json.dumps(
        parse_summary,
        indent=2,
    )
)

print(
    "\nFirst five specification rows:"
)

display(
    specifications_dataframe.head(5)
)


if (
    parse_summary[
        "canonical_file_count"
    ]
    != parse_summary[
        "supported_pending_validation_count"
    ]
):
    raise RuntimeError(
        "Canonical-file accounting failed."
    )


print("\nNOTEBOOK 01 — CELL 8: PASS")

Parsing OpenAPI files:   0%|          | 0/4138 [00:00<?, ?it/s]

{
  "status": "PASS",
  "candidate_file_count": 4138,
  "parse_pass_count": 4124,
  "parse_failure_count": 14,
  "supported_pending_validation_count": 1944,
  "unsupported_or_failed_count": 2194,
  "canonical_file_count": 1944,
  "dialect_counts": {
    "swagger_2_0": 2163,
    "openapi_3_0": 1854,
    "openapi_3_1": 107,
    "unavailable": 14
  },
  "supported_operation_count": 104790,
  "duration_seconds": 1387.295,
  "specifications_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/inventory/specifications.csv",
  "specifications_sha256": "721661cc0395264e78599976fff7b6b85b74022442cce4353bd6926f714fea88"
}

First five specification rows:


,raw_id,spec_id,relative_path,provider_directory,source_sha256,source_bytes,parser,parse_status,document_status,dialect,declared_version,supported,api_title,api_version,path_count,operation_count,canonical_sha256,canonical_relative_path,validation_status,diagnostic
0,raw_036188082e3a8eed83fe,spec_ee48da556848a7e76d19ff38,1forge.com/0.0.1/swagger.yaml,1forge.com,66f7b3337053011fa1cf39af503ccef595dc78501eecac...,1446,OpenAPISafeLoader,PASS,UNSUPPORTED_DIALECT,swagger_2_0,2.0,False,1Forge Finance APIs,0.0.1,2.0,2.0,586548cf39d0a7523ce67de4de5c6d901ef08ffbbcfcdd...,None,NOT_APPLICABLE,Swagger 2.0 is unsupported because no qualifie...
1,raw_6b85b3c54951f0a75ad0,spec_a40eb5c602ff16f61a9d9ecc,1password.com/events/1.2.0/openapi.yaml,1password.com,567a912a79601940095aa6100f29b5cf9c9f12a10b9409...,17339,OpenAPISafeLoader,PASS,SUPPORTED_PENDING_VALIDATION,openapi_3_0,3.0.0,True,Events API,1.2.0,5.0,5.0,81e129d874befc06737b51e3908134fbe93f9d7343e359...,data/canonical/spec_a40eb5c602ff16f61a9d9ecc.json,PENDING,None
2,raw_6291eafb314775009532,spec_3be5364a29b1f6c938f7ede5,1password.local/connect/1.5.7/openapi.yaml,1password.local,0e14a654c095fe0763108359c8369a6e35e3ca2e8309d1...,38611,OpenAPISafeLoader,PASS,SUPPORTED_PENDING_VALIDATION,openapi_3_0,3.0.2,True,1Password Connect,1.5.7,11.0,15.0,09f5060cb1dbd714bdfed9a0c8269ab2dff9b1510c755e...,data/canonical/spec_3be5364a29b1f6c938f7ede5.json,PENDING,None
3,raw_1431dda1595300f49e10,spec_651c5e95e912c33a13d69e11,6-dot-authentiqio.appspot.com/6/openapi.yaml,6-dot-authentiqio.appspot.com,848cd48a639d89d7a1ec58d821ea3c19220c9efbaa88eb...,18985,OpenAPISafeLoader,PASS,SUPPORTED_PENDING_VALIDATION,openapi_3_0,3.0.0,True,Authentiq API,6,5.0,14.0,4aefbe2a26d7e4b8d5c5fb090652cc6d74f77517d22918...,data/canonical/spec_651c5e95e912c33a13d69e11.json,PENDING,None
4,raw_1310fd0b7989354fa91c,spec_1087be7fc84c037678cf970f,ably.io/platform/1.1.0/openapi.yaml,ably.io,87a9a24552041b3a7cd6f175be52ecd36f95e357ccef44...,44030,OpenAPISafeLoader,PASS,SUPPORTED_PENDING_VALIDATION,openapi_3_0,3.0.1,True,Platform API,1.1.0,14.0,22.0,57011bd03c03e6f01bf0c06851d2d900dee18efe26a77a...,data/canonical/spec_1087be7fc84c037678cf970f.json,PENDING,None



NOTEBOOK 01 — CELL 8: PASS


In [9]:
import urllib.parse


def escape_pointer_token(
    value,
) -> str:
    return (
        str(value)
        .replace("~", "~0")
        .replace("/", "~1")
    )


def resolve_local_reference(
    document,
    reference: str,
):
    if reference == "#":
        return (
            True,
            type(document).__name__,
            None,
        )

    if not reference.startswith("#/"):
        return (
            False,
            None,
            (
                "Unsupported local fragment "
                "format."
            ),
        )

    fragment = urllib.parse.unquote(
        reference[1:]
    )

    tokens = fragment.lstrip(
        "/"
    ).split("/")

    current = document

    for raw_token in tokens:
        token = (
            raw_token
            .replace("~1", "/")
            .replace("~0", "~")
        )

        if isinstance(current, dict):
            if token not in current:
                return (
                    False,
                    None,
                    (
                        "Missing mapping key "
                        f"{token!r}."
                    ),
                )

            current = current[token]

        elif isinstance(current, list):
            try:
                index = int(token)
            except ValueError:
                return (
                    False,
                    None,
                    (
                        "Non-integer list index "
                        f"{token!r}."
                    ),
                )

            if (
                index < 0
                or index >= len(current)
            ):
                return (
                    False,
                    None,
                    (
                        "List index outside "
                        f"range: {index}."
                    ),
                )

            current = current[index]

        else:
            return (
                False,
                None,
                (
                    "Pointer traversed into "
                    f"{type(current).__name__}."
                ),
            )

    return (
        True,
        type(current).__name__,
        None,
    )


def classify_external_reference(
    reference: str,
) -> str:
    parsed = urllib.parse.urlparse(
        reference
    )

    if parsed.scheme in {
        "http",
        "https",
    }:
        return "remote_url"

    if parsed.scheme:
        return "external_scheme"

    return "relative_external_file"


def inspect_document_references(
    document,
    spec_id: str,
    source_relative_path: str,
):
    reference_rows = []

    stack = [
        (
            document,
            "#",
        )
    ]

    while stack:
        node, pointer = stack.pop()

        if isinstance(node, dict):
            for key, value in node.items():
                child_pointer = (
                    pointer
                    + "/"
                    + escape_pointer_token(
                        key
                    )
                )

                if key == "$ref":
                    if not isinstance(
                        value,
                        str,
                    ):
                        reference_rows.append({
                            "spec_id": spec_id,
                            "source_relative_path": (
                                source_relative_path
                            ),
                            "source_pointer": (
                                child_pointer
                            ),
                            "reference": None,
                            "reference_kind": (
                                "invalid_non_string"
                            ),
                            "resolution_status": (
                                "FAILED"
                            ),
                            "target_type": None,
                            "diagnostic": (
                                "$ref value is not "
                                "a string."
                            ),
                        })

                    elif value.startswith("#"):
                        (
                            resolved,
                            target_type,
                            diagnostic,
                        ) = resolve_local_reference(
                            document,
                            value,
                        )

                        reference_rows.append({
                            "spec_id": spec_id,
                            "source_relative_path": (
                                source_relative_path
                            ),
                            "source_pointer": (
                                child_pointer
                            ),
                            "reference": value,
                            "reference_kind": (
                                "local"
                            ),
                            "resolution_status": (
                                "PASS"
                                if resolved
                                else "FAILED"
                            ),
                            "target_type": (
                                target_type
                            ),
                            "diagnostic": (
                                diagnostic
                            ),
                        })

                    else:
                        reference_rows.append({
                            "spec_id": spec_id,
                            "source_relative_path": (
                                source_relative_path
                            ),
                            "source_pointer": (
                                child_pointer
                            ),
                            "reference": value,
                            "reference_kind": (
                                classify_external_reference(
                                    value
                                )
                            ),
                            "resolution_status": (
                                "NOT_RESOLVED_BY_POLICY"
                            ),
                            "target_type": None,
                            "diagnostic": (
                                "External references "
                                "are recorded but not "
                                "downloaded."
                            ),
                        })

                if isinstance(
                    value,
                    (dict, list),
                ):
                    stack.append(
                        (
                            value,
                            child_pointer,
                        )
                    )

        elif isinstance(node, list):
            for index, value in enumerate(
                node
            ):
                child_pointer = (
                    pointer
                    + "/"
                    + str(index)
                )

                if isinstance(
                    value,
                    (dict, list),
                ):
                    stack.append(
                        (
                            value,
                            child_pointer,
                        )
                    )

    return reference_rows


reference_audit_started = (
    time.perf_counter()
)

all_reference_rows = []
reference_metrics = {}


pending_reference_rows = (
    specifications_dataframe[
        specifications_dataframe[
            "document_status"
        ]
        == "SUPPORTED_PENDING_VALIDATION"
    ]
)


for row in tqdm(
    pending_reference_rows.itertuples(
        index=False
    ),
    total=len(
        pending_reference_rows
    ),
    desc="Auditing references",
):
    canonical_path = (
        PROJECT_ROOT
        / row.canonical_relative_path
    )

    document = json.loads(
        canonical_path.read_text(
            encoding="utf-8"
        )
    )

    document_reference_rows = (
        inspect_document_references(
            document,
            row.spec_id,
            row.relative_path,
        )
    )

    all_reference_rows.extend(
        document_reference_rows
    )

    local_rows = [
        reference_row
        for reference_row
        in document_reference_rows
        if (
            reference_row[
                "reference_kind"
            ]
            == "local"
        )
    ]

    external_rows = [
        reference_row
        for reference_row
        in document_reference_rows
        if (
            reference_row[
                "reference_kind"
            ]
            not in {
                "local",
                "invalid_non_string",
            }
        )
    ]

    failed_rows = [
        reference_row
        for reference_row
        in document_reference_rows
        if (
            reference_row[
                "resolution_status"
            ]
            == "FAILED"
        )
    ]

    invalid_ref_rows = [
        reference_row
        for reference_row
        in document_reference_rows
        if (
            reference_row[
                "reference_kind"
            ]
            == "invalid_non_string"
        )
    ]

    if invalid_ref_rows:
        reference_status = (
            "INVALID_REFERENCE_VALUE"
        )

    elif failed_rows:
        reference_status = (
            "UNRESOLVED_LOCAL_REFERENCE"
        )

    elif external_rows:
        reference_status = (
            "UNSUPPORTED_EXTERNAL_REFERENCE"
        )

    else:
        reference_status = "PASS"

    reference_metrics[
        row.spec_id
    ] = {
        "reference_status": (
            reference_status
        ),
        "local_ref_count": len(
            local_rows
        ),
        "external_ref_count": len(
            external_rows
        ),
        "failed_ref_count": len(
            failed_rows
        ),
        "invalid_ref_count": len(
            invalid_ref_rows
        ),
    }


reference_status_values = []
local_ref_counts = []
external_ref_counts = []
failed_ref_counts = []
invalid_ref_counts = []


for row in (
    specifications_dataframe.itertuples(
        index=False
    )
):
    metrics = reference_metrics.get(
        row.spec_id
    )

    if metrics is None:
        reference_status_values.append(
            "NOT_ASSESSED"
        )
        local_ref_counts.append(0)
        external_ref_counts.append(0)
        failed_ref_counts.append(0)
        invalid_ref_counts.append(0)

    else:
        reference_status_values.append(
            metrics[
                "reference_status"
            ]
        )
        local_ref_counts.append(
            metrics[
                "local_ref_count"
            ]
        )
        external_ref_counts.append(
            metrics[
                "external_ref_count"
            ]
        )
        failed_ref_counts.append(
            metrics[
                "failed_ref_count"
            ]
        )
        invalid_ref_counts.append(
            metrics[
                "invalid_ref_count"
            ]
        )


specifications_dataframe[
    "reference_status"
] = reference_status_values

specifications_dataframe[
    "local_ref_count"
] = local_ref_counts

specifications_dataframe[
    "external_ref_count"
] = external_ref_counts

specifications_dataframe[
    "failed_ref_count"
] = failed_ref_counts

specifications_dataframe[
    "invalid_ref_count"
] = invalid_ref_counts


status_replacements = {
    "UNSUPPORTED_EXTERNAL_REFERENCE": (
        "UNSUPPORTED_EXTERNAL_REFERENCE"
    ),
    "UNRESOLVED_LOCAL_REFERENCE": (
        "UNRESOLVED_LOCAL_REFERENCE"
    ),
    "INVALID_REFERENCE_VALUE": (
        "INVALID_REFERENCE_VALUE"
    ),
}


for (
    reference_status,
    replacement_status,
) in status_replacements.items():
    mask = (
        specifications_dataframe[
            "document_status"
        ]
        == "SUPPORTED_PENDING_VALIDATION"
    ) & (
        specifications_dataframe[
            "reference_status"
        ]
        == reference_status
    )

    specifications_dataframe.loc[
        mask,
        "document_status",
    ] = replacement_status

    specifications_dataframe.loc[
        mask,
        "validation_status",
    ] = "NOT_APPLICABLE"


write_text(
    specifications_path,
    specifications_dataframe.to_csv(
        index=False
    ),
)


references_path = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "references.jsonl"
)


descriptor, temporary_name = (
    tempfile.mkstemp(
        dir=references_path.parent,
        prefix=".references.",
        suffix=".tmp",
    )
)

try:
    with os.fdopen(
        descriptor,
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        for reference_row in (
            all_reference_rows
        ):
            handle.write(
                json.dumps(
                    reference_row,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                )
                + "\n"
            )

        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_name,
        references_path,
    )

except Exception:
    try:
        os.unlink(
            temporary_name
        )
    except FileNotFoundError:
        pass

    raise


reference_audit_duration = (
    time.perf_counter()
    - reference_audit_started
)


reference_summary = {
    "status": "PASS",

    "specifications_assessed": int(
        len(pending_reference_rows)
    ),

    "reference_row_count": int(
        len(all_reference_rows)
    ),

    "specifications_with_passed_references": int(
        (
            specifications_dataframe[
                "reference_status"
            ]
            == "PASS"
        ).sum()
    ),

    "specifications_with_external_refs": int(
        (
            specifications_dataframe[
                "reference_status"
            ]
            == "UNSUPPORTED_EXTERNAL_REFERENCE"
        ).sum()
    ),

    "specifications_with_unresolved_local_refs": int(
        (
            specifications_dataframe[
                "reference_status"
            ]
            == "UNRESOLVED_LOCAL_REFERENCE"
        ).sum()
    ),

    "specifications_with_invalid_refs": int(
        (
            specifications_dataframe[
                "reference_status"
            ]
            == "INVALID_REFERENCE_VALUE"
        ).sum()
    ),

    "local_reference_count": int(
        sum(local_ref_counts)
    ),

    "external_reference_count": int(
        sum(external_ref_counts)
    ),

    "failed_reference_count": int(
        sum(failed_ref_counts)
    ),

    "duration_seconds": round(
        reference_audit_duration,
        3,
    ),

    "references_path": str(
        references_path
    ),

    "references_sha256": (
        sha256_file(
            references_path
        )
    ),

    "updated_specifications_sha256": (
        sha256_file(
            specifications_path
        )
    ),
}


reference_summary_path = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "reference_summary.json"
)

write_json(
    reference_summary_path,
    reference_summary,
)


print(
    json.dumps(
        reference_summary,
        indent=2,
    )
)

print(
    "\nNOTEBOOK 01 — CELL 9: PASS"
)

Auditing references:   0%|          | 0/1944 [00:00<?, ?it/s]

{
  "status": "PASS",
  "specifications_assessed": 1944,
  "reference_row_count": 854693,
  "specifications_with_passed_references": 1942,
  "specifications_with_external_refs": 1,
  "specifications_with_unresolved_local_refs": 0,
  "specifications_with_invalid_refs": 1,
  "local_reference_count": 854688,
  "external_reference_count": 1,
  "failed_reference_count": 4,
  "duration_seconds": 20.979,
  "references_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/audits/references.jsonl",
  "references_sha256": "6f5b5781b38da7dc9db164ce68aa2bd21edfd68b0fc1ac1feb54825357cc3722",
  "updated_specifications_sha256": "bdecf824fd90cb2e9cd5fb4e973064d7d151450f0e4e6272fb01c50c1773ab51"
}

NOTEBOOK 01 — CELL 9: PASS


In [10]:
import hashlib
import json
import shutil
import subprocess
from pathlib import Path

PINNED_CONVERTER_PACKAGE = "swagger2openapi"
PINNED_CONVERTER_VERSION = "7.0.8"

project_root = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)
report_path = (
    project_root
    / "reports"
    / "audit"
    / "swagger_converter_preflight.json"
)

report_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def run_command(arguments, timeout=120):
    completed = subprocess.run(
        arguments,
        capture_output=True,
        text=True,
        timeout=timeout,
        check=False,
    )

    return {
        "return_code": completed.returncode,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }


node_path = shutil.which("node")
npm_path = shutil.which("npm")

if node_path is None:
    raise RuntimeError(
        "Node.js was not found in the Kaggle runtime."
    )

if npm_path is None:
    raise RuntimeError(
        "npm was not found in the Kaggle runtime."
    )

node_result = run_command(
    [node_path, "--version"]
)
npm_result = run_command(
    [npm_path, "--version"]
)

if node_result["return_code"] != 0:
    raise RuntimeError(
        "Unable to obtain the Node.js version:\n"
        + node_result["stderr"]
    )

if npm_result["return_code"] != 0:
    raise RuntimeError(
        "Unable to obtain the npm version:\n"
        + npm_result["stderr"]
    )

metadata_result = run_command(
    [
        npm_path,
        "view",
        (
            f"{PINNED_CONVERTER_PACKAGE}"
            f"@{PINNED_CONVERTER_VERSION}"
        ),
        "version",
        "dist.integrity",
        "dist.shasum",
        "repository.url",
        "license",
        "--json",
    ],
    timeout=180,
)

if metadata_result["return_code"] != 0:
    raise RuntimeError(
        "Unable to read pinned converter metadata "
        "from the npm registry:\n"
        + metadata_result["stderr"]
    )

try:
    package_metadata = json.loads(
        metadata_result["stdout"]
    )
except json.JSONDecodeError as exception:
    raise RuntimeError(
        "npm returned invalid JSON metadata:\n"
        + metadata_result["stdout"]
    ) from exception

observed_version = str(
    package_metadata.get("version")
)

if observed_version != PINNED_CONVERTER_VERSION:
    raise RuntimeError(
        "Pinned converter version mismatch: "
        f"expected {PINNED_CONVERTER_VERSION}, "
        f"observed {observed_version}."
    )

preflight_report = {
    "status": "PASS",
    "node": {
        "path": node_path,
        "version": node_result["stdout"],
    },
    "npm": {
        "path": npm_path,
        "version": npm_result["stdout"],
    },
    "converter": {
        "package": PINNED_CONVERTER_PACKAGE,
        "pinned_version": PINNED_CONVERTER_VERSION,
        "registry_metadata": package_metadata,
    },
    "conversion_policy": {
        "target_openapi_version": "3.0.3",
        "silent_patching_allowed": False,
        "warnings_must_be_recorded": True,
        "conversion_alone_grants_eligibility": False,
        "semantic_preservation_audit_required": True,
        "openapi_validation_required": True,
        "reference_audit_required": True,
    },
}

serialized_report = json.dumps(
    preflight_report,
    indent=2,
    sort_keys=True,
    ensure_ascii=False,
) + "\n"

temporary_path = report_path.with_suffix(
    ".json.tmp"
)

temporary_path.write_text(
    serialized_report,
    encoding="utf-8",
)

temporary_path.replace(report_path)

report_sha256 = hashlib.sha256(
    report_path.read_bytes()
).hexdigest()

output = {
    **preflight_report,
    "report_path": str(report_path),
    "report_sha256": report_sha256,
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 10: PASS")

{
  "status": "PASS",
  "node": {
    "path": "/tools/node/bin/node",
    "version": "v22.23.4"
  },
  "npm": {
    "path": "/tools/node/bin/npm",
    "version": "10.8.2"
  },
  "converter": {
    "package": "swagger2openapi",
    "pinned_version": "7.0.8",
    "registry_metadata": {
      "version": "7.0.8",
      "dist.integrity": "sha512-upi/0ZGkYgEcLeGieoz8gT74oWHA0E7JivX7aN9mAf+Tc7BQoRBvnIGHoPDw+f9TXTW4s6kGYCZJtauP6OYp7g==",
      "dist.shasum": "12c88d5de776cb1cbba758994930f40ad0afac59",
      "repository.url": "git+https://github.com/Mermade/oas-kit.git",
      "license": "BSD-3-Clause"
    }
  },
  "conversion_policy": {
    "target_openapi_version": "3.0.3",
    "silent_patching_allowed": false,
    "warnings_must_be_recorded": true,
    "conversion_alone_grants_eligibility": false,
    "semantic_preservation_audit_required": true,
    "openapi_validation_required": true,
    "reference_audit_required": true
  },
  "report_path": "/kaggle/working/fresh/stage_01/prose_operation

In [11]:
import hashlib
import json
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

PREFLIGHT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_preflight.json"
)

RUNTIME_TOOL_ROOT = Path(
    "/kaggle/working/stage_01_runtime_tools/swagger2openapi"
)

FROZEN_TOOLCHAIN_ROOT = (
    PROJECT_ROOT
    / "config"
    / "node_toolchain"
)

INSTALLATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_installation.json"
)

PINNED_PACKAGE = "swagger2openapi"
PINNED_VERSION = "7.0.8"


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


if not PREFLIGHT_PATH.is_file():
    raise RuntimeError(
        f"Missing converter preflight report: {PREFLIGHT_PATH}"
    )

preflight = json.loads(
    PREFLIGHT_PATH.read_text(encoding="utf-8")
)

if preflight.get("status") != "PASS":
    raise RuntimeError(
        "Converter preflight status is not PASS."
    )

converter_metadata = preflight["converter"]

if converter_metadata["package"] != PINNED_PACKAGE:
    raise RuntimeError(
        "Preflight converter package mismatch."
    )

if converter_metadata["pinned_version"] != PINNED_VERSION:
    raise RuntimeError(
        "Preflight converter version mismatch."
    )

expected_integrity = converter_metadata[
    "registry_metadata"
]["dist.integrity"]

npm_path = preflight["npm"]["path"]
node_path = preflight["node"]["path"]

RUNTIME_TOOL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FROZEN_TOOLCHAIN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

INSTALLATION_REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

package_definition = {
    "name": "prose-operation-stage-01-converter",
    "version": "1.0.0",
    "private": True,
    "description": (
        "Pinned Swagger 2.0 to OpenAPI 3.0 "
        "conversion toolchain for Stage 01"
    ),
    "license": "UNLICENSED",
    "dependencies": {
        PINNED_PACKAGE: PINNED_VERSION,
    },
}

runtime_package_path = (
    RUNTIME_TOOL_ROOT / "package.json"
)

write_json_atomic(
    runtime_package_path,
    package_definition,
)

installation = subprocess.run(
    [
        npm_path,
        "install",
        "--ignore-scripts",
        "--no-audit",
        "--no-fund",
        "--save-exact",
    ],
    cwd=RUNTIME_TOOL_ROOT,
    capture_output=True,
    text=True,
    timeout=600,
    check=False,
)

if installation.returncode != 0:
    raise RuntimeError(
        "Pinned converter installation failed.\n\n"
        f"STDOUT:\n{installation.stdout}\n\n"
        f"STDERR:\n{installation.stderr}"
    )

runtime_lock_path = (
    RUNTIME_TOOL_ROOT / "package-lock.json"
)

installed_package_path = (
    RUNTIME_TOOL_ROOT
    / "node_modules"
    / PINNED_PACKAGE
    / "package.json"
)

if not runtime_lock_path.is_file():
    raise RuntimeError(
        "npm did not produce package-lock.json."
    )

if not installed_package_path.is_file():
    raise RuntimeError(
        "Installed converter package metadata is missing."
    )

installed_package = json.loads(
    installed_package_path.read_text(
        encoding="utf-8"
    )
)

installed_version = str(
    installed_package.get("version")
)

if installed_version != PINNED_VERSION:
    raise RuntimeError(
        "Installed converter version mismatch: "
        f"expected {PINNED_VERSION}, "
        f"observed {installed_version}."
    )

package_lock = json.loads(
    runtime_lock_path.read_text(
        encoding="utf-8"
    )
)

lock_package_key = (
    f"node_modules/{PINNED_PACKAGE}"
)

locked_converter = package_lock.get(
    "packages",
    {},
).get(lock_package_key)

if locked_converter is None:
    raise RuntimeError(
        "Pinned converter is absent from package-lock.json."
    )

locked_version = str(
    locked_converter.get("version")
)

locked_integrity = locked_converter.get(
    "integrity"
)

if locked_version != PINNED_VERSION:
    raise RuntimeError(
        "Lockfile converter version mismatch."
    )

if locked_integrity != expected_integrity:
    raise RuntimeError(
        "Installed package integrity does not match "
        "the registry preflight record."
    )

module_probe_script = """
const converter = require("swagger2openapi");
const packageInfo = require("swagger2openapi/package.json");

const report = {
    version: packageInfo.version,
    convertObjAvailable:
        typeof converter.convertObj === "function",
    convertFileAvailable:
        typeof converter.convertFile === "function"
};

console.log(JSON.stringify(report));
"""

module_probe = subprocess.run(
    [
        node_path,
        "-e",
        module_probe_script,
    ],
    cwd=RUNTIME_TOOL_ROOT,
    capture_output=True,
    text=True,
    timeout=120,
    check=False,
)

if module_probe.returncode != 0:
    raise RuntimeError(
        "Installed converter could not be imported.\n\n"
        f"STDOUT:\n{module_probe.stdout}\n\n"
        f"STDERR:\n{module_probe.stderr}"
    )

try:
    probe_result = json.loads(
        module_probe.stdout.strip()
    )
except json.JSONDecodeError as exception:
    raise RuntimeError(
        "Converter module probe returned invalid JSON."
    ) from exception

if not probe_result.get("convertObjAvailable"):
    raise RuntimeError(
        "swagger2openapi.convertObj is unavailable."
    )

frozen_package_path = (
    FROZEN_TOOLCHAIN_ROOT / "package.json"
)

frozen_lock_path = (
    FROZEN_TOOLCHAIN_ROOT / "package-lock.json"
)

frozen_package_path.write_bytes(
    runtime_package_path.read_bytes()
)

frozen_lock_path.write_bytes(
    runtime_lock_path.read_bytes()
)

installation_report = {
    "status": "PASS",
    "package": PINNED_PACKAGE,
    "pinned_version": PINNED_VERSION,
    "installed_version": installed_version,
    "expected_registry_integrity": expected_integrity,
    "locked_integrity": locked_integrity,
    "integrity_matches": (
        locked_integrity == expected_integrity
    ),
    "node_version": preflight["node"]["version"],
    "npm_version": preflight["npm"]["version"],
    "runtime_tool_root": str(RUNTIME_TOOL_ROOT),
    "runtime_package_path": str(
        runtime_package_path
    ),
    "runtime_lock_path": str(
        runtime_lock_path
    ),
    "frozen_package": {
        "path": str(frozen_package_path),
        "sha256": sha256_file(
            frozen_package_path
        ),
    },
    "frozen_package_lock": {
        "path": str(frozen_lock_path),
        "sha256": sha256_file(
            frozen_lock_path
        ),
        "lockfile_version": package_lock.get(
            "lockfileVersion"
        ),
        "locked_package_count": max(
            0,
            len(package_lock.get("packages", {})) - 1,
        ),
    },
    "module_probe": probe_result,
    "install_stdout": installation.stdout.strip(),
    "install_stderr": installation.stderr.strip(),
}

write_json_atomic(
    INSTALLATION_REPORT_PATH,
    installation_report,
)

output = {
    **installation_report,
    "installation_report_path": str(
        INSTALLATION_REPORT_PATH
    ),
    "installation_report_sha256": sha256_file(
        INSTALLATION_REPORT_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 11: PASS")

{
  "status": "PASS",
  "package": "swagger2openapi",
  "pinned_version": "7.0.8",
  "installed_version": "7.0.8",
  "expected_registry_integrity": "sha512-upi/0ZGkYgEcLeGieoz8gT74oWHA0E7JivX7aN9mAf+Tc7BQoRBvnIGHoPDw+f9TXTW4s6kGYCZJtauP6OYp7g==",
  "locked_integrity": "sha512-upi/0ZGkYgEcLeGieoz8gT74oWHA0E7JivX7aN9mAf+Tc7BQoRBvnIGHoPDw+f9TXTW4s6kGYCZJtauP6OYp7g==",
  "integrity_matches": true,
  "node_version": "v22.23.4",
  "npm_version": "10.8.2",
  "runtime_tool_root": "/kaggle/working/stage_01_runtime_tools/swagger2openapi",
  "runtime_package_path": "/kaggle/working/stage_01_runtime_tools/swagger2openapi/package.json",
  "runtime_lock_path": "/kaggle/working/stage_01_runtime_tools/swagger2openapi/package-lock.json",
  "frozen_package": {
    "path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/config/node_toolchain/package.json",
    "sha256": "645b762d8805755a5cfaa31402be6fa4aace36b98fbff91042deb17f8aa76ca7"
  },
  "frozen_package_lock": {
    "path": "/kaggle/worki

In [12]:
import hashlib
import json
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

RUNTIME_TOOL_ROOT = Path(
    "/kaggle/working/stage_01_runtime_tools/swagger2openapi"
)

QUALIFICATION_ROOT = (
    PROJECT_ROOT
    / "data"
    / "development_examples"
    / "swagger_conversion_qualification"
)

SOURCE_ROOT = QUALIFICATION_ROOT / "swagger"
OUTPUT_ROOT = QUALIFICATION_ROOT / "openapi"
METADATA_ROOT = QUALIFICATION_ROOT / "metadata"

FROZEN_HARNESS_PATH = (
    PROJECT_ROOT
    / "config"
    / "node_toolchain"
    / "convert_swagger.js"
)

RUNTIME_HARNESS_PATH = (
    RUNTIME_TOOL_ROOT / "convert_swagger.js"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_fixture_execution.json"
)

NODE_PATH = node_path

for directory in [
    SOURCE_ROOT,
    OUTPUT_ROOT,
    METADATA_ROOT,
    REPORT_PATH.parent,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def write_json_atomic(path, value):
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


core_fixture = {
    "swagger": "2.0",
    "info": {
        "title": "Qualification User API",
        "description": (
            "Controlled fixture for semantic-preservation testing."
        ),
        "version": "1.0.0",
    },
    "host": "api.example.test",
    "basePath": "/v1",
    "schemes": ["https"],
    "consumes": ["application/json"],
    "produces": ["application/json"],
    "tags": [
        {
            "name": "users",
            "description": "Operations involving users.",
        }
    ],
    "securityDefinitions": {
        "ApiKeyAuth": {
            "type": "apiKey",
            "name": "X-API-Key",
            "in": "header",
        },
        "OAuth2Auth": {
            "type": "oauth2",
            "flow": "accessCode",
            "authorizationUrl": (
                "https://auth.example.test/authorize"
            ),
            "tokenUrl": (
                "https://auth.example.test/token"
            ),
            "scopes": {
                "users:read": "Read user records.",
            },
        },
    },
    "security": [
        {
            "ApiKeyAuth": [],
        }
    ],
    "parameters": {
        "TraceHeader": {
            "name": "X-Trace-ID",
            "in": "header",
            "required": False,
            "type": "string",
            "minLength": 8,
            "description": "Client-generated trace identifier.",
        }
    },
    "paths": {
        "/users/{userId}": {
            "parameters": [
                {
                    "name": "userId",
                    "in": "path",
                    "required": True,
                    "type": "integer",
                    "format": "int64",
                    "minimum": 1,
                    "description": "Stable user identifier.",
                }
            ],
            "get": {
                "tags": ["users"],
                "summary": "Fetch one user",
                "description": (
                    "Returns the user matching the supplied identifier."
                ),
                "operationId": "getUser",
                "parameters": [
                    {
                        "$ref": "#/parameters/TraceHeader",
                    },
                    {
                        "name": "fields",
                        "in": "query",
                        "required": False,
                        "type": "array",
                        "items": {
                            "type": "string",
                            "enum": ["name", "email"],
                        },
                        "collectionFormat": "csv",
                        "description": (
                            "Optional response fields."
                        ),
                    },
                ],
                "security": [
                    {
                        "OAuth2Auth": ["users:read"],
                    }
                ],
                "responses": {
                    "200": {
                        "description": (
                            "The requested user record."
                        ),
                        "headers": {
                            "X-Rate-Limit": {
                                "type": "integer",
                                "format": "int32",
                            }
                        },
                        "schema": {
                            "$ref": "#/definitions/User",
                        },
                    },
                    "404": {
                        "description": "The user was not found.",
                        "schema": {
                            "$ref": "#/definitions/Error",
                        },
                    },
                },
            },
        },
        "/users": {
            "post": {
                "tags": ["users"],
                "summary": "Create one user",
                "description": (
                    "Creates a user from a JSON request body."
                ),
                "operationId": "createUser",
                "parameters": [
                    {
                        "name": "payload",
                        "in": "body",
                        "required": True,
                        "description": (
                            "Properties of the user to create."
                        ),
                        "schema": {
                            "$ref": "#/definitions/NewUser",
                        },
                    }
                ],
                "responses": {
                    "201": {
                        "description": (
                            "The newly created user."
                        ),
                        "schema": {
                            "$ref": "#/definitions/User",
                        },
                    },
                    "400": {
                        "description": (
                            "The request body was invalid."
                        ),
                        "schema": {
                            "$ref": "#/definitions/Error",
                        },
                    },
                },
            }
        },
    },
    "definitions": {
        "User": {
            "type": "object",
            "description": "A stored user record.",
            "required": ["id", "name"],
            "additionalProperties": False,
            "properties": {
                "id": {
                    "type": "integer",
                    "format": "int64",
                    "minimum": 1,
                    "description": "Unique user identifier.",
                },
                "name": {
                    "type": "string",
                    "minLength": 1,
                    "description": "Display name.",
                },
                "email": {
                    "type": "string",
                    "format": "email",
                    "description": "Contact email address.",
                },
            },
        },
        "NewUser": {
            "type": "object",
            "description": "Input used to create a user.",
            "required": ["name", "email"],
            "properties": {
                "name": {
                    "type": "string",
                    "minLength": 1,
                },
                "email": {
                    "type": "string",
                    "format": "email",
                },
                "role": {
                    "type": "string",
                    "enum": ["reader", "writer"],
                    "default": "reader",
                },
            },
        },
        "Error": {
            "type": "object",
            "description": "A structured API error.",
            "required": ["code", "message"],
            "properties": {
                "code": {
                    "type": "string",
                },
                "message": {
                    "type": "string",
                },
            },
        },
    },
}

multipart_fixture = {
    "swagger": "2.0",
    "info": {
        "title": "Qualification Upload API",
        "description": (
            "Controlled multipart and file-upload fixture."
        ),
        "version": "1.0.0",
    },
    "host": "uploads.example.test",
    "basePath": "/service",
    "schemes": ["https", "http"],
    "produces": ["application/json"],
    "securityDefinitions": {
        "BasicAuth": {
            "type": "basic",
        }
    },
    "paths": {
        "/uploads": {
            "post": {
                "summary": "Upload one file",
                "description": (
                    "Uploads a binary file with optional labels."
                ),
                "operationId": "uploadFile",
                "consumes": ["multipart/form-data"],
                "parameters": [
                    {
                        "name": "file",
                        "in": "formData",
                        "required": True,
                        "type": "file",
                        "description": "Binary file contents.",
                    },
                    {
                        "name": "labels",
                        "in": "formData",
                        "required": False,
                        "type": "array",
                        "collectionFormat": "multi",
                        "items": {
                            "type": "string",
                        },
                        "description": "Labels assigned to the upload.",
                    },
                    {
                        "name": "checksum",
                        "in": "formData",
                        "required": False,
                        "type": "string",
                        "pattern": "^[a-f0-9]{64}$",
                        "description": "Expected SHA-256 checksum.",
                    },
                ],
                "security": [
                    {
                        "BasicAuth": [],
                    }
                ],
                "responses": {
                    "201": {
                        "description": (
                            "The upload was accepted."
                        ),
                        "schema": {
                            "$ref": "#/definitions/UploadReceipt",
                        },
                    },
                    "415": {
                        "description": (
                            "The media type was unsupported."
                        ),
                    },
                },
            }
        }
    },
    "definitions": {
        "UploadReceipt": {
            "type": "object",
            "description": "Receipt for an accepted upload.",
            "required": ["uploadId", "size"],
            "properties": {
                "uploadId": {
                    "type": "string",
                    "description": "Unique upload identifier.",
                },
                "size": {
                    "type": "integer",
                    "format": "int64",
                    "minimum": 0,
                    "description": "Uploaded size in bytes.",
                },
            },
        }
    },
}

fixtures = {
    "core_semantics": core_fixture,
    "multipart_upload": multipart_fixture,
}

for fixture_name, fixture in fixtures.items():
    write_json_atomic(
        SOURCE_ROOT / f"{fixture_name}.json",
        fixture,
    )

conversion_harness = r"""
"use strict";

const fs = require("fs");
const converter = require("swagger2openapi");

const [
    inputPath,
    outputPath,
    metadataPath
] = process.argv.slice(2);

if (!inputPath || !outputPath || !metadataPath) {
    console.error(
        "Expected input, output, and metadata paths."
    );
    process.exit(2);
}

const source = JSON.parse(
    fs.readFileSync(inputPath, "utf8")
);

const options = {
    patch: false,
    warnOnly: false,
    resolve: false,
    targetVersion: "3.0.3",
    rbname: "x-original-body-parameter-name"
};

converter.convertObj(
    source,
    options,
    (error, result) => {
        if (error) {
            console.error(
                error.stack || String(error)
            );
            process.exitCode = 1;
            return;
        }

        const openapi = result.openapi;

        if (!openapi || typeof openapi !== "object") {
            console.error(
                "Converter returned no OpenAPI document."
            );
            process.exitCode = 1;
            return;
        }

        const metadata = {
            converterVersion:
                require(
                    "swagger2openapi/package.json"
                ).version,
            targetVersion: options.targetVersion,
            outputOpenapiVersion:
                openapi.openapi || null,
            warnings: result.warnings || [],
            warningCount:
                (result.warnings || []).length,
            patches: Number(result.patches || 0),
            patchingEnabled: options.patch,
            warnOnlyEnabled: options.warnOnly,
            referenceResolutionEnabled:
                options.resolve
        };

        fs.writeFileSync(
            outputPath,
            JSON.stringify(openapi, null, 2) + "\n",
            "utf8"
        );

        fs.writeFileSync(
            metadataPath,
            JSON.stringify(metadata, null, 2) + "\n",
            "utf8"
        );
    }
);
""".strip() + "\n"

for harness_path in [
    RUNTIME_HARNESS_PATH,
    FROZEN_HARNESS_PATH,
]:
    harness_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    harness_path.write_text(
        conversion_harness,
        encoding="utf-8",
    )

execution_results = []

for fixture_name in sorted(fixtures):
    source_path = (
        SOURCE_ROOT / f"{fixture_name}.json"
    )
    output_path = (
        OUTPUT_ROOT / f"{fixture_name}.json"
    )
    metadata_path = (
        METADATA_ROOT / f"{fixture_name}.json"
    )

    completed = subprocess.run(
        [
            NODE_PATH,
            str(RUNTIME_HARNESS_PATH),
            str(source_path),
            str(output_path),
            str(metadata_path),
        ],
        cwd=RUNTIME_TOOL_ROOT,
        capture_output=True,
        text=True,
        timeout=180,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"Conversion failed for {fixture_name}.\n\n"
            f"STDOUT:\n{completed.stdout}\n\n"
            f"STDERR:\n{completed.stderr}"
        )

    if not output_path.is_file():
        raise RuntimeError(
            f"Missing converted fixture: {output_path}"
        )

    if not metadata_path.is_file():
        raise RuntimeError(
            f"Missing conversion metadata: {metadata_path}"
        )

    metadata = json.loads(
        metadata_path.read_text(
            encoding="utf-8"
        )
    )

    execution_results.append(
        {
            "fixture": fixture_name,
            "source_path": str(source_path),
            "source_sha256": sha256_file(
                source_path
            ),
            "converted_path": str(output_path),
            "converted_sha256": sha256_file(
                output_path
            ),
            "metadata_path": str(metadata_path),
            "metadata_sha256": sha256_file(
                metadata_path
            ),
            "converter_version": metadata[
                "converterVersion"
            ],
            "output_openapi_version": metadata[
                "outputOpenapiVersion"
            ],
            "warning_count": metadata[
                "warningCount"
            ],
            "warnings": metadata["warnings"],
            "patches": metadata["patches"],
        }
    )

problems = []

for result in execution_results:
    if result["converter_version"] != "7.0.8":
        problems.append(
            f"{result['fixture']}: converter version mismatch"
        )

    if result["output_openapi_version"] != "3.0.3":
        problems.append(
            f"{result['fixture']}: unexpected OpenAPI version"
        )

    if result["warning_count"] != 0:
        problems.append(
            f"{result['fixture']}: converter warnings present"
        )

    if result["patches"] != 0:
        problems.append(
            f"{result['fixture']}: converter patches were applied"
        )

report = {
    "status": (
        "PASS"
        if not problems
        else "BLOCKED"
    ),
    "converter": "swagger2openapi",
    "converter_version": "7.0.8",
    "target_openapi_version": "3.0.3",
    "fixture_count": len(execution_results),
    "conversion_results": execution_results,
    "qualification_problems": problems,
    "harness": {
        "runtime_path": str(
            RUNTIME_HARNESS_PATH
        ),
        "frozen_path": str(
            FROZEN_HARNESS_PATH
        ),
        "sha256": sha256_file(
            FROZEN_HARNESS_PATH
        ),
    },
}

write_json_atomic(
    REPORT_PATH,
    report,
)

output = {
    **report,
    "report_path": str(REPORT_PATH),
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

if problems:
    raise RuntimeError(
        "Controlled conversion qualification failed:\n"
        + "\n".join(problems)
    )

print("\nNOTEBOOK 01 — CELL 12: PASS")

{
  "status": "PASS",
  "converter": "swagger2openapi",
  "converter_version": "7.0.8",
  "target_openapi_version": "3.0.3",
  "fixture_count": 2,
  "conversion_results": [
    {
      "fixture": "core_semantics",
      "source_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/development_examples/swagger_conversion_qualification/swagger/core_semantics.json",
      "source_sha256": "6f97ca0e697e4eb9f435e4f4823c0c84a2345971dbf457bc5cc3b0e25368b6e8",
      "converted_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/development_examples/swagger_conversion_qualification/openapi/core_semantics.json",
      "converted_sha256": "8a75e79736e15b4844bd9d3740b95f91294c1c70ff2ffd40193cdba67eda2ec7",
      "metadata_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/development_examples/swagger_conversion_qualification/metadata/core_semantics.json",
      "metadata_sha256": "5d463e9670852ee504a6ad371d2635f4bd30e37cc7eed4a2336f31656f8fa5

In [13]:
import hashlib
import importlib.metadata
import json
from collections import Counter
from pathlib import Path

from openapi_spec_validator import validate

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

QUALIFICATION_ROOT = (
    PROJECT_ROOT
    / "data"
    / "development_examples"
    / "swagger_conversion_qualification"
)

SOURCE_ROOT = QUALIFICATION_ROOT / "swagger"
OUTPUT_ROOT = QUALIFICATION_ROOT / "openapi"
METADATA_ROOT = QUALIFICATION_ROOT / "metadata"

EXECUTION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_fixture_execution.json"
)

QUALIFICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_semantic_qualification.json"
)

HTTP_METHODS = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}

checks = []
failures = []


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


def compact(value, limit=500):
    try:
        text = json.dumps(
            value,
            sort_keys=True,
            ensure_ascii=False,
        )
    except TypeError:
        text = repr(value)

    if len(text) > limit:
        return text[:limit] + "...[truncated]"

    return text


def record_check(
    fixture,
    check_name,
    passed,
    expected=None,
    observed=None,
):
    result = {
        "fixture": fixture,
        "check": check_name,
        "pass": bool(passed),
    }

    if not passed:
        result["expected"] = compact(expected)
        result["observed"] = compact(observed)

        failures.append(
            {
                "fixture": fixture,
                "check": check_name,
                "expected": compact(expected),
                "observed": compact(observed),
            }
        )

    checks.append(result)


def check_equal(
    fixture,
    check_name,
    expected,
    observed,
):
    record_check(
        fixture,
        check_name,
        expected == observed,
        expected,
        observed,
    )


def operation_set(document):
    result = set()

    for path_name, path_item in (
        document.get("paths", {}).items()
    ):
        if not isinstance(path_item, dict):
            continue

        for method_name in path_item:
            method = method_name.lower()

            if method in HTTP_METHODS:
                result.add(
                    (path_name, method)
                )

    return result


def collect_prose(value):
    prose = []

    def visit(current):
        if isinstance(current, dict):
            for key, child in current.items():
                if (
                    key
                    in {
                        "title",
                        "summary",
                        "description",
                    }
                    and isinstance(child, str)
                    and child.strip()
                ):
                    prose.append(child)

                visit(child)

        elif isinstance(current, list):
            for child in current:
                visit(child)

    visit(value)
    return Counter(prose)


def collect_refs(value):
    refs = []

    def visit(current):
        if isinstance(current, dict):
            for key, child in current.items():
                if (
                    key == "$ref"
                    and isinstance(child, str)
                ):
                    refs.append(child)
                else:
                    visit(child)

        elif isinstance(current, list):
            for child in current:
                visit(child)

    visit(value)
    return refs


def remap_swagger_ref(reference):
    mappings = {
        "#/definitions/": "#/components/schemas/",
        "#/parameters/": "#/components/parameters/",
        "#/responses/": "#/components/responses/",
        "#/securityDefinitions/": (
            "#/components/securitySchemes/"
        ),
    }

    for old_prefix, new_prefix in mappings.items():
        if reference.startswith(old_prefix):
            return (
                new_prefix
                + reference[len(old_prefix):]
            )

    return reference


def remap_schema_refs(value):
    if isinstance(value, dict):
        remapped = {}

        for key, child in value.items():
            if (
                key == "$ref"
                and isinstance(child, str)
            ):
                remapped[key] = remap_swagger_ref(
                    child
                )
            else:
                remapped[key] = remap_schema_refs(
                    child
                )

        return remapped

    if isinstance(value, list):
        return [
            remap_schema_refs(child)
            for child in value
        ]

    return value


def resolve_local_pointer(document, reference):
    if not reference.startswith("#/"):
        return True

    current = document

    for token in reference[2:].split("/"):
        token = (
            token
            .replace("~1", "/")
            .replace("~0", "~")
        )

        if isinstance(current, dict) and token in current:
            current = current[token]
        else:
            return False

    return True


def effective_parameters(
    document,
    path_name,
    method,
):
    path_item = document["paths"][path_name]
    operation = path_item[method]

    return (
        list(path_item.get("parameters", []))
        + list(operation.get("parameters", []))
    )


def find_parameter(
    parameters,
    name,
    location,
):
    for parameter in parameters:
        if not isinstance(parameter, dict):
            continue

        if (
            parameter.get("name") == name
            and parameter.get("in") == location
        ):
            return parameter

    return None


def contains_key_value(
    value,
    expected_key,
    expected_value,
):
    if isinstance(value, dict):
        if value.get(expected_key) == expected_value:
            return True

        return any(
            contains_key_value(
                child,
                expected_key,
                expected_value,
            )
            for child in value.values()
        )

    if isinstance(value, list):
        return any(
            contains_key_value(
                child,
                expected_key,
                expected_value,
            )
            for child in value
        )

    return False


if not EXECUTION_REPORT_PATH.is_file():
    raise RuntimeError(
        "Missing Cell 12 fixture-execution report."
    )

execution_report = json.loads(
    EXECUTION_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

if execution_report.get("status") != "PASS":
    raise RuntimeError(
        "Cell 12 fixture-execution status is not PASS."
    )

fixture_names = [
    "core_semantics",
    "multipart_upload",
]

documents = {}

for fixture_name in fixture_names:
    source_path = (
        SOURCE_ROOT / f"{fixture_name}.json"
    )
    converted_path = (
        OUTPUT_ROOT / f"{fixture_name}.json"
    )
    metadata_path = (
        METADATA_ROOT / f"{fixture_name}.json"
    )

    for required_path in [
        source_path,
        converted_path,
        metadata_path,
    ]:
        if not required_path.is_file():
            raise RuntimeError(
                f"Missing qualification file: {required_path}"
            )

    source = json.loads(
        source_path.read_text(
            encoding="utf-8"
        )
    )

    converted = json.loads(
        converted_path.read_text(
            encoding="utf-8"
        )
    )

    metadata = json.loads(
        metadata_path.read_text(
            encoding="utf-8"
        )
    )

    documents[fixture_name] = {
        "source": source,
        "converted": converted,
        "metadata": metadata,
        "source_path": source_path,
        "converted_path": converted_path,
    }

    validation_diagnostic = None

    try:
        validate(converted)
        validation_passed = True
    except Exception as exception:
        validation_passed = False
        validation_diagnostic = (
            f"{type(exception).__name__}: "
            f"{exception}"
        )

    record_check(
        fixture_name,
        "openapi_spec_validation",
        validation_passed,
        "valid OpenAPI 3.0.3 document",
        validation_diagnostic,
    )

    check_equal(
        fixture_name,
        "output_version",
        "3.0.3",
        converted.get("openapi"),
    )

    check_equal(
        fixture_name,
        "converter_warning_count",
        0,
        metadata.get("warningCount"),
    )

    check_equal(
        fixture_name,
        "converter_patch_count",
        0,
        metadata.get("patches"),
    )

    check_equal(
        fixture_name,
        "info_preserved",
        source.get("info"),
        converted.get("info"),
    )

    check_equal(
        fixture_name,
        "operation_set_preserved",
        sorted(operation_set(source)),
        sorted(operation_set(converted)),
    )

    check_equal(
        fixture_name,
        "global_security_preserved",
        source.get("security", []),
        converted.get("security", []),
    )

    expected_definitions = remap_schema_refs(
        source.get("definitions", {})
    )

    observed_definitions = (
        converted
        .get("components", {})
        .get("schemas", {})
    )

    check_equal(
        fixture_name,
        "component_schemas_preserved",
        expected_definitions,
        observed_definitions,
    )

    source_prose = collect_prose(source)
    converted_prose = collect_prose(converted)

    lost_prose = {
        text: source_count - converted_prose[text]
        for text, source_count in source_prose.items()
        if converted_prose[text] < source_count
    }

    record_check(
        fixture_name,
        "all_source_prose_preserved",
        not lost_prose,
        "no lost title, summary, or description",
        lost_prose,
    )

    expected_refs = {
        remap_swagger_ref(reference)
        for reference in collect_refs(source)
        if reference.startswith("#/")
    }

    observed_refs = set(
        collect_refs(converted)
    )

    missing_remapped_refs = sorted(
        expected_refs - observed_refs
    )

    record_check(
        fixture_name,
        "internal_references_remapped",
        not missing_remapped_refs,
        sorted(expected_refs),
        {
            "missing": missing_remapped_refs,
            "observed": sorted(observed_refs),
        },
    )

    unresolved_refs = sorted(
        {
            reference
            for reference in observed_refs
            if reference.startswith("#/")
            and not resolve_local_pointer(
                converted,
                reference,
            )
        }
    )

    record_check(
        fixture_name,
        "no_unresolved_local_references",
        not unresolved_refs,
        [],
        unresolved_refs,
    )

    for path_name, method in sorted(
        operation_set(source)
    ):
        source_operation = (
            source["paths"][path_name][method]
        )

        converted_operation = (
            converted["paths"][path_name][method]
        )

        for field in [
            "operationId",
            "summary",
            "description",
            "tags",
            "deprecated",
            "security",
        ]:
            check_equal(
                fixture_name,
                (
                    f"{method.upper()} {path_name} "
                    f"{field}_preserved"
                ),
                source_operation.get(field),
                converted_operation.get(field),
            )

        source_responses = source_operation.get(
            "responses",
            {}
        )

        converted_responses = converted_operation.get(
            "responses",
            {}
        )

        check_equal(
            fixture_name,
            (
                f"{method.upper()} {path_name} "
                "response_statuses_preserved"
            ),
            sorted(source_responses),
            sorted(converted_responses),
        )

        for status_code, source_response in (
            source_responses.items()
        ):
            converted_response = (
                converted_responses.get(
                    status_code,
                    {}
                )
            )

            check_equal(
                fixture_name,
                (
                    f"{method.upper()} {path_name} "
                    f"response_{status_code}_description"
                ),
                source_response.get("description"),
                converted_response.get("description"),
            )

# Core fixture checks
core = documents["core_semantics"]["converted"]

check_equal(
    "core_semantics",
    "servers_preserved",
    ["https://api.example.test/v1"],
    [
        server.get("url")
        for server in core.get("servers", [])
    ],
)

core_security = (
    core
    .get("components", {})
    .get("securitySchemes", {})
)

check_equal(
    "core_semantics",
    "api_key_security_preserved",
    {
        "type": "apiKey",
        "name": "X-API-Key",
        "in": "header",
    },
    core_security.get("ApiKeyAuth"),
)

oauth = core_security.get(
    "OAuth2Auth",
    {}
)

authorization_code = (
    oauth
    .get("flows", {})
    .get("authorizationCode", {})
)

check_equal(
    "core_semantics",
    "oauth_type_preserved",
    "oauth2",
    oauth.get("type"),
)

check_equal(
    "core_semantics",
    "oauth_authorization_url_preserved",
    "https://auth.example.test/authorize",
    authorization_code.get("authorizationUrl"),
)

check_equal(
    "core_semantics",
    "oauth_token_url_preserved",
    "https://auth.example.test/token",
    authorization_code.get("tokenUrl"),
)

check_equal(
    "core_semantics",
    "oauth_scopes_preserved",
    {
        "users:read": "Read user records.",
    },
    authorization_code.get("scopes"),
)

trace_parameter = (
    core
    .get("components", {})
    .get("parameters", {})
    .get("TraceHeader", {})
)

check_equal(
    "core_semantics",
    "shared_header_identity_preserved",
    {
        "name": "X-Trace-ID",
        "in": "header",
        "required": False,
        "description": (
            "Client-generated trace identifier."
        ),
    },
    {
        "name": trace_parameter.get("name"),
        "in": trace_parameter.get("in"),
        "required": trace_parameter.get(
            "required",
            False,
        ),
        "description": trace_parameter.get(
            "description"
        ),
    },
)

check_equal(
    "core_semantics",
    "shared_header_schema_preserved",
    {
        "type": "string",
        "minLength": 8,
    },
    trace_parameter.get("schema"),
)

get_parameters = effective_parameters(
    core,
    "/users/{userId}",
    "get",
)

user_id_parameter = find_parameter(
    get_parameters,
    "userId",
    "path",
)

fields_parameter = find_parameter(
    get_parameters,
    "fields",
    "query",
)

record_check(
    "core_semantics",
    "path_parameter_present",
    user_id_parameter is not None,
    "userId path parameter",
    user_id_parameter,
)

if user_id_parameter is not None:
    check_equal(
        "core_semantics",
        "path_parameter_schema_preserved",
        {
            "type": "integer",
            "format": "int64",
            "minimum": 1,
        },
        user_id_parameter.get("schema"),
    )

    check_equal(
        "core_semantics",
        "path_parameter_required_preserved",
        True,
        user_id_parameter.get("required"),
    )

record_check(
    "core_semantics",
    "query_array_parameter_present",
    fields_parameter is not None,
    "fields query parameter",
    fields_parameter,
)

if fields_parameter is not None:
    check_equal(
        "core_semantics",
        "query_array_schema_preserved",
        {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["name", "email"],
            },
        },
        fields_parameter.get("schema"),
    )

    check_equal(
        "core_semantics",
        "query_csv_style_preserved",
        "form",
        fields_parameter.get(
            "style",
            "form",
        ),
    )

    check_equal(
        "core_semantics",
        "query_csv_explode_preserved",
        False,
        fields_parameter.get("explode"),
    )

create_user = core["paths"]["/users"]["post"]

create_request = create_user.get(
    "requestBody",
    {}
)

check_equal(
    "core_semantics",
    "json_request_required_preserved",
    True,
    create_request.get("required"),
)

check_equal(
    "core_semantics",
    "json_request_description_preserved",
    "Properties of the user to create.",
    create_request.get("description"),
)

check_equal(
    "core_semantics",
    "json_request_schema_preserved",
    {
        "$ref": "#/components/schemas/NewUser",
    },
    (
        create_request
        .get("content", {})
        .get("application/json", {})
        .get("schema")
    ),
)

# The rbname extension is stored on the converted operation,
# not inside the requestBody object.
record_check(
    "core_semantics",
    "body_parameter_name_trace_preserved",
    contains_key_value(
        create_user,
        "x-original-body-parameter-name",
        "payload",
    ),
    (
        "x-original-body-parameter-name="
        "payload"
    ),
    create_user,
)

check_equal(
    "core_semantics",
    "success_response_schema_preserved",
    {
        "$ref": "#/components/schemas/User",
    },
    (
        create_user["responses"]["201"]
        .get("content", {})
        .get("application/json", {})
        .get("schema")
    ),
)

rate_limit_header = (
    core["paths"]["/users/{userId}"]["get"]
    ["responses"]["200"]
    .get("headers", {})
    .get("X-Rate-Limit", {})
)

check_equal(
    "core_semantics",
    "response_header_schema_preserved",
    {
        "type": "integer",
        "format": "int32",
    },
    rate_limit_header.get("schema"),
)

# Multipart fixture checks
multipart = documents[
    "multipart_upload"
]["converted"]

check_equal(
    "multipart_upload",
    "servers_preserved",
    {
        "https://uploads.example.test/service",
        "http://uploads.example.test/service",
    },
    {
        server.get("url")
        for server in multipart.get(
            "servers",
            []
        )
    },
)

basic_auth = (
    multipart
    .get("components", {})
    .get("securitySchemes", {})
    .get("BasicAuth", {})
)

check_equal(
    "multipart_upload",
    "basic_auth_preserved",
    {
        "type": "http",
        "scheme": "basic",
    },
    basic_auth,
)

upload_operation = (
    multipart["paths"]["/uploads"]["post"]
)

upload_request = upload_operation.get(
    "requestBody",
    {}
)

multipart_media = (
    upload_request
    .get("content", {})
    .get("multipart/form-data", {})
)

multipart_schema = multipart_media.get(
    "schema",
    {}
)

multipart_properties = multipart_schema.get(
    "properties",
    {}
)

check_equal(
    "multipart_upload",
    "required_form_fields_preserved",
    ["file"],
    multipart_schema.get("required"),
)

check_equal(
    "multipart_upload",
    "file_field_preserved",
    {
        "type": "string",
        "format": "binary",
        "description": "Binary file contents.",
    },
    multipart_properties.get("file"),
)

check_equal(
    "multipart_upload",
    "labels_field_preserved",
    {
        "type": "array",
        "items": {
            "type": "string",
        },
        "description": (
            "Labels assigned to the upload."
        ),
    },
    multipart_properties.get("labels"),
)

check_equal(
    "multipart_upload",
    "checksum_field_preserved",
    {
        "type": "string",
        "pattern": "^[a-f0-9]{64}$",
        "description": (
            "Expected SHA-256 checksum."
        ),
    },
    multipart_properties.get("checksum"),
)

labels_encoding = (
    multipart_media
    .get("encoding", {})
    .get("labels", {})
)

effective_labels_style = labels_encoding.get(
    "style",
    "form",
)

effective_labels_explode = labels_encoding.get(
    "explode",
    True,
)

check_equal(
    "multipart_upload",
    "multipart_array_style_preserved",
    "form",
    effective_labels_style,
)

check_equal(
    "multipart_upload",
    "multipart_multi_encoding_preserved",
    True,
    effective_labels_explode,
)

check_equal(
    "multipart_upload",
    "upload_response_schema_preserved",
    {
        "$ref": (
            "#/components/schemas/"
            "UploadReceipt"
        ),
    },
    (
        upload_operation["responses"]["201"]
        .get("content", {})
        .get("application/json", {})
        .get("schema")
    ),
)

passed_check_count = sum(
    1
    for check in checks
    if check["pass"]
)

report = {
    "status": (
        "PASS"
        if not failures
        else "BLOCKED"
    ),
    "converter": "swagger2openapi",
    "converter_version": "7.0.8",
    "validator": "openapi-spec-validator",
    "validator_version": (
        importlib.metadata.version(
            "openapi-spec-validator"
        )
    ),
    "target_openapi_version": "3.0.3",
    "fixture_count": len(fixture_names),
    "check_count": len(checks),
    "passed_check_count": passed_check_count,
    "failed_check_count": len(failures),
    "failures": failures,
    "checks": checks,
    "qualification_decision": (
        "QUALIFIED_FOR_CORPUS_CONVERSION"
        if not failures
        else "NOT_QUALIFIED"
    ),
    "qualification_scope": [
        "operation-set preservation",
        "information and prose preservation",
        "schema and constraint preservation",
        "parameter preservation",
        "request-body conversion",
        "response and media-type preservation",
        "security-scheme conversion",
        "server URL conversion",
        "multipart and binary-file conversion",
        "reference remapping and resolution",
        "OpenAPI 3.0.3 validation",
    ],
    "fixture_hashes": {
        fixture_name: {
            "source_sha256": sha256_file(
                documents[fixture_name][
                    "source_path"
                ]
            ),
            "converted_sha256": sha256_file(
                documents[fixture_name][
                    "converted_path"
                ]
            ),
        }
        for fixture_name in fixture_names
    },
}

write_json_atomic(
    QUALIFICATION_REPORT_PATH,
    report,
)

summary = {
    key: value
    for key, value in report.items()
    if key != "checks"
}

summary["report_path"] = str(
    QUALIFICATION_REPORT_PATH
)

summary["report_sha256"] = sha256_file(
    QUALIFICATION_REPORT_PATH
)

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    )
)

if failures:
    raise RuntimeError(
        "Converter semantic qualification failed. "
        "Review the failure list above."
    )

print("\nNOTEBOOK 01 — CELL 13: PASS")

{
  "status": "PASS",
  "converter": "swagger2openapi",
  "converter_version": "7.0.8",
  "validator": "openapi-spec-validator",
  "validator_version": "0.9.0",
  "target_openapi_version": "3.0.3",
  "fixture_count": 2,
  "check_count": 79,
  "passed_check_count": 79,
  "failed_check_count": 0,
  "failures": [],
  "qualification_decision": "QUALIFIED_FOR_CORPUS_CONVERSION",
  "qualification_scope": [
    "operation-set preservation",
    "information and prose preservation",
    "schema and constraint preservation",
    "parameter preservation",
    "request-body conversion",
    "response and media-type preservation",
    "security-scheme conversion",
    "server URL conversion",
    "multipart and binary-file conversion",
    "reference remapping and resolution",
    "OpenAPI 3.0.3 validation"
  ],
  "fixture_hashes": {
    "core_semantics": {
      "source_sha256": "6f97ca0e697e4eb9f435e4f4823c0c84a2345971dbf457bc5cc3b0e25368b6e8",
      "converted_sha256": "8a75e79736e15b4844bd9d37

In [14]:
import hashlib
import json
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

POLICY_PATH = (
    PROJECT_ROOT
    / "config"
    / "stage_01.yaml"
)

PRE_AMENDMENT_POLICY_PATH = (
    PROJECT_ROOT
    / "config"
    / "stage_01_before_swagger_conversion.yaml"
)

SPECIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "specifications.csv"
)

PREFLIGHT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_preflight.json"
)

INSTALLATION_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_installation.json"
)

QUALIFICATION_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_semantic_qualification.json"
)

AMENDMENT_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "stage_01_swagger_conversion_amendment.json"
)


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


def write_yaml_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    serialized = yaml.safe_dump(
        value,
        sort_keys=False,
        allow_unicode=True,
        default_flow_style=False,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        serialized,
        encoding="utf-8",
    )

    temporary_path.replace(path)


required_paths = [
    POLICY_PATH,
    SPECIFICATIONS_PATH,
    PREFLIGHT_PATH,
    INSTALLATION_PATH,
    QUALIFICATION_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Missing required amendment inputs:\n"
        + "\n".join(missing_paths)
    )

preflight = json.loads(
    PREFLIGHT_PATH.read_text(
        encoding="utf-8"
    )
)

installation = json.loads(
    INSTALLATION_PATH.read_text(
        encoding="utf-8"
    )
)

qualification = json.loads(
    QUALIFICATION_PATH.read_text(
        encoding="utf-8"
    )
)

if preflight.get("status") != "PASS":
    raise RuntimeError(
        "Converter preflight status is not PASS."
    )

if installation.get("status") != "PASS":
    raise RuntimeError(
        "Converter installation status is not PASS."
    )

if qualification.get("status") != "PASS":
    raise RuntimeError(
        "Semantic qualification status is not PASS."
    )

if (
    qualification.get("qualification_decision")
    != "QUALIFIED_FOR_CORPUS_CONVERSION"
):
    raise RuntimeError(
        "Converter has not been qualified "
        "for corpus conversion."
    )

specifications = pd.read_csv(
    SPECIFICATIONS_PATH,
    low_memory=False,
)

swagger_mask = (
    specifications["dialect"]
    .fillna("")
    .eq("swagger_2_0")
)

swagger_file_count = int(
    swagger_mask.sum()
)

swagger_operation_count = int(
    specifications.loc[
        swagger_mask,
        "operation_count",
    ]
    .fillna(0)
    .sum()
)

if swagger_file_count != 2163:
    raise RuntimeError(
        "Unexpected Swagger file count: "
        f"expected 2163, observed {swagger_file_count}."
    )

if not PRE_AMENDMENT_POLICY_PATH.exists():
    PRE_AMENDMENT_POLICY_PATH.write_bytes(
        POLICY_PATH.read_bytes()
    )

pre_amendment_sha256 = sha256_file(
    PRE_AMENDMENT_POLICY_PATH
)

current_policy = yaml.safe_load(
    POLICY_PATH.read_text(
        encoding="utf-8"
    )
)

if not isinstance(current_policy, dict):
    raise RuntimeError(
        "Stage 01 policy is not a YAML mapping."
    )

conversion_policy = {
    "amendment_version": 1,
    "decision": "ENABLED_AFTER_QUALIFICATION",
    "prospective": True,
    "decision_point": (
        "Before corpus conversion, converted-spec "
        "validation, eligibility decisions, "
        "deduplication, and sampling."
    ),
    "reason": (
        "Dialect enumeration identified a substantial "
        "Swagger 2.0 subset. A pinned converter was "
        "therefore qualified before admitting any "
        "converted specification."
    ),
    "source_dialect": "swagger_2_0",
    "target_dialect": "openapi_3_0",
    "target_openapi_version": "3.0.3",
    "converter": {
        "package": "swagger2openapi",
        "version": "7.0.8",
        "registry_integrity": installation[
            "expected_registry_integrity"
        ],
        "repository": preflight[
            "converter"
        ]["registry_metadata"]["repository.url"],
        "package_lock_sha256": installation[
            "frozen_package_lock"
        ]["sha256"],
        "conversion_harness_sha256": (
            json.loads(
                (
                    PROJECT_ROOT
                    / "reports"
                    / "audit"
                    / "swagger_converter_fixture_execution.json"
                ).read_text(encoding="utf-8")
            )["harness"]["sha256"]
        ),
    },
    "qualification": {
        "status": qualification["status"],
        "decision": qualification[
            "qualification_decision"
        ],
        "check_count": qualification[
            "check_count"
        ],
        "passed_check_count": qualification[
            "passed_check_count"
        ],
        "failed_check_count": qualification[
            "failed_check_count"
        ],
        "report_sha256": sha256_file(
            QUALIFICATION_PATH
        ),
        "scope": qualification[
            "qualification_scope"
        ],
    },
    "corpus_scope": {
        "swagger_file_count": swagger_file_count,
        "swagger_operation_count": (
            swagger_operation_count
        ),
    },
    "conversion_options": {
        "patch": False,
        "warn_only": False,
        "resolve_external_references": False,
        "target_version": "3.0.3",
        "body_name_extension": (
            "x-original-body-parameter-name"
        ),
    },
    "admission_requirements": [
        "source parsed successfully under the frozen loader",
        "source dialect is Swagger 2.0",
        "converter completes without an exception",
        "no converter patch is applied",
        "no converter warning is emitted",
        "converted version is OpenAPI 3.0.3",
        "path and HTTP-method operation set is preserved",
        "operation metadata and prose are preserved",
        "parameters and request bodies are semantically preserved",
        "responses and media types are semantically preserved",
        "schemas, constraints, and internal references are preserved",
        "security requirements and schemes are preserved",
        "converted document passes OpenAPI validation",
        "converted document passes reference auditing",
    ],
    "failure_policy": {
        "automatic_repair": False,
        "silent_exclusion": False,
        "failed_conversion_status": (
            "CONVERSION_FAILED"
        ),
        "uncertain_semantics_status": (
            "CONVERSION_SEMANTICS_UNCERTAIN"
        ),
        "validation_failure_status": (
            "CONVERTED_VALIDATION_FAILED"
        ),
        "eligibility": False,
    },
}

current_policy[
    "swagger_2_0_conversion"
] = conversion_policy

write_yaml_atomic(
    POLICY_PATH,
    current_policy,
)

updated_policy_sha256 = sha256_file(
    POLICY_PATH
)

amendment_report = {
    "status": "PASS",
    "amendment": (
        "STAGE_01_SWAGGER_CONVERSION_ENABLED"
    ),
    "prospective": True,
    "pre_amendment_policy": {
        "path": str(
            PRE_AMENDMENT_POLICY_PATH
        ),
        "sha256": pre_amendment_sha256,
    },
    "updated_policy": {
        "path": str(POLICY_PATH),
        "sha256": updated_policy_sha256,
    },
    "converter": {
        "package": "swagger2openapi",
        "version": "7.0.8",
        "integrity_verified": installation[
            "integrity_matches"
        ],
    },
    "qualification": {
        "report_path": str(
            QUALIFICATION_PATH
        ),
        "report_sha256": sha256_file(
            QUALIFICATION_PATH
        ),
        "checks_passed": qualification[
            "passed_check_count"
        ],
        "checks_total": qualification[
            "check_count"
        ],
        "decision": qualification[
            "qualification_decision"
        ],
    },
    "affected_corpus": {
        "swagger_file_count": swagger_file_count,
        "swagger_operation_count": (
            swagger_operation_count
        ),
    },
    "conversion_started_before_amendment": False,
    "eligibility_decided_before_amendment": False,
}

write_json_atomic(
    AMENDMENT_REPORT_PATH,
    amendment_report,
)

output = {
    **amendment_report,
    "amendment_report_path": str(
        AMENDMENT_REPORT_PATH
    ),
    "amendment_report_sha256": sha256_file(
        AMENDMENT_REPORT_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 14: PASS")

{
  "status": "PASS",
  "amendment": "STAGE_01_SWAGGER_CONVERSION_ENABLED",
  "prospective": true,
  "pre_amendment_policy": {
    "path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/config/stage_01_before_swagger_conversion.yaml",
    "sha256": "4679894a4f1c3c82dba5f5f50a410e35e0e94b2d2ae4f1f5d1cf95e85aefba31"
  },
  "updated_policy": {
    "path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/config/stage_01.yaml",
    "sha256": "454b7b2e17f7343f9bc22953c96ec7501cf04b40021a76a3301d22a5078b630e"
  },
  "converter": {
    "package": "swagger2openapi",
    "version": "7.0.8",
    "integrity_verified": true
  },
  "qualification": {
    "report_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/reports/audit/swagger_converter_semantic_qualification.json",
    "report_sha256": "3836347d084d8c0aa1fd5b0a9d75664b19c751396c2eafeabc27880cc3135cc8",
    "checks_passed": 79,
    "checks_total": 79,
    "decision": "QUALIFIED_FOR_CORPUS_CONVERSION"
  },


In [15]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

SPECIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "specifications.csv"
)

specifications = pd.read_csv(
    SPECIFICATIONS_PATH,
    low_memory=False,
)

swagger_rows = (
    specifications.loc[
        specifications["dialect"]
        .fillna("")
        .eq("swagger_2_0")
    ]
    .copy()
    .sort_values(
        [
            "relative_path",
            "raw_id",
        ],
        kind="mergesort",
    )
)

missing_spec_id_mask = (
    swagger_rows["spec_id"].isna()
    | swagger_rows["spec_id"]
    .astype(str)
    .str.strip()
    .isin(["", "nan", "None"])
)

missing_spec_id_rows = swagger_rows.loc[
    missing_spec_id_mask
].copy()

nonmissing_rows = swagger_rows.loc[
    ~missing_spec_id_mask
].copy()

duplicate_spec_id_mask = (
    nonmissing_rows["spec_id"]
    .duplicated(keep=False)
)

duplicate_spec_id_rows = nonmissing_rows.loc[
    duplicate_spec_id_mask
].copy()

preferred_columns = [
    "raw_id",
    "spec_id",
    "relative_path",
    "provider_directory",
    "source_sha256",
    "source_bytes",
    "parser",
    "parse_status",
    "document_status",
    "dialect",
    "declared_version",
    "supported",
    "api_title",
    "api_version",
    "path_count",
    "operation_count",
    "canonical_sha256",
    "canonical_relative_path",
    "validation_status",
    "diagnostic",
]

available_columns = [
    column
    for column in preferred_columns
    if column in swagger_rows.columns
]


def records_without_nan(dataframe):
    records = []

    for row in dataframe[
        available_columns
    ].to_dict(orient="records"):
        cleaned = {}

        for key, value in row.items():
            if pd.isna(value):
                cleaned[key] = None
            elif hasattr(value, "item"):
                cleaned[key] = value.item()
            else:
                cleaned[key] = value

        records.append(cleaned)

    return records


report = {
    "swagger_row_count": int(
        len(swagger_rows)
    ),
    "missing_spec_id_count": int(
        len(missing_spec_id_rows)
    ),
    "genuine_duplicate_spec_id_row_count": int(
        len(duplicate_spec_id_rows)
    ),
    "genuine_duplicate_spec_id_count": int(
        duplicate_spec_id_rows[
            "spec_id"
        ].nunique()
    ),
    "missing_spec_id_rows": records_without_nan(
        missing_spec_id_rows
    ),
    "duplicate_spec_id_rows": records_without_nan(
        duplicate_spec_id_rows
    ),
    "status": (
        "NEEDS_REPAIR"
        if (
            len(missing_spec_id_rows) > 0
            or len(duplicate_spec_id_rows) > 0
        )
        else "PASS"
    ),
}

print(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    )
)

print(
    "\nTEMPORARY CELL 15 DIAGNOSTIC: COMPLETE"
)

{
  "swagger_row_count": 2163,
  "missing_spec_id_count": 2,
  "genuine_duplicate_spec_id_row_count": 0,
  "genuine_duplicate_spec_id_count": 0,
  "missing_spec_id_rows": [
    {
      "raw_id": "raw_a3ae7b4227d333ffd84d",
      "spec_id": null,
      "relative_path": "launchdarkly.com/5.3.0/swagger.yaml",
      "provider_directory": "launchdarkly.com",
      "source_sha256": "243831bbc39d1d89543aed6f6928353386777c31d13af17307bc88fec9c45eba",
      "source_bytes": 136723,
      "parser": "OpenAPISafeLoader",
      "parse_status": "PASS",
      "document_status": "CANONICALIZATION_FAILED",
      "dialect": "swagger_2_0",
      "declared_version": "2.0",
      "supported": false,
      "api_title": "LaunchDarkly REST API",
      "api_version": "5.3.0",
      "path_count": 64.0,
      "operation_count": 105.0,
      "canonical_sha256": null,
      "canonical_relative_path": null,
      "validation_status": "NOT_APPLICABLE",
      "diagnostic": "TypeError(\"'<' not supported between instan

In [16]:
import json
from pathlib import Path

import yaml
from yaml.nodes import MappingNode, ScalarNode, SequenceNode

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

RELATIVE_PATHS = [
    "launchdarkly.com/5.3.0/swagger.yaml",
    "victorops.com/0.0.3/swagger.yaml",
]

loader_class = globals().get(
    "OpenAPISafeLoader",
    yaml.SafeLoader,
)

STRING_TAG = "tag:yaml.org,2002:str"


def key_text(node):
    if isinstance(node, ScalarNode):
        return node.value

    return f"<{type(node).__name__}>"


def audit_node(
    node,
    source_lines,
    path=(),
    nonstring_keys=None,
    collisions=None,
):
    if nonstring_keys is None:
        nonstring_keys = []

    if collisions is None:
        collisions = []

    if isinstance(node, MappingNode):
        lexical_keys = {}

        for key_node, value_node in node.value:
            lexical_key = key_text(key_node)
            line_number = (
                key_node.start_mark.line + 1
            )

            if lexical_key in lexical_keys:
                collisions.append(
                    {
                        "mapping_path": "/".join(path),
                        "lexical_key": lexical_key,
                        "first_line": lexical_keys[
                            lexical_key
                        ],
                        "second_line": line_number,
                    }
                )
            else:
                lexical_keys[
                    lexical_key
                ] = line_number

            if (
                not isinstance(
                    key_node,
                    ScalarNode,
                )
                or key_node.tag != STRING_TAG
            ):
                source_line = (
                    source_lines[
                        line_number - 1
                    ].rstrip()
                    if (
                        0
                        < line_number
                        <= len(source_lines)
                    )
                    else None
                )

                nonstring_keys.append(
                    {
                        "parent_path": "/".join(path),
                        "lexical_key": lexical_key,
                        "yaml_tag": key_node.tag,
                        "line_number": line_number,
                        "source_line": source_line,
                    }
                )

            audit_node(
                value_node,
                source_lines,
                path + (lexical_key,),
                nonstring_keys,
                collisions,
            )

    elif isinstance(node, SequenceNode):
        for index, child in enumerate(
            node.value
        ):
            audit_node(
                child,
                source_lines,
                path + (f"[{index}]",),
                nonstring_keys,
                collisions,
            )

    return nonstring_keys, collisions


file_reports = []

for relative_path in RELATIVE_PATHS:
    source_path = (
        CORPUS_ROOT / relative_path
    )

    if not source_path.is_file():
        raise RuntimeError(
            f"Missing source file: {source_path}"
        )

    source_text = source_path.read_text(
        encoding="utf-8"
    )

    source_lines = source_text.splitlines()

    root_node = yaml.compose(
        source_text,
        Loader=loader_class,
    )

    if root_node is None:
        raise RuntimeError(
            f"Empty YAML document: {relative_path}"
        )

    nonstring_keys, collisions = audit_node(
        root_node,
        source_lines,
    )

    file_reports.append(
        {
            "relative_path": relative_path,
            "nonstring_mapping_key_count": len(
                nonstring_keys
            ),
            "lexical_key_collision_count": len(
                collisions
            ),
            "can_preserve_keys_as_strings": (
                len(collisions) == 0
            ),
            "nonstring_mapping_keys": (
                nonstring_keys
            ),
            "lexical_key_collisions": collisions,
        }
    )

report = {
    "file_count": len(file_reports),
    "files": file_reports,
    "all_keys_can_be_preserved_as_strings": all(
        file_report[
            "can_preserve_keys_as_strings"
        ]
        for file_report in file_reports
    ),
    "status": (
        "SAFE_TO_REPAIR"
        if all(
            file_report[
                "can_preserve_keys_as_strings"
            ]
            for file_report in file_reports
        )
        else "BLOCKED"
    ),
}

print(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    )
)

print(
    "\nTEMPORARY CELL 15 KEY AUDIT: COMPLETE"
)

{
  "file_count": 2,
  "files": [
    {
      "relative_path": "launchdarkly.com/5.3.0/swagger.yaml",
      "nonstring_mapping_key_count": 7,
      "lexical_key_collision_count": 0,
      "can_preserve_keys_as_strings": true,
      "nonstring_mapping_keys": [
        {
          "parent_path": "parameters/DestinationPostRequest/schema/properties",
          "lexical_key": "on",
          "yaml_tag": "tag:yaml.org,2002:bool",
          "line_number": 224,
          "source_line": "        on:"
        },
        {
          "parent_path": "parameters/IntegrationPostRequest/schema/properties",
          "lexical_key": "on",
          "yaml_tag": "tag:yaml.org,2002:bool",
          "line_number": 447,
          "source_line": "        on:"
        },
        {
          "parent_path": "parameters/WebhookPostRequest/schema/properties",
          "lexical_key": "on",
          "yaml_tag": "tag:yaml.org,2002:bool",
          "line_number": 817,
          "source_line": "        on:"
        

In [17]:
import hashlib
import json
from pathlib import Path

import pandas as pd
import yaml
from yaml.constructor import ConstructorError
from yaml.nodes import MappingNode, ScalarNode, SequenceNode
from yaml.resolver import BaseResolver

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

SPECIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "specifications.csv"
)

REPAIR_ROOT = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_source_repairs"
)

REPAIR_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_string_key_canonicalization_repair.json"
)

EXPECTED_REPAIRS = {
    "raw_a3ae7b4227d333ffd84d": {
        "relative_path": (
            "launchdarkly.com/5.3.0/swagger.yaml"
        ),
        "expected_nonstring_key_count": 7,
        "expected_lexical_keys": ["on"] * 7,
    },
    "raw_193bd739ad90054cc023": {
        "relative_path": (
            "victorops.com/0.0.3/swagger.yaml"
        ),
        "expected_nonstring_key_count": 2,
        "expected_lexical_keys": [
            "off",
            "on",
        ],
    },
}

REPAIR_NAMESPACE = (
    "stage01-swagger-string-key-repair-v1"
)

REPAIR_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPAIR_REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def sha256_bytes(value):
    return hashlib.sha256(
        value
    ).hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


if not SPECIFICATIONS_PATH.is_file():
    raise RuntimeError(
        f"Missing specifications inventory: "
        f"{SPECIFICATIONS_PATH}"
    )

original_specifications_sha256 = (
    sha256_file(SPECIFICATIONS_PATH)
)

specifications = pd.read_csv(
    SPECIFICATIONS_PATH,
    low_memory=False,
)

required_columns = {
    "raw_id",
    "spec_id",
    "relative_path",
    "source_sha256",
    "parse_status",
    "document_status",
    "dialect",
    "supported",
    "canonical_sha256",
    "canonical_relative_path",
    "validation_status",
    "diagnostic",
}

missing_columns = sorted(
    required_columns
    - set(specifications.columns)
)

if missing_columns:
    raise RuntimeError(
        "Specifications inventory is missing columns: "
        + ", ".join(missing_columns)
    )

swagger_mask = (
    specifications["dialect"]
    .fillna("")
    .eq("swagger_2_0")
)

missing_id_mask = (
    swagger_mask
    & (
        specifications["spec_id"].isna()
        | specifications["spec_id"]
        .astype(str)
        .str.strip()
        .isin(["", "nan", "None"])
    )
)

observed_missing_raw_ids = set(
    specifications.loc[
        missing_id_mask,
        "raw_id",
    ].astype(str)
)

expected_missing_raw_ids = set(
    EXPECTED_REPAIRS
)

if observed_missing_raw_ids != expected_missing_raw_ids:
    raise RuntimeError(
        "Unexpected missing-spec-ID set.\n"
        f"Expected: {sorted(expected_missing_raw_ids)}\n"
        f"Observed: {sorted(observed_missing_raw_ids)}"
    )

base_loader = globals().get(
    "OpenAPISafeLoader",
    yaml.SafeLoader,
)


class OpenAPIStringKeyLoader(base_loader):
    pass


def construct_string_key_mapping(
    loader,
    node,
    deep=False,
):
    if not isinstance(node, MappingNode):
        raise ConstructorError(
            None,
            None,
            "Expected a mapping node.",
            node.start_mark,
        )

    loader.flatten_mapping(node)

    mapping = {}

    for key_node, value_node in node.value:
        if not isinstance(
            key_node,
            ScalarNode,
        ):
            raise ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                (
                    "OpenAPI/JSON mapping keys "
                    "must be scalar strings"
                ),
                key_node.start_mark,
            )

        # Preserve the exact lexical YAML key.
        # This prevents YAML 1.1 from changing
        # keys such as `on` and `off` to booleans.
        key = key_node.value

        if key in mapping:
            raise ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                (
                    "duplicate mapping key after "
                    f"string preservation: {key!r}"
                ),
                key_node.start_mark,
            )

        value = loader.construct_object(
            value_node,
            deep=deep,
        )

        mapping[key] = value

    return mapping


def construct_timestamp_as_string(
    loader,
    node,
):
    return loader.construct_scalar(node)


OpenAPIStringKeyLoader.add_constructor(
    BaseResolver.DEFAULT_MAPPING_TAG,
    construct_string_key_mapping,
)

OpenAPIStringKeyLoader.add_constructor(
    "tag:yaml.org,2002:timestamp",
    construct_timestamp_as_string,
)


def inspect_mapping_keys(
    node,
    source_lines,
    path=(),
    nonstring_keys=None,
    lexical_collisions=None,
):
    if nonstring_keys is None:
        nonstring_keys = []

    if lexical_collisions is None:
        lexical_collisions = []

    if isinstance(node, MappingNode):
        keys_in_mapping = {}

        for key_node, value_node in node.value:
            if isinstance(
                key_node,
                ScalarNode,
            ):
                lexical_key = key_node.value
            else:
                lexical_key = (
                    f"<{type(key_node).__name__}>"
                )

            line_number = (
                key_node.start_mark.line + 1
            )

            if lexical_key in keys_in_mapping:
                lexical_collisions.append(
                    {
                        "parent_path": "/".join(path),
                        "lexical_key": lexical_key,
                        "first_line": keys_in_mapping[
                            lexical_key
                        ],
                        "second_line": line_number,
                    }
                )
            else:
                keys_in_mapping[
                    lexical_key
                ] = line_number

            if (
                not isinstance(
                    key_node,
                    ScalarNode,
                )
                or key_node.tag
                != "tag:yaml.org,2002:str"
            ):
                source_line = (
                    source_lines[
                        line_number - 1
                    ].rstrip()
                    if (
                        0
                        < line_number
                        <= len(source_lines)
                    )
                    else None
                )

                nonstring_keys.append(
                    {
                        "parent_path": "/".join(path),
                        "lexical_key": lexical_key,
                        "yaml_tag": key_node.tag,
                        "line_number": line_number,
                        "source_line": source_line,
                    }
                )

            inspect_mapping_keys(
                value_node,
                source_lines,
                path + (lexical_key,),
                nonstring_keys,
                lexical_collisions,
            )

    elif isinstance(node, SequenceNode):
        for index, child in enumerate(
            node.value
        ):
            inspect_mapping_keys(
                child,
                source_lines,
                path + (f"[{index}]",),
                nonstring_keys,
                lexical_collisions,
            )

    return (
        nonstring_keys,
        lexical_collisions,
    )


existing_spec_ids = set(
    specifications.loc[
        specifications["spec_id"].notna(),
        "spec_id",
    ].astype(str)
)

repair_results = []

for raw_id in sorted(EXPECTED_REPAIRS):
    expectation = EXPECTED_REPAIRS[
        raw_id
    ]

    row_indices = specifications.index[
        specifications["raw_id"]
        .astype(str)
        .eq(raw_id)
    ].tolist()

    if len(row_indices) != 1:
        raise RuntimeError(
            f"Expected exactly one row for {raw_id}; "
            f"observed {len(row_indices)}."
        )

    row_index = row_indices[0]

    observed_relative_path = str(
        specifications.at[
            row_index,
            "relative_path",
        ]
    )

    expected_relative_path = expectation[
        "relative_path"
    ]

    if (
        observed_relative_path
        != expected_relative_path
    ):
        raise RuntimeError(
            f"Relative-path mismatch for {raw_id}."
        )

    if (
        str(
            specifications.at[
                row_index,
                "parse_status",
            ]
        )
        != "PASS"
    ):
        raise RuntimeError(
            f"Source parse status is not PASS "
            f"for {raw_id}."
        )

    source_path = (
        CORPUS_ROOT
        / observed_relative_path
    )

    if not source_path.is_file():
        raise RuntimeError(
            f"Missing source file: {source_path}"
        )

    observed_source_sha256 = sha256_file(
        source_path
    )

    expected_source_sha256 = str(
        specifications.at[
            row_index,
            "source_sha256",
        ]
    )

    if (
        observed_source_sha256
        != expected_source_sha256
    ):
        raise RuntimeError(
            f"Source hash mismatch for {raw_id}."
        )

    source_text = source_path.read_text(
        encoding="utf-8"
    )

    source_lines = source_text.splitlines()

    source_node = yaml.compose(
        source_text,
        Loader=base_loader,
    )

    if source_node is None:
        raise RuntimeError(
            f"Empty YAML document: {source_path}"
        )

    (
        nonstring_keys,
        lexical_collisions,
    ) = inspect_mapping_keys(
        source_node,
        source_lines,
    )

    if lexical_collisions:
        raise RuntimeError(
            "Lexical key collisions prevent safe "
            f"repair for {raw_id}: "
            f"{lexical_collisions}"
        )

    observed_lexical_keys = [
        item["lexical_key"]
        for item in nonstring_keys
    ]

    expected_lexical_keys = expectation[
        "expected_lexical_keys"
    ]

    if (
        len(nonstring_keys)
        != expectation[
            "expected_nonstring_key_count"
        ]
    ):
        raise RuntimeError(
            f"Unexpected non-string key count "
            f"for {raw_id}."
        )

    if (
        observed_lexical_keys
        != expected_lexical_keys
    ):
        raise RuntimeError(
            f"Unexpected lexical keys for {raw_id}.\n"
            f"Expected: {expected_lexical_keys}\n"
            f"Observed: {observed_lexical_keys}"
        )

    repaired_document = yaml.load(
        source_text,
        Loader=OpenAPIStringKeyLoader,
    )

    if not isinstance(
        repaired_document,
        dict,
    ):
        raise RuntimeError(
            f"Repaired document is not a mapping: "
            f"{raw_id}"
        )

    canonical_text = json.dumps(
        repaired_document,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ) + "\n"

    canonical_bytes = canonical_text.encode(
        "utf-8"
    )

    canonical_sha256 = sha256_bytes(
        canonical_bytes
    )

    identity_record = {
        "namespace": REPAIR_NAMESPACE,
        "relative_path": (
            observed_relative_path
        ),
        "source_sha256": (
            observed_source_sha256
        ),
        "canonical_sha256": (
            canonical_sha256
        ),
    }

    identity_bytes = json.dumps(
        identity_record,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")

    repaired_spec_id = (
        "spec_"
        + hashlib.sha256(
            identity_bytes
        ).hexdigest()[:24]
    )

    if repaired_spec_id in existing_spec_ids:
        raise RuntimeError(
            "Repaired spec ID collides with an "
            f"existing ID: {repaired_spec_id}"
        )

    existing_spec_ids.add(
        repaired_spec_id
    )

    repaired_source_path = (
        REPAIR_ROOT
        / f"{repaired_spec_id}.json"
    )

    temporary_repair_path = (
        repaired_source_path.with_suffix(
            ".json.tmp"
        )
    )

    temporary_repair_path.write_text(
        canonical_text,
        encoding="utf-8",
    )

    temporary_repair_path.replace(
        repaired_source_path
    )

    if (
        sha256_file(repaired_source_path)
        != canonical_sha256
    ):
        raise RuntimeError(
            "Repaired canonical source hash "
            f"verification failed for {raw_id}."
        )

    specifications.at[
        row_index,
        "spec_id",
    ] = repaired_spec_id

    specifications.at[
        row_index,
        "canonical_sha256",
    ] = canonical_sha256

    specifications.at[
        row_index,
        "document_status",
    ] = "UNSUPPORTED_DIALECT"

    specifications.at[
        row_index,
        "supported",
    ] = False

    specifications.at[
        row_index,
        "validation_status",
    ] = "NOT_APPLICABLE"

    specifications.at[
        row_index,
        "diagnostic",
    ] = (
        "Swagger 2.0 pending qualified conversion. "
        "Canonicalization repaired by preserving "
        "all YAML mapping keys as their exact "
        "lexical strings."
    )

    repair_results.append(
        {
            "raw_id": raw_id,
            "relative_path": (
                observed_relative_path
            ),
            "source_sha256": (
                observed_source_sha256
            ),
            "repaired_spec_id": (
                repaired_spec_id
            ),
            "canonical_sha256": (
                canonical_sha256
            ),
            "repaired_source_path": str(
                repaired_source_path
            ),
            "repaired_source_relative_path": (
                repaired_source_path
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "nonstring_key_count": len(
                nonstring_keys
            ),
            "preserved_keys": (
                nonstring_keys
            ),
            "lexical_collision_count": 0,
        }
    )

remaining_missing_mask = (
    specifications["dialect"]
    .fillna("")
    .eq("swagger_2_0")
    & (
        specifications["spec_id"].isna()
        | specifications["spec_id"]
        .astype(str)
        .str.strip()
        .isin(["", "nan", "None"])
    )
)

if remaining_missing_mask.any():
    remaining_paths = (
        specifications.loc[
            remaining_missing_mask,
            "relative_path",
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Swagger rows still have missing IDs:\n"
        + "\n".join(remaining_paths)
    )

swagger_spec_ids = (
    specifications.loc[
        specifications["dialect"]
        .fillna("")
        .eq("swagger_2_0"),
        "spec_id",
    ]
    .astype(str)
)

if not swagger_spec_ids.is_unique:
    duplicated = (
        swagger_spec_ids.loc[
            swagger_spec_ids.duplicated(
                keep=False
            )
        ]
        .tolist()
    )

    raise RuntimeError(
        "Swagger spec IDs remain duplicated: "
        + ", ".join(duplicated[:20])
    )

temporary_inventory_path = (
    SPECIFICATIONS_PATH.with_suffix(
        ".csv.tmp"
    )
)

specifications.to_csv(
    temporary_inventory_path,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
)

temporary_inventory_path.replace(
    SPECIFICATIONS_PATH
)

reloaded = pd.read_csv(
    SPECIFICATIONS_PATH,
    low_memory=False,
)

reloaded_swagger = reloaded.loc[
    reloaded["dialect"]
    .fillna("")
    .eq("swagger_2_0")
]

reloaded_missing_count = int(
    (
        reloaded_swagger["spec_id"].isna()
        | reloaded_swagger["spec_id"]
        .astype(str)
        .str.strip()
        .isin(["", "nan", "None"])
    ).sum()
)

if reloaded_missing_count != 0:
    raise RuntimeError(
        "Rewritten inventory still contains "
        "missing Swagger spec IDs."
    )

if not reloaded_swagger[
    "spec_id"
].is_unique:
    raise RuntimeError(
        "Rewritten inventory contains duplicate "
        "Swagger spec IDs."
    )

report = {
    "status": "PASS",
    "repair_namespace": (
        REPAIR_NAMESPACE
    ),
    "repair_reason": (
        "PyYAML YAML 1.1 implicitly resolved "
        "unquoted OpenAPI property names `on` "
        "and `off` as booleans. OpenAPI and JSON "
        "require mapping keys to be strings, so "
        "their exact lexical forms were preserved."
    ),
    "files_repaired": len(
        repair_results
    ),
    "mapping_keys_repaired": sum(
        result["nonstring_key_count"]
        for result in repair_results
    ),
    "lexical_collisions": 0,
    "repair_results": repair_results,
    "specifications_inventory": {
        "path": str(
            SPECIFICATIONS_PATH
        ),
        "sha256_before": (
            original_specifications_sha256
        ),
        "sha256_after": sha256_file(
            SPECIFICATIONS_PATH
        ),
        "swagger_row_count": int(
            len(reloaded_swagger)
        ),
        "missing_swagger_spec_ids": (
            reloaded_missing_count
        ),
        "duplicate_swagger_spec_ids": 0,
    },
}

write_json_atomic(
    REPAIR_REPORT_PATH,
    report,
)

output = {
    **report,
    "repair_report_path": str(
        REPAIR_REPORT_PATH
    ),
    "repair_report_sha256": sha256_file(
        REPAIR_REPORT_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 15: PASS")

{
  "status": "PASS",
  "repair_namespace": "stage01-swagger-string-key-repair-v1",
  "repair_reason": "PyYAML YAML 1.1 implicitly resolved unquoted OpenAPI property names `on` and `off` as booleans. OpenAPI and JSON require mapping keys to be strings, so their exact lexical forms were preserved.",
  "files_repaired": 2,
  "mapping_keys_repaired": 9,
  "lexical_collisions": 0,
  "repair_results": [
    {
      "raw_id": "raw_193bd739ad90054cc023",
      "relative_path": "victorops.com/0.0.3/swagger.yaml",
      "source_sha256": "99c4e1026f1fd0ef8b533627c7283e9ea840850bdd954b806e8cf1d6616ccfce",
      "repaired_spec_id": "spec_7ad0e8882b6b6e5afd981259",
      "canonical_sha256": "97e625d2f22b7d13cfe06b62428daa0c3bf874985cff655728b74846ae491b78",
      "repaired_source_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/audits/swagger_source_repairs/spec_7ad0e8882b6b6e5afd981259.json",
      "repaired_source_relative_path": "data/audits/swagger_source_repairs/spec_7ad0

In [18]:
import hashlib
import json
import subprocess
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

RUNTIME_TOOL_ROOT = Path(
    "/kaggle/working/stage_01_runtime_tools/swagger2openapi"
)

SPECIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "specifications.csv"
)

QUALIFICATION_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converter_semantic_qualification.json"
)

AMENDMENT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "stage_01_swagger_conversion_amendment.json"
)

REPAIR_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_string_key_canonicalization_repair.json"
)

CONVERSION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "swagger_conversion_manifest.json"
)

CANONICAL_ROOT = (
    PROJECT_ROOT
    / "data"
    / "canonical"
)

ITEM_METADATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_conversion_items"
)

RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_conversion_results.jsonl"
)

SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_conversion_execution.json"
)

RUNTIME_HARNESS_PATH = (
    RUNTIME_TOOL_ROOT
    / "batch_convert_swagger.js"
)

FROZEN_HARNESS_PATH = (
    PROJECT_ROOT
    / "config"
    / "node_toolchain"
    / "batch_convert_swagger.js"
)

NODE_PATH = Path(node_path)
CONCURRENCY = 2

for directory in [
    CONVERSION_MANIFEST_PATH.parent,
    CANONICAL_ROOT,
    ITEM_METADATA_ROOT,
    RESULTS_PATH.parent,
    SUMMARY_PATH.parent,
    FROZEN_HARNESS_PATH.parent,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


required_paths = [
    SPECIFICATIONS_PATH,
    QUALIFICATION_PATH,
    AMENDMENT_PATH,
    REPAIR_REPORT_PATH,
    NODE_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise RuntimeError(
        "Missing conversion prerequisites:\n"
        + "\n".join(missing_paths)
    )

qualification = json.loads(
    QUALIFICATION_PATH.read_text(
        encoding="utf-8"
    )
)

amendment = json.loads(
    AMENDMENT_PATH.read_text(
        encoding="utf-8"
    )
)

repair_report = json.loads(
    REPAIR_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

if qualification.get("status") != "PASS":
    raise RuntimeError(
        "Converter semantic qualification is not PASS."
    )

if (
    qualification.get("qualification_decision")
    != "QUALIFIED_FOR_CORPUS_CONVERSION"
):
    raise RuntimeError(
        "Converter is not qualified for corpus conversion."
    )

if amendment.get("status") != "PASS":
    raise RuntimeError(
        "Conversion amendment status is not PASS."
    )

if not amendment.get("prospective"):
    raise RuntimeError(
        "Conversion amendment is not prospective."
    )

if repair_report.get("status") != "PASS":
    raise RuntimeError(
        "String-key repair status is not PASS."
    )

specifications = pd.read_csv(
    SPECIFICATIONS_PATH,
    low_memory=False,
)

required_columns = {
    "raw_id",
    "spec_id",
    "relative_path",
    "provider_directory",
    "source_sha256",
    "source_bytes",
    "parse_status",
    "dialect",
    "operation_count",
}

missing_columns = sorted(
    required_columns
    - set(specifications.columns)
)

if missing_columns:
    raise RuntimeError(
        "Specifications inventory is missing columns: "
        + ", ".join(missing_columns)
    )

swagger_rows = (
    specifications.loc[
        specifications["dialect"]
        .fillna("")
        .eq("swagger_2_0")
    ]
    .copy()
    .sort_values(
        [
            "relative_path",
            "spec_id",
        ],
        kind="mergesort",
    )
)

if len(swagger_rows) != 2163:
    raise RuntimeError(
        "Unexpected Swagger row count: "
        f"expected 2163, observed {len(swagger_rows)}."
    )

missing_spec_id_mask = (
    swagger_rows["spec_id"].isna()
    | swagger_rows["spec_id"]
    .astype(str)
    .str.strip()
    .isin(["", "nan", "None"])
)

if missing_spec_id_mask.any():
    missing_paths = (
        swagger_rows.loc[
            missing_spec_id_mask,
            "relative_path",
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Swagger rows still have missing spec IDs:\n"
        + "\n".join(missing_paths)
    )

if not swagger_rows["spec_id"].is_unique:
    duplicated = (
        swagger_rows.loc[
            swagger_rows["spec_id"].duplicated(
                keep=False
            ),
            "spec_id",
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Duplicate Swagger spec IDs detected: "
        + ", ".join(duplicated[:20])
    )

nonpassing_parse_rows = swagger_rows.loc[
    ~swagger_rows["parse_status"]
    .fillna("")
    .eq("PASS")
]

if not nonpassing_parse_rows.empty:
    raise RuntimeError(
        "Swagger manifest contains parse-failed rows."
    )

manifest_records = []
resolved_corpus_root = CORPUS_ROOT.resolve()

for row in swagger_rows.itertuples(
    index=False
):
    relative_path = str(row.relative_path)

    source_path = (
        CORPUS_ROOT / relative_path
    ).resolve()

    try:
        source_path.relative_to(
            resolved_corpus_root
        )
    except ValueError as exception:
        raise RuntimeError(
            "Source path escapes the corpus root: "
            f"{relative_path}"
        ) from exception

    if not source_path.is_file():
        raise RuntimeError(
            f"Missing source file: {source_path}"
        )

    manifest_records.append(
        {
            "raw_id": str(row.raw_id),
            "spec_id": str(row.spec_id),
            "relative_path": relative_path,
            "provider_directory": str(
                row.provider_directory
            ),
            "source_sha256": str(
                row.source_sha256
            ),
            "source_bytes": int(
                row.source_bytes
            ),
            "source_operation_count": int(
                float(row.operation_count)
            ),
        }
    )

conversion_manifest = {
    "manifest_version": 1,
    "source_dialect": "swagger_2_0",
    "target_openapi_version": "3.0.3",
    "converter": {
        "package": "swagger2openapi",
        "version": "7.0.8",
    },
    "conversion_options": {
        "patch": False,
        "warn_only": False,
        "resolve": False,
        "target_version": "3.0.3",
        "body_name_extension": (
            "x-original-body-parameter-name"
        ),
    },
    "provenance": {
        "specifications_sha256": sha256_file(
            SPECIFICATIONS_PATH
        ),
        "qualification_sha256": sha256_file(
            QUALIFICATION_PATH
        ),
        "amendment_sha256": sha256_file(
            AMENDMENT_PATH
        ),
        "repair_report_sha256": sha256_file(
            REPAIR_REPORT_PATH
        ),
    },
    "record_count": len(
        manifest_records
    ),
    "records": manifest_records,
}

write_json_atomic(
    CONVERSION_MANIFEST_PATH,
    conversion_manifest,
)

batch_harness = r"""
"use strict";

const crypto = require("crypto");
const fs = require("fs");
const path = require("path");
const converter = require("swagger2openapi");

const fsp = fs.promises;

const [
    manifestPath,
    corpusRootArgument,
    canonicalRootArgument,
    itemMetadataRootArgument,
    resultsPath,
    summaryPath,
    concurrencyArgument
] = process.argv.slice(2);

if (
    !manifestPath
    || !corpusRootArgument
    || !canonicalRootArgument
    || !itemMetadataRootArgument
    || !resultsPath
    || !summaryPath
) {
    console.error(
        "Missing batch-conversion arguments."
    );
    process.exit(2);
}

const corpusRoot = path.resolve(
    corpusRootArgument
);

const canonicalRoot = path.resolve(
    canonicalRootArgument
);

const itemMetadataRoot = path.resolve(
    itemMetadataRootArgument
);

const concurrency = Math.max(
    1,
    Number(concurrencyArgument || 1)
);

const HARNESS_VERSION = 1;
const TARGET_VERSION = "3.0.3";

const CONVERTER_VERSION =
    require(
        "swagger2openapi/package.json"
    ).version;

function sha256Buffer(value) {
    return crypto
        .createHash("sha256")
        .update(value)
        .digest("hex");
}

async function sha256File(filePath) {
    return sha256Buffer(
        await fsp.readFile(filePath)
    );
}

async function fileExists(filePath) {
    try {
        await fsp.access(
            filePath,
            fs.constants.F_OK
        );

        return true;
    } catch {
        return false;
    }
}

async function writeAtomic(
    filePath,
    content
) {
    await fsp.mkdir(
        path.dirname(filePath),
        {
            recursive: true
        }
    );

    const temporaryPath =
        `${filePath}.${process.pid}.tmp`;

    await fsp.writeFile(
        temporaryPath,
        content,
        "utf8"
    );

    await fsp.rename(
        temporaryPath,
        filePath
    );
}

function normalizeWarnings(warnings) {
    if (!Array.isArray(warnings)) {
        return [];
    }

    return warnings.map(
        (warning) => {
            if (
                typeof warning === "string"
            ) {
                return warning;
            }

            if (
                warning
                && typeof warning.message
                    === "string"
            ) {
                return warning.message;
            }

            try {
                return JSON.stringify(
                    warning
                );
            } catch {
                return String(warning);
            }
        }
    );
}

function countOperations(document) {
    const methods = new Set([
        "get",
        "put",
        "post",
        "delete",
        "options",
        "head",
        "patch",
        "trace"
    ]);

    let count = 0;

    for (
        const pathItem
        of Object.values(
            document.paths || {}
        )
    ) {
        if (
            !pathItem
            || typeof pathItem !== "object"
            || Array.isArray(pathItem)
        ) {
            continue;
        }

        for (
            const method
            of Object.keys(pathItem)
        ) {
            if (
                methods.has(
                    method.toLowerCase()
                )
            ) {
                count += 1;
            }
        }
    }

    return count;
}

function convertFile(sourcePath) {
    return new Promise(
        (resolve, reject) => {
            const options = {
                patch: false,
                warnOnly: false,
                resolve: false,
                targetVersion:
                    TARGET_VERSION,
                rbname:
                    "x-original-body-parameter-name"
            };

            converter.convertFile(
                sourcePath,
                options,
                (error, result) => {
                    if (error) {
                        reject(error);
                        return;
                    }

                    resolve({
                        openapi:
                            result.openapi,
                        warnings:
                            normalizeWarnings(
                                result.warnings
                            ),
                        patches:
                            Number(
                                result.patches
                                || 0
                            )
                    });
                }
            );
        }
    );
}

function safeDiagnostic(error) {
    const text = (
        error
        && (
            error.stack
            || error.message
        )
    )
        ? String(
            error.stack
            || error.message
        )
        : String(error);

    return text.slice(0, 12000);
}

async function loadReusableResult(
    record,
    outputPath,
    itemPath
) {
    if (
        !(await fileExists(outputPath))
        || !(await fileExists(itemPath))
    ) {
        return null;
    }

    try {
        const item = JSON.parse(
            await fsp.readFile(
                itemPath,
                "utf8"
            )
        );

        if (
            item.status !== "SUCCESS"
            || item.harness_version
                !== HARNESS_VERSION
            || item.converter_version
                !== CONVERTER_VERSION
            || item.target_openapi_version
                !== TARGET_VERSION
            || item.source_sha256
                !== record.source_sha256
        ) {
            return null;
        }

        const observedOutputHash =
            await sha256File(
                outputPath
            );

        if (
            observedOutputHash
            !== item.converted_sha256
        ) {
            return null;
        }

        return item;
    } catch {
        return null;
    }
}

async function processRecord(record) {
    const sourcePath = path.resolve(
        corpusRoot,
        record.relative_path
    );

    const relativeCheck = path.relative(
        corpusRoot,
        sourcePath
    );

    if (
        relativeCheck.startsWith("..")
        || path.isAbsolute(relativeCheck)
    ) {
        return {
            result: {
                harness_version:
                    HARNESS_VERSION,
                spec_id: record.spec_id,
                raw_id: record.raw_id,
                relative_path:
                    record.relative_path,
                source_sha256:
                    record.source_sha256,
                status:
                    "SOURCE_PATH_REJECTED",
                diagnostic:
                    "Source path escapes corpus root."
            },
            resumed: false
        };
    }

    if (
        !/^spec_[a-zA-Z0-9]+$/.test(
            record.spec_id
        )
    ) {
        return {
            result: {
                harness_version:
                    HARNESS_VERSION,
                spec_id: record.spec_id,
                raw_id: record.raw_id,
                relative_path:
                    record.relative_path,
                source_sha256:
                    record.source_sha256,
                status:
                    "SPEC_ID_REJECTED",
                diagnostic:
                    "Unsafe spec_id."
            },
            resumed: false
        };
    }

    const outputPath = path.join(
        canonicalRoot,
        `${record.spec_id}.json`
    );

    const itemPath = path.join(
        itemMetadataRoot,
        `${record.spec_id}.json`
    );

    const reusable =
        await loadReusableResult(
            record,
            outputPath,
            itemPath
        );

    if (reusable) {
        return {
            result: reusable,
            resumed: true
        };
    }

    let result;

    try {
        const sourceBytes =
            await fsp.readFile(
                sourcePath
            );

        const observedSourceHash =
            sha256Buffer(sourceBytes);

        if (
            observedSourceHash
            !== record.source_sha256
        ) {
            result = {
                harness_version:
                    HARNESS_VERSION,
                converter_version:
                    CONVERTER_VERSION,
                target_openapi_version:
                    TARGET_VERSION,
                spec_id: record.spec_id,
                raw_id: record.raw_id,
                relative_path:
                    record.relative_path,
                provider_directory:
                    record.provider_directory,
                source_sha256:
                    record.source_sha256,
                observed_source_sha256:
                    observedSourceHash,
                source_bytes:
                    sourceBytes.length,
                source_operation_count:
                    record.source_operation_count,
                status:
                    "SOURCE_HASH_MISMATCH",
                diagnostic:
                    "Observed source hash "
                    + "differs from the "
                    + "frozen inventory."
            };
        } else {
            const converted =
                await convertFile(
                    sourcePath
                );

            if (
                !converted.openapi
                || typeof converted.openapi
                    !== "object"
            ) {
                throw new Error(
                    "Converter returned no "
                    + "OpenAPI document."
                );
            }

            const serialized =
                JSON.stringify(
                    converted.openapi,
                    null,
                    2
                ) + "\n";

            const convertedBytes =
                Buffer.from(
                    serialized,
                    "utf8"
                );

            const convertedHash =
                sha256Buffer(
                    convertedBytes
                );

            await writeAtomic(
                outputPath,
                serialized
            );

            const outputOperationCount =
                countOperations(
                    converted.openapi
                );

            result = {
                harness_version:
                    HARNESS_VERSION,
                converter_version:
                    CONVERTER_VERSION,
                target_openapi_version:
                    TARGET_VERSION,
                spec_id: record.spec_id,
                raw_id: record.raw_id,
                relative_path:
                    record.relative_path,
                provider_directory:
                    record.provider_directory,
                source_sha256:
                    record.source_sha256,
                observed_source_sha256:
                    observedSourceHash,
                source_bytes:
                    sourceBytes.length,
                source_operation_count:
                    record.source_operation_count,
                status: "SUCCESS",
                output_relative_path:
                    path.posix.join(
                        "data",
                        "canonical",
                        `${record.spec_id}.json`
                    ),
                converted_sha256:
                    convertedHash,
                converted_bytes:
                    convertedBytes.length,
                output_openapi_version:
                    converted.openapi.openapi
                    || null,
                output_operation_count:
                    outputOperationCount,
                operation_count_matches:
                    outputOperationCount
                    === record
                        .source_operation_count,
                warning_count:
                    converted.warnings.length,
                warnings:
                    converted.warnings,
                patches:
                    converted.patches
            };
        }
    } catch (error) {
        result = {
            harness_version:
                HARNESS_VERSION,
            converter_version:
                CONVERTER_VERSION,
            target_openapi_version:
                TARGET_VERSION,
            spec_id: record.spec_id,
            raw_id: record.raw_id,
            relative_path:
                record.relative_path,
            provider_directory:
                record.provider_directory,
            source_sha256:
                record.source_sha256,
            source_bytes:
                record.source_bytes,
            source_operation_count:
                record.source_operation_count,
            status:
                "CONVERSION_FAILED",
            diagnostic:
                safeDiagnostic(error)
        };
    }

    await writeAtomic(
        itemPath,
        JSON.stringify(
            result,
            null,
            2
        ) + "\n"
    );

    return {
        result,
        resumed: false
    };
}

async function main() {
    const startedAt = Date.now();

    await fsp.mkdir(
        canonicalRoot,
        {
            recursive: true
        }
    );

    await fsp.mkdir(
        itemMetadataRoot,
        {
            recursive: true
        }
    );

    const manifest = JSON.parse(
        await fsp.readFile(
            manifestPath,
            "utf8"
        )
    );

    const records =
        manifest.records || [];

    const results = new Array(
        records.length
    );

    let nextIndex = 0;
    let completedCount = 0;
    let resumedCount = 0;

    async function worker() {
        while (true) {
            const currentIndex =
                nextIndex++;

            if (
                currentIndex
                >= records.length
            ) {
                return;
            }

            const outcome =
                await processRecord(
                    records[currentIndex]
                );

            results[currentIndex] =
                outcome.result;

            if (outcome.resumed) {
                resumedCount += 1;
            }

            completedCount += 1;

            if (
                completedCount % 25 === 0
                || completedCount
                    === records.length
            ) {
                const elapsedSeconds =
                    (
                        Date.now()
                        - startedAt
                    ) / 1000;

                const rate = (
                    elapsedSeconds > 0
                    ? completedCount
                        / elapsedSeconds
                    : 0
                );

                const remaining =
                    records.length
                    - completedCount;

                const etaSeconds = (
                    rate > 0
                    ? remaining / rate
                    : null
                );

                console.log(
                    JSON.stringify({
                        progress:
                            completedCount,
                        total:
                            records.length,
                        resumed:
                            resumedCount,
                        elapsed_seconds:
                            Number(
                                elapsedSeconds
                                .toFixed(1)
                            ),
                        rate_per_second:
                            Number(
                                rate.toFixed(3)
                            ),
                        eta_seconds:
                            etaSeconds === null
                            ? null
                            : Number(
                                etaSeconds
                                .toFixed(1)
                            )
                    })
                );
            }
        }
    }

    await Promise.all(
        Array.from(
            {
                length: concurrency
            },
            () => worker()
        )
    );

    const sortedResults = [
        ...results
    ].sort(
        (left, right) => {
            const pathComparison =
                String(
                    left.relative_path
                ).localeCompare(
                    String(
                        right.relative_path
                    )
                );

            if (pathComparison !== 0) {
                return pathComparison;
            }

            return String(
                left.spec_id
            ).localeCompare(
                String(
                    right.spec_id
                )
            );
        }
    );

    const jsonl =
        sortedResults
        .map(
            (record) =>
                JSON.stringify(record)
        )
        .join("\n")
        + "\n";

    await writeAtomic(
        resultsPath,
        jsonl
    );

    const countStatus = (
        status
    ) => sortedResults.filter(
        (record) =>
            record.status === status
    ).length;

    const successful =
        sortedResults.filter(
            (record) =>
                record.status
                === "SUCCESS"
        );

    const failures =
        sortedResults.filter(
            (record) =>
                record.status
                !== "SUCCESS"
        );

    const sourceIntegrityFailures =
        countStatus(
            "SOURCE_HASH_MISMATCH"
        )
        + countStatus(
            "SOURCE_PATH_REJECTED"
        )
        + countStatus(
            "SPEC_ID_REJECTED"
        );

    const summary = {
        status: (
            sortedResults.length
                === records.length
            && sourceIntegrityFailures === 0
            ? "PASS"
            : "BLOCKED"
        ),
        converter:
            "swagger2openapi",
        converter_version:
            CONVERTER_VERSION,
        target_openapi_version:
            TARGET_VERSION,
        concurrency,
        source_record_count:
            records.length,
        result_record_count:
            sortedResults.length,
        resumed_success_count:
            resumedCount,
        conversion_success_count:
            successful.length,
        conversion_failure_count:
            failures.length,
        source_integrity_failure_count:
            sourceIntegrityFailures,
        success_with_warnings_count:
            successful.filter(
                (record) =>
                    record.warning_count > 0
            ).length,
        success_with_patches_count:
            successful.filter(
                (record) =>
                    record.patches > 0
            ).length,
        output_version_mismatch_count:
            successful.filter(
                (record) =>
                    record
                    .output_openapi_version
                    !== TARGET_VERSION
            ).length,
        operation_count_mismatch_count:
            successful.filter(
                (record) =>
                    !record
                    .operation_count_matches
            ).length,
        converted_output_bytes:
            successful.reduce(
                (
                    total,
                    record
                ) =>
                    total
                    + Number(
                        record
                        .converted_bytes
                        || 0
                    ),
                0
            ),
        results_sha256:
            sha256Buffer(
                Buffer.from(
                    jsonl,
                    "utf8"
                )
            ),
        duration_seconds:
            Number(
                (
                    (
                        Date.now()
                        - startedAt
                    ) / 1000
                ).toFixed(3)
            ),
        failure_examples:
            failures.slice(0, 20),
        results_path:
            resultsPath
    };

    await writeAtomic(
        summaryPath,
        JSON.stringify(
            summary,
            null,
            2
        ) + "\n"
    );

    console.log(
        JSON.stringify({
            completed: true,
            status: summary.status,
            successes:
                summary
                .conversion_success_count,
            failures:
                summary
                .conversion_failure_count
        })
    );

    if (summary.status !== "PASS") {
        process.exitCode = 1;
    }
}

main().catch((error) => {
    console.error(
        error.stack || String(error)
    );

    process.exitCode = 1;
});
""".strip() + "\n"

for harness_path in [
    RUNTIME_HARNESS_PATH,
    FROZEN_HARNESS_PATH,
]:
    harness_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    harness_path.write_text(
        batch_harness,
        encoding="utf-8",
    )

manifest_sha256 = sha256_file(
    CONVERSION_MANIFEST_PATH
)

harness_sha256 = sha256_file(
    FROZEN_HARNESS_PATH
)

started = time.perf_counter()

process = subprocess.Popen(
    [
        str(NODE_PATH),
        str(RUNTIME_HARNESS_PATH),
        str(CONVERSION_MANIFEST_PATH),
        str(CORPUS_ROOT),
        str(CANONICAL_ROOT),
        str(ITEM_METADATA_ROOT),
        str(RESULTS_PATH),
        str(SUMMARY_PATH),
        str(CONCURRENCY),
    ],
    cwd=RUNTIME_TOOL_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

captured_lines = []

if process.stdout is None:
    raise RuntimeError(
        "Unable to capture conversion progress."
    )

for line in process.stdout:
    line = line.rstrip()

    if line:
        captured_lines.append(line)
        print(line, flush=True)

return_code = process.wait()

wall_duration_seconds = round(
    time.perf_counter() - started,
    3,
)

if return_code != 0:
    raise RuntimeError(
        "Batch Swagger conversion failed.\n\n"
        + "\n".join(
            captured_lines[-50:]
        )
    )

if not SUMMARY_PATH.is_file():
    raise RuntimeError(
        "Batch conversion produced no summary report."
    )

if not RESULTS_PATH.is_file():
    raise RuntimeError(
        "Batch conversion produced no results ledger."
    )

summary = json.loads(
    SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

summary.update(
    {
        "conversion_manifest_path": str(
            CONVERSION_MANIFEST_PATH
        ),
        "conversion_manifest_sha256": (
            manifest_sha256
        ),
        "frozen_harness_path": str(
            FROZEN_HARNESS_PATH
        ),
        "frozen_harness_sha256": (
            harness_sha256
        ),
        "results_path": str(
            RESULTS_PATH
        ),
        "results_sha256": sha256_file(
            RESULTS_PATH
        ),
        "wall_duration_seconds": (
            wall_duration_seconds
        ),
    }
)

write_json_atomic(
    SUMMARY_PATH,
    summary,
)

output = {
    **summary,
    "summary_path": str(
        SUMMARY_PATH
    ),
    "summary_sha256": sha256_file(
        SUMMARY_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

if summary.get("status") != "PASS":
    raise RuntimeError(
        "Corpus-wide conversion execution "
        "did not reach PASS."
    )

print("\nNOTEBOOK 01 — CELL 16: PASS")

{"progress":25,"total":2163,"resumed":0,"elapsed_seconds":0.6,"rate_per_second":42.808,"eta_seconds":49.9}
{"progress":50,"total":2163,"resumed":0,"elapsed_seconds":3.3,"rate_per_second":14.93,"eta_seconds":141.5}
{"progress":75,"total":2163,"resumed":0,"elapsed_seconds":4.6,"rate_per_second":16.383,"eta_seconds":127.5}
{"progress":100,"total":2163,"resumed":0,"elapsed_seconds":5,"rate_per_second":19.818,"eta_seconds":104.1}
{"progress":125,"total":2163,"resumed":0,"elapsed_seconds":5.5,"rate_per_second":22.752,"eta_seconds":89.6}
{"progress":150,"total":2163,"resumed":0,"elapsed_seconds":5.9,"rate_per_second":25.493,"eta_seconds":79}
{"progress":175,"total":2163,"resumed":0,"elapsed_seconds":6.3,"rate_per_second":27.681,"eta_seconds":71.8}
{"progress":200,"total":2163,"resumed":0,"elapsed_seconds":6.7,"rate_per_second":29.828,"eta_seconds":65.8}
{"progress":225,"total":2163,"resumed":0,"elapsed_seconds":7.2,"rate_per_second":31.311,"eta_seconds":61.9}
{"progress":250,"total":2163,"res

In [19]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_conversion_results.jsonl"
)

EXECUTION_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_conversion_execution.json"
)

DISPOSITIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_conversion_dispositions.csv"
)

RECONCILIATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_conversion_reconciliation.json"
)


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


required_paths = [
    RESULTS_PATH,
    EXECUTION_SUMMARY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Missing conversion outputs:\n"
        + "\n".join(missing_paths)
    )

execution_summary = json.loads(
    EXECUTION_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

if execution_summary.get("status") != "PASS":
    raise RuntimeError(
        "Conversion execution status is not PASS."
    )

observed_results_sha256 = sha256_file(
    RESULTS_PATH
)

expected_results_sha256 = execution_summary[
    "results_sha256"
]

if (
    observed_results_sha256
    != expected_results_sha256
):
    raise RuntimeError(
        "Conversion-results ledger hash mismatch."
    )

conversion_records = []

with RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line_number, line in enumerate(
        file,
        start=1,
    ):
        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as exception:
            raise RuntimeError(
                "Invalid JSON in conversion ledger "
                f"at line {line_number}."
            ) from exception

        conversion_records.append(record)

if len(conversion_records) != 2163:
    raise RuntimeError(
        "Unexpected conversion-record count: "
        f"expected 2163, "
        f"observed {len(conversion_records)}."
    )

spec_ids = [
    str(record.get("spec_id"))
    for record in conversion_records
]

if len(spec_ids) != len(set(spec_ids)):
    duplicated = [
        spec_id
        for spec_id, count in Counter(
            spec_ids
        ).items()
        if count > 1
    ]

    raise RuntimeError(
        "Duplicate spec IDs in conversion ledger: "
        + ", ".join(duplicated[:20])
    )


def classify_failure(diagnostic):
    text = str(diagnostic or "")

    if "Could not resolve reference" in text:
        return "UNRESOLVED_SOURCE_REFERENCE"

    if (
        "Nested collectionFormats "
        "are not supported"
        in text
    ):
        return "NESTED_COLLECTION_FORMAT"

    if (
        "collectionFormat is only applicable "
        "to param.type array"
        in text
    ):
        return "INVALID_COLLECTION_FORMAT"

    return "OTHER_CONVERSION_FAILURE"


disposition_rows = []
integrity_problems = []
failure_records = []
patched_records = []
warning_records = []

for record in conversion_records:
    status = str(record.get("status"))
    spec_id = str(record.get("spec_id"))
    relative_path = str(
        record.get("relative_path")
    )

    reasons = []
    failure_category = None
    output_integrity_pass = False

    warning_count = int(
        record.get("warning_count") or 0
    )

    patch_count = int(
        record.get("patches") or 0
    )

    operation_count_matches = bool(
        record.get(
            "operation_count_matches",
            False,
        )
    )

    output_version = record.get(
        "output_openapi_version"
    )

    converted_sha256 = record.get(
        "converted_sha256"
    )

    converted_bytes = record.get(
        "converted_bytes"
    )

    output_relative_path = record.get(
        "output_relative_path"
    )

    if status != "SUCCESS":
        failure_category = classify_failure(
            record.get("diagnostic")
        )

        reasons.append(
            "CONVERSION_FAILED"
        )

        failure_records.append(
            {
                "spec_id": spec_id,
                "relative_path": relative_path,
                "failure_category": (
                    failure_category
                ),
                "diagnostic": record.get(
                    "diagnostic"
                ),
            }
        )

    else:
        if not output_relative_path:
            integrity_problems.append(
                {
                    "spec_id": spec_id,
                    "problem": (
                        "missing output_relative_path"
                    ),
                }
            )
        else:
            output_path = (
                PROJECT_ROOT
                / str(output_relative_path)
            )

            if not output_path.is_file():
                integrity_problems.append(
                    {
                        "spec_id": spec_id,
                        "problem": (
                            "converted output is missing"
                        ),
                        "path": str(output_path),
                    }
                )
            else:
                observed_hash = sha256_file(
                    output_path
                )

                observed_bytes = (
                    output_path.stat().st_size
                )

                hash_matches = (
                    observed_hash
                    == converted_sha256
                )

                bytes_match = (
                    observed_bytes
                    == int(converted_bytes)
                )

                output_integrity_pass = (
                    hash_matches
                    and bytes_match
                )

                if not output_integrity_pass:
                    integrity_problems.append(
                        {
                            "spec_id": spec_id,
                            "problem": (
                                "converted output "
                                "integrity mismatch"
                            ),
                            "expected_sha256": (
                                converted_sha256
                            ),
                            "observed_sha256": (
                                observed_hash
                            ),
                            "expected_bytes": int(
                                converted_bytes
                            ),
                            "observed_bytes": (
                                observed_bytes
                            ),
                        }
                    )

        if warning_count > 0:
            reasons.append(
                "CONVERTER_WARNINGS"
            )

            warning_records.append(
                {
                    "spec_id": spec_id,
                    "relative_path": (
                        relative_path
                    ),
                    "warning_count": (
                        warning_count
                    ),
                    "warnings": record.get(
                        "warnings",
                        [],
                    ),
                }
            )

        if patch_count > 0:
            reasons.append(
                "CONVERTER_PATCHES"
            )

            patched_records.append(
                {
                    "spec_id": spec_id,
                    "relative_path": (
                        relative_path
                    ),
                    "patch_count": (
                        patch_count
                    ),
                }
            )

        if output_version != "3.0.3":
            reasons.append(
                "OUTPUT_VERSION_MISMATCH"
            )

        if not operation_count_matches:
            reasons.append(
                "OPERATION_COUNT_MISMATCH"
            )

        if not output_integrity_pass:
            reasons.append(
                "OUTPUT_INTEGRITY_FAILURE"
            )

    if status != "SUCCESS":
        initial_disposition = (
            "EXCLUDE_CONVERSION_FAILED"
        )
    elif reasons:
        initial_disposition = (
            "QUARANTINE_CONVERSION_UNCERTAIN"
        )
    else:
        initial_disposition = (
            "PENDING_STRICT_AUDIT"
        )

    disposition_rows.append(
        {
            "spec_id": spec_id,
            "raw_id": record.get("raw_id"),
            "relative_path": relative_path,
            "provider_directory": record.get(
                "provider_directory"
            ),
            "execution_status": status,
            "failure_category": (
                failure_category
            ),
            "warning_count": warning_count,
            "patch_count": patch_count,
            "output_openapi_version": (
                output_version
            ),
            "source_operation_count": (
                record.get(
                    "source_operation_count"
                )
            ),
            "output_operation_count": (
                record.get(
                    "output_operation_count"
                )
            ),
            "operation_count_matches": (
                operation_count_matches
            ),
            "output_integrity_pass": (
                output_integrity_pass
            ),
            "converted_sha256": (
                converted_sha256
            ),
            "converted_bytes": (
                converted_bytes
            ),
            "output_relative_path": (
                output_relative_path
            ),
            "initial_disposition": (
                initial_disposition
            ),
            "disposition_reasons": (
                ";".join(reasons)
            ),
            "eligible_now": False,
        }
    )

if integrity_problems:
    raise RuntimeError(
        "Converted-output integrity problems:\n"
        + json.dumps(
            integrity_problems[:20],
            indent=2,
            ensure_ascii=False,
        )
    )

dispositions = pd.DataFrame(
    disposition_rows
).sort_values(
    [
        "relative_path",
        "spec_id",
    ],
    kind="mergesort",
)

DISPOSITIONS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

temporary_dispositions_path = (
    DISPOSITIONS_PATH.with_suffix(
        ".csv.tmp"
    )
)

dispositions.to_csv(
    temporary_dispositions_path,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
)

temporary_dispositions_path.replace(
    DISPOSITIONS_PATH
)

disposition_counts = {
    str(key): int(value)
    for key, value in (
        dispositions[
            "initial_disposition"
        ]
        .value_counts()
        .sort_index()
        .items()
    )
}

patch_count_distribution = {
    str(key): int(value)
    for key, value in sorted(
        Counter(
            int(record["patch_count"])
            for record in patched_records
        ).items()
    )
}

failure_category_counts = {
    str(key): int(value)
    for key, value in sorted(
        Counter(
            record["failure_category"]
            for record in failure_records
        ).items()
    )
}

strict_candidate_count = int(
    (
        dispositions[
            "initial_disposition"
        ]
        == "PENDING_STRICT_AUDIT"
    ).sum()
)

report = {
    "status": "PASS",
    "source_record_count": len(
        conversion_records
    ),
    "conversion_success_count": int(
        (
            dispositions[
                "execution_status"
            ]
            == "SUCCESS"
        ).sum()
    ),
    "conversion_failure_count": len(
        failure_records
    ),
    "output_integrity_problem_count": len(
        integrity_problems
    ),
    "warning_quarantine_count": len(
        warning_records
    ),
    "patched_output_quarantine_count": len(
        patched_records
    ),
    "strict_audit_candidate_count": (
        strict_candidate_count
    ),
    "disposition_counts": (
        disposition_counts
    ),
    "failure_category_counts": (
        failure_category_counts
    ),
    "patch_count_distribution": (
        patch_count_distribution
    ),
    "failure_records": failure_records,
    "patched_records": patched_records,
    "warning_records": warning_records,
    "policy": {
        "conversion_failure": "EXCLUDE",
        "converter_warning": "QUARANTINE",
        "converter_patch": "QUARANTINE",
        "version_mismatch": "QUARANTINE",
        "operation_count_mismatch": (
            "QUARANTINE"
        ),
        "clean_conversion": (
            "PENDING_STRICT_AUDIT"
        ),
    },
    "dispositions_path": str(
        DISPOSITIONS_PATH
    ),
    "dispositions_sha256": sha256_file(
        DISPOSITIONS_PATH
    ),
    "conversion_results_sha256": (
        observed_results_sha256
    ),
}

write_json_atomic(
    RECONCILIATION_REPORT_PATH,
    report,
)

printed_output = {
    "status": report["status"],
    "source_record_count": (
        report["source_record_count"]
    ),
    "conversion_success_count": (
        report["conversion_success_count"]
    ),
    "conversion_failure_count": (
        report["conversion_failure_count"]
    ),
    "output_integrity_problem_count": (
        report[
            "output_integrity_problem_count"
        ]
    ),
    "warning_quarantine_count": (
        report["warning_quarantine_count"]
    ),
    "patched_output_quarantine_count": (
        report[
            "patched_output_quarantine_count"
        ]
    ),
    "strict_audit_candidate_count": (
        report[
            "strict_audit_candidate_count"
        ]
    ),
    "disposition_counts": (
        disposition_counts
    ),
    "failure_category_counts": (
        failure_category_counts
    ),
    "patch_count_distribution": (
        patch_count_distribution
    ),
    "failure_records": failure_records,
    "patched_record_examples": (
        patched_records[:20]
    ),
    "dispositions_path": str(
        DISPOSITIONS_PATH
    ),
    "dispositions_sha256": sha256_file(
        DISPOSITIONS_PATH
    ),
    "report_path": str(
        RECONCILIATION_REPORT_PATH
    ),
    "report_sha256": sha256_file(
        RECONCILIATION_REPORT_PATH
    ),
}

print(
    json.dumps(
        printed_output,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 17: PASS")

{
  "status": "PASS",
  "source_record_count": 2163,
  "conversion_success_count": 2159,
  "conversion_failure_count": 4,
  "output_integrity_problem_count": 0,
  "warning_quarantine_count": 0,
  "patched_output_quarantine_count": 48,
  "strict_audit_candidate_count": 2111,
  "disposition_counts": {
    "EXCLUDE_CONVERSION_FAILED": 4,
    "PENDING_STRICT_AUDIT": 2111,
    "QUARANTINE_CONVERSION_UNCERTAIN": 48
  },
  "failure_category_counts": {
    "INVALID_COLLECTION_FORMAT": 2,
    "NESTED_COLLECTION_FORMAT": 1,
    "UNRESOLVED_SOURCE_REFERENCE": 1
  },
  "patch_count_distribution": {
    "1": 13,
    "2": 12,
    "3": 8,
    "4": 6,
    "7": 1,
    "8": 1,
    "10": 1,
    "11": 1,
    "12": 1,
    "19": 1,
    "20": 1,
    "21": 1,
    "38": 1
  },
  "failure_records": [
    {
      "spec_id": "spec_7d2c770f502832d9feb19328",
      "relative_path": "azure.com/azsadmin-RegionHealth/2016-05-01/swagger.yaml",
      "failure_category": "UNRESOLVED_SOURCE_REFERENCE",
      "diagnostic":

In [20]:
import hashlib
import importlib.metadata
import json
import time
from collections import Counter
from pathlib import Path

import pandas as pd
from openapi_spec_validator import validate

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

DISPOSITIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_conversion_dispositions.csv"
)

RECONCILIATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_conversion_reconciliation.json"
)

VALIDATION_ITEM_ROOT = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_validation_items"
)

VALIDATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_converted_validation.jsonl"
)

VALIDATION_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converted_validation_summary.json"
)

VALIDATOR_VERSION = (
    importlib.metadata.version(
        "openapi-spec-validator"
    )
)

VALIDATION_ITEM_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

VALIDATION_RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

VALIDATION_SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


def collect_references(document):
    references = []
    malformed_references = []

    stack = [document]

    while stack:
        current = stack.pop()

        if isinstance(current, dict):
            for key, value in current.items():
                if key == "$ref":
                    if isinstance(value, str):
                        references.append(value)
                    else:
                        malformed_references.append(
                            repr(value)
                        )
                else:
                    stack.append(value)

        elif isinstance(current, list):
            stack.extend(current)

    return (
        references,
        malformed_references,
    )


def resolve_local_reference(
    document,
    reference,
):
    if reference == "#":
        return True

    if not reference.startswith("#/"):
        return False

    current = document

    for token in reference[2:].split("/"):
        token = (
            token
            .replace("~1", "/")
            .replace("~0", "~")
        )

        if (
            isinstance(current, dict)
            and token in current
        ):
            current = current[token]
        else:
            return False

    return True


def compact_diagnostic(exception):
    text = (
        f"{type(exception).__name__}: "
        f"{exception}"
    )

    return text[:12000]


required_paths = [
    DISPOSITIONS_PATH,
    RECONCILIATION_REPORT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Missing validation prerequisites:\n"
        + "\n".join(missing_paths)
    )

reconciliation = json.loads(
    RECONCILIATION_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

if reconciliation.get("status") != "PASS":
    raise RuntimeError(
        "Conversion reconciliation is not PASS."
    )

if (
    sha256_file(DISPOSITIONS_PATH)
    != reconciliation[
        "dispositions_sha256"
    ]
):
    raise RuntimeError(
        "Conversion dispositions hash mismatch."
    )

dispositions = pd.read_csv(
    DISPOSITIONS_PATH,
    low_memory=False,
)

successful = (
    dispositions.loc[
        dispositions["execution_status"]
        .eq("SUCCESS")
    ]
    .copy()
    .sort_values(
        [
            "relative_path",
            "spec_id",
        ],
        kind="mergesort",
    )
)

if len(successful) != 2159:
    raise RuntimeError(
        "Unexpected successful-conversion count: "
        f"expected 2159, observed {len(successful)}."
    )

if not successful["spec_id"].is_unique:
    raise RuntimeError(
        "Successful conversion IDs are not unique."
    )

started = time.perf_counter()
results = []
resumed_count = 0

for position, row in enumerate(
    successful.itertuples(index=False),
    start=1,
):
    spec_id = str(row.spec_id)

    converted_path = (
        PROJECT_ROOT
        / str(row.output_relative_path)
    )

    checkpoint_path = (
        VALIDATION_ITEM_ROOT
        / f"{spec_id}.json"
    )

    expected_converted_sha256 = str(
        row.converted_sha256
    )

    reusable = None

    if checkpoint_path.is_file():
        try:
            candidate = json.loads(
                checkpoint_path.read_text(
                    encoding="utf-8"
                )
            )

            if (
                candidate.get("spec_id")
                == spec_id
                and candidate.get(
                    "converted_sha256"
                )
                == expected_converted_sha256
                and candidate.get(
                    "validator_version"
                )
                == VALIDATOR_VERSION
                and candidate.get(
                    "audit_version"
                )
                == 1
            ):
                reusable = candidate
        except Exception:
            reusable = None

    if reusable is not None:
        results.append(reusable)
        resumed_count += 1

    else:
        result = {
            "audit_version": 1,
            "spec_id": spec_id,
            "raw_id": str(row.raw_id),
            "relative_path": str(
                row.relative_path
            ),
            "provider_directory": str(
                row.provider_directory
            ),
            "initial_disposition": str(
                row.initial_disposition
            ),
            "patch_count": int(
                row.patch_count
            ),
            "warning_count": int(
                row.warning_count
            ),
            "converted_path": str(
                converted_path
            ),
            "converted_sha256": (
                expected_converted_sha256
            ),
            "validator": (
                "openapi-spec-validator"
            ),
            "validator_version": (
                VALIDATOR_VERSION
            ),
        }

        if not converted_path.is_file():
            result.update(
                {
                    "output_integrity_pass": False,
                    "json_load_status": "NOT_RUN",
                    "openapi_version": None,
                    "validation_status": (
                        "NOT_RUN"
                    ),
                    "validation_diagnostic": (
                        "Converted file is missing."
                    ),
                    "reference_status": (
                        "NOT_RUN"
                    ),
                    "strict_validation_pass": (
                        False
                    ),
                }
            )

        else:
            observed_hash = sha256_file(
                converted_path
            )

            output_integrity_pass = (
                observed_hash
                == expected_converted_sha256
            )

            result[
                "observed_converted_sha256"
            ] = observed_hash

            result[
                "output_integrity_pass"
            ] = output_integrity_pass

            if not output_integrity_pass:
                result.update(
                    {
                        "json_load_status": (
                            "NOT_RUN"
                        ),
                        "openapi_version": None,
                        "validation_status": (
                            "NOT_RUN"
                        ),
                        "validation_diagnostic": (
                            "Converted file hash "
                            "does not match the ledger."
                        ),
                        "reference_status": (
                            "NOT_RUN"
                        ),
                        "strict_validation_pass": (
                            False
                        ),
                    }
                )

            else:
                try:
                    document = json.loads(
                        converted_path.read_text(
                            encoding="utf-8"
                        )
                    )

                    result[
                        "json_load_status"
                    ] = "PASS"

                except Exception as exception:
                    document = None

                    result[
                        "json_load_status"
                    ] = "FAIL"

                    result[
                        "json_load_diagnostic"
                    ] = compact_diagnostic(
                        exception
                    )

                if document is not None:
                    openapi_version = (
                        document.get("openapi")
                        if isinstance(
                            document,
                            dict,
                        )
                        else None
                    )

                    result[
                        "openapi_version"
                    ] = openapi_version

                    (
                        references,
                        malformed_references,
                    ) = collect_references(
                        document
                    )

                    local_references = [
                        reference
                        for reference in references
                        if reference.startswith("#")
                    ]

                    external_references = [
                        reference
                        for reference in references
                        if not reference.startswith(
                            "#"
                        )
                    ]

                    unresolved_local_references = [
                        reference
                        for reference
                        in local_references
                        if not resolve_local_reference(
                            document,
                            reference,
                        )
                    ]

                    unique_external_references = (
                        sorted(
                            set(
                                external_references
                            )
                        )
                    )

                    unique_unresolved_local = (
                        sorted(
                            set(
                                unresolved_local_references
                            )
                        )
                    )

                    result.update(
                        {
                            "reference_count": len(
                                references
                            ),
                            "local_reference_count": (
                                len(
                                    local_references
                                )
                            ),
                            "external_reference_count": (
                                len(
                                    external_references
                                )
                            ),
                            "unique_external_reference_count": (
                                len(
                                    unique_external_references
                                )
                            ),
                            "external_reference_examples": (
                                unique_external_references[
                                    :20
                                ]
                            ),
                            "unresolved_local_reference_count": (
                                len(
                                    unresolved_local_references
                                )
                            ),
                            "unique_unresolved_local_reference_count": (
                                len(
                                    unique_unresolved_local
                                )
                            ),
                            "unresolved_local_reference_examples": (
                                unique_unresolved_local[
                                    :20
                                ]
                            ),
                            "malformed_reference_count": (
                                len(
                                    malformed_references
                                )
                            ),
                            "malformed_reference_examples": (
                                malformed_references[
                                    :20
                                ]
                            ),
                        }
                    )

                    if malformed_references:
                        reference_status = (
                            "INVALID_REFERENCES"
                        )
                    elif (
                        unresolved_local_references
                    ):
                        reference_status = (
                            "UNRESOLVED_LOCAL_REFERENCES"
                        )
                    elif external_references:
                        reference_status = (
                            "EXTERNAL_REFERENCES"
                        )
                    else:
                        reference_status = "PASS"

                    result[
                        "reference_status"
                    ] = reference_status

                    validation_started = (
                        time.perf_counter()
                    )

                    try:
                        validate(document)

                        validation_status = (
                            "PASS"
                        )

                        validation_diagnostic = (
                            None
                        )

                    except Exception as exception:
                        validation_status = (
                            "FAIL"
                        )

                        validation_diagnostic = (
                            compact_diagnostic(
                                exception
                            )
                        )

                    result[
                        "validation_duration_seconds"
                    ] = round(
                        time.perf_counter()
                        - validation_started,
                        6,
                    )

                    result[
                        "validation_status"
                    ] = validation_status

                    result[
                        "validation_diagnostic"
                    ] = validation_diagnostic

                    result[
                        "strict_validation_pass"
                    ] = bool(
                        output_integrity_pass
                        and openapi_version
                        == "3.0.3"
                        and validation_status
                        == "PASS"
                        and reference_status
                        == "PASS"
                    )

        write_json_atomic(
            checkpoint_path,
            result,
        )

        results.append(result)

    if (
        position % 25 == 0
        or position == len(successful)
    ):
        elapsed = (
            time.perf_counter()
            - started
        )

        rate = (
            position / elapsed
            if elapsed > 0
            else 0
        )

        remaining = (
            len(successful)
            - position
        )

        eta = (
            remaining / rate
            if rate > 0
            else None
        )

        print(
            json.dumps(
                {
                    "progress": position,
                    "total": len(successful),
                    "resumed": resumed_count,
                    "elapsed_seconds": round(
                        elapsed,
                        1,
                    ),
                    "rate_per_second": round(
                        rate,
                        3,
                    ),
                    "eta_seconds": (
                        round(eta, 1)
                        if eta is not None
                        else None
                    ),
                }
            ),
            flush=True,
        )

results = sorted(
    results,
    key=lambda item: (
        item["relative_path"],
        item["spec_id"],
    ),
)

results_jsonl = (
    "\n".join(
        json.dumps(
            result,
            sort_keys=True,
            ensure_ascii=False,
        )
        for result in results
    )
    + "\n"
)

temporary_results_path = (
    VALIDATION_RESULTS_PATH.with_suffix(
        ".jsonl.tmp"
    )
)

temporary_results_path.write_text(
    results_jsonl,
    encoding="utf-8",
)

temporary_results_path.replace(
    VALIDATION_RESULTS_PATH
)

validation_status_counts = Counter(
    result.get("validation_status")
    for result in results
)

reference_status_counts = Counter(
    result.get("reference_status")
    for result in results
)

initial_disposition_counts = Counter(
    result.get("initial_disposition")
    for result in results
)

strict_clean_results = [
    result
    for result in results
    if result.get(
        "initial_disposition"
    ) == "PENDING_STRICT_AUDIT"
]

patched_results = [
    result
    for result in results
    if result.get(
        "initial_disposition"
    )
    == "QUARANTINE_CONVERSION_UNCERTAIN"
]

strict_clean_pass_count = sum(
    bool(
        result.get(
            "strict_validation_pass"
        )
    )
    for result in strict_clean_results
)

patched_strict_pass_count = sum(
    bool(
        result.get(
            "strict_validation_pass"
        )
    )
    for result in patched_results
)

output_integrity_problem_count = sum(
    not bool(
        result.get(
            "output_integrity_pass"
        )
    )
    for result in results
)

validation_failure_examples = [
    {
        "spec_id": result["spec_id"],
        "relative_path": (
            result["relative_path"]
        ),
        "initial_disposition": (
            result["initial_disposition"]
        ),
        "reference_status": (
            result.get("reference_status")
        ),
        "diagnostic": (
            result.get(
                "validation_diagnostic"
            )
        ),
    }
    for result in results
    if result.get(
        "validation_status"
    ) != "PASS"
][:20]

reference_failure_examples = [
    {
        "spec_id": result["spec_id"],
        "relative_path": (
            result["relative_path"]
        ),
        "initial_disposition": (
            result["initial_disposition"]
        ),
        "reference_status": (
            result.get("reference_status")
        ),
        "external_reference_examples": (
            result.get(
                "external_reference_examples",
                [],
            )
        ),
        "unresolved_local_reference_examples": (
            result.get(
                "unresolved_local_reference_examples",
                [],
            )
        ),
    }
    for result in results
    if result.get(
        "reference_status"
    ) != "PASS"
][:20]

summary = {
    "status": (
        "PASS"
        if (
            len(results) == 2159
            and output_integrity_problem_count
            == 0
        )
        else "BLOCKED"
    ),
    "validator": (
        "openapi-spec-validator"
    ),
    "validator_version": (
        VALIDATOR_VERSION
    ),
    "audited_conversion_count": len(
        results
    ),
    "resumed_checkpoint_count": (
        resumed_count
    ),
    "output_integrity_problem_count": (
        output_integrity_problem_count
    ),
    "validation_status_counts": {
        str(key): int(value)
        for key, value in sorted(
            validation_status_counts.items(),
            key=lambda item: str(item[0]),
        )
    },
    "reference_status_counts": {
        str(key): int(value)
        for key, value in sorted(
            reference_status_counts.items(),
            key=lambda item: str(item[0]),
        )
    },
    "initial_disposition_counts": {
        str(key): int(value)
        for key, value in sorted(
            initial_disposition_counts.items(),
            key=lambda item: str(item[0]),
        )
    },
    "strict_clean_candidate_count": len(
        strict_clean_results
    ),
    "strict_clean_validation_pass_count": (
        strict_clean_pass_count
    ),
    "strict_clean_validation_failure_count": (
        len(strict_clean_results)
        - strict_clean_pass_count
    ),
    "patched_quarantine_count": len(
        patched_results
    ),
    "patched_strict_validation_pass_count": (
        patched_strict_pass_count
    ),
    "patched_strict_validation_failure_count": (
        len(patched_results)
        - patched_strict_pass_count
    ),
    "validation_failure_examples": (
        validation_failure_examples
    ),
    "reference_failure_examples": (
        reference_failure_examples
    ),
    "duration_seconds": round(
        time.perf_counter() - started,
        3,
    ),
    "results_path": str(
        VALIDATION_RESULTS_PATH
    ),
    "results_sha256": sha256_file(
        VALIDATION_RESULTS_PATH
    ),
}

write_json_atomic(
    VALIDATION_SUMMARY_PATH,
    summary,
)

output = {
    **summary,
    "summary_path": str(
        VALIDATION_SUMMARY_PATH
    ),
    "summary_sha256": sha256_file(
        VALIDATION_SUMMARY_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

if summary["status"] != "PASS":
    raise RuntimeError(
        "Converted-spec validation audit "
        "did not reach PASS."
    )

print("\nNOTEBOOK 01 — CELL 18: PASS")

/usr/local/lib/python3.13/dist-packages/openapi_schema_validator/_regex.py:25: FutureWarning: Possible nested set at position 1
  re.compile(pattern)


{"progress": 25, "total": 2159, "resumed": 0, "elapsed_seconds": 0.9, "rate_per_second": 29.103, "eta_seconds": 73.3}
{"progress": 50, "total": 2159, "resumed": 0, "elapsed_seconds": 18.1, "rate_per_second": 2.769, "eta_seconds": 761.6}
{"progress": 75, "total": 2159, "resumed": 0, "elapsed_seconds": 22.9, "rate_per_second": 3.273, "eta_seconds": 636.8}
{"progress": 100, "total": 2159, "resumed": 0, "elapsed_seconds": 24.5, "rate_per_second": 4.087, "eta_seconds": 503.8}
{"progress": 125, "total": 2159, "resumed": 0, "elapsed_seconds": 25.8, "rate_per_second": 4.841, "eta_seconds": 420.2}
{"progress": 150, "total": 2159, "resumed": 0, "elapsed_seconds": 27.0, "rate_per_second": 5.559, "eta_seconds": 361.4}
{"progress": 175, "total": 2159, "resumed": 0, "elapsed_seconds": 28.6, "rate_per_second": 6.119, "eta_seconds": 324.3}
{"progress": 200, "total": 2159, "resumed": 0, "elapsed_seconds": 30.1, "rate_per_second": 6.648, "eta_seconds": 294.7}
{"progress": 225, "total": 2159, "resumed": 

In [21]:
import hashlib
import json
import time
from collections import Counter
from pathlib import Path

import yaml
from yaml.constructor import ConstructorError
from yaml.nodes import MappingNode, ScalarNode
from yaml.resolver import BaseResolver

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

CONVERSION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "swagger_conversion_manifest.json"
)

VALIDATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_converted_validation.jsonl"
)

VALIDATION_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converted_validation_summary.json"
)

AUDIT_ITEM_ROOT = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_semantic_core_items"
)

AUDIT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_semantic_core_results.jsonl"
)

AUDIT_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_semantic_core_summary.json"
)

AUDIT_VERSION = 1

HTTP_METHODS = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}

AUDIT_ITEM_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIT_RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIT_SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


def compact(value, limit=1000):
    try:
        text = json.dumps(
            value,
            sort_keys=True,
            ensure_ascii=False,
        )
    except Exception:
        text = repr(value)

    if len(text) > limit:
        return text[:limit] + "...[truncated]"

    return text


loader_base = getattr(
    yaml,
    "CSafeLoader",
    yaml.SafeLoader,
)


class OpenAPIStringKeyLoader(loader_base):
    pass


def construct_string_key_mapping(
    loader,
    node,
    deep=False,
):
    if not isinstance(node, MappingNode):
        raise ConstructorError(
            None,
            None,
            "Expected a mapping node.",
            node.start_mark,
        )

    loader.flatten_mapping(node)
    mapping = {}

    for key_node, value_node in node.value:
        if not isinstance(
            key_node,
            ScalarNode,
        ):
            raise ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                (
                    "OpenAPI mapping keys must "
                    "be scalar strings"
                ),
                key_node.start_mark,
            )

        key = key_node.value

        if key in mapping:
            raise ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                (
                    "duplicate mapping key after "
                    f"string preservation: {key!r}"
                ),
                key_node.start_mark,
            )

        mapping[key] = (
            loader.construct_object(
                value_node,
                deep=deep,
            )
        )

    return mapping


def construct_timestamp_as_string(
    loader,
    node,
):
    return loader.construct_scalar(node)


OpenAPIStringKeyLoader.add_constructor(
    BaseResolver.DEFAULT_MAPPING_TAG,
    construct_string_key_mapping,
)

OpenAPIStringKeyLoader.add_constructor(
    "tag:yaml.org,2002:timestamp",
    construct_timestamp_as_string,
)


def operation_map(document):
    operations = {}

    for path_name, path_item in (
        document.get("paths", {}).items()
    ):
        if not isinstance(path_item, dict):
            continue

        for method_name, operation in (
            path_item.items()
        ):
            method = method_name.lower()

            if (
                method in HTTP_METHODS
                and isinstance(
                    operation,
                    dict,
                )
            ):
                operations[
                    (path_name, method)
                ] = operation

    return operations


def collect_prose(value):
    result = []

    stack = [value]

    while stack:
        current = stack.pop()

        if isinstance(current, dict):
            for key, child in current.items():
                if (
                    key
                    in {
                        "title",
                        "summary",
                        "description",
                    }
                    and isinstance(child, str)
                    and child.strip()
                ):
                    result.append(child)

                stack.append(child)

        elif isinstance(current, list):
            stack.extend(current)

    return Counter(result)


def collect_refs(value):
    result = []
    stack = [value]

    while stack:
        current = stack.pop()

        if isinstance(current, dict):
            for key, child in current.items():
                if (
                    key == "$ref"
                    and isinstance(child, str)
                ):
                    result.append(child)
                else:
                    stack.append(child)

        elif isinstance(current, list):
            stack.extend(current)

    return result


def remap_ref(reference):
    if reference.startswith(
        "#/definitions/"
    ):
        return (
            "#/components/schemas/"
            + reference[
                len("#/definitions/"):
            ]
        )

    if reference.startswith(
        "#/parameters/"
    ):
        return (
            "#/components/parameters/"
            + reference[
                len("#/parameters/"):
            ]
        )

    if reference.startswith(
        "#/responses/"
    ):
        return (
            "#/components/responses/"
            + reference[
                len("#/responses/"):
            ]
        )

    return reference


def normalize_schema(value):
    if isinstance(value, list):
        return [
            normalize_schema(child)
            for child in value
        ]

    if not isinstance(value, dict):
        return value

    normalized = {}

    nullable_value = None

    for key, child in value.items():
        if (
            key == "$ref"
            and isinstance(child, str)
        ):
            normalized[key] = remap_ref(
                child
            )

        elif key == "x-nullable":
            nullable_value = bool(child)

        elif (
            key == "discriminator"
            and isinstance(child, str)
        ):
            normalized[key] = {
                "propertyName": child,
            }

        else:
            normalized[key] = (
                normalize_schema(child)
            )

    if nullable_value is not None:
        normalized["nullable"] = (
            nullable_value
        )

    return normalized


def expected_servers(source):
    host = source.get("host")
    schemes = source.get("schemes")

    if (
        not isinstance(host, str)
        or not host
        or not isinstance(schemes, list)
        or not schemes
    ):
        return None

    base_path = source.get(
        "basePath",
        "",
    )

    if not isinstance(base_path, str):
        return None

    return {
        f"{scheme}://{host}{base_path}"
        for scheme in schemes
        if isinstance(scheme, str)
    }


def audit_document(
    source,
    converted,
):
    categories = {}
    diagnostics = {}

    categories["source_dialect"] = (
        source.get("swagger") == "2.0"
    )

    if not categories["source_dialect"]:
        diagnostics["source_dialect"] = {
            "observed": source.get(
                "swagger"
            )
        }

    categories["target_version"] = (
        converted.get("openapi")
        == "3.0.3"
    )

    if not categories["target_version"]:
        diagnostics["target_version"] = {
            "observed": converted.get(
                "openapi"
            )
        }

    categories["info_preserved"] = (
        source.get("info")
        == converted.get("info")
    )

    if not categories["info_preserved"]:
        diagnostics["info_preserved"] = {
            "source": compact(
                source.get("info")
            ),
            "converted": compact(
                converted.get("info")
            ),
        }

    source_operations = operation_map(
        source
    )

    converted_operations = operation_map(
        converted
    )

    source_operation_set = set(
        source_operations
    )

    converted_operation_set = set(
        converted_operations
    )

    categories[
        "operation_set_preserved"
    ] = (
        source_operation_set
        == converted_operation_set
    )

    if not categories[
        "operation_set_preserved"
    ]:
        diagnostics[
            "operation_set_preserved"
        ] = {
            "missing": sorted(
                source_operation_set
                - converted_operation_set
            )[:50],
            "added": sorted(
                converted_operation_set
                - source_operation_set
            )[:50],
        }

    metadata_failures = []
    response_failures = []

    metadata_fields = [
        "operationId",
        "summary",
        "description",
        "tags",
        "deprecated",
        "security",
        "externalDocs",
    ]

    for operation_key in sorted(
        source_operation_set
        & converted_operation_set
    ):
        source_operation = (
            source_operations[
                operation_key
            ]
        )

        converted_operation = (
            converted_operations[
                operation_key
            ]
        )

        for field in metadata_fields:
            if (
                field in source_operation
                and source_operation[field]
                != converted_operation.get(
                    field
                )
            ):
                metadata_failures.append(
                    {
                        "operation": (
                            f"{operation_key[1].upper()} "
                            f"{operation_key[0]}"
                        ),
                        "field": field,
                    }
                )

        source_extensions = {
            key: value
            for key, value
            in source_operation.items()
            if key.startswith("x-")
        }

        for key, value in (
            source_extensions.items()
        ):
            if (
                converted_operation.get(key)
                != value
            ):
                metadata_failures.append(
                    {
                        "operation": (
                            f"{operation_key[1].upper()} "
                            f"{operation_key[0]}"
                        ),
                        "field": key,
                    }
                )

        source_responses = (
            source_operation.get(
                "responses",
                {},
            )
        )

        converted_responses = (
            converted_operation.get(
                "responses",
                {},
            )
        )

        if (
            set(source_responses)
            != set(converted_responses)
        ):
            response_failures.append(
                {
                    "operation": (
                        f"{operation_key[1].upper()} "
                        f"{operation_key[0]}"
                    ),
                    "problem": (
                        "response_status_set"
                    ),
                }
            )

        for status_code in (
            set(source_responses)
            & set(converted_responses)
        ):
            source_response = (
                source_responses[
                    status_code
                ]
            )

            converted_response = (
                converted_responses[
                    status_code
                ]
            )

            if (
                isinstance(
                    source_response,
                    dict,
                )
                and isinstance(
                    converted_response,
                    dict,
                )
                and source_response.get(
                    "description"
                )
                != converted_response.get(
                    "description"
                )
            ):
                response_failures.append(
                    {
                        "operation": (
                            f"{operation_key[1].upper()} "
                            f"{operation_key[0]}"
                        ),
                        "status": status_code,
                        "problem": (
                            "response_description"
                        ),
                    }
                )

    categories[
        "operation_metadata_preserved"
    ] = not metadata_failures

    if metadata_failures:
        diagnostics[
            "operation_metadata_preserved"
        ] = metadata_failures[:50]

    categories[
        "response_contract_surface_preserved"
    ] = not response_failures

    if response_failures:
        diagnostics[
            "response_contract_surface_preserved"
        ] = response_failures[:50]

    source_prose = collect_prose(
        source
    )

    converted_prose = collect_prose(
        converted
    )

    lost_prose = {
        text: count
        - converted_prose[text]
        for text, count
        in source_prose.items()
        if converted_prose[text] < count
    }

    categories[
        "all_prose_preserved"
    ] = not lost_prose

    if lost_prose:
        diagnostics[
            "all_prose_preserved"
        ] = {
            "lost_count": sum(
                lost_prose.values()
            ),
            "examples": list(
                lost_prose.items()
            )[:30],
        }

    source_definitions = (
        source.get(
            "definitions",
            {},
        )
    )

    converted_schemas = (
        converted
        .get("components", {})
        .get("schemas", {})
    )

    schema_failures = []

    if not isinstance(
        source_definitions,
        dict,
    ):
        schema_failures.append(
            "source definitions is not a mapping"
        )

    elif not isinstance(
        converted_schemas,
        dict,
    ):
        schema_failures.append(
            "converted components.schemas "
            "is not a mapping"
        )

    else:
        for name, source_schema in (
            source_definitions.items()
        ):
            if name not in converted_schemas:
                schema_failures.append(
                    f"missing schema: {name}"
                )
                continue

            if (
                normalize_schema(
                    source_schema
                )
                != normalize_schema(
                    converted_schemas[
                        name
                    ]
                )
            ):
                schema_failures.append(
                    f"schema mismatch: {name}"
                )

    categories[
        "definitions_preserved"
    ] = not schema_failures

    if schema_failures:
        diagnostics[
            "definitions_preserved"
        ] = schema_failures[:50]

    source_definition_refs = {
        remap_ref(reference)
        for reference in collect_refs(
            source
        )
        if reference.startswith(
            "#/definitions/"
        )
    }

    converted_refs = set(
        collect_refs(converted)
    )

    missing_definition_refs = sorted(
        source_definition_refs
        - converted_refs
    )

    categories[
        "definition_references_preserved"
    ] = not missing_definition_refs

    if missing_definition_refs:
        diagnostics[
            "definition_references_preserved"
        ] = missing_definition_refs[:50]

    top_level_failures = []

    for field in [
        "tags",
        "externalDocs",
        "security",
    ]:
        if (
            field in source
            and source[field]
            != converted.get(field)
        ):
            top_level_failures.append(
                field
            )

    source_top_extensions = {
        key: value
        for key, value in source.items()
        if key.startswith("x-")
    }

    for key, value in (
        source_top_extensions.items()
    ):
        if converted.get(key) != value:
            top_level_failures.append(
                key
            )

    categories[
        "top_level_metadata_preserved"
    ] = not top_level_failures

    if top_level_failures:
        diagnostics[
            "top_level_metadata_preserved"
        ] = top_level_failures[:50]

    expected_server_urls = (
        expected_servers(source)
    )

    if expected_server_urls is None:
        categories[
            "declared_servers_preserved"
        ] = True

    else:
        observed_server_urls = {
            server.get("url")
            for server in converted.get(
                "servers",
                [],
            )
            if isinstance(server, dict)
        }

        categories[
            "declared_servers_preserved"
        ] = (
            expected_server_urls
            == observed_server_urls
        )

        if not categories[
            "declared_servers_preserved"
        ]:
            diagnostics[
                "declared_servers_preserved"
            ] = {
                "expected": sorted(
                    expected_server_urls
                ),
                "observed": sorted(
                    str(value)
                    for value
                    in observed_server_urls
                ),
            }

    return categories, diagnostics


required_paths = [
    CONVERSION_MANIFEST_PATH,
    VALIDATION_RESULTS_PATH,
    VALIDATION_SUMMARY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Missing semantic-audit inputs:\n"
        + "\n".join(missing_paths)
    )

validation_summary = json.loads(
    VALIDATION_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

if validation_summary.get("status") != "PASS":
    raise RuntimeError(
        "Converted validation summary is not PASS."
    )

if (
    sha256_file(
        VALIDATION_RESULTS_PATH
    )
    != validation_summary[
        "results_sha256"
    ]
):
    raise RuntimeError(
        "Converted validation ledger hash mismatch."
    )

manifest = json.loads(
    CONVERSION_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

manifest_by_spec_id = {
    record["spec_id"]: record
    for record in manifest["records"]
}

validation_records = []

with VALIDATION_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line in file:
        line = line.strip()

        if line:
            validation_records.append(
                json.loads(line)
            )

candidate_records = sorted(
    [
        record
        for record in validation_records
        if (
            record.get(
                "initial_disposition"
            )
            == "PENDING_STRICT_AUDIT"
            and record.get(
                "strict_validation_pass"
            )
            is True
        )
    ],
    key=lambda item: (
        item["relative_path"],
        item["spec_id"],
    ),
)

if len(candidate_records) != 1868:
    raise RuntimeError(
        "Unexpected strict semantic-audit "
        "candidate count: "
        f"expected 1868, "
        f"observed {len(candidate_records)}."
    )

started = time.perf_counter()
results = []
resumed_count = 0

for position, validation_record in enumerate(
    candidate_records,
    start=1,
):
    spec_id = validation_record[
        "spec_id"
    ]

    manifest_record = (
        manifest_by_spec_id.get(
            spec_id
        )
    )

    if manifest_record is None:
        raise RuntimeError(
            "Missing conversion-manifest record "
            f"for {spec_id}."
        )

    source_path = (
        CORPUS_ROOT
        / manifest_record[
            "relative_path"
        ]
    )

    converted_path = Path(
        validation_record[
            "converted_path"
        ]
    )

    checkpoint_path = (
        AUDIT_ITEM_ROOT
        / f"{spec_id}.json"
    )

    expected_source_sha256 = (
        manifest_record[
            "source_sha256"
        ]
    )

    expected_converted_sha256 = (
        validation_record[
            "converted_sha256"
        ]
    )

    reusable = None

    if checkpoint_path.is_file():
        try:
            candidate = json.loads(
                checkpoint_path.read_text(
                    encoding="utf-8"
                )
            )

            if (
                candidate.get(
                    "audit_version"
                )
                == AUDIT_VERSION
                and candidate.get(
                    "spec_id"
                )
                == spec_id
                and candidate.get(
                    "source_sha256"
                )
                == expected_source_sha256
                and candidate.get(
                    "converted_sha256"
                )
                == expected_converted_sha256
            ):
                reusable = candidate
        except Exception:
            reusable = None

    if reusable is not None:
        results.append(reusable)
        resumed_count += 1

    else:
        categories = {}
        diagnostics = {}

        observed_source_sha256 = (
            sha256_file(source_path)
            if source_path.is_file()
            else None
        )

        observed_converted_sha256 = (
            sha256_file(converted_path)
            if converted_path.is_file()
            else None
        )

        categories[
            "source_integrity"
        ] = (
            observed_source_sha256
            == expected_source_sha256
        )

        categories[
            "converted_integrity"
        ] = (
            observed_converted_sha256
            == expected_converted_sha256
        )

        if not categories[
            "source_integrity"
        ]:
            diagnostics[
                "source_integrity"
            ] = {
                "expected": (
                    expected_source_sha256
                ),
                "observed": (
                    observed_source_sha256
                ),
            }

        if not categories[
            "converted_integrity"
        ]:
            diagnostics[
                "converted_integrity"
            ] = {
                "expected": (
                    expected_converted_sha256
                ),
                "observed": (
                    observed_converted_sha256
                ),
            }

        if (
            categories[
                "source_integrity"
            ]
            and categories[
                "converted_integrity"
            ]
        ):
            try:
                source_document = yaml.load(
                    source_path.read_text(
                        encoding="utf-8"
                    ),
                    Loader=(
                        OpenAPIStringKeyLoader
                    ),
                )

                converted_document = (
                    json.loads(
                        converted_path.read_text(
                            encoding="utf-8"
                        )
                    )
                )

                (
                    semantic_categories,
                    semantic_diagnostics,
                ) = audit_document(
                    source_document,
                    converted_document,
                )

                categories.update(
                    semantic_categories
                )

                diagnostics.update(
                    semantic_diagnostics
                )

            except Exception as exception:
                categories[
                    "document_comparison"
                ] = False

                diagnostics[
                    "document_comparison"
                ] = (
                    f"{type(exception).__name__}: "
                    f"{exception}"
                )[:12000]

        core_semantic_pass = all(
            categories.values()
        )

        result = {
            "audit_version": (
                AUDIT_VERSION
            ),
            "spec_id": spec_id,
            "raw_id": manifest_record[
                "raw_id"
            ],
            "relative_path": (
                manifest_record[
                    "relative_path"
                ]
            ),
            "provider_directory": (
                manifest_record[
                    "provider_directory"
                ]
            ),
            "source_sha256": (
                expected_source_sha256
            ),
            "converted_sha256": (
                expected_converted_sha256
            ),
            "category_results": categories,
            "failed_categories": sorted(
                category
                for category, passed
                in categories.items()
                if not passed
            ),
            "diagnostics": diagnostics,
            "core_semantic_pass": (
                core_semantic_pass
            ),
        }

        write_json_atomic(
            checkpoint_path,
            result,
        )

        results.append(result)

    if (
        position % 25 == 0
        or position
        == len(candidate_records)
    ):
        elapsed = (
            time.perf_counter()
            - started
        )

        rate = (
            position / elapsed
            if elapsed > 0
            else 0
        )

        remaining = (
            len(candidate_records)
            - position
        )

        eta = (
            remaining / rate
            if rate > 0
            else None
        )

        print(
            json.dumps(
                {
                    "progress": position,
                    "total": len(
                        candidate_records
                    ),
                    "resumed": resumed_count,
                    "elapsed_seconds": round(
                        elapsed,
                        1,
                    ),
                    "rate_per_second": round(
                        rate,
                        3,
                    ),
                    "eta_seconds": (
                        round(eta, 1)
                        if eta is not None
                        else None
                    ),
                }
            ),
            flush=True,
        )

results = sorted(
    results,
    key=lambda item: (
        item["relative_path"],
        item["spec_id"],
    ),
)

results_text = (
    "\n".join(
        json.dumps(
            result,
            sort_keys=True,
            ensure_ascii=False,
        )
        for result in results
    )
    + "\n"
)

temporary_results_path = (
    AUDIT_RESULTS_PATH.with_suffix(
        ".jsonl.tmp"
    )
)

temporary_results_path.write_text(
    results_text,
    encoding="utf-8",
)

temporary_results_path.replace(
    AUDIT_RESULTS_PATH
)

semantic_pass_count = sum(
    bool(
        result[
            "core_semantic_pass"
        ]
    )
    for result in results
)

category_failure_counts = Counter()

for result in results:
    category_failure_counts.update(
        result["failed_categories"]
    )

failure_examples = [
    {
        "spec_id": result["spec_id"],
        "relative_path": (
            result["relative_path"]
        ),
        "failed_categories": (
            result["failed_categories"]
        ),
        "diagnostics": (
            result["diagnostics"]
        ),
    }
    for result in results
    if not result[
        "core_semantic_pass"
    ]
][:20]

summary = {
    "status": (
        "PASS"
        if len(results) == 1868
        else "BLOCKED"
    ),
    "audit_version": AUDIT_VERSION,
    "audited_specification_count": len(
        results
    ),
    "resumed_checkpoint_count": (
        resumed_count
    ),
    "core_semantic_pass_count": (
        semantic_pass_count
    ),
    "core_semantic_failure_count": (
        len(results)
        - semantic_pass_count
    ),
    "category_failure_counts": {
        str(key): int(value)
        for key, value in sorted(
            category_failure_counts.items()
        )
    },
    "failure_examples": (
        failure_examples
    ),
    "duration_seconds": round(
        time.perf_counter() - started,
        3,
    ),
    "results_path": str(
        AUDIT_RESULTS_PATH
    ),
    "results_sha256": sha256_file(
        AUDIT_RESULTS_PATH
    ),
}

write_json_atomic(
    AUDIT_SUMMARY_PATH,
    summary,
)

output = {
    **summary,
    "summary_path": str(
        AUDIT_SUMMARY_PATH
    ),
    "summary_sha256": sha256_file(
        AUDIT_SUMMARY_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

if summary["status"] != "PASS":
    raise RuntimeError(
        "Core semantic audit did not "
        "account for every candidate."
    )

print("\nNOTEBOOK 01 — CELL 19: PASS")

{"progress": 25, "total": 1868, "resumed": 0, "elapsed_seconds": 0.3, "rate_per_second": 81.658, "eta_seconds": 22.6}
{"progress": 50, "total": 1868, "resumed": 0, "elapsed_seconds": 4.6, "rate_per_second": 10.932, "eta_seconds": 166.3}
{"progress": 75, "total": 1868, "resumed": 0, "elapsed_seconds": 5.4, "rate_per_second": 13.857, "eta_seconds": 129.4}
{"progress": 100, "total": 1868, "resumed": 0, "elapsed_seconds": 6.9, "rate_per_second": 14.407, "eta_seconds": 122.7}
{"progress": 125, "total": 1868, "resumed": 0, "elapsed_seconds": 7.2, "rate_per_second": 17.328, "eta_seconds": 100.6}
{"progress": 150, "total": 1868, "resumed": 0, "elapsed_seconds": 7.4, "rate_per_second": 20.134, "eta_seconds": 85.3}
{"progress": 175, "total": 1868, "resumed": 0, "elapsed_seconds": 7.7, "rate_per_second": 22.606, "eta_seconds": 74.9}
{"progress": 200, "total": 1868, "resumed": 0, "elapsed_seconds": 8.0, "rate_per_second": 24.875, "eta_seconds": 67.1}
{"progress": 225, "total": 1868, "resumed": 0, 

In [22]:
import hashlib
import json
import time
from collections import Counter
from pathlib import Path
from urllib.parse import urlsplit

import yaml
from yaml.constructor import ConstructorError
from yaml.nodes import MappingNode, ScalarNode
from yaml.resolver import BaseResolver

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

CONVERSION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "swagger_conversion_manifest.json"
)

VALIDATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_converted_validation.jsonl"
)

VALIDATION_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_converted_validation_summary.json"
)

VERSION_1_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_semantic_core_summary.json"
)

AUDIT_ITEM_ROOT = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_semantic_v2_items"
)

AUDIT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_semantic_v2_results.jsonl"
)

AUDIT_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_semantic_v2_summary.json"
)

AUDIT_VERSION = 2

HTTP_METHODS = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}

for directory in [
    AUDIT_ITEM_ROOT,
    AUDIT_RESULTS_PATH.parent,
    AUDIT_SUMMARY_PATH.parent,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


loader_base = getattr(
    yaml,
    "CSafeLoader",
    yaml.SafeLoader,
)


class OpenAPIStringKeyLoader(loader_base):
    pass


def construct_string_key_mapping(
    loader,
    node,
    deep=False,
):
    if not isinstance(node, MappingNode):
        raise ConstructorError(
            None,
            None,
            "Expected a mapping node.",
            node.start_mark,
        )

    loader.flatten_mapping(node)
    mapping = {}

    for key_node, value_node in node.value:
        if not isinstance(
            key_node,
            ScalarNode,
        ):
            raise ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                "Mapping key is not scalar.",
                key_node.start_mark,
            )

        key = key_node.value

        if key in mapping:
            raise ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                f"Duplicate mapping key: {key!r}",
                key_node.start_mark,
            )

        mapping[key] = (
            loader.construct_object(
                value_node,
                deep=deep,
            )
        )

    return mapping


def construct_timestamp_as_string(
    loader,
    node,
):
    return loader.construct_scalar(node)


OpenAPIStringKeyLoader.add_constructor(
    BaseResolver.DEFAULT_MAPPING_TAG,
    construct_string_key_mapping,
)

OpenAPIStringKeyLoader.add_constructor(
    "tag:yaml.org,2002:timestamp",
    construct_timestamp_as_string,
)


def remap_ref(reference):
    mappings = {
        "#/definitions/": (
            "#/components/schemas/"
        ),
        "#/parameters/": (
            "#/components/parameters/"
        ),
        "#/responses/": (
            "#/components/responses/"
        ),
    }

    for old_prefix, new_prefix in (
        mappings.items()
    ):
        if reference.startswith(
            old_prefix
        ):
            return (
                new_prefix
                + reference[
                    len(old_prefix):
                ]
            )

    return reference


def normalize_references(value):
    if isinstance(value, list):
        return [
            normalize_references(child)
            for child in value
        ]

    if not isinstance(value, dict):
        return value

    result = {}

    for key, child in value.items():
        if (
            key == "$ref"
            and isinstance(child, str)
        ):
            result[key] = remap_ref(child)
        else:
            result[key] = (
                normalize_references(child)
            )

    return result


def normalize_schema(value):
    if isinstance(value, list):
        return [
            normalize_schema(child)
            for child in value
        ]

    if not isinstance(value, dict):
        return value

    normalized = {}
    nullable_value = None

    for key, child in value.items():
        if (
            key == "$ref"
            and isinstance(child, str)
        ):
            normalized[key] = remap_ref(
                child
            )

        elif key == "x-nullable":
            nullable_value = bool(child)

        elif (
            key == "discriminator"
            and isinstance(child, str)
        ):
            normalized[key] = {
                "propertyName": child,
            }

        else:
            normalized[key] = (
                normalize_schema(child)
            )

    if nullable_value is not None:
        normalized["nullable"] = (
            nullable_value
        )

    # Swagger permits $ref siblings in places
    # where OpenAPI 3 requires an allOf wrapper.
    if (
        "$ref" in normalized
        and len(normalized) > 1
    ):
        reference = normalized.pop(
            "$ref"
        )

        existing_all_of = (
            normalized.pop(
                "allOf",
                [],
            )
        )

        normalized["allOf"] = (
            [
                {
                    "$ref": reference,
                }
            ]
            + existing_all_of
        )

    return normalized


def operation_map(document):
    result = {}

    for path_name, path_item in (
        document.get("paths", {}).items()
    ):
        if not isinstance(path_item, dict):
            continue

        for method_name, operation in (
            path_item.items()
        ):
            method = method_name.lower()

            if (
                method in HTTP_METHODS
                and isinstance(
                    operation,
                    dict,
                )
            ):
                result[
                    (path_name, method)
                ] = operation

    return result


def collect_prose_set(value):
    result = set()
    stack = [value]

    while stack:
        current = stack.pop()

        if isinstance(current, dict):
            for key, child in current.items():
                if (
                    key
                    in {
                        "title",
                        "summary",
                        "description",
                    }
                    and isinstance(child, str)
                    and child.strip()
                ):
                    result.add(child)

                stack.append(child)

        elif isinstance(current, list):
            stack.extend(current)

    return result


def collect_refs(value):
    refs = []
    stack = [value]

    while stack:
        current = stack.pop()

        if isinstance(current, dict):
            for key, child in current.items():
                if (
                    key == "$ref"
                    and isinstance(child, str)
                ):
                    refs.append(child)
                else:
                    stack.append(child)

        elif isinstance(current, list):
            stack.extend(current)

    return refs


def normalize_root_url(url):
    if not isinstance(url, str):
        return url

    parsed = urlsplit(url)

    if parsed.path in {"", "/"}:
        result = (
            f"{parsed.scheme}://"
            f"{parsed.netloc}"
        )

        if parsed.query:
            result += f"?{parsed.query}"

        if parsed.fragment:
            result += f"#{parsed.fragment}"

        return result

    return url


def expected_servers(source):
    host = source.get("host")
    schemes = source.get("schemes")

    if (
        not isinstance(host, str)
        or not host
        or not isinstance(schemes, list)
        or not schemes
    ):
        return None

    base_path = source.get(
        "basePath",
        "",
    )

    if not isinstance(base_path, str):
        return None

    return {
        normalize_root_url(
            f"{scheme}://{host}{base_path}"
        )
        for scheme in schemes
        if isinstance(scheme, str)
    }


def audit_document(
    source,
    converted,
):
    categories = {}
    diagnostics = {}

    categories["source_dialect"] = (
        source.get("swagger") == "2.0"
    )

    categories["target_version"] = (
        converted.get("openapi")
        == "3.0.3"
    )

    categories["info_preserved"] = (
        source.get("info")
        == converted.get("info")
    )

    source_operations = operation_map(
        source
    )

    converted_operations = operation_map(
        converted
    )

    source_operation_set = set(
        source_operations
    )

    converted_operation_set = set(
        converted_operations
    )

    categories[
        "operation_set_preserved"
    ] = (
        source_operation_set
        == converted_operation_set
    )

    if not categories[
        "operation_set_preserved"
    ]:
        diagnostics[
            "operation_set_preserved"
        ] = {
            "missing": sorted(
                source_operation_set
                - converted_operation_set
            )[:30],
            "added": sorted(
                converted_operation_set
                - source_operation_set
            )[:30],
        }

    metadata_failures = []
    response_failures = []

    ordinary_metadata_fields = [
        "operationId",
        "summary",
        "description",
        "tags",
        "deprecated",
        "security",
        "externalDocs",
    ]

    for operation_key in sorted(
        source_operation_set
        & converted_operation_set
    ):
        source_operation = (
            source_operations[
                operation_key
            ]
        )

        converted_operation = (
            converted_operations[
                operation_key
            ]
        )

        operation_label = (
            f"{operation_key[1].upper()} "
            f"{operation_key[0]}"
        )

        for field in ordinary_metadata_fields:
            if (
                field in source_operation
                and normalize_references(
                    source_operation[field]
                )
                != normalize_references(
                    converted_operation.get(
                        field
                    )
                )
            ):
                metadata_failures.append(
                    {
                        "operation": (
                            operation_label
                        ),
                        "field": field,
                    }
                )

        for key, value in (
            source_operation.items()
        ):
            if not key.startswith("x-"):
                continue

            if (
                normalize_references(value)
                != normalize_references(
                    converted_operation.get(
                        key
                    )
                )
            ):
                metadata_failures.append(
                    {
                        "operation": (
                            operation_label
                        ),
                        "field": key,
                    }
                )

        source_responses = (
            source_operation.get(
                "responses",
                {},
            )
        )

        converted_responses = (
            converted_operation.get(
                "responses",
                {},
            )
        )

        if (
            set(source_responses)
            != set(converted_responses)
        ):
            response_failures.append(
                {
                    "operation": (
                        operation_label
                    ),
                    "problem": (
                        "response_status_set"
                    ),
                }
            )

        for status_code in (
            set(source_responses)
            & set(converted_responses)
        ):
            source_response = (
                source_responses[
                    status_code
                ]
            )

            converted_response = (
                converted_responses[
                    status_code
                ]
            )

            if (
                isinstance(
                    source_response,
                    dict,
                )
                and isinstance(
                    converted_response,
                    dict,
                )
                and source_response.get(
                    "description"
                )
                != converted_response.get(
                    "description"
                )
            ):
                response_failures.append(
                    {
                        "operation": (
                            operation_label
                        ),
                        "status": status_code,
                        "problem": (
                            "response_description"
                        ),
                    }
                )

    categories[
        "operation_metadata_preserved"
    ] = not metadata_failures

    if metadata_failures:
        diagnostics[
            "operation_metadata_preserved"
        ] = metadata_failures[:30]

    categories[
        "response_contract_surface_preserved"
    ] = not response_failures

    if response_failures:
        diagnostics[
            "response_contract_surface_preserved"
        ] = response_failures[:30]

    source_prose = collect_prose_set(
        source
    )

    converted_prose = collect_prose_set(
        converted
    )

    lost_prose = sorted(
        source_prose - converted_prose
    )

    categories[
        "all_unique_prose_preserved"
    ] = not lost_prose

    if lost_prose:
        diagnostics[
            "all_unique_prose_preserved"
        ] = {
            "lost_unique_count": len(
                lost_prose
            ),
            "examples": lost_prose[:30],
        }

    source_definitions = (
        source.get(
            "definitions",
            {},
        )
    )

    converted_schemas = (
        converted
        .get("components", {})
        .get("schemas", {})
    )

    schema_failures = []

    if not isinstance(
        source_definitions,
        dict,
    ):
        schema_failures.append(
            "source definitions is not a mapping"
        )

    elif not isinstance(
        converted_schemas,
        dict,
    ):
        schema_failures.append(
            "components.schemas is not a mapping"
        )

    else:
        for name, source_schema in (
            source_definitions.items()
        ):
            converted_schema = (
                converted_schemas.get(name)
            )

            if converted_schema is None:
                schema_failures.append(
                    f"missing schema: {name}"
                )

            elif (
                normalize_schema(
                    source_schema
                )
                != normalize_schema(
                    converted_schema
                )
            ):
                schema_failures.append(
                    f"schema mismatch: {name}"
                )

    categories[
        "definitions_preserved"
    ] = not schema_failures

    if schema_failures:
        diagnostics[
            "definitions_preserved"
        ] = schema_failures[:30]

    expected_definition_refs = {
        remap_ref(reference)
        for reference in collect_refs(
            source
        )
        if reference.startswith(
            "#/definitions/"
        )
    }

    converted_refs = set(
        collect_refs(converted)
    )

    missing_refs = sorted(
        expected_definition_refs
        - converted_refs
    )

    categories[
        "definition_references_preserved"
    ] = not missing_refs

    if missing_refs:
        diagnostics[
            "definition_references_preserved"
        ] = missing_refs[:30]

    top_level_failures = []

    for field in [
        "tags",
        "externalDocs",
        "security",
    ]:
        if (
            field in source
            and normalize_references(
                source[field]
            )
            != normalize_references(
                converted.get(field)
            )
        ):
            top_level_failures.append(
                field
            )

    for key, value in source.items():
        if not key.startswith("x-"):
            continue

        if (
            normalize_references(value)
            != normalize_references(
                converted.get(key)
            )
        ):
            top_level_failures.append(
                key
            )

    categories[
        "top_level_metadata_preserved"
    ] = not top_level_failures

    if top_level_failures:
        diagnostics[
            "top_level_metadata_preserved"
        ] = top_level_failures[:30]

    expected_server_urls = (
        expected_servers(source)
    )

    if expected_server_urls is None:
        categories[
            "declared_servers_preserved"
        ] = True

    else:
        observed_server_urls = {
            normalize_root_url(
                server.get("url")
            )
            for server in converted.get(
                "servers",
                [],
            )
            if isinstance(server, dict)
        }

        categories[
            "declared_servers_preserved"
        ] = (
            expected_server_urls
            == observed_server_urls
        )

        if not categories[
            "declared_servers_preserved"
        ]:
            diagnostics[
                "declared_servers_preserved"
            ] = {
                "expected": sorted(
                    str(value)
                    for value
                    in expected_server_urls
                ),
                "observed": sorted(
                    str(value)
                    for value
                    in observed_server_urls
                ),
            }

    return categories, diagnostics


required_paths = [
    CONVERSION_MANIFEST_PATH,
    VALIDATION_RESULTS_PATH,
    VALIDATION_SUMMARY_PATH,
    VERSION_1_SUMMARY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Missing audit prerequisites:\n"
        + "\n".join(missing_paths)
    )

validation_summary = json.loads(
    VALIDATION_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

version_1_summary = json.loads(
    VERSION_1_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

if validation_summary.get("status") != "PASS":
    raise RuntimeError(
        "Converted validation summary is not PASS."
    )

if version_1_summary.get("status") != "PASS":
    raise RuntimeError(
        "Version 1 semantic audit is not PASS."
    )

if (
    sha256_file(
        VALIDATION_RESULTS_PATH
    )
    != validation_summary[
        "results_sha256"
    ]
):
    raise RuntimeError(
        "Validation-results hash mismatch."
    )

manifest = json.loads(
    CONVERSION_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

manifest_by_spec_id = {
    record["spec_id"]: record
    for record in manifest["records"]
}

validation_records = []

with VALIDATION_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line in file:
        line = line.strip()

        if line:
            validation_records.append(
                json.loads(line)
            )

candidate_records = sorted(
    [
        record
        for record in validation_records
        if (
            record.get(
                "initial_disposition"
            )
            == "PENDING_STRICT_AUDIT"
            and record.get(
                "strict_validation_pass"
            )
            is True
        )
    ],
    key=lambda item: (
        item["relative_path"],
        item["spec_id"],
    ),
)

if len(candidate_records) != 1868:
    raise RuntimeError(
        "Expected 1868 strict candidates; "
        f"observed {len(candidate_records)}."
    )

started = time.perf_counter()
results = []
resumed_count = 0

for position, validation_record in enumerate(
    candidate_records,
    start=1,
):
    spec_id = validation_record[
        "spec_id"
    ]

    manifest_record = (
        manifest_by_spec_id.get(spec_id)
    )

    if manifest_record is None:
        raise RuntimeError(
            f"Missing manifest record: {spec_id}"
        )

    source_path = (
        CORPUS_ROOT
        / manifest_record[
            "relative_path"
        ]
    )

    converted_path = Path(
        validation_record[
            "converted_path"
        ]
    )

    source_sha256 = manifest_record[
        "source_sha256"
    ]

    converted_sha256 = (
        validation_record[
            "converted_sha256"
        ]
    )

    checkpoint_path = (
        AUDIT_ITEM_ROOT
        / f"{spec_id}.json"
    )

    reusable = None

    if checkpoint_path.is_file():
        try:
            candidate = json.loads(
                checkpoint_path.read_text(
                    encoding="utf-8"
                )
            )

            if (
                candidate.get(
                    "audit_version"
                )
                == AUDIT_VERSION
                and candidate.get(
                    "source_sha256"
                )
                == source_sha256
                and candidate.get(
                    "converted_sha256"
                )
                == converted_sha256
            ):
                reusable = candidate
        except Exception:
            reusable = None

    if reusable is not None:
        results.append(reusable)
        resumed_count += 1

    else:
        categories = {
            "source_integrity": (
                source_path.is_file()
                and sha256_file(source_path)
                == source_sha256
            ),
            "converted_integrity": (
                converted_path.is_file()
                and sha256_file(
                    converted_path
                )
                == converted_sha256
            ),
        }

        diagnostics = {}

        if (
            categories[
                "source_integrity"
            ]
            and categories[
                "converted_integrity"
            ]
        ):
            try:
                source_document = yaml.load(
                    source_path.read_text(
                        encoding="utf-8"
                    ),
                    Loader=(
                        OpenAPIStringKeyLoader
                    ),
                )

                converted_document = (
                    json.loads(
                        converted_path.read_text(
                            encoding="utf-8"
                        )
                    )
                )

                (
                    semantic_categories,
                    semantic_diagnostics,
                ) = audit_document(
                    source_document,
                    converted_document,
                )

                categories.update(
                    semantic_categories
                )

                diagnostics.update(
                    semantic_diagnostics
                )

            except Exception as exception:
                categories[
                    "document_comparison"
                ] = False

                diagnostics[
                    "document_comparison"
                ] = (
                    f"{type(exception).__name__}: "
                    f"{exception}"
                )[:12000]

        representation_aware_pass = all(
            categories.values()
        )

        result = {
            "audit_version": (
                AUDIT_VERSION
            ),
            "spec_id": spec_id,
            "raw_id": manifest_record[
                "raw_id"
            ],
            "relative_path": (
                manifest_record[
                    "relative_path"
                ]
            ),
            "provider_directory": (
                manifest_record[
                    "provider_directory"
                ]
            ),
            "source_sha256": (
                source_sha256
            ),
            "converted_sha256": (
                converted_sha256
            ),
            "category_results": categories,
            "failed_categories": sorted(
                category
                for category, passed
                in categories.items()
                if not passed
            ),
            "diagnostics": diagnostics,
            "representation_aware_pass": (
                representation_aware_pass
            ),
        }

        write_json_atomic(
            checkpoint_path,
            result,
        )

        results.append(result)

    if (
        position % 25 == 0
        or position
        == len(candidate_records)
    ):
        elapsed = (
            time.perf_counter()
            - started
        )

        rate = (
            position / elapsed
            if elapsed > 0
            else 0
        )

        remaining = (
            len(candidate_records)
            - position
        )

        eta = (
            remaining / rate
            if rate > 0
            else None
        )

        print(
            json.dumps(
                {
                    "progress": position,
                    "total": len(
                        candidate_records
                    ),
                    "resumed": resumed_count,
                    "elapsed_seconds": round(
                        elapsed,
                        1,
                    ),
                    "rate_per_second": round(
                        rate,
                        3,
                    ),
                    "eta_seconds": (
                        round(eta, 1)
                        if eta is not None
                        else None
                    ),
                }
            ),
            flush=True,
        )

results = sorted(
    results,
    key=lambda item: (
        item["relative_path"],
        item["spec_id"],
    ),
)

results_text = (
    "\n".join(
        json.dumps(
            result,
            sort_keys=True,
            ensure_ascii=False,
        )
        for result in results
    )
    + "\n"
)

temporary_results_path = (
    AUDIT_RESULTS_PATH.with_suffix(
        ".jsonl.tmp"
    )
)

temporary_results_path.write_text(
    results_text,
    encoding="utf-8",
)

temporary_results_path.replace(
    AUDIT_RESULTS_PATH
)

pass_count = sum(
    bool(
        result[
            "representation_aware_pass"
        ]
    )
    for result in results
)

category_failure_counts = Counter()

for result in results:
    category_failure_counts.update(
        result["failed_categories"]
    )

failure_examples = [
    {
        "spec_id": result["spec_id"],
        "relative_path": (
            result["relative_path"]
        ),
        "failed_categories": (
            result["failed_categories"]
        ),
        "diagnostics": (
            result["diagnostics"]
        ),
    }
    for result in results
    if not result[
        "representation_aware_pass"
    ]
][:20]

summary = {
    "status": (
        "PASS"
        if len(results) == 1868
        else "BLOCKED"
    ),
    "audit_version": AUDIT_VERSION,
    "supersedes_audit_version": 1,
    "recalibrations": [
        (
            "Root URLs differing only by a "
            "trailing slash are equivalent."
        ),
        (
            "Repeated identical prose may be "
            "represented once in a shared component."
        ),
        (
            "References inside extensions are "
            "compared after OpenAPI remapping."
        ),
        (
            "Swagger ref siblings are compared "
            "with the equivalent allOf form."
        ),
    ],
    "audited_specification_count": len(
        results
    ),
    "resumed_checkpoint_count": (
        resumed_count
    ),
    "representation_aware_pass_count": (
        pass_count
    ),
    "representation_aware_failure_count": (
        len(results) - pass_count
    ),
    "version_1_pass_count": (
        version_1_summary[
            "core_semantic_pass_count"
        ]
    ),
    "newly_resolved_false_positive_count": (
        pass_count
        - version_1_summary[
            "core_semantic_pass_count"
        ]
    ),
    "category_failure_counts": {
        str(key): int(value)
        for key, value in sorted(
            category_failure_counts.items()
        )
    },
    "failure_examples": (
        failure_examples
    ),
    "duration_seconds": round(
        time.perf_counter() - started,
        3,
    ),
    "results_path": str(
        AUDIT_RESULTS_PATH
    ),
    "results_sha256": sha256_file(
        AUDIT_RESULTS_PATH
    ),
}

write_json_atomic(
    AUDIT_SUMMARY_PATH,
    summary,
)

output = {
    **summary,
    "summary_path": str(
        AUDIT_SUMMARY_PATH
    ),
    "summary_sha256": sha256_file(
        AUDIT_SUMMARY_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

if summary["status"] != "PASS":
    raise RuntimeError(
        "Representation-aware semantic audit "
        "did not account for every candidate."
    )

print("\nNOTEBOOK 01 — CELL 20: PASS")

{"progress": 25, "total": 1868, "resumed": 0, "elapsed_seconds": 0.3, "rate_per_second": 76.857, "eta_seconds": 24.0}
{"progress": 50, "total": 1868, "resumed": 0, "elapsed_seconds": 4.8, "rate_per_second": 10.503, "eta_seconds": 173.1}
{"progress": 75, "total": 1868, "resumed": 0, "elapsed_seconds": 5.6, "rate_per_second": 13.446, "eta_seconds": 133.3}
{"progress": 100, "total": 1868, "resumed": 0, "elapsed_seconds": 7.1, "rate_per_second": 14.115, "eta_seconds": 125.3}
{"progress": 125, "total": 1868, "resumed": 0, "elapsed_seconds": 7.3, "rate_per_second": 17.013, "eta_seconds": 102.4}
{"progress": 150, "total": 1868, "resumed": 0, "elapsed_seconds": 7.6, "rate_per_second": 19.79, "eta_seconds": 86.8}
{"progress": 175, "total": 1868, "resumed": 0, "elapsed_seconds": 7.9, "rate_per_second": 22.22, "eta_seconds": 76.2}
{"progress": 200, "total": 1868, "resumed": 0, "elapsed_seconds": 8.2, "rate_per_second": 24.442, "eta_seconds": 68.2}
{"progress": 225, "total": 1868, "resumed": 0, "e

In [23]:
import hashlib
import json
import time
from collections import Counter
from pathlib import Path

import yaml
from yaml.constructor import ConstructorError
from yaml.nodes import MappingNode, ScalarNode
from yaml.resolver import BaseResolver

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "swagger_conversion_manifest.json"
)

VALIDATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_converted_validation.jsonl"
)

SEMANTIC_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_semantic_v2_results.jsonl"
)

SEMANTIC_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_semantic_v2_summary.json"
)

ITEM_ROOT = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_contract_audit_items"
)

RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_contract_audit_results.jsonl"
)

SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_contract_audit_summary.json"
)

AUDIT_VERSION = 1

HTTP_METHODS = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}

SCHEMA_FIELDS = {
    "type",
    "format",
    "items",
    "default",
    "maximum",
    "exclusiveMaximum",
    "minimum",
    "exclusiveMinimum",
    "maxLength",
    "minLength",
    "pattern",
    "maxItems",
    "minItems",
    "uniqueItems",
    "enum",
    "multipleOf",
    "readOnly",
    "example",
}

ITEM_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


loader_base = getattr(
    yaml,
    "CSafeLoader",
    yaml.SafeLoader,
)


class OpenAPIStringKeyLoader(loader_base):
    pass


def construct_string_key_mapping(
    loader,
    node,
    deep=False,
):
    if not isinstance(node, MappingNode):
        raise ConstructorError(
            None,
            None,
            "Expected mapping node.",
            node.start_mark,
        )

    loader.flatten_mapping(node)
    mapping = {}

    for key_node, value_node in node.value:
        if not isinstance(
            key_node,
            ScalarNode,
        ):
            raise ConstructorError(
                "while constructing mapping",
                node.start_mark,
                "Mapping key is not scalar.",
                key_node.start_mark,
            )

        key = key_node.value

        if key in mapping:
            raise ConstructorError(
                "while constructing mapping",
                node.start_mark,
                f"Duplicate key: {key!r}",
                key_node.start_mark,
            )

        mapping[key] = (
            loader.construct_object(
                value_node,
                deep=deep,
            )
        )

    return mapping


def construct_timestamp_as_string(
    loader,
    node,
):
    return loader.construct_scalar(node)


OpenAPIStringKeyLoader.add_constructor(
    BaseResolver.DEFAULT_MAPPING_TAG,
    construct_string_key_mapping,
)

OpenAPIStringKeyLoader.add_constructor(
    "tag:yaml.org,2002:timestamp",
    construct_timestamp_as_string,
)


def remap_ref(reference):
    mappings = {
        "#/definitions/": (
            "#/components/schemas/"
        ),
        "#/parameters/": (
            "#/components/parameters/"
        ),
        "#/responses/": (
            "#/components/responses/"
        ),
    }

    for old_prefix, new_prefix in (
        mappings.items()
    ):
        if reference.startswith(
            old_prefix
        ):
            return (
                new_prefix
                + reference[
                    len(old_prefix):
                ]
            )

    return reference


def normalize_refs(value):
    if isinstance(value, list):
        return [
            normalize_refs(child)
            for child in value
        ]

    if not isinstance(value, dict):
        return value

    result = {}

    for key, child in value.items():
        if (
            key == "$ref"
            and isinstance(child, str)
        ):
            result[key] = remap_ref(
                child
            )
        else:
            result[key] = (
                normalize_refs(child)
            )

    return result


def normalize_schema(value):
    if isinstance(value, list):
        return [
            normalize_schema(child)
            for child in value
        ]

    if not isinstance(value, dict):
        return value

    result = {}
    nullable_value = None

    for key, child in value.items():
        if (
            key == "$ref"
            and isinstance(child, str)
        ):
            result[key] = remap_ref(
                child
            )

        elif key == "x-nullable":
            nullable_value = bool(child)

        elif (
            key == "discriminator"
            and isinstance(child, str)
        ):
            result[key] = {
                "propertyName": child,
            }

        else:
            result[key] = (
                normalize_schema(child)
            )

    if nullable_value is not None:
        result["nullable"] = (
            nullable_value
        )

    if (
        "$ref" in result
        and len(result) > 1
    ):
        reference = result.pop("$ref")

        existing_all_of = result.pop(
            "allOf",
            [],
        )

        result["allOf"] = (
            [
                {
                    "$ref": reference,
                }
            ]
            + existing_all_of
        )

    return result


def resolve_pointer(
    document,
    reference,
):
    if not isinstance(reference, str):
        return None

    if not reference.startswith("#/"):
        return None

    current = document

    for token in reference[2:].split("/"):
        token = (
            token
            .replace("~1", "/")
            .replace("~0", "~")
        )

        if (
            isinstance(current, dict)
            and token in current
        ):
            current = current[token]
        else:
            return None

    return current


def dereference(
    document,
    value,
):
    current = value
    visited = set()

    while (
        isinstance(current, dict)
        and isinstance(
            current.get("$ref"),
            str,
        )
        and current["$ref"].startswith(
            "#/"
        )
    ):
        reference = current["$ref"]

        if reference in visited:
            return None

        visited.add(reference)

        resolved = resolve_pointer(
            document,
            reference,
        )

        if not isinstance(resolved, dict):
            return None

        siblings = {
            key: child
            for key, child in current.items()
            if key != "$ref"
        }

        current = {
            **resolved,
            **siblings,
        }

    return current


def operation_map(document):
    result = {}

    for path_name, path_item in (
        document.get("paths", {}).items()
    ):
        if not isinstance(path_item, dict):
            continue

        for method_name, operation in (
            path_item.items()
        ):
            method = method_name.lower()

            if (
                method in HTTP_METHODS
                and isinstance(
                    operation,
                    dict,
                )
            ):
                result[
                    (path_name, method)
                ] = operation

    return result


def effective_parameters(
    document,
    path_name,
    method,
):
    path_item = document["paths"][
        path_name
    ]

    operation = path_item[method]

    combined = (
        list(
            path_item.get(
                "parameters",
                [],
            )
        )
        + list(
            operation.get(
                "parameters",
                [],
            )
        )
    )

    result = {}

    for parameter in combined:
        resolved = dereference(
            document,
            parameter,
        )

        if not isinstance(resolved, dict):
            continue

        name = resolved.get("name")
        location = resolved.get("in")

        if (
            isinstance(name, str)
            and isinstance(location, str)
        ):
            result[
                (name, location)
            ] = resolved

    return result


def source_parameter_schema(
    parameter,
):
    result = {}

    for field in SCHEMA_FIELDS:
        if field in parameter:
            result[field] = (
                parameter[field]
            )

    if result.get("type") == "file":
        result["type"] = "string"
        result["format"] = "binary"

    return normalize_schema(result)


def expected_parameter_style(
    parameter,
):
    if parameter.get("type") != "array":
        return None

    location = parameter.get("in")

    collection_format = parameter.get(
        "collectionFormat",
        "csv",
    )

    if location == "query":
        mappings = {
            "csv": ("form", False),
            "ssv": (
                "spaceDelimited",
                False,
            ),
            "pipes": (
                "pipeDelimited",
                False,
            ),
            "multi": ("form", True),
        }

        return mappings.get(
            collection_format
        )

    if location in {
        "path",
        "header",
    }:
        return ("simple", False)

    return None


def observed_parameter_style(
    parameter,
):
    location = parameter.get("in")

    default_style = (
        "form"
        if location in {
            "query",
            "cookie",
        }
        else "simple"
    )

    style = parameter.get(
        "style",
        default_style,
    )

    default_explode = (
        style == "form"
    )

    explode = parameter.get(
        "explode",
        default_explode,
    )

    return style, bool(explode)


def effective_consumes(
    source,
    operation,
):
    value = operation.get(
        "consumes",
        source.get("consumes", []),
    )

    return (
        list(value)
        if isinstance(value, list)
        else []
    )


def effective_produces(
    source,
    operation,
):
    value = operation.get(
        "produces",
        source.get("produces", []),
    )

    return (
        list(value)
        if isinstance(value, list)
        else []
    )


def compare_nonbody_parameters(
    source,
    converted,
    path_name,
    method,
):
    source_parameters = (
        effective_parameters(
            source,
            path_name,
            method,
        )
    )

    converted_parameters = (
        effective_parameters(
            converted,
            path_name,
            method,
        )
    )

    source_nonbody = {
        identity: parameter
        for identity, parameter
        in source_parameters.items()
        if identity[1]
        not in {
            "body",
            "formData",
        }
    }

    converted_nonbody = {
        identity: parameter
        for identity, parameter
        in converted_parameters.items()
        if identity[1]
        not in {
            "body",
            "formData",
        }
    }

    failures = []

    if (
        set(source_nonbody)
        != set(converted_nonbody)
    ):
        failures.append(
            {
                "problem": (
                    "parameter_identity_set"
                ),
                "missing": sorted(
                    set(source_nonbody)
                    - set(converted_nonbody)
                ),
                "added": sorted(
                    set(converted_nonbody)
                    - set(source_nonbody)
                ),
            }
        )

    for identity in (
        set(source_nonbody)
        & set(converted_nonbody)
    ):
        source_parameter = (
            source_nonbody[identity]
        )

        converted_parameter = (
            converted_nonbody[identity]
        )

        source_required = bool(
            source_parameter.get(
                "required",
                False,
            )
        )

        converted_required = bool(
            converted_parameter.get(
                "required",
                False,
            )
        )

        if (
            source_required
            != converted_required
        ):
            failures.append(
                {
                    "parameter": identity,
                    "problem": "required",
                }
            )

        if (
            "description"
            in source_parameter
            and source_parameter[
                "description"
            ]
            != converted_parameter.get(
                "description"
            )
        ):
            failures.append(
                {
                    "parameter": identity,
                    "problem": "description",
                }
            )

        if (
            "allowEmptyValue"
            in source_parameter
            and bool(
                source_parameter[
                    "allowEmptyValue"
                ]
            )
            != bool(
                converted_parameter.get(
                    "allowEmptyValue",
                    False,
                )
            )
        ):
            failures.append(
                {
                    "parameter": identity,
                    "problem": (
                        "allowEmptyValue"
                    ),
                }
            )

        expected_schema = (
            source_parameter_schema(
                source_parameter
            )
        )

        observed_schema = (
            normalize_schema(
                converted_parameter.get(
                    "schema",
                    {},
                )
            )
        )

        if (
            expected_schema
            != observed_schema
        ):
            failures.append(
                {
                    "parameter": identity,
                    "problem": "schema",
                }
            )

        expected_style = (
            expected_parameter_style(
                source_parameter
            )
        )

        if (
            expected_style is not None
            and expected_style
            != observed_parameter_style(
                converted_parameter
            )
        ):
            failures.append(
                {
                    "parameter": identity,
                    "problem": (
                        "collection_serialization"
                    ),
                    "expected": (
                        expected_style
                    ),
                    "observed": (
                        observed_parameter_style(
                            converted_parameter
                        )
                    ),
                }
            )

        source_extensions = {
            key: value
            for key, value
            in source_parameter.items()
            if key.startswith("x-")
            and key != "x-nullable"
        }

        for key, value in (
            source_extensions.items()
        ):
            if (
                normalize_refs(value)
                != normalize_refs(
                    converted_parameter.get(
                        key
                    )
                )
            ):
                failures.append(
                    {
                        "parameter": identity,
                        "problem": key,
                    }
                )

    return failures


def compare_request_body(
    source,
    converted,
    path_name,
    method,
):
    source_operation = (
        source["paths"][path_name][method]
    )

    converted_operation = (
        converted[
            "paths"
        ][path_name][method]
    )

    source_parameters = (
        effective_parameters(
            source,
            path_name,
            method,
        )
    )

    body_parameters = [
        parameter
        for (name, location), parameter
        in source_parameters.items()
        if location == "body"
    ]

    form_parameters = [
        parameter
        for (name, location), parameter
        in source_parameters.items()
        if location == "formData"
    ]

    output_request_body = (
        converted_operation.get(
            "requestBody"
        )
    )

    if output_request_body is not None:
        output_request_body = (
            dereference(
                converted,
                output_request_body,
            )
        )

    failures = []

    if (
        not body_parameters
        and not form_parameters
    ):
        if output_request_body is not None:
            failures.append(
                {
                    "problem": (
                        "unexpected_request_body"
                    )
                }
            )

        return failures

    if not isinstance(
        output_request_body,
        dict,
    ):
        return [
            {
                "problem": (
                    "missing_request_body"
                )
            }
        ]

    output_content = (
        output_request_body.get(
            "content",
            {},
        )
    )

    consumes = effective_consumes(
        source,
        source_operation,
    )

    if body_parameters:
        if len(body_parameters) != 1:
            failures.append(
                {
                    "problem": (
                        "multiple_body_parameters"
                    )
                }
            )

            return failures

        body_parameter = (
            body_parameters[0]
        )

        if bool(
            body_parameter.get(
                "required",
                False,
            )
        ) != bool(
            output_request_body.get(
                "required",
                False,
            )
        ):
            failures.append(
                {
                    "problem": (
                        "body_required"
                    )
                }
            )

        if (
            "description"
            in body_parameter
            and body_parameter[
                "description"
            ]
            != output_request_body.get(
                "description"
            )
        ):
            failures.append(
                {
                    "problem": (
                        "body_description"
                    )
                }
            )

        expected_schema = (
            normalize_schema(
                body_parameter.get(
                    "schema",
                    {},
                )
            )
        )

        if consumes:
            if set(output_content) != set(
                consumes
            ):
                failures.append(
                    {
                        "problem": (
                            "request_media_types"
                        ),
                        "expected": sorted(
                            consumes
                        ),
                        "observed": sorted(
                            output_content
                        ),
                    }
                )

            media_types = [
                media_type
                for media_type in consumes
                if media_type
                in output_content
            ]
        else:
            media_types = list(
                output_content
            )

        if not media_types:
            failures.append(
                {
                    "problem": (
                        "missing_request_content"
                    )
                }
            )

        for media_type in media_types:
            observed_schema = (
                normalize_schema(
                    output_content[
                        media_type
                    ].get(
                        "schema",
                        {},
                    )
                )
            )

            if (
                expected_schema
                != observed_schema
            ):
                failures.append(
                    {
                        "problem": (
                            "body_schema"
                        ),
                        "media_type": (
                            media_type
                        ),
                    }
                )

        body_name = body_parameter.get(
            "name"
        )

        if (
            isinstance(body_name, str)
            and converted_operation.get(
                "x-original-body-parameter-name"
            )
            != body_name
        ):
            failures.append(
                {
                    "problem": (
                        "body_name_trace"
                    )
                }
            )

    if form_parameters:
        if consumes:
            if set(output_content) != set(
                consumes
            ):
                failures.append(
                    {
                        "problem": (
                            "form_media_types"
                        ),
                        "expected": sorted(
                            consumes
                        ),
                        "observed": sorted(
                            output_content
                        ),
                    }
                )

            media_types = [
                media_type
                for media_type in consumes
                if media_type
                in output_content
            ]
        else:
            media_types = list(
                output_content
            )

        if not media_types:
            failures.append(
                {
                    "problem": (
                        "missing_form_content"
                    )
                }
            )

        expected_required = sorted(
            parameter["name"]
            for parameter in form_parameters
            if bool(
                parameter.get(
                    "required",
                    False,
                )
            )
        )

        for media_type in media_types:
            media = output_content[
                media_type
            ]

            schema = media.get(
                "schema",
                {},
            )

            properties = schema.get(
                "properties",
                {},
            )

            observed_required = sorted(
                schema.get(
                    "required",
                    [],
                )
            )

            if (
                expected_required
                != observed_required
            ):
                failures.append(
                    {
                        "problem": (
                            "form_required_fields"
                        ),
                        "media_type": (
                            media_type
                        ),
                    }
                )

            source_names = {
                parameter["name"]
                for parameter in form_parameters
            }

            if set(properties) != source_names:
                failures.append(
                    {
                        "problem": (
                            "form_property_set"
                        ),
                        "media_type": (
                            media_type
                        ),
                    }
                )

            for parameter in (
                form_parameters
            ):
                name = parameter["name"]

                if name not in properties:
                    continue

                expected_property = (
                    source_parameter_schema(
                        parameter
                    )
                )

                if (
                    "description"
                    in parameter
                ):
                    expected_property[
                        "description"
                    ] = parameter[
                        "description"
                    ]

                observed_property = (
                    normalize_schema(
                        properties[name]
                    )
                )

                if (
                    expected_property
                    != observed_property
                ):
                    failures.append(
                        {
                            "problem": (
                                "form_property_schema"
                            ),
                            "parameter": name,
                            "media_type": (
                                media_type
                            ),
                        }
                    )

                expected_style = (
                    expected_parameter_style(
                        parameter
                    )
                )

                if (
                    expected_style
                    is not None
                ):
                    encoding = (
                        media.get(
                            "encoding",
                            {},
                        ).get(
                            name,
                            {},
                        )
                    )

                    observed_style = (
                        encoding.get(
                            "style",
                            "form",
                        ),
                        bool(
                            encoding.get(
                                "explode",
                                True,
                            )
                        ),
                    )

                    if (
                        expected_style
                        != observed_style
                    ):
                        failures.append(
                            {
                                "problem": (
                                    "form_serialization"
                                ),
                                "parameter": name,
                            }
                        )

    return failures


def response_headers_equal(
    source_headers,
    converted_headers,
):
    failures = []

    if set(source_headers) != set(
        converted_headers
    ):
        failures.append(
            "header_name_set"
        )

    for name in (
        set(source_headers)
        & set(converted_headers)
    ):
        source_header = (
            source_headers[name]
        )

        converted_header = (
            converted_headers[name]
        )

        if not isinstance(
            converted_header,
            dict,
        ):
            failures.append(
                f"{name}: invalid output header"
            )
            continue

        if (
            "description"
            in source_header
            and source_header[
                "description"
            ]
            != converted_header.get(
                "description"
            )
        ):
            failures.append(
                f"{name}: description"
            )

        expected_schema = (
            source_parameter_schema(
                source_header
            )
        )

        observed_schema = (
            normalize_schema(
                converted_header.get(
                    "schema",
                    {},
                )
            )
        )

        if (
            expected_schema
            != observed_schema
        ):
            failures.append(
                f"{name}: schema"
            )

    return failures


def compare_responses(
    source,
    converted,
    path_name,
    method,
):
    source_operation = (
        source["paths"][path_name][method]
    )

    converted_operation = (
        converted[
            "paths"
        ][path_name][method]
    )

    source_responses = (
        source_operation.get(
            "responses",
            {},
        )
    )

    converted_responses = (
        converted_operation.get(
            "responses",
            {},
        )
    )

    produces = effective_produces(
        source,
        source_operation,
    )

    failures = []

    for status_code in (
        set(source_responses)
        & set(converted_responses)
    ):
        source_response = dereference(
            source,
            source_responses[
                status_code
            ],
        )

        converted_response = dereference(
            converted,
            converted_responses[
                status_code
            ],
        )

        if (
            not isinstance(
                source_response,
                dict,
            )
            or not isinstance(
                converted_response,
                dict,
            )
        ):
            failures.append(
                {
                    "status": status_code,
                    "problem": (
                        "unresolvable_response"
                    ),
                }
            )
            continue

        source_schema = (
            source_response.get(
                "schema"
            )
        )

        output_content = (
            converted_response.get(
                "content",
                {},
            )
        )

        if source_schema is not None:
            expected_schema = (
                normalize_schema(
                    source_schema
                )
            )

            if produces:
                if set(output_content) != set(
                    produces
                ):
                    failures.append(
                        {
                            "status": status_code,
                            "problem": (
                                "response_media_types"
                            ),
                        }
                    )

                media_types = [
                    media_type
                    for media_type in produces
                    if media_type
                    in output_content
                ]
            else:
                media_types = list(
                    output_content
                )

            if not media_types:
                failures.append(
                    {
                        "status": status_code,
                        "problem": (
                            "missing_response_content"
                        ),
                    }
                )

            for media_type in media_types:
                observed_schema = (
                    normalize_schema(
                        output_content[
                            media_type
                        ].get(
                            "schema",
                            {},
                        )
                    )
                )

                if (
                    expected_schema
                    != observed_schema
                ):
                    failures.append(
                        {
                            "status": status_code,
                            "problem": (
                                "response_schema"
                            ),
                            "media_type": (
                                media_type
                            ),
                        }
                    )

        source_examples = (
            source_response.get(
                "examples",
                {},
            )
        )

        for media_type, example in (
            source_examples.items()
        ):
            observed_example = (
                output_content.get(
                    media_type,
                    {},
                ).get("example")
            )

            if example != observed_example:
                failures.append(
                    {
                        "status": status_code,
                        "problem": (
                            "response_example"
                        ),
                        "media_type": (
                            media_type
                        ),
                    }
                )

        header_failures = (
            response_headers_equal(
                source_response.get(
                    "headers",
                    {},
                ),
                converted_response.get(
                    "headers",
                    {},
                ),
            )
        )

        for header_failure in (
            header_failures
        ):
            failures.append(
                {
                    "status": status_code,
                    "problem": (
                        "response_header"
                    ),
                    "detail": (
                        header_failure
                    ),
                }
            )

    return failures


def expected_security_scheme(
    scheme,
):
    scheme_type = scheme.get("type")
    expected = {}

    if "description" in scheme:
        expected["description"] = (
            scheme["description"]
        )

    for key, value in scheme.items():
        if key.startswith("x-"):
            expected[key] = (
                normalize_refs(value)
            )

    if scheme_type == "apiKey":
        expected.update(
            {
                "type": "apiKey",
                "name": scheme.get(
                    "name"
                ),
                "in": scheme.get("in"),
            }
        )

    elif scheme_type == "basic":
        expected.update(
            {
                "type": "http",
                "scheme": "basic",
            }
        )

    elif scheme_type == "oauth2":
        flow_name = scheme.get("flow")

        flow_mapping = {
            "implicit": "implicit",
            "password": "password",
            "application": (
                "clientCredentials"
            ),
            "accessCode": (
                "authorizationCode"
            ),
        }

        converted_flow_name = (
            flow_mapping.get(flow_name)
        )

        flow = {
            "scopes": scheme.get(
                "scopes",
                {},
            )
        }

        if flow_name in {
            "implicit",
            "accessCode",
        }:
            flow["authorizationUrl"] = (
                scheme.get(
                    "authorizationUrl"
                )
            )

        if flow_name in {
            "password",
            "application",
            "accessCode",
        }:
            flow["tokenUrl"] = (
                scheme.get("tokenUrl")
            )

        expected.update(
            {
                "type": "oauth2",
                "flows": {
                    converted_flow_name: (
                        flow
                    )
                },
            }
        )

    else:
        expected = None

    return expected


def compare_security_schemes(
    source,
    converted,
):
    source_schemes = source.get(
        "securityDefinitions",
        {},
    )

    converted_schemes = (
        converted
        .get("components", {})
        .get("securitySchemes", {})
    )

    failures = []

    if set(source_schemes) != set(
        converted_schemes
    ):
        failures.append(
            {
                "problem": (
                    "security_scheme_name_set"
                )
            }
        )

    for name in (
        set(source_schemes)
        & set(converted_schemes)
    ):
        expected = (
            expected_security_scheme(
                source_schemes[name]
            )
        )

        observed = normalize_refs(
            converted_schemes[name]
        )

        if expected is None:
            failures.append(
                {
                    "scheme": name,
                    "problem": (
                        "unsupported_source_scheme"
                    ),
                }
            )

        elif expected != observed:
            failures.append(
                {
                    "scheme": name,
                    "problem": (
                        "security_scheme_mapping"
                    ),
                }
            )

    return failures


def audit_contract(
    source,
    converted,
):
    categories = {}
    diagnostics = {}

    source_operations = operation_map(
        source
    )

    converted_operations = operation_map(
        converted
    )

    common_operations = sorted(
        set(source_operations)
        & set(converted_operations)
    )

    parameter_failures = []
    request_failures = []
    response_failures = []

    for path_name, method in (
        common_operations
    ):
        label = (
            f"{method.upper()} {path_name}"
        )

        observed_parameter_failures = (
            compare_nonbody_parameters(
                source,
                converted,
                path_name,
                method,
            )
        )

        for failure in (
            observed_parameter_failures
        ):
            parameter_failures.append(
                {
                    "operation": label,
                    **failure,
                }
            )

        observed_request_failures = (
            compare_request_body(
                source,
                converted,
                path_name,
                method,
            )
        )

        for failure in (
            observed_request_failures
        ):
            request_failures.append(
                {
                    "operation": label,
                    **failure,
                }
            )

        observed_response_failures = (
            compare_responses(
                source,
                converted,
                path_name,
                method,
            )
        )

        for failure in (
            observed_response_failures
        ):
            response_failures.append(
                {
                    "operation": label,
                    **failure,
                }
            )

    security_failures = (
        compare_security_schemes(
            source,
            converted,
        )
    )

    categories[
        "nonbody_parameters_preserved"
    ] = not parameter_failures

    categories[
        "request_contracts_preserved"
    ] = not request_failures

    categories[
        "response_contracts_preserved"
    ] = not response_failures

    categories[
        "security_schemes_preserved"
    ] = not security_failures

    if parameter_failures:
        diagnostics[
            "nonbody_parameters_preserved"
        ] = parameter_failures[:50]

    if request_failures:
        diagnostics[
            "request_contracts_preserved"
        ] = request_failures[:50]

    if response_failures:
        diagnostics[
            "response_contracts_preserved"
        ] = response_failures[:50]

    if security_failures:
        diagnostics[
            "security_schemes_preserved"
        ] = security_failures[:50]

    return categories, diagnostics


required_paths = [
    MANIFEST_PATH,
    VALIDATION_RESULTS_PATH,
    SEMANTIC_RESULTS_PATH,
    SEMANTIC_SUMMARY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Missing contract-audit inputs:\n"
        + "\n".join(missing_paths)
    )

semantic_summary = json.loads(
    SEMANTIC_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

if semantic_summary.get("status") != "PASS":
    raise RuntimeError(
        "Semantic-v2 audit status is not PASS."
    )

if (
    sha256_file(
        SEMANTIC_RESULTS_PATH
    )
    != semantic_summary[
        "results_sha256"
    ]
):
    raise RuntimeError(
        "Semantic-v2 results hash mismatch."
    )

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

manifest_by_spec_id = {
    record["spec_id"]: record
    for record in manifest["records"]
}

validation_by_spec_id = {}

with VALIDATION_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line in file:
        line = line.strip()

        if line:
            record = json.loads(line)

            validation_by_spec_id[
                record["spec_id"]
            ] = record

semantic_candidates = []

with SEMANTIC_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        if record.get(
            "representation_aware_pass"
        ) is True:
            semantic_candidates.append(
                record
            )

semantic_candidates = sorted(
    semantic_candidates,
    key=lambda item: (
        item["relative_path"],
        item["spec_id"],
    ),
)

if len(semantic_candidates) != 226:
    raise RuntimeError(
        "Expected 226 deep-audit candidates; "
        f"observed {len(semantic_candidates)}."
    )

started = time.perf_counter()
results = []
resumed_count = 0

for position, semantic_record in enumerate(
    semantic_candidates,
    start=1,
):
    spec_id = semantic_record[
        "spec_id"
    ]

    manifest_record = (
        manifest_by_spec_id[spec_id]
    )

    validation_record = (
        validation_by_spec_id[spec_id]
    )

    source_path = (
        CORPUS_ROOT
        / manifest_record[
            "relative_path"
        ]
    )

    converted_path = Path(
        validation_record[
            "converted_path"
        ]
    )

    checkpoint_path = (
        ITEM_ROOT
        / f"{spec_id}.json"
    )

    source_sha256 = manifest_record[
        "source_sha256"
    ]

    converted_sha256 = (
        validation_record[
            "converted_sha256"
        ]
    )

    reusable = None

    if checkpoint_path.is_file():
        try:
            candidate = json.loads(
                checkpoint_path.read_text(
                    encoding="utf-8"
                )
            )

            if (
                candidate.get(
                    "audit_version"
                )
                == AUDIT_VERSION
                and candidate.get(
                    "source_sha256"
                )
                == source_sha256
                and candidate.get(
                    "converted_sha256"
                )
                == converted_sha256
            ):
                reusable = candidate
        except Exception:
            reusable = None

    if reusable is not None:
        results.append(reusable)
        resumed_count += 1

    else:
        categories = {}
        diagnostics = {}

        try:
            source = yaml.load(
                source_path.read_text(
                    encoding="utf-8"
                ),
                Loader=(
                    OpenAPIStringKeyLoader
                ),
            )

            converted = json.loads(
                converted_path.read_text(
                    encoding="utf-8"
                )
            )

            (
                categories,
                diagnostics,
            ) = audit_contract(
                source,
                converted,
            )

        except Exception as exception:
            categories = {
                "contract_comparison": False,
            }

            diagnostics = {
                "contract_comparison": (
                    f"{type(exception).__name__}: "
                    f"{exception}"
                )[:12000],
            }

        contract_equivalence_pass = all(
            categories.values()
        )

        result = {
            "audit_version": (
                AUDIT_VERSION
            ),
            "spec_id": spec_id,
            "raw_id": manifest_record[
                "raw_id"
            ],
            "relative_path": (
                manifest_record[
                    "relative_path"
                ]
            ),
            "provider_directory": (
                manifest_record[
                    "provider_directory"
                ]
            ),
            "source_sha256": (
                source_sha256
            ),
            "converted_sha256": (
                converted_sha256
            ),
            "category_results": categories,
            "failed_categories": sorted(
                category
                for category, passed
                in categories.items()
                if not passed
            ),
            "diagnostics": diagnostics,
            "contract_equivalence_pass": (
                contract_equivalence_pass
            ),
        }

        write_json_atomic(
            checkpoint_path,
            result,
        )

        results.append(result)

    if (
        position % 25 == 0
        or position
        == len(semantic_candidates)
    ):
        elapsed = (
            time.perf_counter()
            - started
        )

        rate = (
            position / elapsed
            if elapsed > 0
            else 0
        )

        remaining = (
            len(semantic_candidates)
            - position
        )

        eta = (
            remaining / rate
            if rate > 0
            else None
        )

        print(
            json.dumps(
                {
                    "progress": position,
                    "total": len(
                        semantic_candidates
                    ),
                    "resumed": resumed_count,
                    "elapsed_seconds": round(
                        elapsed,
                        1,
                    ),
                    "rate_per_second": round(
                        rate,
                        3,
                    ),
                    "eta_seconds": (
                        round(eta, 1)
                        if eta is not None
                        else None
                    ),
                }
            ),
            flush=True,
        )

results = sorted(
    results,
    key=lambda item: (
        item["relative_path"],
        item["spec_id"],
    ),
)

results_text = (
    "\n".join(
        json.dumps(
            result,
            sort_keys=True,
            ensure_ascii=False,
        )
        for result in results
    )
    + "\n"
)

temporary_results_path = (
    RESULTS_PATH.with_suffix(
        ".jsonl.tmp"
    )
)

temporary_results_path.write_text(
    results_text,
    encoding="utf-8",
)

temporary_results_path.replace(
    RESULTS_PATH
)

pass_count = sum(
    bool(
        result[
            "contract_equivalence_pass"
        ]
    )
    for result in results
)

category_failure_counts = Counter()

for result in results:
    category_failure_counts.update(
        result["failed_categories"]
    )

failure_examples = [
    {
        "spec_id": result["spec_id"],
        "relative_path": (
            result["relative_path"]
        ),
        "failed_categories": (
            result["failed_categories"]
        ),
        "diagnostics": (
            result["diagnostics"]
        ),
    }
    for result in results
    if not result[
        "contract_equivalence_pass"
    ]
][:20]

summary = {
    "status": (
        "PASS"
        if len(results) == 226
        else "BLOCKED"
    ),
    "audit_version": AUDIT_VERSION,
    "audited_specification_count": len(
        results
    ),
    "resumed_checkpoint_count": (
        resumed_count
    ),
    "contract_equivalence_pass_count": (
        pass_count
    ),
    "contract_equivalence_failure_count": (
        len(results) - pass_count
    ),
    "category_failure_counts": {
        str(key): int(value)
        for key, value in sorted(
            category_failure_counts.items()
        )
    },
    "failure_examples": (
        failure_examples
    ),
    "duration_seconds": round(
        time.perf_counter() - started,
        3,
    ),
    "results_path": str(
        RESULTS_PATH
    ),
    "results_sha256": sha256_file(
        RESULTS_PATH
    ),
}

write_json_atomic(
    SUMMARY_PATH,
    summary,
)

output = {
    **summary,
    "summary_path": str(
        SUMMARY_PATH
    ),
    "summary_sha256": sha256_file(
        SUMMARY_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

if summary["status"] != "PASS":
    raise RuntimeError(
        "Contract audit did not account "
        "for every candidate."
    )

print("\nNOTEBOOK 01 — CELL 21: PASS")

{"progress": 25, "total": 226, "resumed": 0, "elapsed_seconds": 0.2, "rate_per_second": 103.451, "eta_seconds": 1.9}
{"progress": 50, "total": 226, "resumed": 0, "elapsed_seconds": 0.4, "rate_per_second": 119.364, "eta_seconds": 1.5}
{"progress": 75, "total": 226, "resumed": 0, "elapsed_seconds": 0.6, "rate_per_second": 129.066, "eta_seconds": 1.2}
{"progress": 100, "total": 226, "resumed": 0, "elapsed_seconds": 2.0, "rate_per_second": 48.925, "eta_seconds": 2.6}
{"progress": 125, "total": 226, "resumed": 0, "elapsed_seconds": 2.3, "rate_per_second": 55.068, "eta_seconds": 1.8}
{"progress": 150, "total": 226, "resumed": 0, "elapsed_seconds": 2.5, "rate_per_second": 59.036, "eta_seconds": 1.3}
{"progress": 175, "total": 226, "resumed": 0, "elapsed_seconds": 2.7, "rate_per_second": 64.894, "eta_seconds": 0.8}
{"progress": 200, "total": 226, "resumed": 0, "elapsed_seconds": 2.9, "rate_per_second": 69.433, "eta_seconds": 0.4}
{"progress": 225, "total": 226, "resumed": 0, "elapsed_seconds":

In [24]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import yaml
from yaml.constructor import ConstructorError
from yaml.nodes import MappingNode, ScalarNode
from yaml.resolver import BaseResolver

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "swagger_conversion_manifest.json"
)

VALIDATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_converted_validation.jsonl"
)

CONTRACT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_contract_audit_results.jsonl"
)

CONTRACT_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_contract_audit_summary.json"
)

ADJUSTED_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_contract_adjusted_results.jsonl"
)

ADJUSTED_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_contract_adjusted_summary.json"
)

ADJUSTMENT_VERSION = 1


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


loader_base = getattr(
    yaml,
    "CSafeLoader",
    yaml.SafeLoader,
)


class OpenAPIStringKeyLoader(loader_base):
    pass


def construct_string_key_mapping(
    loader,
    node,
    deep=False,
):
    if not isinstance(node, MappingNode):
        raise ConstructorError(
            None,
            None,
            "Expected mapping node.",
            node.start_mark,
        )

    loader.flatten_mapping(node)
    mapping = {}

    for key_node, value_node in node.value:
        if not isinstance(
            key_node,
            ScalarNode,
        ):
            raise ConstructorError(
                "while constructing mapping",
                node.start_mark,
                "Mapping key is not scalar.",
                key_node.start_mark,
            )

        key = key_node.value

        if key in mapping:
            raise ConstructorError(
                "while constructing mapping",
                node.start_mark,
                f"Duplicate key: {key!r}",
                key_node.start_mark,
            )

        mapping[key] = (
            loader.construct_object(
                value_node,
                deep=deep,
            )
        )

    return mapping


def construct_timestamp_as_string(
    loader,
    node,
):
    return loader.construct_scalar(node)


OpenAPIStringKeyLoader.add_constructor(
    BaseResolver.DEFAULT_MAPPING_TAG,
    construct_string_key_mapping,
)

OpenAPIStringKeyLoader.add_constructor(
    "tag:yaml.org,2002:timestamp",
    construct_timestamp_as_string,
)


def resolve_pointer(
    document,
    reference,
):
    if (
        not isinstance(reference, str)
        or not reference.startswith("#/")
    ):
        return None

    current = document

    for token in reference[2:].split("/"):
        token = (
            token
            .replace("~1", "/")
            .replace("~0", "~")
        )

        if (
            isinstance(current, dict)
            and token in current
        ):
            current = current[token]
        else:
            return None

    return current


def dereference(
    document,
    value,
):
    current = value
    visited = set()

    while (
        isinstance(current, dict)
        and isinstance(
            current.get("$ref"),
            str,
        )
        and current["$ref"].startswith(
            "#/"
        )
    ):
        reference = current["$ref"]

        if reference in visited:
            return None

        visited.add(reference)

        resolved = resolve_pointer(
            document,
            reference,
        )

        if not isinstance(resolved, dict):
            return None

        siblings = {
            key: child
            for key, child in current.items()
            if key != "$ref"
        }

        current = {
            **resolved,
            **siblings,
        }

    return current


def effective_parameters(
    document,
    path_name,
    method,
):
    path_item = document["paths"][
        path_name
    ]

    operation = path_item[method]

    combined = (
        list(
            path_item.get(
                "parameters",
                [],
            )
        )
        + list(
            operation.get(
                "parameters",
                [],
            )
        )
    )

    result = {}

    for parameter in combined:
        resolved = dereference(
            document,
            parameter,
        )

        if not isinstance(resolved, dict):
            continue

        name = resolved.get("name")
        location = resolved.get("in")

        if (
            isinstance(name, str)
            and isinstance(location, str)
        ):
            result[
                (name, location)
            ] = resolved

    return result


required_paths = [
    MANIFEST_PATH,
    VALIDATION_RESULTS_PATH,
    CONTRACT_RESULTS_PATH,
    CONTRACT_SUMMARY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Missing adjustment inputs:\n"
        + "\n".join(missing_paths)
    )

contract_summary = json.loads(
    CONTRACT_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

if contract_summary.get("status") != "PASS":
    raise RuntimeError(
        "Contract audit status is not PASS."
    )

if (
    sha256_file(CONTRACT_RESULTS_PATH)
    != contract_summary[
        "results_sha256"
    ]
):
    raise RuntimeError(
        "Contract-results hash mismatch."
    )

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

manifest_by_spec_id = {
    record["spec_id"]: record
    for record in manifest["records"]
}

validation_by_spec_id = {}

with VALIDATION_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line in file:
        line = line.strip()

        if line:
            record = json.loads(line)

            validation_by_spec_id[
                record["spec_id"]
            ] = record

contract_records = []

with CONTRACT_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line in file:
        line = line.strip()

        if line:
            contract_records.append(
                json.loads(line)
            )

if len(contract_records) != 226:
    raise RuntimeError(
        "Expected 226 contract records; "
        f"observed {len(contract_records)}."
    )

adjusted_results = []
mapping_records = []
assessment_problems = []

for contract_record in contract_records:
    spec_id = contract_record[
        "spec_id"
    ]

    failed_categories = set(
        contract_record.get(
            "failed_categories",
            [],
        )
    )

    adjusted_diagnostics = dict(
        contract_record.get(
            "diagnostics",
            {},
        )
    )

    parameter_failures = list(
        adjusted_diagnostics.get(
            "nonbody_parameters_preserved",
            [],
        )
    )

    x_example_failures = [
        failure
        for failure in parameter_failures
        if failure.get("problem")
        == "x-example"
    ]

    remaining_parameter_failures = [
        failure
        for failure in parameter_failures
        if failure.get("problem")
        != "x-example"
    ]

    if x_example_failures:
        manifest_record = (
            manifest_by_spec_id.get(
                spec_id
            )
        )

        validation_record = (
            validation_by_spec_id.get(
                spec_id
            )
        )

        if (
            manifest_record is None
            or validation_record is None
        ):
            assessment_problems.append(
                {
                    "spec_id": spec_id,
                    "problem": (
                        "missing provenance record"
                    ),
                }
            )

            remaining_parameter_failures.extend(
                x_example_failures
            )

        else:
            source_path = (
                CORPUS_ROOT
                / manifest_record[
                    "relative_path"
                ]
            )

            converted_path = Path(
                validation_record[
                    "converted_path"
                ]
            )

            try:
                source = yaml.load(
                    source_path.read_text(
                        encoding="utf-8"
                    ),
                    Loader=(
                        OpenAPIStringKeyLoader
                    ),
                )

                converted = json.loads(
                    converted_path.read_text(
                        encoding="utf-8"
                    )
                )

                source_parameter_cache = {}
                converted_parameter_cache = {}

                for failure in x_example_failures:
                    operation_label = failure[
                        "operation"
                    ]

                    method, path_name = (
                        operation_label.split(
                            " ",
                            1,
                        )
                    )

                    method = method.lower()

                    identity_value = failure[
                        "parameter"
                    ]

                    identity = (
                        str(identity_value[0]),
                        str(identity_value[1]),
                    )

                    cache_key = (
                        path_name,
                        method,
                    )

                    if (
                        cache_key
                        not in source_parameter_cache
                    ):
                        source_parameter_cache[
                            cache_key
                        ] = effective_parameters(
                            source,
                            path_name,
                            method,
                        )

                        converted_parameter_cache[
                            cache_key
                        ] = effective_parameters(
                            converted,
                            path_name,
                            method,
                        )

                    source_parameter = (
                        source_parameter_cache[
                            cache_key
                        ].get(identity)
                    )

                    converted_parameter = (
                        converted_parameter_cache[
                            cache_key
                        ].get(identity)
                    )

                    if (
                        not isinstance(
                            source_parameter,
                            dict,
                        )
                        or not isinstance(
                            converted_parameter,
                            dict,
                        )
                    ):
                        mapping_status = (
                            "PARAMETER_NOT_FOUND"
                        )

                        observed_value = None

                    else:
                        expected_value = (
                            source_parameter.get(
                                "x-example"
                            )
                        )

                        observed_value = (
                            converted_parameter.get(
                                "example"
                            )
                        )

                        if (
                            expected_value
                            == observed_value
                            and "example"
                            in converted_parameter
                        ):
                            mapping_status = (
                                "PRESERVED_AS_EXAMPLE"
                            )
                        else:
                            mapping_status = (
                                "VALUE_MISMATCH"
                            )

                    mapping_record = {
                        "spec_id": spec_id,
                        "relative_path": (
                            contract_record[
                                "relative_path"
                            ]
                        ),
                        "operation": (
                            operation_label
                        ),
                        "parameter": list(
                            identity
                        ),
                        "mapping": (
                            "x-example -> example"
                        ),
                        "status": (
                            mapping_status
                        ),
                    }

                    if (
                        isinstance(
                            source_parameter,
                            dict,
                        )
                    ):
                        mapping_record[
                            "source_x_example"
                        ] = (
                            source_parameter.get(
                                "x-example"
                            )
                        )

                    mapping_record[
                        "converted_example"
                    ] = observed_value

                    mapping_records.append(
                        mapping_record
                    )

                    if (
                        mapping_status
                        != "PRESERVED_AS_EXAMPLE"
                    ):
                        remaining_parameter_failures.append(
                            {
                                **failure,
                                "adjustment_status": (
                                    mapping_status
                                ),
                            }
                        )

            except Exception as exception:
                assessment_problems.append(
                    {
                        "spec_id": spec_id,
                        "problem": (
                            f"{type(exception).__name__}: "
                            f"{exception}"
                        )[:12000],
                    }
                )

                remaining_parameter_failures.extend(
                    x_example_failures
                )

    if remaining_parameter_failures:
        adjusted_diagnostics[
            "nonbody_parameters_preserved"
        ] = remaining_parameter_failures

        failed_categories.add(
            "nonbody_parameters_preserved"
        )

    else:
        adjusted_diagnostics.pop(
            "nonbody_parameters_preserved",
            None,
        )

        failed_categories.discard(
            "nonbody_parameters_preserved"
        )

    adjusted_pass = (
        len(failed_categories) == 0
    )

    adjusted_results.append(
        {
            **contract_record,
            "adjustment_version": (
                ADJUSTMENT_VERSION
            ),
            "adjustment": (
                "verified_x_example_promotion"
            ),
            "adjusted_failed_categories": (
                sorted(failed_categories)
            ),
            "adjusted_diagnostics": (
                adjusted_diagnostics
            ),
            "adjusted_contract_equivalence_pass": (
                adjusted_pass
            ),
        }
    )

if assessment_problems:
    raise RuntimeError(
        "Unable to assess every x-example mapping:\n"
        + json.dumps(
            assessment_problems[:20],
            indent=2,
            ensure_ascii=False,
        )
    )

adjusted_results = sorted(
    adjusted_results,
    key=lambda item: (
        item["relative_path"],
        item["spec_id"],
    ),
)

results_text = (
    "\n".join(
        json.dumps(
            result,
            sort_keys=True,
            ensure_ascii=False,
        )
        for result in adjusted_results
    )
    + "\n"
)

temporary_results_path = (
    ADJUSTED_RESULTS_PATH.with_suffix(
        ".jsonl.tmp"
    )
)

temporary_results_path.write_text(
    results_text,
    encoding="utf-8",
)

temporary_results_path.replace(
    ADJUSTED_RESULTS_PATH
)

original_pass_count = sum(
    bool(
        record.get(
            "contract_equivalence_pass"
        )
    )
    for record in contract_records
)

adjusted_pass_count = sum(
    bool(
        record.get(
            "adjusted_contract_equivalence_pass"
        )
    )
    for record in adjusted_results
)

mapping_status_counts = Counter(
    record["status"]
    for record in mapping_records
)

remaining_failure_counts = Counter()

for result in adjusted_results:
    remaining_failure_counts.update(
        result[
            "adjusted_failed_categories"
        ]
    )

remaining_failure_examples = [
    {
        "spec_id": result["spec_id"],
        "relative_path": (
            result["relative_path"]
        ),
        "failed_categories": (
            result[
                "adjusted_failed_categories"
            ]
        ),
        "diagnostics": (
            result[
                "adjusted_diagnostics"
            ]
        ),
    }
    for result in adjusted_results
    if not result[
        "adjusted_contract_equivalence_pass"
    ]
][:20]

summary = {
    "status": "PASS",
    "adjustment_version": (
        ADJUSTMENT_VERSION
    ),
    "adjustment": (
        "VERIFY_X_EXAMPLE_TO_EXAMPLE_PROMOTION"
    ),
    "documentation": (
        "OAS-Kit documents x-example within "
        "parameters as converted to example."
    ),
    "audited_specification_count": len(
        adjusted_results
    ),
    "original_contract_pass_count": (
        original_pass_count
    ),
    "x_example_mapping_count": len(
        mapping_records
    ),
    "x_example_mapping_status_counts": {
        str(key): int(value)
        for key, value in sorted(
            mapping_status_counts.items()
        )
    },
    "adjusted_contract_pass_count": (
        adjusted_pass_count
    ),
    "newly_accepted_specification_count": (
        adjusted_pass_count
        - original_pass_count
    ),
    "adjusted_contract_failure_count": (
        len(adjusted_results)
        - adjusted_pass_count
    ),
    "remaining_failure_category_counts": {
        str(key): int(value)
        for key, value in sorted(
            remaining_failure_counts.items()
        )
    },
    "remaining_failure_examples": (
        remaining_failure_examples
    ),
    "mapping_records": mapping_records,
    "results_path": str(
        ADJUSTED_RESULTS_PATH
    ),
    "results_sha256": sha256_file(
        ADJUSTED_RESULTS_PATH
    ),
}

write_json_atomic(
    ADJUSTED_SUMMARY_PATH,
    summary,
)

printed_output = {
    key: value
    for key, value in summary.items()
    if key not in {
        "mapping_records",
        "remaining_failure_examples",
    }
}

printed_output[
    "remaining_failure_examples"
] = remaining_failure_examples[:10]

printed_output["summary_path"] = str(
    ADJUSTED_SUMMARY_PATH
)

printed_output["summary_sha256"] = (
    sha256_file(
        ADJUSTED_SUMMARY_PATH
    )
)

print(
    json.dumps(
        printed_output,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 22: PASS")

{
  "status": "PASS",
  "adjustment_version": 1,
  "adjustment": "VERIFY_X_EXAMPLE_TO_EXAMPLE_PROMOTION",
  "documentation": "OAS-Kit documents x-example within parameters as converted to example.",
  "audited_specification_count": 226,
  "original_contract_pass_count": 149,
  "x_example_mapping_count": 165,
  "x_example_mapping_status_counts": {
    "PRESERVED_AS_EXAMPLE": 163,
    "VALUE_MISMATCH": 2
  },
  "adjusted_contract_pass_count": 171,
  "newly_accepted_specification_count": 22,
  "adjusted_contract_failure_count": 55,
  "remaining_failure_category_counts": {
    "nonbody_parameters_preserved": 12,
    "request_contracts_preserved": 14,
    "response_contracts_preserved": 36,
    "security_schemes_preserved": 1
  },
  "results_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/audits/swagger_contract_adjusted_results.jsonl",
  "results_sha256": "1627f3986358ad3b482d4e08d38e8e341a35147b99e1b7e97e448c763d3bec1d",
  "remaining_failure_examples": [
    {
     

In [25]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import yaml
from yaml.constructor import ConstructorError
from yaml.nodes import MappingNode, ScalarNode
from yaml.resolver import BaseResolver

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "swagger_conversion_manifest.json"
)

SEMANTIC_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_semantic_v2_results.jsonl"
)

CONTRACT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "swagger_contract_adjusted_results.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "swagger_extension_transformation_taxonomy.json"
)

DOCUMENTED_EXTENSIONS = {
    "x-ms-paths",
    "x-ms-skip-url-encoding",
    "x-ms-odata",
    "x-ms-parameterized-host",
    "x-amazon-apigateway-any-method",
    "x-servers",
    "x-anyOf",
    "x-oneOf",
    "x-not",
    "x-required",
    "x-deprecated",
    "x-links",
    "x-callbacks",
    "x-example",
}


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


loader_base = getattr(
    yaml,
    "CSafeLoader",
    yaml.SafeLoader,
)


class OpenAPIStringKeyLoader(loader_base):
    pass


def construct_string_key_mapping(
    loader,
    node,
    deep=False,
):
    if not isinstance(node, MappingNode):
        raise ConstructorError(
            None,
            None,
            "Expected mapping node.",
            node.start_mark,
        )

    loader.flatten_mapping(node)
    mapping = {}

    for key_node, value_node in node.value:
        if not isinstance(
            key_node,
            ScalarNode,
        ):
            raise ConstructorError(
                "while constructing mapping",
                node.start_mark,
                "Mapping key is not scalar.",
                key_node.start_mark,
            )

        key = key_node.value

        if key in mapping:
            raise ConstructorError(
                "while constructing mapping",
                node.start_mark,
                f"Duplicate key: {key!r}",
                key_node.start_mark,
            )

        mapping[key] = (
            loader.construct_object(
                value_node,
                deep=deep,
            )
        )

    return mapping


def construct_timestamp_as_string(
    loader,
    node,
):
    return loader.construct_scalar(node)


OpenAPIStringKeyLoader.add_constructor(
    BaseResolver.DEFAULT_MAPPING_TAG,
    construct_string_key_mapping,
)

OpenAPIStringKeyLoader.add_constructor(
    "tag:yaml.org,2002:timestamp",
    construct_timestamp_as_string,
)


def collect_documented_extensions(
    document,
):
    counter = Counter()
    stack = [document]

    while stack:
        current = stack.pop()

        if isinstance(current, dict):
            for key, value in current.items():
                if key in DOCUMENTED_EXTENSIONS:
                    counter[key] += 1

                stack.append(value)

        elif isinstance(current, list):
            stack.extend(current)

    return counter


required_paths = [
    MANIFEST_PATH,
    SEMANTIC_RESULTS_PATH,
    CONTRACT_RESULTS_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Missing taxonomy inputs:\n"
        + "\n".join(missing_paths)
    )

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

manifest_by_spec_id = {
    record["spec_id"]: record
    for record in manifest["records"]
}

semantic_records = []

with SEMANTIC_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line in file:
        line = line.strip()

        if line:
            semantic_records.append(
                json.loads(line)
            )

contract_records = []

with CONTRACT_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line in file:
        line = line.strip()

        if line:
            contract_records.append(
                json.loads(line)
            )

semantic_failures = [
    record
    for record in semantic_records
    if not record.get(
        "representation_aware_pass",
        False,
    )
]

contract_failures = [
    record
    for record in contract_records
    if not record.get(
        "adjusted_contract_equivalence_pass",
        False,
    )
]

semantic_category_counts = Counter()
operation_metadata_field_counts = Counter()
top_level_field_counts = Counter()

for record in semantic_failures:
    semantic_category_counts.update(
        record.get(
            "failed_categories",
            [],
        )
    )

    diagnostics = record.get(
        "diagnostics",
        {},
    )

    for failure in diagnostics.get(
        "operation_metadata_preserved",
        [],
    ):
        field = failure.get("field")

        if field:
            operation_metadata_field_counts[
                field
            ] += 1

    for field in diagnostics.get(
        "top_level_metadata_preserved",
        [],
    ):
        top_level_field_counts[
            str(field)
        ] += 1

contract_category_counts = Counter()
contract_problem_counts = Counter()

for record in contract_failures:
    failed_categories = record.get(
        "adjusted_failed_categories",
        [],
    )

    contract_category_counts.update(
        failed_categories
    )

    diagnostics = record.get(
        "adjusted_diagnostics",
        {},
    )

    for category in failed_categories:
        failures = diagnostics.get(
            category,
            [],
        )

        if isinstance(failures, list):
            for failure in failures:
                if isinstance(
                    failure,
                    dict,
                ):
                    problem = failure.get(
                        "problem",
                        "UNSPECIFIED",
                    )
                else:
                    problem = str(failure)

                contract_problem_counts[
                    f"{category}::{problem}"
                ] += 1

extension_document_counts = Counter()
extension_occurrence_counts = Counter()
parse_problems = []

for position, record in enumerate(
    semantic_failures,
    start=1,
):
    spec_id = record["spec_id"]

    manifest_record = (
        manifest_by_spec_id.get(spec_id)
    )

    if manifest_record is None:
        parse_problems.append(
            {
                "spec_id": spec_id,
                "problem": (
                    "missing manifest record"
                ),
            }
        )
        continue

    source_path = (
        CORPUS_ROOT
        / manifest_record[
            "relative_path"
        ]
    )

    try:
        source = yaml.load(
            source_path.read_text(
                encoding="utf-8"
            ),
            Loader=(
                OpenAPIStringKeyLoader
            ),
        )

        extension_counts = (
            collect_documented_extensions(
                source
            )
        )

        for extension, count in (
            extension_counts.items()
        ):
            extension_document_counts[
                extension
            ] += 1

            extension_occurrence_counts[
                extension
            ] += count

    except Exception as exception:
        parse_problems.append(
            {
                "spec_id": spec_id,
                "relative_path": record[
                    "relative_path"
                ],
                "problem": (
                    f"{type(exception).__name__}: "
                    f"{exception}"
                )[:2000],
            }
        )

    if (
        position % 100 == 0
        or position
        == len(semantic_failures)
    ):
        print(
            json.dumps(
                {
                    "progress": position,
                    "total": len(
                        semantic_failures
                    ),
                }
            ),
            flush=True,
        )

if parse_problems:
    raise RuntimeError(
        "Unable to inspect every failed source:\n"
        + json.dumps(
            parse_problems[:20],
            indent=2,
            ensure_ascii=False,
        )
    )

report = {
    "status": "PASS",
    "semantic_failure_specification_count": (
        len(semantic_failures)
    ),
    "contract_failure_specification_count": (
        len(contract_failures)
    ),
    "semantic_failure_category_counts": {
        str(key): int(value)
        for key, value in sorted(
            semantic_category_counts.items()
        )
    },
    "operation_metadata_field_counts": {
        str(key): int(value)
        for key, value in sorted(
            operation_metadata_field_counts.items()
        )
    },
    "top_level_metadata_field_counts": {
        str(key): int(value)
        for key, value in sorted(
            top_level_field_counts.items()
        )
    },
    "contract_failure_category_counts": {
        str(key): int(value)
        for key, value in sorted(
            contract_category_counts.items()
        )
    },
    "contract_problem_counts": {
        str(key): int(value)
        for key, value in sorted(
            contract_problem_counts.items()
        )
    },
    "documented_extension_document_counts": {
        str(key): int(value)
        for key, value in sorted(
            extension_document_counts.items()
        )
    },
    "documented_extension_occurrence_counts": {
        str(key): int(value)
        for key, value in sorted(
            extension_occurrence_counts.items()
        )
    },
    "documentation": {
        "source": (
            "OAS-Kit specification-extension "
            "conversion table"
        ),
        "url": (
            "https://mermade.github.io/"
            "oas-kit/extensions.html"
        ),
    },
    "semantic_results_sha256": (
        sha256_file(
            SEMANTIC_RESULTS_PATH
        )
    ),
    "contract_results_sha256": (
        sha256_file(
            CONTRACT_RESULTS_PATH
        )
    ),
}

write_json_atomic(
    REPORT_PATH,
    report,
)

output = {
    **report,
    "report_path": str(
        REPORT_PATH
    ),
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 23: PASS")

{"progress": 100, "total": 1642}
{"progress": 200, "total": 1642}
{"progress": 300, "total": 1642}
{"progress": 400, "total": 1642}
{"progress": 500, "total": 1642}
{"progress": 600, "total": 1642}
{"progress": 700, "total": 1642}
{"progress": 800, "total": 1642}
{"progress": 900, "total": 1642}
{"progress": 1000, "total": 1642}
{"progress": 1100, "total": 1642}
{"progress": 1200, "total": 1642}
{"progress": 1300, "total": 1642}
{"progress": 1400, "total": 1642}
{"progress": 1500, "total": 1642}
{"progress": 1600, "total": 1642}
{"progress": 1642, "total": 1642}
{
  "status": "PASS",
  "semantic_failure_specification_count": 1642,
  "contract_failure_specification_count": 55,
  "semantic_failure_category_counts": {
    "all_unique_prose_preserved": 1179,
    "declared_servers_preserved": 80,
    "definition_references_preserved": 34,
    "definitions_preserved": 1356,
    "operation_metadata_preserved": 1482,
    "top_level_metadata_preserved": 83
  },
  "operation_metadata_field_count

In [26]:
from pathlib import Path
from collections import Counter
import copy
import hashlib
import json
import re

import yaml


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

ADJUSTED_V1_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_results.jsonl"
)

ADJUSTED_V1_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports/audit/swagger_contract_adjusted_summary.json"
)

ADJUSTED_V2_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v2_results.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_x_ms_skip_url_encoding_verification.json"
)

DOCUMENTATION_URL = (
    "https://mermade.github.io/oas-kit/extensions.html"
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_jsonl(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def write_jsonl_atomic(path: Path, records):
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                )
                + "\n"
            )

    temporary_path.replace(path)


def write_json_atomic(path: Path, value):
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


class OpenAPIStringKeySafeLoader(yaml.SafeLoader):
    pass


# Prevent YAML 1.1 from interpreting mapping keys such as
# "on" and "off" as Boolean values.
for resolver_character, resolvers in list(
    OpenAPIStringKeySafeLoader.yaml_implicit_resolvers.items()
):
    OpenAPIStringKeySafeLoader.yaml_implicit_resolvers[
        resolver_character
    ] = [
        resolver
        for resolver in resolvers
        if resolver[0] != "tag:yaml.org,2002:bool"
    ]

OpenAPIStringKeySafeLoader.add_implicit_resolver(
    "tag:yaml.org,2002:bool",
    re.compile(
        r"^(?:true|True|TRUE|false|False|FALSE)$"
    ),
    list("tTfF"),
)


def load_document(path: Path):
    text = path.read_text(
        encoding="utf-8-sig",
        errors="strict",
    )

    if path.suffix.lower() == ".json":
        return json.loads(text)

    return yaml.load(
        text,
        Loader=OpenAPIStringKeySafeLoader,
    )


def decode_pointer_token(token: str) -> str:
    return token.replace("~1", "/").replace("~0", "~")


def resolve_local_reference(document, value):
    current = value
    visited = set()

    while (
        isinstance(current, dict)
        and isinstance(current.get("$ref"), str)
        and current["$ref"].startswith("#/")
    ):
        reference = current["$ref"]

        if reference in visited:
            raise RuntimeError(
                f"Cyclic local reference: {reference}"
            )

        visited.add(reference)
        resolved = document

        for token in reference[2:].split("/"):
            token = decode_pointer_token(token)

            if not isinstance(resolved, dict):
                raise KeyError(reference)

            resolved = resolved[token]

        current = resolved

    return current


def find_parameter(
    document,
    method,
    path_template,
    parameter_name,
    parameter_location,
):
    paths = document.get("paths", {})

    if (
        not isinstance(paths, dict)
        or path_template not in paths
    ):
        return None

    path_item = resolve_local_reference(
        document,
        paths[path_template],
    )

    if not isinstance(path_item, dict):
        return None

    operation = resolve_local_reference(
        document,
        path_item.get(method.lower(), {}),
    )

    if not isinstance(operation, dict):
        return None

    # Operation-level parameters override path-level parameters.
    parameter_groups = [
        operation.get("parameters", []),
        path_item.get("parameters", []),
    ]

    for parameter_group in parameter_groups:
        if not isinstance(parameter_group, list):
            continue

        for parameter in parameter_group:
            try:
                parameter = resolve_local_reference(
                    document,
                    parameter,
                )
            except Exception:
                continue

            if not isinstance(parameter, dict):
                continue

            if (
                parameter.get("name") == parameter_name
                and parameter.get("in") == parameter_location
            ):
                return parameter

    return None


required_inputs = [
    ADJUSTED_V1_PATH,
    ADJUSTED_V1_SUMMARY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file() or path.stat().st_size == 0
]

if missing_inputs:
    raise RuntimeError(
        "Missing required inputs:\n"
        + "\n".join(missing_inputs)
    )


adjusted_v1_records = read_jsonl(ADJUSTED_V1_PATH)

adjusted_v1_summary = json.loads(
    ADJUSTED_V1_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

expected_prior_pass_count = int(
    adjusted_v1_summary[
        "adjusted_contract_pass_count"
    ]
)

prior_pass_count_from_records = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in adjusted_v1_records
)

if prior_pass_count_from_records != expected_prior_pass_count:
    raise RuntimeError(
        "Cell 22 record/summary mismatch: "
        f"summary reports {expected_prior_pass_count} passes, "
        f"records contain {prior_pass_count_from_records}."
    )


target_diagnostics = []

for record in adjusted_v1_records:
    diagnostics = record.get(
        "adjusted_diagnostics",
        {},
    )

    if not isinstance(diagnostics, dict):
        continue

    for category, category_diagnostics in diagnostics.items():
        if not isinstance(category_diagnostics, list):
            continue

        for diagnostic in category_diagnostics:
            if not isinstance(diagnostic, dict):
                continue

            if (
                diagnostic.get("problem")
                != "x-ms-skip-url-encoding"
            ):
                continue

            parameter_identity = diagnostic.get(
                "parameter"
            )

            if (
                not isinstance(parameter_identity, list)
                or len(parameter_identity) != 2
            ):
                raise RuntimeError(
                    "Malformed x-ms-skip-url-encoding "
                    "diagnostic: "
                    + json.dumps(
                        diagnostic,
                        ensure_ascii=False,
                    )
                )

            target_diagnostics.append(
                {
                    "spec_id": record.get("spec_id"),
                    "relative_path": record.get(
                        "relative_path"
                    ),
                    "category": category,
                    "operation": diagnostic.get(
                        "operation"
                    ),
                    "parameter_name": (
                        parameter_identity[0]
                    ),
                    "parameter_location": (
                        parameter_identity[1]
                    ),
                }
            )


if not target_diagnostics:
    raise RuntimeError(
        "No x-ms-skip-url-encoding diagnostics were found."
    )


source_documents = {}
converted_documents = {}
verification_records = []

for target in target_diagnostics:
    spec_id = target["spec_id"]
    relative_path = target["relative_path"]
    operation_identity = target["operation"]

    if (
        not isinstance(operation_identity, str)
        or " " not in operation_identity
    ):
        raise RuntimeError(
            f"Malformed operation identity: "
            f"{operation_identity!r}"
        )

    method, path_template = operation_identity.split(
        " ",
        1,
    )

    source_path = CORPUS_ROOT / relative_path

    converted_path = (
        PROJECT_ROOT
        / "data/canonical"
        / f"{spec_id}.json"
    )

    if not source_path.is_file():
        raise FileNotFoundError(source_path)

    if not converted_path.is_file():
        raise FileNotFoundError(converted_path)

    if spec_id not in source_documents:
        source_documents[spec_id] = load_document(
            source_path
        )

    if spec_id not in converted_documents:
        converted_documents[spec_id] = load_document(
            converted_path
        )

    source_parameter = find_parameter(
        source_documents[spec_id],
        method,
        path_template,
        target["parameter_name"],
        target["parameter_location"],
    )

    converted_parameter = find_parameter(
        converted_documents[spec_id],
        method,
        path_template,
        target["parameter_name"],
        target["parameter_location"],
    )

    source_extension_value = (
        source_parameter.get(
            "x-ms-skip-url-encoding"
        )
        if isinstance(source_parameter, dict)
        else None
    )

    converted_allow_reserved = (
        converted_parameter.get("allowReserved")
        if isinstance(converted_parameter, dict)
        else None
    )

    if source_parameter is None:
        mapping_status = "SOURCE_PARAMETER_NOT_FOUND"

    elif converted_parameter is None:
        mapping_status = "CONVERTED_PARAMETER_NOT_FOUND"

    elif source_extension_value is not True:
        mapping_status = "SOURCE_EXTENSION_NOT_TRUE"

    elif converted_allow_reserved is True:
        mapping_status = (
            "PRESERVED_AS_ALLOW_RESERVED_TRUE"
        )

    else:
        mapping_status = (
            "ALLOW_RESERVED_VALUE_MISMATCH"
        )

    verification_records.append(
        {
            **target,
            "source_extension_value": (
                source_extension_value
            ),
            "converted_allow_reserved": (
                converted_allow_reserved
            ),
            "mapping_status": mapping_status,
            "mapping_verified": (
                mapping_status
                == "PRESERVED_AS_ALLOW_RESERVED_TRUE"
            ),
            "source_path": str(source_path),
            "converted_path": str(converted_path),
        }
    )


verification_lookup = {
    (
        record["spec_id"],
        record["operation"],
        record["parameter_name"],
        record["parameter_location"],
    ): record
    for record in verification_records
}

verified_mapping_count = sum(
    record["mapping_verified"]
    for record in verification_records
)

mapping_failure_count = (
    len(verification_records)
    - verified_mapping_count
)

if mapping_failure_count:
    failed_mappings = [
        record
        for record in verification_records
        if not record["mapping_verified"]
    ]

    raise RuntimeError(
        "One or more documented transformations failed:\n"
        + json.dumps(
            failed_mappings,
            indent=2,
            ensure_ascii=False,
        )
    )


adjusted_v2_records = []
newly_accepted_spec_ids = []

for original_record in adjusted_v1_records:
    record = copy.deepcopy(original_record)

    prior_adjusted_pass = (
        record.get(
            "adjusted_contract_equivalence_pass"
        ) is True
    )

    prior_diagnostics = record.get(
        "adjusted_diagnostics",
        {},
    )

    prior_failed_categories = record.get(
        "adjusted_failed_categories",
        [],
    )

    if not isinstance(prior_diagnostics, dict):
        prior_diagnostics = {}

    if not isinstance(prior_failed_categories, list):
        prior_failed_categories = []

    if prior_adjusted_pass:
        filtered_diagnostics = {}
        new_failed_categories = []
        new_pass = True

    else:
        filtered_diagnostics = {}

        for category, category_diagnostics in (
            prior_diagnostics.items()
        ):
            if not isinstance(category_diagnostics, list):
                filtered_diagnostics[category] = (
                    category_diagnostics
                )
                continue

            retained_diagnostics = []

            for diagnostic in category_diagnostics:
                if (
                    not isinstance(diagnostic, dict)
                    or diagnostic.get("problem")
                    != "x-ms-skip-url-encoding"
                ):
                    retained_diagnostics.append(
                        diagnostic
                    )
                    continue

                parameter_identity = diagnostic.get(
                    "parameter",
                    [],
                )

                lookup_key = (
                    record.get("spec_id"),
                    diagnostic.get("operation"),
                    (
                        parameter_identity[0]
                        if len(parameter_identity) > 0
                        else None
                    ),
                    (
                        parameter_identity[1]
                        if len(parameter_identity) > 1
                        else None
                    ),
                )

                verification = verification_lookup.get(
                    lookup_key
                )

                # Remove only a successfully verified mapping.
                if not (
                    verification
                    and verification["mapping_verified"]
                ):
                    retained_diagnostics.append(
                        diagnostic
                    )

            if retained_diagnostics:
                filtered_diagnostics[category] = (
                    retained_diagnostics
                )

        new_failed_categories = [
            category
            for category in prior_failed_categories
            if category in filtered_diagnostics
        ]

        for category in filtered_diagnostics:
            if category not in new_failed_categories:
                new_failed_categories.append(category)

        new_pass = len(new_failed_categories) == 0

    if not prior_adjusted_pass and new_pass:
        newly_accepted_spec_ids.append(
            record.get("spec_id")
        )

    record["adjustment_version"] = 2

    record["adjustment"] = (
        "verified_x_ms_skip_url_encoding_promotion"
    )

    record["prior_adjusted_contract_equivalence_pass"] = (
        prior_adjusted_pass
    )

    record["adjusted_contract_equivalence_pass"] = (
        new_pass
    )

    record["adjusted_failed_categories"] = (
        new_failed_categories
    )

    record["adjusted_diagnostics"] = (
        filtered_diagnostics
    )

    adjusted_v2_records.append(record)


write_jsonl_atomic(
    ADJUSTED_V2_PATH,
    adjusted_v2_records,
)


adjusted_contract_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in adjusted_v2_records
)

adjusted_contract_failure_count = (
    len(adjusted_v2_records)
    - adjusted_contract_pass_count
)

expected_adjusted_pass_count = (
    expected_prior_pass_count
    + len(newly_accepted_spec_ids)
)

if adjusted_contract_pass_count != expected_adjusted_pass_count:
    raise RuntimeError(
        "Adjustment accounting mismatch: "
        f"expected {expected_adjusted_pass_count} passes, "
        f"observed {adjusted_contract_pass_count}."
    )

if adjusted_contract_pass_count < expected_prior_pass_count:
    raise RuntimeError(
        "Cell 24 regressed specifications already "
        "accepted by Cell 22."
    )


remaining_failure_category_counts = Counter()

for record in adjusted_v2_records:
    if record.get(
        "adjusted_contract_equivalence_pass"
    ) is True:
        continue

    remaining_failure_category_counts.update(
        record.get(
            "adjusted_failed_categories",
            [],
        )
    )

mapping_status_counts = Counter(
    record["mapping_status"]
    for record in verification_records
)


report = {
    "status": "PASS",
    "adjustment_version": 2,
    "adjustment": (
        "VERIFY_X_MS_SKIP_URL_ENCODING_TO_ALLOW_RESERVED"
    ),
    "documentation": {
        "statement": (
            "OAS-Kit documents Swagger query parameter "
            "x-ms-skip-url-encoding as conversion to "
            "OpenAPI allowReserved."
        ),
        "url": DOCUMENTATION_URL,
    },
    "audited_specification_count": len(
        adjusted_v1_records
    ),
    "prior_adjusted_contract_pass_count": (
        expected_prior_pass_count
    ),
    "target_diagnostic_count": len(
        target_diagnostics
    ),
    "target_specification_count": len(
        {
            record["spec_id"]
            for record in verification_records
        }
    ),
    "verified_mapping_count": (
        verified_mapping_count
    ),
    "mapping_failure_count": (
        mapping_failure_count
    ),
    "mapping_status_counts": dict(
        sorted(mapping_status_counts.items())
    ),
    "newly_accepted_specification_count": len(
        newly_accepted_spec_ids
    ),
    "newly_accepted_spec_ids": sorted(
        spec_id
        for spec_id in newly_accepted_spec_ids
        if spec_id
    ),
    "adjusted_contract_pass_count": (
        adjusted_contract_pass_count
    ),
    "adjusted_contract_failure_count": (
        adjusted_contract_failure_count
    ),
    "remaining_failure_category_counts": dict(
        sorted(
            remaining_failure_category_counts.items()
        )
    ),
    "no_prior_acceptance_regression": True,
    "verification_records": verification_records,
    "input_results": {
        "path": str(ADJUSTED_V1_PATH),
        "sha256": sha256_file(
            ADJUSTED_V1_PATH
        ),
    },
    "adjusted_results": {
        "path": str(ADJUSTED_V2_PATH),
        "sha256": sha256_file(
            ADJUSTED_V2_PATH
        ),
    },
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

report_sha256 = sha256_file(REPORT_PATH)

display_report = {
    key: value
    for key, value in report.items()
    if key != "verification_records"
}

display_report["report_sha256"] = report_sha256

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 24: PASS")

{
  "status": "PASS",
  "adjustment_version": 2,
  "adjustment": "VERIFY_X_MS_SKIP_URL_ENCODING_TO_ALLOW_RESERVED",
  "documentation": {
    "statement": "OAS-Kit documents Swagger query parameter x-ms-skip-url-encoding as conversion to OpenAPI allowReserved.",
    "url": "https://mermade.github.io/oas-kit/extensions.html"
  },
  "audited_specification_count": 226,
  "prior_adjusted_contract_pass_count": 171,
  "target_diagnostic_count": 18,
  "target_specification_count": 4,
  "verified_mapping_count": 18,
  "mapping_failure_count": 0,
  "mapping_status_counts": {
    "PRESERVED_AS_ALLOW_RESERVED_TRUE": 18
  },
  "newly_accepted_specification_count": 4,
  "newly_accepted_spec_ids": [
    "spec_04546721f70ce2e7dcc31920",
    "spec_2dc10cc1094ff2d202c222c1",
    "spec_8203b807edb55cdd3db7b61a",
    "spec_db5bef157e6eda6d341e27a3"
  ],
  "adjusted_contract_pass_count": 175,
  "adjusted_contract_failure_count": 51,
  "remaining_failure_category_counts": {
    "nonbody_parameters_preserved

In [27]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

INPUT_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v2_results.jsonl"
)

DETAIL_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_remaining_nonbody_parameter_differences.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_remaining_nonbody_parameter_triage.json"
)


required_helpers = [
    "load_document",
    "find_parameter",
    "write_jsonl_atomic",
    "write_json_atomic",
    "sha256_file",
]

missing_helpers = [
    name
    for name in required_helpers
    if name not in globals()
]

if missing_helpers:
    raise RuntimeError(
        "Required Cell 24 helpers are unavailable: "
        + ", ".join(missing_helpers)
        + ". Rerun the corrected Cell 24 first."
    )


def read_jsonl_local(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def json_difference_paths(
    source_value,
    converted_value,
    current_path="",
):
    differences = []

    if type(source_value) is not type(converted_value):
        return [
            current_path or "/"
        ]

    if isinstance(source_value, dict):
        all_keys = sorted(
            set(source_value) | set(converted_value)
        )

        for key in all_keys:
            child_path = (
                f"{current_path}/{key}"
                if current_path
                else f"/{key}"
            )

            if (
                key not in source_value
                or key not in converted_value
            ):
                differences.append(child_path)
                continue

            differences.extend(
                json_difference_paths(
                    source_value[key],
                    converted_value[key],
                    child_path,
                )
            )

        return differences

    if isinstance(source_value, list):
        if source_value != converted_value:
            differences.append(current_path or "/")

        return differences

    if source_value != converted_value:
        differences.append(current_path or "/")

    return differences


SCHEMA_KEYS = {
    "$ref",
    "type",
    "format",
    "title",
    "description",
    "default",
    "enum",
    "multipleOf",
    "maximum",
    "exclusiveMaximum",
    "minimum",
    "exclusiveMinimum",
    "maxLength",
    "minLength",
    "pattern",
    "maxItems",
    "minItems",
    "uniqueItems",
    "maxProperties",
    "minProperties",
    "required",
    "items",
    "allOf",
    "oneOf",
    "anyOf",
    "not",
    "properties",
    "additionalProperties",
    "readOnly",
    "writeOnly",
    "nullable",
    "example",
}


def extract_swagger_parameter_schema(parameter):
    if not isinstance(parameter, dict):
        return None

    if isinstance(parameter.get("schema"), dict):
        return parameter["schema"]

    schema = {
        key: parameter[key]
        for key in SCHEMA_KEYS
        if key in parameter
    }

    return schema


def compact_parameter_projection(
    source_parameter,
    converted_parameter,
):
    source_schema = extract_swagger_parameter_schema(
        source_parameter
    )

    converted_schema = (
        converted_parameter.get("schema")
        if isinstance(converted_parameter, dict)
        else None
    )

    return {
        "source_schema": source_schema,
        "converted_schema": converted_schema,
        "source_collection_format": (
            source_parameter.get("collectionFormat")
            if isinstance(source_parameter, dict)
            else None
        ),
        "converted_style": (
            converted_parameter.get("style")
            if isinstance(converted_parameter, dict)
            else None
        ),
        "converted_explode": (
            converted_parameter.get("explode")
            if isinstance(converted_parameter, dict)
            else None
        ),
        "source_x_example": (
            source_parameter.get("x-example")
            if isinstance(source_parameter, dict)
            else None
        ),
        "converted_example": (
            converted_parameter.get("example")
            if isinstance(converted_parameter, dict)
            else None
        ),
        "source_x_eg": (
            source_parameter.get("x-eg")
            if isinstance(source_parameter, dict)
            else None
        ),
        "converted_x_eg": (
            converted_parameter.get("x-eg")
            if isinstance(converted_parameter, dict)
            else None
        ),
    }


if not INPUT_PATH.is_file():
    raise FileNotFoundError(INPUT_PATH)


records = read_jsonl_local(INPUT_PATH)

remaining_records = [
    record
    for record in records
    if (
        record.get(
            "adjusted_contract_equivalence_pass"
        ) is not True
        and "nonbody_parameters_preserved"
        in record.get(
            "adjusted_failed_categories",
            [],
        )
    )
]


source_cache = {}
converted_cache = {}
detail_records = []

for record in remaining_records:
    spec_id = record["spec_id"]
    relative_path = record["relative_path"]

    source_path = CORPUS_ROOT / relative_path

    converted_path = (
        PROJECT_ROOT
        / "data/canonical"
        / f"{spec_id}.json"
    )

    if not source_path.is_file():
        raise FileNotFoundError(source_path)

    if not converted_path.is_file():
        raise FileNotFoundError(converted_path)

    if spec_id not in source_cache:
        source_cache[spec_id] = load_document(
            source_path
        )

    if spec_id not in converted_cache:
        converted_cache[spec_id] = load_document(
            converted_path
        )

    diagnostics = record.get(
        "adjusted_diagnostics",
        {},
    ).get(
        "nonbody_parameters_preserved",
        [],
    )

    for diagnostic_index, diagnostic in enumerate(
        diagnostics
    ):
        operation_identity = diagnostic.get(
            "operation"
        )

        parameter_identity = diagnostic.get(
            "parameter"
        )

        if (
            not isinstance(operation_identity, str)
            or " " not in operation_identity
        ):
            raise RuntimeError(
                f"Malformed operation identity: "
                f"{operation_identity!r}"
            )

        if (
            not isinstance(parameter_identity, list)
            or len(parameter_identity) != 2
        ):
            raise RuntimeError(
                "Malformed parameter identity: "
                + json.dumps(
                    parameter_identity,
                    ensure_ascii=False,
                )
            )

        method, path_template = operation_identity.split(
            " ",
            1,
        )

        parameter_name = parameter_identity[0]
        parameter_location = parameter_identity[1]

        source_parameter = find_parameter(
            source_cache[spec_id],
            method,
            path_template,
            parameter_name,
            parameter_location,
        )

        converted_parameter = find_parameter(
            converted_cache[spec_id],
            method,
            path_template,
            parameter_name,
            parameter_location,
        )

        raw_difference_paths = json_difference_paths(
            source_parameter,
            converted_parameter,
        )

        projection = compact_parameter_projection(
            source_parameter,
            converted_parameter,
        )

        schema_difference_paths = (
            json_difference_paths(
                projection["source_schema"],
                projection["converted_schema"],
            )
        )

        problem = diagnostic.get(
            "problem",
            "UNKNOWN"
        )

        detail_records.append(
            {
                "spec_id": spec_id,
                "raw_id": record.get("raw_id"),
                "relative_path": relative_path,
                "provider_directory": record.get(
                    "provider_directory"
                ),
                "operation": operation_identity,
                "parameter_name": parameter_name,
                "parameter_location": (
                    parameter_location
                ),
                "problem": problem,
                "diagnostic_index": diagnostic_index,
                "diagnostic": diagnostic,
                "raw_difference_paths": sorted(
                    set(raw_difference_paths)
                ),
                "schema_difference_paths": sorted(
                    set(schema_difference_paths)
                ),
                "projection": projection,
                "source_parameter": source_parameter,
                "converted_parameter": (
                    converted_parameter
                ),
            }
        )


write_jsonl_atomic(
    DETAIL_PATH,
    detail_records,
)


problem_counts = Counter(
    record["problem"]
    for record in detail_records
)

specifications_by_problem = defaultdict(set)

for record in detail_records:
    specifications_by_problem[
        record["problem"]
    ].add(record["spec_id"])


schema_pattern_counts = Counter(
    tuple(record["schema_difference_paths"])
    for record in detail_records
    if record["problem"] == "schema"
)

raw_pattern_counts = Counter(
    (
        record["problem"],
        tuple(record["raw_difference_paths"]),
    )
    for record in detail_records
)


problem_examples = {}

for problem in sorted(problem_counts):
    candidates = [
        record
        for record in detail_records
        if record["problem"] == problem
    ]

    seen_patterns = set()
    examples = []

    for record in candidates:
        pattern = (
            tuple(record["schema_difference_paths"])
            if problem == "schema"
            else tuple(record["raw_difference_paths"])
        )

        if pattern in seen_patterns:
            continue

        seen_patterns.add(pattern)

        examples.append(
            {
                "spec_id": record["spec_id"],
                "relative_path": record[
                    "relative_path"
                ],
                "operation": record["operation"],
                "parameter": [
                    record["parameter_name"],
                    record["parameter_location"],
                ],
                "problem": problem,
                "raw_difference_paths": record[
                    "raw_difference_paths"
                ],
                "schema_difference_paths": record[
                    "schema_difference_paths"
                ],
                "projection": record["projection"],
            }
        )

        if len(examples) >= 12:
            break

    problem_examples[problem] = examples


report = {
    "status": "PASS",
    "audit": (
        "REMAINING_NONBODY_PARAMETER_DIFFERENCE_TRIAGE"
    ),
    "input_specification_count": len(records),
    "remaining_nonbody_failure_specification_count": (
        len(remaining_records)
    ),
    "diagnostic_count": len(detail_records),
    "problem_counts": dict(
        sorted(problem_counts.items())
    ),
    "specification_counts_by_problem": {
        problem: len(spec_ids)
        for problem, spec_ids in sorted(
            specifications_by_problem.items()
        )
    },
    "schema_difference_pattern_counts": [
        {
            "difference_paths": list(pattern),
            "count": count,
        }
        for pattern, count in schema_pattern_counts.most_common()
    ],
    "raw_difference_pattern_count": len(
        raw_pattern_counts
    ),
    "problem_examples": problem_examples,
    "details_path": str(DETAIL_PATH),
    "details_sha256": sha256_file(DETAIL_PATH),
    "input_sha256": sha256_file(INPUT_PATH),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(REPORT_PATH),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 25: PASS")

{
  "status": "PASS",
  "audit": "REMAINING_NONBODY_PARAMETER_DIFFERENCE_TRIAGE",
  "input_specification_count": 226,
  "remaining_nonbody_failure_specification_count": 8,
  "diagnostic_count": 86,
  "problem_counts": {
    "schema": 57,
    "x-eg": 27,
    "x-example": 2
  },
  "specification_counts_by_problem": {
    "schema": 7,
    "x-eg": 1,
    "x-example": 1
  },
  "schema_difference_pattern_counts": [
    {
      "difference_paths": [
        "/description",
        "/required"
      ],
      "count": 26
    },
    {
      "difference_paths": [
        "/description",
        "/items/collectionFormat",
        "/required"
      ],
      "count": 23
    },
    {
      "difference_paths": [
        "/description",
        "/items/enum",
        "/required"
      ],
      "count": 4
    },
    {
      "difference_paths": [
        "/description",
        "/example",
        "/required"
      ],
      "count": 4
    }
  ],
  "raw_difference_pattern_count": 9,
  "problem_examples": 

In [28]:
from pathlib import Path
from collections import Counter
import copy
import hashlib
import json


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

INPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v2_results.jsonl"
)

TRIAGE_DETAILS_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_remaining_nonbody_parameter_differences.jsonl"
)

OUTPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v3_results.jsonl"
)

ASSESSMENTS_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_nonbody_parameter_representation_assessments.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_nonbody_parameter_representation_adjustment.json"
)


def read_jsonl_cell26(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def canonical_json(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


PARAMETER_SCHEMA_KEYS = {
    "$ref",
    "type",
    "format",
    "default",
    "enum",
    "multipleOf",
    "maximum",
    "exclusiveMaximum",
    "minimum",
    "exclusiveMinimum",
    "maxLength",
    "minLength",
    "pattern",
    "maxItems",
    "minItems",
    "uniqueItems",
    "maxProperties",
    "minProperties",
    "items",
    "allOf",
    "oneOf",
    "anyOf",
    "not",
    "properties",
    "additionalProperties",
    "readOnly",
    "writeOnly",
    "nullable",
}


def build_swagger_nonbody_schema(parameter):
    if not isinstance(parameter, dict):
        return None

    schema = {
        key: copy.deepcopy(parameter[key])
        for key in PARAMETER_SCHEMA_KEYS
        if key in parameter
    }

    return schema


def remove_inapplicable_nested_collection_formats(
    value,
    current_path="",
):
    removed = []

    if isinstance(value, dict):
        value_type = value.get("type")

        if (
            "collectionFormat" in value
            and value_type != "array"
        ):
            removed.append(
                {
                    "path": (
                        f"{current_path}/collectionFormat"
                        if current_path
                        else "/collectionFormat"
                    ),
                    "value": value["collectionFormat"],
                    "containing_type": value_type,
                    "reason": (
                        "collectionFormat only affects "
                        "array-valued Items Objects"
                    ),
                }
            )

            value.pop("collectionFormat")

        for key in list(value.keys()):
            child_path = (
                f"{current_path}/{key}"
                if current_path
                else f"/{key}"
            )

            removed.extend(
                remove_inapplicable_nested_collection_formats(
                    value[key],
                    child_path,
                )
            )

    elif isinstance(value, list):
        for index, item in enumerate(value):
            child_path = (
                f"{current_path}/{index}"
                if current_path
                else f"/{index}"
            )

            removed.extend(
                remove_inapplicable_nested_collection_formats(
                    item,
                    child_path,
                )
            )

    return removed


def normalized_required(parameter):
    if not isinstance(parameter, dict):
        return None

    location = parameter.get("in")

    if location == "path":
        return parameter.get("required") is True

    return bool(parameter.get("required", False))


def effective_oas3_serialization(parameter):
    if not isinstance(parameter, dict):
        return {
            "style": None,
            "explode": None,
        }

    location = parameter.get("in")
    style = parameter.get("style")

    if style is None:
        if location in {"query", "cookie"}:
            style = "form"
        elif location in {"path", "header"}:
            style = "simple"

    explode = parameter.get("explode")

    if explode is None:
        explode = style == "form"

    return {
        "style": style,
        "explode": explode,
    }


def expected_oas3_serialization(
    source_parameter,
):
    if not isinstance(source_parameter, dict):
        return {
            "applicable": False,
            "style": None,
            "explode": None,
            "collection_format": None,
        }

    if source_parameter.get("type") != "array":
        return {
            "applicable": False,
            "style": None,
            "explode": None,
            "collection_format": None,
        }

    location = source_parameter.get("in")

    collection_format = source_parameter.get(
        "collectionFormat",
        "csv",
    )

    mappings = {
        ("query", "csv"): ("form", False),
        ("query", "multi"): ("form", True),
        ("query", "ssv"): ("spaceDelimited", False),
        ("query", "pipes"): ("pipeDelimited", False),
        ("path", "csv"): ("simple", False),
        ("header", "csv"): ("simple", False),
    }

    mapping = mappings.get(
        (location, collection_format)
    )

    if mapping is None:
        return {
            "applicable": True,
            "style": None,
            "explode": None,
            "collection_format": collection_format,
            "supported_mapping": False,
        }

    return {
        "applicable": True,
        "style": mapping[0],
        "explode": mapping[1],
        "collection_format": collection_format,
        "supported_mapping": True,
    }


required_inputs = [
    INPUT_RESULTS_PATH,
    TRIAGE_DETAILS_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file() or path.stat().st_size == 0
]

if missing_inputs:
    raise RuntimeError(
        "Missing required inputs:\n"
        + "\n".join(missing_inputs)
    )


input_records = read_jsonl_cell26(
    INPUT_RESULTS_PATH
)

triage_records = read_jsonl_cell26(
    TRIAGE_DETAILS_PATH
)


assessments = []

for detail in triage_records:
    problem = detail["problem"]

    source_parameter = detail.get(
        "source_parameter"
    )

    converted_parameter = detail.get(
        "converted_parameter"
    )

    checks = {}
    resolution_status = "RETAIN_FAILURE"
    resolution_reason = None

    if problem == "x-eg":
        source_value = (
            source_parameter.get("x-eg")
            if isinstance(source_parameter, dict)
            else None
        )

        converted_value = (
            converted_parameter.get("x-eg")
            if isinstance(converted_parameter, dict)
            else None
        )

        checks = {
            "source_x_eg": source_value,
            "converted_x_eg": converted_value,
            "value_preserved": (
                source_value == converted_value
            ),
        }

        if (
            isinstance(source_parameter, dict)
            and "x-eg" in source_parameter
            and source_value == converted_value
        ):
            resolution_status = (
                "RESOLVED_EXTENSION_PRESERVED"
            )
            resolution_reason = (
                "The x-eg extension and its exact value "
                "were retained in the converted parameter."
            )

        else:
            resolution_reason = (
                "The x-eg extension was missing or changed."
            )

    elif problem == "x-example":
        source_value = (
            source_parameter.get("x-example")
            if isinstance(source_parameter, dict)
            else None
        )

        converted_value = (
            converted_parameter.get("example")
            if isinstance(converted_parameter, dict)
            else None
        )

        checks = {
            "source_x_example": source_value,
            "converted_example": converted_value,
            "value_preserved": (
                source_value == converted_value
            ),
        }

        if (
            isinstance(source_parameter, dict)
            and "x-example" in source_parameter
            and source_value == converted_value
        ):
            resolution_status = (
                "RESOLVED_X_EXAMPLE_PROMOTION"
            )
            resolution_reason = (
                "The x-example value was promoted to "
                "example without changing its value."
            )

        else:
            resolution_reason = (
                "The x-example value changed during "
                "conversion."
            )

    elif problem == "schema":
        source_schema = (
            build_swagger_nonbody_schema(
                source_parameter
            )
        )

        converted_schema = copy.deepcopy(
            converted_parameter.get("schema")
            if isinstance(converted_parameter, dict)
            else None
        )

        removed_inapplicable_keywords = []

        if isinstance(source_schema, dict):
            removed_inapplicable_keywords = (
                remove_inapplicable_nested_collection_formats(
                    source_schema
                )
            )

        description_preserved = (
            (
                source_parameter.get("description")
                if isinstance(source_parameter, dict)
                else None
            )
            ==
            (
                converted_parameter.get("description")
                if isinstance(converted_parameter, dict)
                else None
            )
        )

        required_preserved = (
            normalized_required(source_parameter)
            ==
            normalized_required(converted_parameter)
        )

        source_has_example = (
            isinstance(source_parameter, dict)
            and "example" in source_parameter
        )

        source_example = (
            source_parameter.get("example")
            if isinstance(source_parameter, dict)
            else None
        )

        converted_example = (
            converted_parameter.get("example")
            if isinstance(converted_parameter, dict)
            else None
        )

        example_preserved = (
            not source_has_example
            or source_example == converted_example
        )

        schemas_equal = (
            canonical_json(source_schema)
            == canonical_json(converted_schema)
        )

        expected_serialization = (
            expected_oas3_serialization(
                source_parameter
            )
        )

        observed_serialization = (
            effective_oas3_serialization(
                converted_parameter
            )
        )

        if not expected_serialization["applicable"]:
            serialization_preserved = True

        elif not expected_serialization.get(
            "supported_mapping",
            False,
        ):
            serialization_preserved = False

        else:
            serialization_preserved = (
                expected_serialization["style"]
                == observed_serialization["style"]
                and expected_serialization["explode"]
                == observed_serialization["explode"]
            )

        inapplicable_keyword_removal_safe = all(
            item.get("containing_type") != "array"
            for item in removed_inapplicable_keywords
        )

        checks = {
            "description_preserved": (
                description_preserved
            ),
            "required_preserved": (
                required_preserved
            ),
            "example_preserved": (
                example_preserved
            ),
            "normalized_schemas_equal": (
                schemas_equal
            ),
            "serialization_preserved": (
                serialization_preserved
            ),
            "inapplicable_keyword_removal_safe": (
                inapplicable_keyword_removal_safe
            ),
            "removed_inapplicable_keywords": (
                removed_inapplicable_keywords
            ),
            "source_schema_normalized": (
                source_schema
            ),
            "converted_schema": (
                converted_schema
            ),
            "expected_serialization": (
                expected_serialization
            ),
            "observed_serialization": (
                observed_serialization
            ),
            "source_parameter_example": (
                source_example
            ),
            "converted_parameter_example": (
                converted_example
            ),
        }

        if (
            description_preserved
            and required_preserved
            and example_preserved
            and schemas_equal
            and serialization_preserved
            and inapplicable_keyword_removal_safe
        ):
            resolution_status = (
                "RESOLVED_REPRESENTATION_EQUIVALENT"
            )
            resolution_reason = (
                "Parameter metadata, normalized schema, "
                "example, and serialization semantics "
                "were preserved."
            )

        else:
            resolution_reason = (
                "At least one parameter contract value "
                "or schema value changed."
            )

    else:
        checks = {
            "unsupported_problem_type": problem
        }

        resolution_reason = (
            "No approved equivalence rule exists for "
            "this problem type."
        )

    assessments.append(
        {
            "spec_id": detail["spec_id"],
            "relative_path": detail[
                "relative_path"
            ],
            "operation": detail["operation"],
            "parameter_name": detail[
                "parameter_name"
            ],
            "parameter_location": detail[
                "parameter_location"
            ],
            "problem": problem,
            "resolution_status": (
                resolution_status
            ),
            "resolved": (
                resolution_status.startswith(
                    "RESOLVED_"
                )
            ),
            "resolution_reason": (
                resolution_reason
            ),
            "checks": checks,
        }
    )


write_jsonl_atomic(
    ASSESSMENTS_PATH,
    assessments,
)


assessment_lookup = {
    (
        assessment["spec_id"],
        assessment["operation"],
        assessment["parameter_name"],
        assessment["parameter_location"],
        assessment["problem"],
    ): assessment
    for assessment in assessments
}


output_records = []
newly_accepted_spec_ids = []

for original_record in input_records:
    record = copy.deepcopy(original_record)

    prior_pass = (
        record.get(
            "adjusted_contract_equivalence_pass"
        ) is True
    )

    prior_diagnostics = record.get(
        "adjusted_diagnostics",
        {},
    )

    prior_failed_categories = record.get(
        "adjusted_failed_categories",
        [],
    )

    if not isinstance(prior_diagnostics, dict):
        prior_diagnostics = {}

    if not isinstance(prior_failed_categories, list):
        prior_failed_categories = []

    updated_diagnostics = copy.deepcopy(
        prior_diagnostics
    )

    nonbody_diagnostics = updated_diagnostics.get(
        "nonbody_parameters_preserved",
        [],
    )

    retained_nonbody_diagnostics = []

    for diagnostic in nonbody_diagnostics:
        parameter_identity = diagnostic.get(
            "parameter",
            [],
        )

        lookup_key = (
            record.get("spec_id"),
            diagnostic.get("operation"),
            (
                parameter_identity[0]
                if len(parameter_identity) > 0
                else None
            ),
            (
                parameter_identity[1]
                if len(parameter_identity) > 1
                else None
            ),
            diagnostic.get("problem"),
        )

        assessment = assessment_lookup.get(
            lookup_key
        )

        if not (
            assessment
            and assessment["resolved"]
        ):
            retained_nonbody_diagnostics.append(
                diagnostic
            )

    if retained_nonbody_diagnostics:
        updated_diagnostics[
            "nonbody_parameters_preserved"
        ] = retained_nonbody_diagnostics

    else:
        updated_diagnostics.pop(
            "nonbody_parameters_preserved",
            None,
        )

    updated_failed_categories = [
        category
        for category in prior_failed_categories
        if category in updated_diagnostics
    ]

    for category in updated_diagnostics:
        if category not in updated_failed_categories:
            updated_failed_categories.append(
                category
            )

    updated_pass = (
        prior_pass
        or len(updated_failed_categories) == 0
    )

    if not prior_pass and updated_pass:
        newly_accepted_spec_ids.append(
            record.get("spec_id")
        )

    record["adjustment_version"] = 3

    record["adjustment"] = (
        "verified_nonbody_parameter_"
        "representation_equivalence"
    )

    record[
        "prior_adjusted_contract_equivalence_pass"
    ] = prior_pass

    record[
        "adjusted_contract_equivalence_pass"
    ] = updated_pass

    record["adjusted_failed_categories"] = (
        updated_failed_categories
    )

    record["adjusted_diagnostics"] = (
        updated_diagnostics
    )

    output_records.append(record)


write_jsonl_atomic(
    OUTPUT_RESULTS_PATH,
    output_records,
)


prior_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in input_records
)

adjusted_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in output_records
)

adjusted_failure_count = (
    len(output_records)
    - adjusted_pass_count
)

expected_pass_count = (
    prior_pass_count
    + len(newly_accepted_spec_ids)
)

if adjusted_pass_count != expected_pass_count:
    raise RuntimeError(
        "Adjustment accounting mismatch: "
        f"expected {expected_pass_count} passes, "
        f"observed {adjusted_pass_count}."
    )


resolution_status_counts = Counter(
    assessment["resolution_status"]
    for assessment in assessments
)

problem_resolution_counts = Counter(
    (
        assessment["problem"],
        (
            "RESOLVED"
            if assessment["resolved"]
            else "RETAINED_FAILURE"
        ),
    )
    for assessment in assessments
)

remaining_failure_category_counts = Counter()

for record in output_records:
    if record.get(
        "adjusted_contract_equivalence_pass"
    ) is True:
        continue

    remaining_failure_category_counts.update(
        record.get(
            "adjusted_failed_categories",
            [],
        )
    )


report = {
    "status": "PASS",
    "adjustment_version": 3,
    "adjustment": (
        "VERIFY_NONBODY_PARAMETER_"
        "REPRESENTATION_EQUIVALENCE"
    ),
    "prior_adjusted_contract_pass_count": (
        prior_pass_count
    ),
    "assessment_count": len(assessments),
    "resolution_status_counts": dict(
        sorted(resolution_status_counts.items())
    ),
    "problem_resolution_counts": {
        f"{problem}::{decision}": count
        for (
            problem,
            decision,
        ), count in sorted(
            problem_resolution_counts.items()
        )
    },
    "newly_accepted_specification_count": len(
        newly_accepted_spec_ids
    ),
    "newly_accepted_spec_ids": sorted(
        spec_id
        for spec_id in newly_accepted_spec_ids
        if spec_id
    ),
    "adjusted_contract_pass_count": (
        adjusted_pass_count
    ),
    "adjusted_contract_failure_count": (
        adjusted_failure_count
    ),
    "remaining_failure_category_counts": dict(
        sorted(
            remaining_failure_category_counts.items()
        )
    ),
    "assessments_path": str(
        ASSESSMENTS_PATH
    ),
    "assessments_sha256": sha256_file(
        ASSESSMENTS_PATH
    ),
    "adjusted_results_path": str(
        OUTPUT_RESULTS_PATH
    ),
    "adjusted_results_sha256": sha256_file(
        OUTPUT_RESULTS_PATH
    ),
    "input_results_sha256": sha256_file(
        INPUT_RESULTS_PATH
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 26: PASS")

{
  "status": "PASS",
  "adjustment_version": 3,
  "adjustment": "VERIFY_NONBODY_PARAMETER_REPRESENTATION_EQUIVALENCE",
  "prior_adjusted_contract_pass_count": 175,
  "assessment_count": 86,
  "resolution_status_counts": {
    "RESOLVED_EXTENSION_PRESERVED": 27,
    "RESOLVED_REPRESENTATION_EQUIVALENT": 53,
    "RETAIN_FAILURE": 6
  },
  "problem_resolution_counts": {
    "schema::RESOLVED": 53,
    "schema::RETAINED_FAILURE": 4,
    "x-eg::RESOLVED": 27,
    "x-example::RETAINED_FAILURE": 2
  },
  "newly_accepted_specification_count": 1,
  "newly_accepted_spec_ids": [
    "spec_bb856d25a21609cdddb86c3c"
  ],
  "adjusted_contract_pass_count": 176,
  "adjusted_contract_failure_count": 50,
  "remaining_failure_category_counts": {
    "nonbody_parameters_preserved": 2,
    "request_contracts_preserved": 14,
    "response_contracts_preserved": 36,
    "security_schemes_preserved": 1
  },
  "assessments_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/audits/swagger_no

In [29]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

INPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v3_results.jsonl"
)

DETAIL_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_remaining_request_contract_differences.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_remaining_request_contract_triage.json"
)


required_helpers = [
    "load_document",
    "resolve_local_reference",
    "write_jsonl_atomic",
    "write_json_atomic",
    "sha256_file",
]

missing_helpers = [
    name
    for name in required_helpers
    if name not in globals()
]

if missing_helpers:
    raise RuntimeError(
        "Required earlier-cell helpers are unavailable: "
        + ", ".join(missing_helpers)
    )


def read_jsonl_cell27(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def get_operation(
    document,
    method,
    path_template,
):
    paths = document.get("paths", {})

    if (
        not isinstance(paths, dict)
        or path_template not in paths
    ):
        return None, None

    path_item = resolve_local_reference(
        document,
        paths[path_template],
    )

    if not isinstance(path_item, dict):
        return None, None

    operation = resolve_local_reference(
        document,
        path_item.get(method.lower(), {}),
    )

    if not isinstance(operation, dict):
        return path_item, None

    return path_item, operation


def operation_parameters(
    document,
    path_item,
    operation,
):
    combined = {}

    parameter_groups = [
        (
            path_item.get("parameters", [])
            if isinstance(path_item, dict)
            else []
        ),
        (
            operation.get("parameters", [])
            if isinstance(operation, dict)
            else []
        ),
    ]

    for parameter_group in parameter_groups:
        if not isinstance(parameter_group, list):
            continue

        for parameter in parameter_group:
            try:
                parameter = resolve_local_reference(
                    document,
                    parameter,
                )
            except Exception:
                continue

            if not isinstance(parameter, dict):
                continue

            identity = (
                parameter.get("name"),
                parameter.get("in"),
            )

            combined[identity] = parameter

    return list(combined.values())


def project_body_parameter(parameter):
    return {
        "name": parameter.get("name"),
        "required": bool(
            parameter.get("required", False)
        ),
        "description": parameter.get("description"),
        "schema": parameter.get("schema"),
        "x_extensions": {
            key: value
            for key, value in parameter.items()
            if key.startswith("x-")
        },
    }


def project_form_parameter(parameter):
    schema_keys = [
        "type",
        "format",
        "items",
        "default",
        "enum",
        "pattern",
        "minimum",
        "maximum",
        "minLength",
        "maxLength",
        "minItems",
        "maxItems",
        "collectionFormat",
    ]

    return {
        "name": parameter.get("name"),
        "required": bool(
            parameter.get("required", False)
        ),
        "description": parameter.get("description"),
        "schema_fields": {
            key: parameter[key]
            for key in schema_keys
            if key in parameter
        },
        "x_extensions": {
            key: value
            for key, value in parameter.items()
            if key.startswith("x-")
        },
    }


def project_converted_request_body(
    document,
    operation,
):
    if not isinstance(operation, dict):
        return None

    request_body = operation.get("requestBody")

    if request_body is None:
        return None

    try:
        request_body = resolve_local_reference(
            document,
            request_body,
        )
    except Exception:
        return {
            "resolution_failed": True,
            "raw": request_body,
        }

    if not isinstance(request_body, dict):
        return None

    content = request_body.get("content", {})

    projected_content = {}

    if isinstance(content, dict):
        for media_type, media_object in content.items():
            if not isinstance(media_object, dict):
                projected_content[media_type] = (
                    media_object
                )
                continue

            projected_content[media_type] = {
                "schema": media_object.get("schema"),
                "encoding": media_object.get("encoding"),
                "example": media_object.get("example"),
                "examples": media_object.get("examples"),
            }

    return {
        "required": bool(
            request_body.get("required", False)
        ),
        "description": request_body.get(
            "description"
        ),
        "content": projected_content,
        "x_extensions": {
            key: value
            for key, value in request_body.items()
            if key.startswith("x-")
        },
    }


def compact_signature(snapshot):
    body_parameters = snapshot[
        "source_body_parameters"
    ]

    form_parameters = snapshot[
        "source_form_parameters"
    ]

    converted_request_body = snapshot[
        "converted_request_body"
    ]

    if isinstance(converted_request_body, dict):
        converted_media_types = sorted(
            converted_request_body.get(
                "content",
                {},
            ).keys()
        )
    else:
        converted_media_types = []

    return {
        "source_consumes": snapshot[
            "source_effective_consumes"
        ],
        "source_body_parameter_count": len(
            body_parameters
        ),
        "source_form_parameter_count": len(
            form_parameters
        ),
        "converted_request_body_present": (
            converted_request_body is not None
        ),
        "converted_media_types": (
            converted_media_types
        ),
    }


if not INPUT_RESULTS_PATH.is_file():
    raise FileNotFoundError(INPUT_RESULTS_PATH)


input_records = read_jsonl_cell27(
    INPUT_RESULTS_PATH
)

remaining_records = [
    record
    for record in input_records
    if (
        record.get(
            "adjusted_contract_equivalence_pass"
        ) is not True
        and "request_contracts_preserved"
        in record.get(
            "adjusted_failed_categories",
            [],
        )
    )
]


source_cache = {}
converted_cache = {}
operation_snapshot_cache = {}
detail_records = []

for record in remaining_records:
    spec_id = record["spec_id"]
    relative_path = record["relative_path"]

    source_path = CORPUS_ROOT / relative_path

    converted_path = (
        PROJECT_ROOT
        / "data/canonical"
        / f"{spec_id}.json"
    )

    if not source_path.is_file():
        raise FileNotFoundError(source_path)

    if not converted_path.is_file():
        raise FileNotFoundError(converted_path)

    if spec_id not in source_cache:
        source_cache[spec_id] = load_document(
            source_path
        )

    if spec_id not in converted_cache:
        converted_cache[spec_id] = load_document(
            converted_path
        )

    source_document = source_cache[spec_id]
    converted_document = converted_cache[spec_id]

    diagnostics = record.get(
        "adjusted_diagnostics",
        {},
    ).get(
        "request_contracts_preserved",
        [],
    )

    for diagnostic_index, diagnostic in enumerate(
        diagnostics
    ):
        operation_identity = diagnostic.get(
            "operation"
        )

        if (
            not isinstance(operation_identity, str)
            or " " not in operation_identity
        ):
            raise RuntimeError(
                f"Malformed operation identity: "
                f"{operation_identity!r}"
            )

        method, path_template = operation_identity.split(
            " ",
            1,
        )

        cache_key = (
            spec_id,
            method.lower(),
            path_template,
        )

        if cache_key not in operation_snapshot_cache:
            (
                source_path_item,
                source_operation,
            ) = get_operation(
                source_document,
                method,
                path_template,
            )

            (
                converted_path_item,
                converted_operation,
            ) = get_operation(
                converted_document,
                method,
                path_template,
            )

            if source_operation is None:
                raise RuntimeError(
                    "Source operation not found: "
                    f"{operation_identity}"
                )

            if converted_operation is None:
                raise RuntimeError(
                    "Converted operation not found: "
                    f"{operation_identity}"
                )

            parameters = operation_parameters(
                source_document,
                source_path_item,
                source_operation,
            )

            body_parameters = [
                project_body_parameter(parameter)
                for parameter in parameters
                if parameter.get("in") == "body"
            ]

            form_parameters = [
                project_form_parameter(parameter)
                for parameter in parameters
                if parameter.get("in") == "formData"
            ]

            effective_consumes = (
                source_operation.get("consumes")
                if "consumes" in source_operation
                else source_document.get(
                    "consumes",
                    [],
                )
            )

            if not isinstance(effective_consumes, list):
                effective_consumes = []

            snapshot = {
                "source_effective_consumes": sorted(
                    effective_consumes
                ),
                "source_body_parameters": (
                    body_parameters
                ),
                "source_form_parameters": (
                    form_parameters
                ),
                "converted_request_body": (
                    project_converted_request_body(
                        converted_document,
                        converted_operation,
                    )
                ),
            }

            operation_snapshot_cache[
                cache_key
            ] = snapshot

        snapshot = operation_snapshot_cache[
            cache_key
        ]

        detail_records.append(
            {
                "spec_id": spec_id,
                "raw_id": record.get("raw_id"),
                "relative_path": relative_path,
                "provider_directory": record.get(
                    "provider_directory"
                ),
                "operation": operation_identity,
                "problem": diagnostic.get(
                    "problem",
                    "UNKNOWN",
                ),
                "diagnostic_index": diagnostic_index,
                "diagnostic": diagnostic,
                "signature": compact_signature(
                    snapshot
                ),
                "snapshot": snapshot,
            }
        )


write_jsonl_atomic(
    DETAIL_PATH,
    detail_records,
)


problem_counts = Counter(
    record["problem"]
    for record in detail_records
)

specifications_by_problem = defaultdict(set)

for record in detail_records:
    specifications_by_problem[
        record["problem"]
    ].add(record["spec_id"])


signature_counts = Counter(
    (
        record["problem"],
        json.dumps(
            record["signature"],
            sort_keys=True,
            ensure_ascii=False,
        ),
    )
    for record in detail_records
)


problem_examples = {}

for problem in sorted(problem_counts):
    candidates = [
        record
        for record in detail_records
        if record["problem"] == problem
    ]

    seen_signatures = set()
    examples = []

    for record in candidates:
        signature_text = json.dumps(
            record["signature"],
            sort_keys=True,
            ensure_ascii=False,
        )

        if signature_text in seen_signatures:
            continue

        seen_signatures.add(signature_text)

        examples.append(
            {
                "spec_id": record["spec_id"],
                "relative_path": record[
                    "relative_path"
                ],
                "operation": record["operation"],
                "problem": problem,
                "diagnostic": record["diagnostic"],
                "signature": record["signature"],
                "snapshot": record["snapshot"],
            }
        )

        if len(examples) >= 5:
            break

    problem_examples[problem] = examples


report = {
    "status": "PASS",
    "audit": (
        "REMAINING_REQUEST_CONTRACT_DIFFERENCE_TRIAGE"
    ),
    "input_specification_count": len(
        input_records
    ),
    "remaining_request_failure_specification_count": (
        len(remaining_records)
    ),
    "diagnostic_count": len(detail_records),
    "unique_operation_count": len(
        operation_snapshot_cache
    ),
    "problem_counts": dict(
        sorted(problem_counts.items())
    ),
    "specification_counts_by_problem": {
        problem: len(spec_ids)
        for problem, spec_ids in sorted(
            specifications_by_problem.items()
        )
    },
    "signature_pattern_count": len(
        signature_counts
    ),
    "signature_pattern_counts": [
        {
            "problem": problem,
            "signature": json.loads(
                signature_text
            ),
            "count": count,
        }
        for (
            problem,
            signature_text,
        ), count in signature_counts.most_common()
    ],
    "problem_examples": problem_examples,
    "details_path": str(DETAIL_PATH),
    "details_sha256": sha256_file(
        DETAIL_PATH
    ),
    "input_sha256": sha256_file(
        INPUT_RESULTS_PATH
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 27: PASS")

{
  "status": "PASS",
  "audit": "REMAINING_REQUEST_CONTRACT_DIFFERENCE_TRIAGE",
  "input_specification_count": 226,
  "remaining_request_failure_specification_count": 14,
  "diagnostic_count": 112,
  "unique_operation_count": 79,
  "problem_counts": {
    "body_description": 23,
    "body_schema": 1,
    "form_media_types": 37,
    "form_property_schema": 17,
    "missing_form_content": 26,
    "missing_request_body": 8
  },
  "specification_counts_by_problem": {
    "body_description": 3,
    "body_schema": 1,
    "form_media_types": 6,
    "form_property_schema": 4,
    "missing_form_content": 2,
    "missing_request_body": 1
  },
  "signature_pattern_count": 44,
  "signature_pattern_counts": [
    {
      "problem": "body_description",
      "signature": {
        "converted_media_types": [
          "application/json",
          "application/x-www-form-urlencoded",
          "application/xml",
          "text/json",
          "text/xml"
        ],
        "converted_request_body_p

In [30]:
from pathlib import Path
from collections import Counter
import copy
import json


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

INPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v3_results.jsonl"
)

TRIAGE_DETAILS_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_remaining_request_contract_differences.jsonl"
)

OUTPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v4_results.jsonl"
)

ASSESSMENTS_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_request_contract_representation_assessments.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_request_contract_representation_adjustment.json"
)


VALID_FORM_MEDIA_TYPES = {
    "application/x-www-form-urlencoded",
    "multipart/form-data",
}


def read_jsonl_cell28(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def canonical_json_cell28(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def normalized_form_property(
    form_parameter,
):
    if not isinstance(form_parameter, dict):
        return None

    schema_fields = copy.deepcopy(
        form_parameter.get(
            "schema_fields",
            {},
        )
    )

    if schema_fields.get("type") == "file":
        schema_fields["type"] = "string"
        schema_fields["format"] = "binary"

    description = form_parameter.get(
        "description"
    )

    if description is not None:
        schema_fields["description"] = description

    extensions = form_parameter.get(
        "x_extensions",
        {},
    )

    if isinstance(extensions, dict):
        schema_fields.update(
            copy.deepcopy(extensions)
        )

    return schema_fields


def converted_content_media_types(snapshot):
    request_body = snapshot.get(
        "converted_request_body"
    )

    if not isinstance(request_body, dict):
        return set()

    content = request_body.get("content", {})

    if not isinstance(content, dict):
        return set()

    return set(content.keys())


def form_media_assessment(snapshot):
    declared_consumes = set(
        snapshot.get(
            "source_effective_consumes",
            [],
        )
    )

    expected_valid_form_media = (
        declared_consumes
        & VALID_FORM_MEDIA_TYPES
    )

    invalid_or_conflicting_media = (
        declared_consumes
        - VALID_FORM_MEDIA_TYPES
    )

    observed_media = (
        converted_content_media_types(snapshot)
    )

    exact_media_preservation = (
        bool(expected_valid_form_media)
        and not invalid_or_conflicting_media
        and observed_media
        == expected_valid_form_media
    )

    return {
        "declared_consumes": sorted(
            declared_consumes
        ),
        "expected_valid_form_media": sorted(
            expected_valid_form_media
        ),
        "invalid_or_conflicting_media": sorted(
            invalid_or_conflicting_media
        ),
        "observed_media": sorted(
            observed_media
        ),
        "exact_media_preservation": (
            exact_media_preservation
        ),
    }


def assess_all_form_properties(
    snapshot,
    media_types,
):
    source_parameters = snapshot.get(
        "source_form_parameters",
        [],
    )

    expected_properties = {
        parameter["name"]: normalized_form_property(
            parameter
        )
        for parameter in source_parameters
    }

    expected_required = sorted(
        parameter["name"]
        for parameter in source_parameters
        if parameter.get("required") is True
    )

    request_body = snapshot.get(
        "converted_request_body"
    )

    if not isinstance(request_body, dict):
        return {
            "pass": False,
            "reason": "converted_request_body_missing",
        }

    content = request_body.get("content", {})

    if not isinstance(content, dict):
        return {
            "pass": False,
            "reason": "converted_content_missing",
        }

    media_results = {}

    for media_type in sorted(media_types):
        media_object = content.get(media_type)

        if not isinstance(media_object, dict):
            media_results[media_type] = {
                "pass": False,
                "reason": "media_object_missing",
            }
            continue

        schema = media_object.get("schema")

        if not isinstance(schema, dict):
            media_results[media_type] = {
                "pass": False,
                "reason": "schema_missing",
            }
            continue

        observed_properties = schema.get(
            "properties",
            {},
        )

        observed_required = sorted(
            schema.get("required", [])
            if isinstance(
                schema.get("required", []),
                list,
            )
            else []
        )

        properties_equal = (
            canonical_json_cell28(
                expected_properties
            )
            == canonical_json_cell28(
                observed_properties
            )
        )

        required_equal = (
            expected_required
            == observed_required
        )

        media_results[media_type] = {
            "pass": (
                schema.get("type") == "object"
                and properties_equal
                and required_equal
            ),
            "schema_type": schema.get("type"),
            "properties_equal": properties_equal,
            "required_equal": required_equal,
            "expected_required": (
                expected_required
            ),
            "observed_required": (
                observed_required
            ),
        }

    return {
        "pass": (
            bool(media_results)
            and all(
                result["pass"]
                for result in media_results.values()
            )
        ),
        "media_results": media_results,
    }


required_inputs = [
    INPUT_RESULTS_PATH,
    TRIAGE_DETAILS_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file() or path.stat().st_size == 0
]

if missing_inputs:
    raise RuntimeError(
        "Missing required inputs:\n"
        + "\n".join(missing_inputs)
    )


input_records = read_jsonl_cell28(
    INPUT_RESULTS_PATH
)

detail_records = read_jsonl_cell28(
    TRIAGE_DETAILS_PATH
)


assessments = []

for detail in detail_records:
    problem = detail["problem"]
    snapshot = detail["snapshot"]
    diagnostic = detail["diagnostic"]

    resolved = False
    status = "RETAIN_FAILURE"
    reason = None
    checks = {}

    body_parameters = snapshot.get(
        "source_body_parameters",
        [],
    )

    request_body = snapshot.get(
        "converted_request_body"
    )

    if problem == "body_description":
        source_descriptions = [
            parameter.get("description")
            for parameter in body_parameters
        ]

        converted_description = (
            request_body.get("description")
            if isinstance(request_body, dict)
            else None
        )

        description_preserved = (
            len(source_descriptions) == 1
            and source_descriptions[0]
            == converted_description
        )

        checks = {
            "source_descriptions": (
                source_descriptions
            ),
            "converted_description": (
                converted_description
            ),
            "description_preserved": (
                description_preserved
            ),
        }

        if description_preserved:
            resolved = True
            status = (
                "RESOLVED_BODY_DESCRIPTION_PRESERVED"
            )
            reason = (
                "The body-parameter description equals "
                "the OpenAPI requestBody description."
            )
        else:
            reason = (
                "The request-body description was missing "
                "or changed."
            )

    elif problem == "body_schema":
        media_type = diagnostic.get(
            "media_type"
        )

        source_schemas = [
            parameter.get("schema")
            for parameter in body_parameters
        ]

        converted_schema = None

        if isinstance(request_body, dict):
            content = request_body.get(
                "content",
                {},
            )

            if isinstance(content, dict):
                media_object = content.get(
                    media_type
                )

                if isinstance(media_object, dict):
                    converted_schema = (
                        media_object.get("schema")
                    )

        schema_preserved = (
            len(source_schemas) == 1
            and canonical_json_cell28(
                source_schemas[0]
            )
            == canonical_json_cell28(
                converted_schema
            )
        )

        checks = {
            "media_type": media_type,
            "schema_preserved": (
                schema_preserved
            ),
            "source_schema": (
                source_schemas[0]
                if len(source_schemas) == 1
                else None
            ),
            "converted_schema": (
                converted_schema
            ),
        }

        if schema_preserved:
            resolved = True
            status = (
                "RESOLVED_BODY_SCHEMA_PRESERVED"
            )
            reason = (
                "The body schema is byte-independent "
                "canonical-JSON equivalent."
            )
        else:
            reason = (
                "The body schema changed during conversion."
            )

    elif problem == "form_property_schema":
        parameter_name = diagnostic.get(
            "parameter"
        )

        media_type = diagnostic.get(
            "media_type"
        )

        source_parameter = next(
            (
                parameter
                for parameter in snapshot.get(
                    "source_form_parameters",
                    [],
                )
                if parameter.get("name")
                == parameter_name
            ),
            None,
        )

        expected_property = (
            normalized_form_property(
                source_parameter
            )
        )

        observed_property = None
        observed_required = False

        if isinstance(request_body, dict):
            content = request_body.get(
                "content",
                {},
            )

            if isinstance(content, dict):
                media_object = content.get(
                    media_type
                )

                if isinstance(media_object, dict):
                    schema = media_object.get(
                        "schema",
                        {},
                    )

                    if isinstance(schema, dict):
                        properties = schema.get(
                            "properties",
                            {},
                        )

                        if isinstance(properties, dict):
                            observed_property = (
                                properties.get(
                                    parameter_name
                                )
                            )

                        required_names = schema.get(
                            "required",
                            [],
                        )

                        if isinstance(
                            required_names,
                            list,
                        ):
                            observed_required = (
                                parameter_name
                                in required_names
                            )

        expected_required = bool(
            source_parameter.get(
                "required",
                False,
            )
            if isinstance(source_parameter, dict)
            else False
        )

        property_preserved = (
            canonical_json_cell28(
                expected_property
            )
            == canonical_json_cell28(
                observed_property
            )
        )

        required_preserved = (
            expected_required
            == observed_required
        )

        checks = {
            "media_type": media_type,
            "parameter": parameter_name,
            "property_preserved": (
                property_preserved
            ),
            "required_preserved": (
                required_preserved
            ),
            "expected_property": (
                expected_property
            ),
            "observed_property": (
                observed_property
            ),
            "expected_required": (
                expected_required
            ),
            "observed_required": (
                observed_required
            ),
        }

        if (
            property_preserved
            and required_preserved
        ):
            resolved = True
            status = (
                "RESOLVED_FORM_PROPERTY_PRESERVED"
            )
            reason = (
                "The form parameter was preserved as an "
                "OpenAPI requestBody object property."
            )
        else:
            reason = (
                "The converted form property or required "
                "membership changed."
            )

    elif problem == "form_media_types":
        media_check = form_media_assessment(
            snapshot
        )

        checks = media_check

        if media_check[
            "exact_media_preservation"
        ]:
            resolved = True
            status = (
                "RESOLVED_FORM_MEDIA_PRESERVED"
            )
            reason = (
                "All valid declared form media types were "
                "preserved exactly."
            )
        else:
            reason = (
                "Declared media types were missing, invalid, "
                "conflicting, or changed."
            )

    elif problem == "missing_form_content":
        media_check = form_media_assessment(
            snapshot
        )

        property_check = (
            assess_all_form_properties(
                snapshot,
                media_check[
                    "expected_valid_form_media"
                ],
            )
        )

        checks = {
            "media_check": media_check,
            "property_check": property_check,
        }

        if (
            media_check[
                "exact_media_preservation"
            ]
            and property_check["pass"]
        ):
            resolved = True
            status = (
                "RESOLVED_FORM_CONTENT_PRESERVED"
            )
            reason = (
                "Valid form media and every form property "
                "were preserved."
            )
        else:
            reason = (
                "The declared form media or property set "
                "was not preserved exactly."
            )

    elif problem == "missing_request_body":
        checks = {
            "source_body_parameter_count": len(
                body_parameters
            ),
            "converted_request_body_present": (
                request_body is not None
            ),
        }

        if (
            len(body_parameters) == 1
            and isinstance(request_body, dict)
        ):
            resolved = True
            status = (
                "RESOLVED_REQUEST_BODY_PRESENT"
            )
            reason = (
                "The converted request body exists."
            )
        else:
            reason = (
                "A source body parameter has no converted "
                "requestBody."
            )

    else:
        checks = {
            "unsupported_problem": problem
        }

        reason = (
            "No approved rule exists for this problem."
        )

    assessments.append(
        {
            "spec_id": detail["spec_id"],
            "relative_path": detail[
                "relative_path"
            ],
            "operation": detail["operation"],
            "problem": problem,
            "parameter": diagnostic.get(
                "parameter"
            ),
            "media_type": diagnostic.get(
                "media_type"
            ),
            "resolved": resolved,
            "resolution_status": status,
            "resolution_reason": reason,
            "checks": checks,
        }
    )


write_jsonl_atomic(
    ASSESSMENTS_PATH,
    assessments,
)


assessment_lookup = {
    (
        assessment["spec_id"],
        assessment["operation"],
        assessment["problem"],
        assessment.get("parameter"),
        assessment.get("media_type"),
    ): assessment
    for assessment in assessments
}


output_records = []
newly_accepted_spec_ids = []

for original_record in input_records:
    record = copy.deepcopy(original_record)

    prior_pass = (
        record.get(
            "adjusted_contract_equivalence_pass"
        ) is True
    )

    prior_diagnostics = copy.deepcopy(
        record.get(
            "adjusted_diagnostics",
            {},
        )
    )

    prior_failed_categories = list(
        record.get(
            "adjusted_failed_categories",
            [],
        )
    )

    request_diagnostics = prior_diagnostics.get(
        "request_contracts_preserved",
        [],
    )

    retained_request_diagnostics = []

    for diagnostic in request_diagnostics:
        lookup_key = (
            record.get("spec_id"),
            diagnostic.get("operation"),
            diagnostic.get("problem"),
            diagnostic.get("parameter"),
            diagnostic.get("media_type"),
        )

        assessment = assessment_lookup.get(
            lookup_key
        )

        if not (
            assessment
            and assessment["resolved"]
        ):
            retained_request_diagnostics.append(
                diagnostic
            )

    if retained_request_diagnostics:
        prior_diagnostics[
            "request_contracts_preserved"
        ] = retained_request_diagnostics
    else:
        prior_diagnostics.pop(
            "request_contracts_preserved",
            None,
        )

    updated_failed_categories = [
        category
        for category in prior_failed_categories
        if category in prior_diagnostics
    ]

    for category in prior_diagnostics:
        if category not in updated_failed_categories:
            updated_failed_categories.append(
                category
            )

    updated_pass = (
        prior_pass
        or len(updated_failed_categories) == 0
    )

    if not prior_pass and updated_pass:
        newly_accepted_spec_ids.append(
            record.get("spec_id")
        )

    record["adjustment_version"] = 4

    record["adjustment"] = (
        "verified_request_contract_"
        "representation_equivalence"
    )

    record[
        "prior_adjusted_contract_equivalence_pass"
    ] = prior_pass

    record[
        "adjusted_contract_equivalence_pass"
    ] = updated_pass

    record["adjusted_failed_categories"] = (
        updated_failed_categories
    )

    record["adjusted_diagnostics"] = (
        prior_diagnostics
    )

    output_records.append(record)


write_jsonl_atomic(
    OUTPUT_RESULTS_PATH,
    output_records,
)


prior_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in input_records
)

adjusted_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in output_records
)

adjusted_failure_count = (
    len(output_records)
    - adjusted_pass_count
)

expected_pass_count = (
    prior_pass_count
    + len(newly_accepted_spec_ids)
)

if adjusted_pass_count != expected_pass_count:
    raise RuntimeError(
        "Adjustment accounting mismatch: "
        f"expected {expected_pass_count}, "
        f"observed {adjusted_pass_count}."
    )


resolution_status_counts = Counter(
    assessment["resolution_status"]
    for assessment in assessments
)

problem_resolution_counts = Counter(
    (
        assessment["problem"],
        (
            "RESOLVED"
            if assessment["resolved"]
            else "RETAINED_FAILURE"
        ),
    )
    for assessment in assessments
)

remaining_failure_category_counts = Counter()

for record in output_records:
    if record.get(
        "adjusted_contract_equivalence_pass"
    ) is True:
        continue

    remaining_failure_category_counts.update(
        record.get(
            "adjusted_failed_categories",
            [],
        )
    )


report = {
    "status": "PASS",
    "adjustment_version": 4,
    "adjustment": (
        "VERIFY_REQUEST_CONTRACT_"
        "REPRESENTATION_EQUIVALENCE"
    ),
    "prior_adjusted_contract_pass_count": (
        prior_pass_count
    ),
    "assessment_count": len(assessments),
    "resolution_status_counts": dict(
        sorted(resolution_status_counts.items())
    ),
    "problem_resolution_counts": {
        f"{problem}::{decision}": count
        for (
            problem,
            decision,
        ), count in sorted(
            problem_resolution_counts.items()
        )
    },
    "newly_accepted_specification_count": len(
        newly_accepted_spec_ids
    ),
    "newly_accepted_spec_ids": sorted(
        spec_id
        for spec_id in newly_accepted_spec_ids
        if spec_id
    ),
    "adjusted_contract_pass_count": (
        adjusted_pass_count
    ),
    "adjusted_contract_failure_count": (
        adjusted_failure_count
    ),
    "remaining_failure_category_counts": dict(
        sorted(
            remaining_failure_category_counts.items()
        )
    ),
    "strict_media_policy": (
        "Invalid, conflicting, or discarded consumes "
        "values are not treated as equivalent."
    ),
    "assessments_path": str(
        ASSESSMENTS_PATH
    ),
    "assessments_sha256": sha256_file(
        ASSESSMENTS_PATH
    ),
    "adjusted_results_path": str(
        OUTPUT_RESULTS_PATH
    ),
    "adjusted_results_sha256": sha256_file(
        OUTPUT_RESULTS_PATH
    ),
    "input_results_sha256": sha256_file(
        INPUT_RESULTS_PATH
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 28: PASS")

{
  "status": "PASS",
  "adjustment_version": 4,
  "adjustment": "VERIFY_REQUEST_CONTRACT_REPRESENTATION_EQUIVALENCE",
  "prior_adjusted_contract_pass_count": 176,
  "assessment_count": 112,
  "resolution_status_counts": {
    "RESOLVED_BODY_SCHEMA_PRESERVED": 1,
    "RESOLVED_FORM_PROPERTY_PRESERVED": 16,
    "RETAIN_FAILURE": 95
  },
  "problem_resolution_counts": {
    "body_description::RETAINED_FAILURE": 23,
    "body_schema::RESOLVED": 1,
    "form_media_types::RETAINED_FAILURE": 37,
    "form_property_schema::RESOLVED": 16,
    "form_property_schema::RETAINED_FAILURE": 1,
    "missing_form_content::RETAINED_FAILURE": 26,
    "missing_request_body::RETAINED_FAILURE": 8
  },
  "newly_accepted_specification_count": 3,
  "newly_accepted_spec_ids": [
    "spec_3036f112c6e4e81ab78af239",
    "spec_57219eb791bdab99188a8a0f",
    "spec_e62c5fdbf01eece7339013d2"
  ],
  "adjusted_contract_pass_count": 179,
  "adjusted_contract_failure_count": 47,
  "remaining_failure_category_counts": {
 

In [31]:
from pathlib import Path
from collections import Counter, defaultdict
import json


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

ASSESSMENTS_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_request_contract_representation_assessments.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_request_contract_residual_triage.json"
)


def read_jsonl_cell29(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def normalize_empty_prose(value):
    if value is None:
        return None

    if isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            return None

        return stripped

    return value


if not ASSESSMENTS_PATH.is_file():
    raise FileNotFoundError(ASSESSMENTS_PATH)


assessments = read_jsonl_cell29(
    ASSESSMENTS_PATH
)

unresolved = [
    record
    for record in assessments
    if not record.get("resolved", False)
]


body_description_records = [
    record
    for record in unresolved
    if record.get("problem")
    == "body_description"
]

form_property_records = [
    record
    for record in unresolved
    if record.get("problem")
    == "form_property_schema"
]


body_description_classifications = []

for record in body_description_records:
    checks = record.get("checks", {})

    source_descriptions = checks.get(
        "source_descriptions",
        [],
    )

    converted_description = checks.get(
        "converted_description"
    )

    if len(source_descriptions) == 1:
        source_description = (
            source_descriptions[0]
        )
    else:
        source_description = None

    normalized_source = normalize_empty_prose(
        source_description
    )

    normalized_converted = normalize_empty_prose(
        converted_description
    )

    if (
        len(source_descriptions) == 1
        and normalized_source
        == normalized_converted
    ):
        classification = (
            "EMPTY_OR_WHITESPACE_EQUIVALENT"
        )
    else:
        classification = (
            "SUBSTANTIVE_DESCRIPTION_DIFFERENCE"
        )

    body_description_classifications.append(
        {
            "spec_id": record["spec_id"],
            "relative_path": record[
                "relative_path"
            ],
            "operation": record["operation"],
            "source_description": (
                source_description
            ),
            "converted_description": (
                converted_description
            ),
            "source_description_length": (
                len(source_description)
                if isinstance(
                    source_description,
                    str,
                )
                else None
            ),
            "converted_description_length": (
                len(converted_description)
                if isinstance(
                    converted_description,
                    str,
                )
                else None
            ),
            "normalized_source": (
                normalized_source
            ),
            "normalized_converted": (
                normalized_converted
            ),
            "classification": classification,
        }
    )


description_classification_counts = Counter(
    record["classification"]
    for record in body_description_classifications
)

description_specification_counts = defaultdict(
    set
)

for record in body_description_classifications:
    description_specification_counts[
        record["classification"]
    ].add(record["spec_id"])


description_examples = {}

for classification in sorted(
    description_classification_counts
):
    description_examples[classification] = [
        record
        for record in body_description_classifications
        if record["classification"]
        == classification
    ][:10]


form_property_details = []

for record in form_property_records:
    checks = record.get("checks", {})

    form_property_details.append(
        {
            "spec_id": record["spec_id"],
            "relative_path": record[
                "relative_path"
            ],
            "operation": record["operation"],
            "parameter": record.get(
                "parameter"
            ),
            "media_type": record.get(
                "media_type"
            ),
            "property_preserved": checks.get(
                "property_preserved"
            ),
            "required_preserved": checks.get(
                "required_preserved"
            ),
            "expected_property": checks.get(
                "expected_property"
            ),
            "observed_property": checks.get(
                "observed_property"
            ),
            "expected_required": checks.get(
                "expected_required"
            ),
            "observed_required": checks.get(
                "observed_required"
            ),
        }
    )


other_unresolved_problem_counts = Counter(
    record.get("problem", "UNKNOWN")
    for record in unresolved
    if record.get("problem")
    not in {
        "body_description",
        "form_property_schema",
    }
)


report = {
    "status": "PASS",
    "audit": (
        "REQUEST_CONTRACT_RESIDUAL_TRIAGE"
    ),
    "assessment_count": len(assessments),
    "unresolved_assessment_count": len(
        unresolved
    ),
    "body_description_assessment_count": len(
        body_description_records
    ),
    "body_description_classification_counts": (
        dict(
            sorted(
                description_classification_counts.items()
            )
        )
    ),
    "body_description_specification_counts": {
        classification: len(spec_ids)
        for classification, spec_ids in sorted(
            description_specification_counts.items()
        )
    },
    "body_description_examples": (
        description_examples
    ),
    "unresolved_form_property_count": len(
        form_property_records
    ),
    "unresolved_form_property_details": (
        form_property_details
    ),
    "other_unresolved_problem_counts": dict(
        sorted(
            other_unresolved_problem_counts.items()
        )
    ),
    "assessments_sha256": sha256_file(
        ASSESSMENTS_PATH
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 29: PASS")

{
  "status": "PASS",
  "audit": "REQUEST_CONTRACT_RESIDUAL_TRIAGE",
  "assessment_count": 112,
  "unresolved_assessment_count": 95,
  "body_description_assessment_count": 23,
  "body_description_classification_counts": {
    "EMPTY_OR_WHITESPACE_EQUIVALENT": 23
  },
  "body_description_specification_counts": {
    "EMPTY_OR_WHITESPACE_EQUIVALENT": 3
  },
  "body_description_examples": {
    "EMPTY_OR_WHITESPACE_EQUIVALENT": [
      {
        "spec_id": "spec_3cf8dc18bab5dfe538d03fce",
        "relative_path": "avaza.com/v1/swagger.yaml",
        "operation": "POST /ScheduleSeries/AddBooking",
        "source_description": "",
        "converted_description": null,
        "source_description_length": 0,
        "converted_description_length": null,
        "normalized_source": null,
        "normalized_converted": null,
        "classification": "EMPTY_OR_WHITESPACE_EQUIVALENT"
      },
      {
        "spec_id": "spec_3cf8dc18bab5dfe538d03fce",
        "relative_path": "avaza.com/v1/

In [32]:
from pathlib import Path
from collections import Counter
import copy
import json


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

INPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v4_results.jsonl"
)

REQUEST_ASSESSMENTS_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_request_contract_representation_assessments.jsonl"
)

OUTPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v5_results.jsonl"
)

RESIDUAL_ASSESSMENTS_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_request_contract_residual_assessments.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_request_contract_residual_adjustment.json"
)


def read_jsonl_cell30(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def canonical_json_cell30(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def normalize_empty_prose_cell30(value):
    if value is None:
        return None

    if isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            return None

        return stripped

    return value


def remove_inapplicable_collection_formats(
    value,
    current_path="",
):
    removed = []

    if isinstance(value, dict):
        value_type = value.get("type")

        if (
            "collectionFormat" in value
            and value_type != "array"
        ):
            removed.append(
                {
                    "path": (
                        f"{current_path}/collectionFormat"
                        if current_path
                        else "/collectionFormat"
                    ),
                    "value": value[
                        "collectionFormat"
                    ],
                    "containing_type": value_type,
                }
            )

            value.pop("collectionFormat")

        for key in list(value.keys()):
            child_path = (
                f"{current_path}/{key}"
                if current_path
                else f"/{key}"
            )

            removed.extend(
                remove_inapplicable_collection_formats(
                    value[key],
                    child_path,
                )
            )

    elif isinstance(value, list):
        for index, item in enumerate(value):
            child_path = (
                f"{current_path}/{index}"
                if current_path
                else f"/{index}"
            )

            removed.extend(
                remove_inapplicable_collection_formats(
                    item,
                    child_path,
                )
            )

    return removed


required_inputs = [
    INPUT_RESULTS_PATH,
    REQUEST_ASSESSMENTS_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file() or path.stat().st_size == 0
]

if missing_inputs:
    raise RuntimeError(
        "Missing required inputs:\n"
        + "\n".join(missing_inputs)
    )


input_records = read_jsonl_cell30(
    INPUT_RESULTS_PATH
)

request_assessments = read_jsonl_cell30(
    REQUEST_ASSESSMENTS_PATH
)

unresolved_candidates = [
    assessment
    for assessment in request_assessments
    if (
        not assessment.get("resolved", False)
        and assessment.get("problem")
        in {
            "body_description",
            "form_property_schema",
        }
    )
]


residual_assessments = []

for assessment in unresolved_candidates:
    problem = assessment["problem"]
    checks = assessment.get("checks", {})

    resolved = False
    status = "RETAIN_FAILURE"
    reason = None
    additional_checks = {}

    if problem == "body_description":
        source_descriptions = checks.get(
            "source_descriptions",
            [],
        )

        converted_description = checks.get(
            "converted_description"
        )

        if len(source_descriptions) == 1:
            source_description = (
                source_descriptions[0]
            )
        else:
            source_description = None

        normalized_source = (
            normalize_empty_prose_cell30(
                source_description
            )
        )

        normalized_converted = (
            normalize_empty_prose_cell30(
                converted_description
            )
        )

        resolved = (
            len(source_descriptions) == 1
            and normalized_source
            == normalized_converted
        )

        additional_checks = {
            "source_description": (
                source_description
            ),
            "converted_description": (
                converted_description
            ),
            "normalized_source": (
                normalized_source
            ),
            "normalized_converted": (
                normalized_converted
            ),
        }

        if resolved:
            status = (
                "RESOLVED_EMPTY_DESCRIPTION_EQUIVALENT"
            )
            reason = (
                "An empty or whitespace-only source "
                "description contains no prose and is "
                "equivalent to an absent description."
            )
        else:
            reason = (
                "The descriptions contain different "
                "nonempty prose."
            )

    elif problem == "form_property_schema":
        expected_property = copy.deepcopy(
            checks.get("expected_property")
        )

        observed_property = copy.deepcopy(
            checks.get("observed_property")
        )

        removed_keywords = []

        if isinstance(expected_property, dict):
            removed_keywords = (
                remove_inapplicable_collection_formats(
                    expected_property
                )
            )

        properties_equal = (
            canonical_json_cell30(
                expected_property
            )
            == canonical_json_cell30(
                observed_property
            )
        )

        required_equal = (
            checks.get("expected_required")
            == checks.get("observed_required")
        )

        removed_keywords_safe = (
            bool(removed_keywords)
            and all(
                item.get("containing_type")
                != "array"
                for item in removed_keywords
            )
        )

        resolved = (
            properties_equal
            and required_equal
            and removed_keywords_safe
        )

        additional_checks = {
            "normalized_expected_property": (
                expected_property
            ),
            "observed_property": (
                observed_property
            ),
            "properties_equal": (
                properties_equal
            ),
            "required_equal": (
                required_equal
            ),
            "removed_inapplicable_keywords": (
                removed_keywords
            ),
            "removed_keywords_safe": (
                removed_keywords_safe
            ),
        }

        if resolved:
            status = (
                "RESOLVED_INAPPLICABLE_ITEM_"
                "COLLECTION_FORMAT"
            )
            reason = (
                "The only difference was collectionFormat "
                "inside a non-array Items Object, where the "
                "keyword has no serialization effect."
            )
        else:
            reason = (
                "A substantive form-property difference "
                "remains after normalization."
            )

    residual_assessments.append(
        {
            "spec_id": assessment["spec_id"],
            "relative_path": assessment[
                "relative_path"
            ],
            "operation": assessment["operation"],
            "problem": problem,
            "parameter": assessment.get(
                "parameter"
            ),
            "media_type": assessment.get(
                "media_type"
            ),
            "resolved": resolved,
            "resolution_status": status,
            "resolution_reason": reason,
            "checks": additional_checks,
        }
    )


write_jsonl_atomic(
    RESIDUAL_ASSESSMENTS_PATH,
    residual_assessments,
)


resolution_lookup = {
    (
        assessment["spec_id"],
        assessment["operation"],
        assessment["problem"],
        assessment.get("parameter"),
        assessment.get("media_type"),
    ): assessment
    for assessment in residual_assessments
}


output_records = []
newly_accepted_spec_ids = []

for original_record in input_records:
    record = copy.deepcopy(original_record)

    prior_pass = (
        record.get(
            "adjusted_contract_equivalence_pass"
        ) is True
    )

    diagnostics = copy.deepcopy(
        record.get(
            "adjusted_diagnostics",
            {},
        )
    )

    prior_failed_categories = list(
        record.get(
            "adjusted_failed_categories",
            [],
        )
    )

    request_diagnostics = diagnostics.get(
        "request_contracts_preserved",
        [],
    )

    retained_request_diagnostics = []

    for diagnostic in request_diagnostics:
        lookup_key = (
            record.get("spec_id"),
            diagnostic.get("operation"),
            diagnostic.get("problem"),
            diagnostic.get("parameter"),
            diagnostic.get("media_type"),
        )

        resolution = resolution_lookup.get(
            lookup_key
        )

        if not (
            resolution
            and resolution["resolved"]
        ):
            retained_request_diagnostics.append(
                diagnostic
            )

    if retained_request_diagnostics:
        diagnostics[
            "request_contracts_preserved"
        ] = retained_request_diagnostics
    else:
        diagnostics.pop(
            "request_contracts_preserved",
            None,
        )

    updated_failed_categories = [
        category
        for category in prior_failed_categories
        if category in diagnostics
    ]

    for category in diagnostics:
        if category not in updated_failed_categories:
            updated_failed_categories.append(
                category
            )

    updated_pass = (
        prior_pass
        or len(updated_failed_categories) == 0
    )

    if not prior_pass and updated_pass:
        newly_accepted_spec_ids.append(
            record.get("spec_id")
        )

    record["adjustment_version"] = 5

    record["adjustment"] = (
        "verified_request_contract_"
        "residual_equivalence"
    )

    record[
        "prior_adjusted_contract_equivalence_pass"
    ] = prior_pass

    record[
        "adjusted_contract_equivalence_pass"
    ] = updated_pass

    record["adjusted_failed_categories"] = (
        updated_failed_categories
    )

    record["adjusted_diagnostics"] = diagnostics

    output_records.append(record)


write_jsonl_atomic(
    OUTPUT_RESULTS_PATH,
    output_records,
)


prior_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in input_records
)

adjusted_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in output_records
)

adjusted_failure_count = (
    len(output_records)
    - adjusted_pass_count
)

expected_pass_count = (
    prior_pass_count
    + len(newly_accepted_spec_ids)
)

if adjusted_pass_count != expected_pass_count:
    raise RuntimeError(
        "Adjustment accounting mismatch: "
        f"expected {expected_pass_count}, "
        f"observed {adjusted_pass_count}."
    )


resolution_status_counts = Counter(
    assessment["resolution_status"]
    for assessment in residual_assessments
)

remaining_failure_category_counts = Counter()

for record in output_records:
    if record.get(
        "adjusted_contract_equivalence_pass"
    ) is True:
        continue

    remaining_failure_category_counts.update(
        record.get(
            "adjusted_failed_categories",
            [],
        )
    )


report = {
    "status": "PASS",
    "adjustment_version": 5,
    "adjustment": (
        "VERIFY_REQUEST_CONTRACT_RESIDUAL_EQUIVALENCE"
    ),
    "prior_adjusted_contract_pass_count": (
        prior_pass_count
    ),
    "residual_assessment_count": len(
        residual_assessments
    ),
    "resolution_status_counts": dict(
        sorted(
            resolution_status_counts.items()
        )
    ),
    "resolved_assessment_count": sum(
        assessment["resolved"]
        for assessment in residual_assessments
    ),
    "retained_failure_count": sum(
        not assessment["resolved"]
        for assessment in residual_assessments
    ),
    "newly_accepted_specification_count": len(
        newly_accepted_spec_ids
    ),
    "newly_accepted_spec_ids": sorted(
        spec_id
        for spec_id in newly_accepted_spec_ids
        if spec_id
    ),
    "adjusted_contract_pass_count": (
        adjusted_pass_count
    ),
    "adjusted_contract_failure_count": (
        adjusted_failure_count
    ),
    "remaining_failure_category_counts": dict(
        sorted(
            remaining_failure_category_counts.items()
        )
    ),
    "residual_assessments_path": str(
        RESIDUAL_ASSESSMENTS_PATH
    ),
    "residual_assessments_sha256": sha256_file(
        RESIDUAL_ASSESSMENTS_PATH
    ),
    "adjusted_results_path": str(
        OUTPUT_RESULTS_PATH
    ),
    "adjusted_results_sha256": sha256_file(
        OUTPUT_RESULTS_PATH
    ),
    "input_results_sha256": sha256_file(
        INPUT_RESULTS_PATH
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 30: PASS")

{
  "status": "PASS",
  "adjustment_version": 5,
  "adjustment": "VERIFY_REQUEST_CONTRACT_RESIDUAL_EQUIVALENCE",
  "prior_adjusted_contract_pass_count": 179,
  "residual_assessment_count": 24,
  "resolution_status_counts": {
    "RESOLVED_EMPTY_DESCRIPTION_EQUIVALENT": 23,
    "RESOLVED_INAPPLICABLE_ITEM_COLLECTION_FORMAT": 1
  },
  "resolved_assessment_count": 24,
  "retained_failure_count": 0,
  "newly_accepted_specification_count": 3,
  "newly_accepted_spec_ids": [
    "spec_05bad46c4c418e6faa4290ff",
    "spec_1d81b27802b8adf756f91e43",
    "spec_f1d1f7f6e23b909f5c0d6a92"
  ],
  "adjusted_contract_pass_count": 182,
  "adjusted_contract_failure_count": 44,
  "remaining_failure_category_counts": {
    "nonbody_parameters_preserved": 2,
    "request_contracts_preserved": 7,
    "response_contracts_preserved": 36,
    "security_schemes_preserved": 1
  },
  "residual_assessments_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/audits/swagger_request_contract_residu

In [33]:
from pathlib import Path
from collections import Counter, defaultdict
import copy
import hashlib
import json


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

INPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v5_results.jsonl"
)

DETAIL_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_response_contract_representation_triage.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_response_contract_representation_triage.json"
)


def read_jsonl_cell31(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def canonical_json_cell31(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def canonical_hash_cell31(value):
    return hashlib.sha256(
        canonical_json_cell31(value).encode(
            "utf-8"
        )
    ).hexdigest()


def compact_value_cell31(value, limit=300):
    serialized = canonical_json_cell31(value)

    return {
        "type": type(value).__name__,
        "bytes": len(
            serialized.encode("utf-8")
        ),
        "sha256": hashlib.sha256(
            serialized.encode("utf-8")
        ).hexdigest(),
        "preview": (
            serialized
            if len(serialized) <= limit
            else serialized[:limit] + "..."
        ),
    }


def remap_swagger_references(value):
    if isinstance(value, dict):
        result = {}

        for key, child in value.items():
            if (
                key == "$ref"
                and isinstance(child, str)
            ):
                if child.startswith(
                    "#/definitions/"
                ):
                    child = child.replace(
                        "#/definitions/",
                        "#/components/schemas/",
                        1,
                    )

                elif child.startswith(
                    "#/parameters/"
                ):
                    child = child.replace(
                        "#/parameters/",
                        "#/components/parameters/",
                        1,
                    )

                elif child.startswith(
                    "#/responses/"
                ):
                    child = child.replace(
                        "#/responses/",
                        "#/components/responses/",
                        1,
                    )

            result[key] = remap_swagger_references(
                child
            )

        return result

    if isinstance(value, list):
        return [
            remap_swagger_references(item)
            for item in value
        ]

    return value


def get_operation_cell31(
    document,
    method,
    path_template,
):
    paths = document.get("paths", {})

    if (
        not isinstance(paths, dict)
        or path_template not in paths
    ):
        return None

    path_item = resolve_local_reference(
        document,
        paths[path_template],
    )

    if not isinstance(path_item, dict):
        return None

    operation = resolve_local_reference(
        document,
        path_item.get(method.lower(), {}),
    )

    return (
        operation
        if isinstance(operation, dict)
        else None
    )


def get_response_cell31(
    document,
    operation,
    status,
):
    if not isinstance(operation, dict):
        return None

    responses = operation.get("responses", {})

    if not isinstance(responses, dict):
        return None

    candidates = [
        status,
        str(status),
    ]

    try:
        candidates.append(int(status))
    except Exception:
        pass

    response = None

    for candidate in candidates:
        if candidate in responses:
            response = responses[candidate]
            break

    if response is None:
        return None

    try:
        response = resolve_local_reference(
            document,
            response,
        )
    except Exception:
        return None

    return (
        response
        if isinstance(response, dict)
        else None
    )


def effective_produces_cell31(
    document,
    operation,
):
    if (
        isinstance(operation, dict)
        and "produces" in operation
    ):
        produces = operation.get("produces")
    else:
        produces = document.get("produces", [])

    if not isinstance(produces, list):
        return []

    return sorted(
        str(value)
        for value in produces
    )


if not INPUT_RESULTS_PATH.is_file():
    raise FileNotFoundError(
        INPUT_RESULTS_PATH
    )


input_records = read_jsonl_cell31(
    INPUT_RESULTS_PATH
)

remaining_records = [
    record
    for record in input_records
    if (
        record.get(
            "adjusted_contract_equivalence_pass"
        ) is not True
        and "response_contracts_preserved"
        in record.get(
            "adjusted_failed_categories",
            [],
        )
    )
]


source_cache = {}
converted_cache = {}
detail_records = []

for record in remaining_records:
    spec_id = record["spec_id"]
    relative_path = record["relative_path"]

    source_path = CORPUS_ROOT / relative_path

    converted_path = (
        PROJECT_ROOT
        / "data/canonical"
        / f"{spec_id}.json"
    )

    if not source_path.is_file():
        raise FileNotFoundError(source_path)

    if not converted_path.is_file():
        raise FileNotFoundError(converted_path)

    if spec_id not in source_cache:
        source_cache[spec_id] = load_document(
            source_path
        )

    if spec_id not in converted_cache:
        converted_cache[spec_id] = load_document(
            converted_path
        )

    source_document = source_cache[spec_id]
    converted_document = converted_cache[spec_id]

    diagnostics = record.get(
        "adjusted_diagnostics",
        {},
    ).get(
        "response_contracts_preserved",
        [],
    )

    for diagnostic in diagnostics:
        operation_identity = diagnostic.get(
            "operation"
        )

        status_code = diagnostic.get(
            "status"
        )

        problem = diagnostic.get(
            "problem",
            "UNKNOWN",
        )

        media_type = diagnostic.get(
            "media_type"
        )

        if (
            not isinstance(operation_identity, str)
            or " " not in operation_identity
        ):
            raise RuntimeError(
                "Malformed operation identity: "
                f"{operation_identity!r}"
            )

        method, path_template = (
            operation_identity.split(" ", 1)
        )

        source_operation = get_operation_cell31(
            source_document,
            method,
            path_template,
        )

        converted_operation = (
            get_operation_cell31(
                converted_document,
                method,
                path_template,
            )
        )

        source_response = get_response_cell31(
            source_document,
            source_operation,
            status_code,
        )

        converted_response = get_response_cell31(
            converted_document,
            converted_operation,
            status_code,
        )

        source_produces = (
            effective_produces_cell31(
                source_document,
                source_operation,
            )
        )

        source_schema = (
            source_response.get("schema")
            if isinstance(source_response, dict)
            else None
        )

        normalized_source_schema = (
            remap_swagger_references(
                source_schema
            )
        )

        source_examples = (
            source_response.get("examples", {})
            if isinstance(source_response, dict)
            else {}
        )

        if not isinstance(source_examples, dict):
            source_examples = {}

        converted_content = (
            converted_response.get("content", {})
            if isinstance(converted_response, dict)
            else {}
        )

        if not isinstance(converted_content, dict):
            converted_content = {}

        converted_media_types = sorted(
            converted_content.keys()
        )

        converted_media_object = (
            converted_content.get(media_type)
            if media_type is not None
            else None
        )

        converted_schema = (
            converted_media_object.get("schema")
            if isinstance(
                converted_media_object,
                dict,
            )
            else None
        )

        source_example = (
            source_examples.get(media_type)
            if media_type is not None
            else None
        )

        converted_example = None

        if isinstance(
            converted_media_object,
            dict,
        ):
            if "example" in converted_media_object:
                converted_example = (
                    converted_media_object.get(
                        "example"
                    )
                )

            elif isinstance(
                converted_media_object.get(
                    "examples"
                ),
                dict,
            ):
                converted_example = (
                    converted_media_object.get(
                        "examples"
                    )
                )

        if problem == "response_schema":
            if converted_media_object is None:
                comparison_status = (
                    "TARGET_MEDIA_MISSING"
                )
            elif (
                canonical_json_cell31(
                    normalized_source_schema
                )
                == canonical_json_cell31(
                    converted_schema
                )
            ):
                comparison_status = (
                    "SCHEMA_PRESERVED_AFTER_REF_REMAP"
                )
            else:
                comparison_status = (
                    "SCHEMA_VALUE_MISMATCH"
                )

        elif problem == "response_example":
            if media_type not in source_examples:
                comparison_status = (
                    "SOURCE_EXAMPLE_MISSING"
                )
            elif converted_media_object is None:
                comparison_status = (
                    "TARGET_MEDIA_MISSING"
                )
            elif (
                canonical_json_cell31(
                    source_example
                )
                == canonical_json_cell31(
                    converted_example
                )
            ):
                comparison_status = (
                    "EXAMPLE_PRESERVED"
                )
            else:
                comparison_status = (
                    "EXAMPLE_VALUE_MISMATCH"
                )

        elif problem == "response_media_types":
            expected_media = sorted(
                diagnostic.get(
                    "expected",
                    source_produces,
                )
            )

            observed_media = sorted(
                diagnostic.get(
                    "observed",
                    converted_media_types,
                )
            )

            if expected_media == observed_media:
                comparison_status = (
                    "MEDIA_TYPES_PRESERVED"
                )
            else:
                comparison_status = (
                    "MEDIA_TYPES_DIFFER"
                )

        elif problem == "missing_response_content":
            if converted_content:
                comparison_status = (
                    "CONTENT_PRESENT"
                )
            else:
                comparison_status = (
                    "CONTENT_MISSING"
                )

        else:
            comparison_status = (
                "UNSUPPORTED_PROBLEM"
            )

        detail_records.append(
            {
                "spec_id": spec_id,
                "relative_path": relative_path,
                "operation": operation_identity,
                "status": status_code,
                "problem": problem,
                "media_type": media_type,
                "comparison_status": (
                    comparison_status
                ),
                "source_produces": (
                    source_produces
                ),
                "converted_media_types": (
                    converted_media_types
                ),
                "source_response_present": (
                    source_response is not None
                ),
                "converted_response_present": (
                    converted_response is not None
                ),
                "source_schema_present": (
                    source_schema is not None
                ),
                "source_schema_fingerprint": (
                    compact_value_cell31(
                        normalized_source_schema
                    )
                    if source_schema is not None
                    else None
                ),
                "converted_schema_fingerprint": (
                    compact_value_cell31(
                        converted_schema
                    )
                    if converted_schema is not None
                    else None
                ),
                "source_example_fingerprint": (
                    compact_value_cell31(
                        source_example
                    )
                    if media_type in source_examples
                    else None
                ),
                "converted_example_fingerprint": (
                    compact_value_cell31(
                        converted_example
                    )
                    if converted_media_object
                    is not None
                    else None
                ),
                "diagnostic": diagnostic,
            }
        )


write_jsonl_atomic(
    DETAIL_PATH,
    detail_records,
)


problem_counts = Counter(
    record["problem"]
    for record in detail_records
)

comparison_status_counts = Counter(
    record["comparison_status"]
    for record in detail_records
)

problem_status_counts = Counter(
    (
        record["problem"],
        record["comparison_status"],
    )
    for record in detail_records
)

specifications_by_problem = defaultdict(set)

for record in detail_records:
    specifications_by_problem[
        record["problem"]
    ].add(record["spec_id"])


mismatch_examples = {}

for problem in sorted(problem_counts):
    candidates = [
        record
        for record in detail_records
        if (
            record["problem"] == problem
            and record["comparison_status"]
            not in {
                "SCHEMA_PRESERVED_AFTER_REF_REMAP",
                "EXAMPLE_PRESERVED",
                "MEDIA_TYPES_PRESERVED",
                "CONTENT_PRESENT",
            }
        )
    ]

    seen = set()
    examples = []

    for record in candidates:
        identity = (
            record["comparison_status"],
            canonical_json_cell31(
                record.get(
                    "source_schema_fingerprint"
                )
            ),
            canonical_json_cell31(
                record.get(
                    "converted_schema_fingerprint"
                )
            ),
            canonical_json_cell31(
                record.get(
                    "source_example_fingerprint"
                )
            ),
            canonical_json_cell31(
                record.get(
                    "converted_example_fingerprint"
                )
            ),
        )

        if identity in seen:
            continue

        seen.add(identity)

        examples.append(record)

        if len(examples) >= 10:
            break

    mismatch_examples[problem] = examples


report = {
    "status": "PASS",
    "audit": (
        "RESPONSE_CONTRACT_REPRESENTATION_TRIAGE"
    ),
    "input_specification_count": len(
        input_records
    ),
    "remaining_response_failure_specification_count": (
        len(remaining_records)
    ),
    "diagnostic_count": len(detail_records),
    "problem_counts": dict(
        sorted(problem_counts.items())
    ),
    "comparison_status_counts": dict(
        sorted(
            comparison_status_counts.items()
        )
    ),
    "problem_status_counts": {
        f"{problem}::{status}": count
        for (
            problem,
            status,
        ), count in sorted(
            problem_status_counts.items()
        )
    },
    "specification_counts_by_problem": {
        problem: len(spec_ids)
        for problem, spec_ids in sorted(
            specifications_by_problem.items()
        )
    },
    "mismatch_examples": mismatch_examples,
    "details_path": str(DETAIL_PATH),
    "details_sha256": sha256_file(
        DETAIL_PATH
    ),
    "input_results_sha256": sha256_file(
        INPUT_RESULTS_PATH
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 31: PASS")

{
  "status": "PASS",
  "audit": "RESPONSE_CONTRACT_REPRESENTATION_TRIAGE",
  "input_specification_count": 226,
  "remaining_response_failure_specification_count": 36,
  "diagnostic_count": 460,
  "problem_counts": {
    "missing_response_content": 10,
    "response_example": 390,
    "response_media_types": 11,
    "response_schema": 49
  },
  "comparison_status_counts": {
    "CONTENT_PRESENT": 10,
    "EXAMPLE_VALUE_MISMATCH": 390,
    "MEDIA_TYPES_DIFFER": 11,
    "SCHEMA_VALUE_MISMATCH": 49
  },
  "problem_status_counts": {
    "missing_response_content::CONTENT_PRESENT": 10,
    "response_example::EXAMPLE_VALUE_MISMATCH": 390,
    "response_media_types::MEDIA_TYPES_DIFFER": 11,
    "response_schema::SCHEMA_VALUE_MISMATCH": 49
  },
  "specification_counts_by_problem": {
    "missing_response_content": 1,
    "response_example": 24,
    "response_media_types": 2,
    "response_schema": 11
  },
  "mismatch_examples": {
    "missing_response_content": [],
    "response_example": [
  

In [34]:
from pathlib import Path
from collections import Counter
from datetime import date, datetime, time
import copy
import json


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CORPUS_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/apis-dataset/APIs"
)

INPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v5_results.jsonl"
)

TRIAGE_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_response_contract_representation_triage.jsonl"
)

OUTPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v6_results.jsonl"
)

ASSESSMENTS_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "swagger_response_contract_equivalence_assessments.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "swagger_response_contract_equivalence_adjustment.json"
)


def read_jsonl_cell32(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def canonical_json_cell32(value):
    def json_default(item):
        if isinstance(
            item,
            (date, datetime, time),
        ):
            return item.isoformat()

        raise TypeError(
            f"Object of type "
            f"{item.__class__.__name__} "
            f"is not JSON serializable"
        )

    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=json_default,
    )


def decode_pointer_cell32(token):
    return token.replace("~1", "/").replace(
        "~0",
        "~",
    )


def resolve_reference_cell32(
    document,
    reference,
):
    if (
        not isinstance(reference, str)
        or not reference.startswith("#/")
    ):
        raise KeyError(reference)

    current = document

    for token in reference[2:].split("/"):
        token = decode_pointer_cell32(token)

        if not isinstance(current, dict):
            raise KeyError(reference)

        current = current[token]

    return current


def remap_reference_cell32(reference):
    if not isinstance(reference, str):
        return reference

    replacements = [
        (
            "#/definitions/",
            "#/components/schemas/",
        ),
        (
            "#/parameters/",
            "#/components/parameters/",
        ),
        (
            "#/responses/",
            "#/components/responses/",
        ),
    ]

    for source_prefix, target_prefix in replacements:
        if reference.startswith(source_prefix):
            return reference.replace(
                source_prefix,
                target_prefix,
                1,
            )

    return reference


def normalize_schema_cell32(
    document,
    value,
    reference_stack=None,
    depth=0,
):
    if reference_stack is None:
        reference_stack = []

    if depth > 100:
        raise RuntimeError(
            "Schema normalization exceeded depth 100."
        )

    if isinstance(value, (date, datetime, time)):
        return value.isoformat()

    if isinstance(value, dict):
        reference = value.get("$ref")

        if (
            isinstance(reference, str)
            and reference.startswith("#/")
        ):
            mapped_reference = (
                remap_reference_cell32(reference)
            )

            if reference in reference_stack:
                return {
                    "$recursiveRef": mapped_reference
                }

            try:
                resolved = resolve_reference_cell32(
                    document,
                    reference,
                )
            except Exception:
                return {
                    "$unresolvedRef": mapped_reference
                }

            # JSON Reference siblings do not contribute to the
            # referenced schema semantics.
            return normalize_schema_cell32(
                document,
                resolved,
                reference_stack + [reference],
                depth + 1,
            )

        normalized = {}

        for key, child in value.items():
            normalized[key] = (
                normalize_schema_cell32(
                    document,
                    child,
                    reference_stack,
                    depth + 1,
                )
            )

        if normalized.get("type") == "file":
            normalized["type"] = "string"
            normalized["format"] = "binary"

        return normalized

    if isinstance(value, list):
        return [
            normalize_schema_cell32(
                document,
                item,
                reference_stack,
                depth + 1,
            )
            for item in value
        ]

    return value


def get_operation_cell32(
    document,
    method,
    path_template,
):
    paths = document.get("paths", {})

    if (
        not isinstance(paths, dict)
        or path_template not in paths
    ):
        return None

    path_item = paths[path_template]

    if (
        isinstance(path_item, dict)
        and isinstance(
            path_item.get("$ref"),
            str,
        )
    ):
        try:
            path_item = resolve_reference_cell32(
                document,
                path_item["$ref"],
            )
        except Exception:
            return None

    if not isinstance(path_item, dict):
        return None

    operation = path_item.get(
        method.lower()
    )

    if (
        isinstance(operation, dict)
        and isinstance(
            operation.get("$ref"),
            str,
        )
    ):
        try:
            operation = resolve_reference_cell32(
                document,
                operation["$ref"],
            )
        except Exception:
            return None

    return (
        operation
        if isinstance(operation, dict)
        else None
    )


def get_response_cell32(
    document,
    operation,
    status,
):
    if not isinstance(operation, dict):
        return None

    responses = operation.get("responses", {})

    if not isinstance(responses, dict):
        return None

    candidates = [
        status,
        str(status),
    ]

    try:
        candidates.append(int(status))
    except Exception:
        pass

    response = None

    for candidate in candidates:
        if candidate in responses:
            response = responses[candidate]
            break

    if response is None:
        return None

    if (
        isinstance(response, dict)
        and isinstance(
            response.get("$ref"),
            str,
        )
    ):
        try:
            response = resolve_reference_cell32(
                document,
                response["$ref"],
            )
        except Exception:
            return None

    return (
        response
        if isinstance(response, dict)
        else None
    )


def effective_produces_cell32(
    document,
    operation,
):
    if (
        isinstance(operation, dict)
        and "produces" in operation
    ):
        produces = operation.get("produces")
    else:
        produces = document.get("produces", [])

    if not isinstance(produces, list):
        return []

    return sorted(
        str(value)
        for value in produces
    )


def converted_example_candidates(
    media_object,
):
    candidates = []

    if not isinstance(media_object, dict):
        return candidates

    if "example" in media_object:
        candidates.append(
            {
                "location": "example",
                "value": media_object.get(
                    "example"
                ),
            }
        )

    examples = media_object.get("examples")

    if isinstance(examples, dict):
        for example_name, example_object in (
            examples.items()
        ):
            if (
                isinstance(example_object, dict)
                and "value" in example_object
            ):
                value = example_object["value"]
            else:
                value = example_object

            candidates.append(
                {
                    "location": (
                        f"examples/{example_name}/value"
                    ),
                    "value": value,
                }
            )

    return candidates


required_inputs = [
    INPUT_RESULTS_PATH,
    TRIAGE_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file() or path.stat().st_size == 0
]

if missing_inputs:
    raise RuntimeError(
        "Missing required inputs:\n"
        + "\n".join(missing_inputs)
    )


input_records = read_jsonl_cell32(
    INPUT_RESULTS_PATH
)

triage_records = read_jsonl_cell32(
    TRIAGE_PATH
)


source_cache = {}
converted_cache = {}
assessments = []

for triage in triage_records:
    spec_id = triage["spec_id"]
    relative_path = triage["relative_path"]
    operation_identity = triage["operation"]
    status_code = triage["status"]
    problem = triage["problem"]
    media_type = triage.get("media_type")

    method, path_template = (
        operation_identity.split(" ", 1)
    )

    source_path = CORPUS_ROOT / relative_path

    converted_path = (
        PROJECT_ROOT
        / "data/canonical"
        / f"{spec_id}.json"
    )

    if not source_path.is_file():
        raise FileNotFoundError(source_path)

    if not converted_path.is_file():
        raise FileNotFoundError(converted_path)

    if spec_id not in source_cache:
        source_cache[spec_id] = load_document(
            source_path
        )

    if spec_id not in converted_cache:
        converted_cache[spec_id] = load_document(
            converted_path
        )

    source_document = source_cache[spec_id]
    converted_document = converted_cache[spec_id]

    source_operation = get_operation_cell32(
        source_document,
        method,
        path_template,
    )

    converted_operation = get_operation_cell32(
        converted_document,
        method,
        path_template,
    )

    source_response = get_response_cell32(
        source_document,
        source_operation,
        status_code,
    )

    converted_response = get_response_cell32(
        converted_document,
        converted_operation,
        status_code,
    )

    source_produces = effective_produces_cell32(
        source_document,
        source_operation,
    )

    source_schema = (
        source_response.get("schema")
        if isinstance(source_response, dict)
        else None
    )

    source_examples = (
        source_response.get("examples", {})
        if isinstance(source_response, dict)
        else {}
    )

    if not isinstance(source_examples, dict):
        source_examples = {}

    converted_content = (
        converted_response.get("content", {})
        if isinstance(converted_response, dict)
        else {}
    )

    if not isinstance(converted_content, dict):
        converted_content = {}

    converted_media_object = (
        converted_content.get(media_type)
        if media_type is not None
        else None
    )

    resolved = False
    resolution_status = "RETAIN_FAILURE"
    reason = None
    checks = {}

    if problem == "response_example":
        source_example_exists = (
            media_type in source_examples
        )

        source_example = source_examples.get(
            media_type
        )

        example_candidates = (
            converted_example_candidates(
                converted_media_object
            )
        )

        matching_candidates = [
            candidate
            for candidate in example_candidates
            if canonical_json_cell32(
                candidate["value"]
            )
            == canonical_json_cell32(
                source_example
            )
        ]

        resolved = (
            source_example_exists
            and bool(matching_candidates)
        )

        checks = {
            "source_example_exists": (
                source_example_exists
            ),
            "candidate_count": len(
                example_candidates
            ),
            "matching_locations": [
                candidate["location"]
                for candidate in matching_candidates
            ],
        }

        if resolved:
            resolution_status = (
                "RESOLVED_EXAMPLE_OBJECT_PROMOTION"
            )
            reason = (
                "The Swagger response example was "
                "preserved as an OpenAPI 3 named "
                "Example Object value."
            )
        else:
            reason = (
                "No converted example value exactly "
                "matches the source example."
            )

    elif problem == "response_schema":
        converted_schema = (
            converted_media_object.get("schema")
            if isinstance(
                converted_media_object,
                dict,
            )
            else None
        )

        normalized_source = (
            normalize_schema_cell32(
                source_document,
                source_schema,
            )
        )

        normalized_converted = (
            normalize_schema_cell32(
                converted_document,
                converted_schema,
            )
        )

        resolved = (
            source_schema is not None
            and converted_schema is not None
            and canonical_json_cell32(
                normalized_source
            )
            == canonical_json_cell32(
                normalized_converted
            )
        )

        checks = {
            "source_schema_present": (
                source_schema is not None
            ),
            "converted_schema_present": (
                converted_schema is not None
            ),
            "normalized_equal": resolved,
        }

        if resolved:
            resolution_status = (
                "RESOLVED_SCHEMA_AFTER_REFERENCE_"
                "AND_FILE_NORMALIZATION"
            )
            reason = (
                "The schemas are equivalent after local "
                "reference expansion, ignored $ref sibling "
                "removal, timestamp normalization, and "
                "file-to-binary normalization."
            )
        else:
            reason = (
                "A schema difference remains after approved "
                "normalization."
            )

    elif problem == "missing_response_content":
        expected_media = set(
            source_examples.keys()
        )

        if source_schema is not None:
            expected_media.update(
                source_produces
            )

        observed_media = set(
            converted_content.keys()
        )

        if expected_media:
            resolved = expected_media.issubset(
                observed_media
            )
        else:
            resolved = bool(converted_content)

        checks = {
            "expected_media": sorted(
                expected_media
            ),
            "observed_media": sorted(
                observed_media
            ),
            "expected_media_present": (
                expected_media.issubset(
                    observed_media
                )
                if expected_media
                else bool(converted_content)
            ),
        }

        if resolved:
            resolution_status = (
                "RESOLVED_RESPONSE_CONTENT_PRESENT"
            )
            reason = (
                "The converted response contains every "
                "media type implied by the source response."
            )
        else:
            reason = (
                "One or more expected response media types "
                "are missing."
            )

    elif problem == "response_media_types":
        expected_media = set(
            triage.get(
                "diagnostic",
                {},
            ).get(
                "expected",
                source_produces,
            )
        )

        observed_media = set(
            converted_content.keys()
        )

        resolved = (
            expected_media == observed_media
        )

        checks = {
            "expected_media": sorted(
                expected_media
            ),
            "observed_media": sorted(
                observed_media
            ),
            "exact_match": resolved,
        }

        if resolved:
            resolution_status = (
                "RESOLVED_RESPONSE_MEDIA_PRESERVED"
            )
            reason = (
                "The response media-type sets match exactly."
            )
        else:
            reason = (
                "Declared response media types were "
                "discarded or changed."
            )

    else:
        checks = {
            "unsupported_problem": problem
        }

        reason = (
            "No approved equivalence rule exists."
        )

    assessments.append(
        {
            "spec_id": spec_id,
            "relative_path": relative_path,
            "operation": operation_identity,
            "status": status_code,
            "problem": problem,
            "media_type": media_type,
            "resolved": resolved,
            "resolution_status": (
                resolution_status
            ),
            "resolution_reason": reason,
            "checks": checks,
        }
    )


write_jsonl_atomic(
    ASSESSMENTS_PATH,
    assessments,
)


assessment_lookup = {
    (
        assessment["spec_id"],
        assessment["operation"],
        str(assessment["status"]),
        assessment["problem"],
        assessment.get("media_type"),
    ): assessment
    for assessment in assessments
}


output_records = []
newly_accepted_spec_ids = []

for original_record in input_records:
    record = copy.deepcopy(original_record)

    prior_pass = (
        record.get(
            "adjusted_contract_equivalence_pass"
        ) is True
    )

    diagnostics = copy.deepcopy(
        record.get(
            "adjusted_diagnostics",
            {},
        )
    )

    prior_failed_categories = list(
        record.get(
            "adjusted_failed_categories",
            [],
        )
    )

    response_diagnostics = diagnostics.get(
        "response_contracts_preserved",
        [],
    )

    retained_response_diagnostics = []

    for diagnostic in response_diagnostics:
        lookup_key = (
            record.get("spec_id"),
            diagnostic.get("operation"),
            str(diagnostic.get("status")),
            diagnostic.get("problem"),
            diagnostic.get("media_type"),
        )

        assessment = assessment_lookup.get(
            lookup_key
        )

        if not (
            assessment
            and assessment["resolved"]
        ):
            retained_response_diagnostics.append(
                diagnostic
            )

    if retained_response_diagnostics:
        diagnostics[
            "response_contracts_preserved"
        ] = retained_response_diagnostics
    else:
        diagnostics.pop(
            "response_contracts_preserved",
            None,
        )

    updated_failed_categories = [
        category
        for category in prior_failed_categories
        if category in diagnostics
    ]

    for category in diagnostics:
        if category not in updated_failed_categories:
            updated_failed_categories.append(
                category
            )

    updated_pass = (
        prior_pass
        or len(updated_failed_categories) == 0
    )

    if not prior_pass and updated_pass:
        newly_accepted_spec_ids.append(
            record.get("spec_id")
        )

    record["adjustment_version"] = 6

    record["adjustment"] = (
        "verified_response_contract_equivalence"
    )

    record[
        "prior_adjusted_contract_equivalence_pass"
    ] = prior_pass

    record[
        "adjusted_contract_equivalence_pass"
    ] = updated_pass

    record["adjusted_failed_categories"] = (
        updated_failed_categories
    )

    record["adjusted_diagnostics"] = (
        diagnostics
    )

    output_records.append(record)


write_jsonl_atomic(
    OUTPUT_RESULTS_PATH,
    output_records,
)


prior_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in input_records
)

adjusted_pass_count = sum(
    record.get(
        "adjusted_contract_equivalence_pass"
    ) is True
    for record in output_records
)

adjusted_failure_count = (
    len(output_records)
    - adjusted_pass_count
)

expected_pass_count = (
    prior_pass_count
    + len(newly_accepted_spec_ids)
)

if adjusted_pass_count != expected_pass_count:
    raise RuntimeError(
        "Adjustment accounting mismatch: "
        f"expected {expected_pass_count}, "
        f"observed {adjusted_pass_count}."
    )


resolution_status_counts = Counter(
    assessment["resolution_status"]
    for assessment in assessments
)

problem_resolution_counts = Counter(
    (
        assessment["problem"],
        (
            "RESOLVED"
            if assessment["resolved"]
            else "RETAINED_FAILURE"
        ),
    )
    for assessment in assessments
)

remaining_failure_category_counts = Counter()

for record in output_records:
    if record.get(
        "adjusted_contract_equivalence_pass"
    ) is True:
        continue

    remaining_failure_category_counts.update(
        record.get(
            "adjusted_failed_categories",
            [],
        )
    )


report = {
    "status": "PASS",
    "adjustment_version": 6,
    "adjustment": (
        "VERIFY_RESPONSE_CONTRACT_EQUIVALENCE"
    ),
    "prior_adjusted_contract_pass_count": (
        prior_pass_count
    ),
    "assessment_count": len(assessments),
    "resolution_status_counts": dict(
        sorted(
            resolution_status_counts.items()
        )
    ),
    "problem_resolution_counts": {
        f"{problem}::{decision}": count
        for (
            problem,
            decision,
        ), count in sorted(
            problem_resolution_counts.items()
        )
    },
    "newly_accepted_specification_count": len(
        newly_accepted_spec_ids
    ),
    "newly_accepted_spec_ids": sorted(
        spec_id
        for spec_id in newly_accepted_spec_ids
        if spec_id
    ),
    "adjusted_contract_pass_count": (
        adjusted_pass_count
    ),
    "adjusted_contract_failure_count": (
        adjusted_failure_count
    ),
    "remaining_failure_category_counts": dict(
        sorted(
            remaining_failure_category_counts.items()
        )
    ),
    "strict_media_policy": (
        "Dropped or changed response media types remain "
        "failures."
    ),
    "assessments_path": str(
        ASSESSMENTS_PATH
    ),
    "assessments_sha256": sha256_file(
        ASSESSMENTS_PATH
    ),
    "adjusted_results_path": str(
        OUTPUT_RESULTS_PATH
    ),
    "adjusted_results_sha256": sha256_file(
        OUTPUT_RESULTS_PATH
    ),
    "input_results_sha256": sha256_file(
        INPUT_RESULTS_PATH
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 32: PASS")

{
  "status": "PASS",
  "adjustment_version": 6,
  "adjustment": "VERIFY_RESPONSE_CONTRACT_EQUIVALENCE",
  "prior_adjusted_contract_pass_count": 182,
  "assessment_count": 460,
  "resolution_status_counts": {
    "RESOLVED_EXAMPLE_OBJECT_PROMOTION": 390,
    "RESOLVED_SCHEMA_AFTER_REFERENCE_AND_FILE_NORMALIZATION": 46,
    "RETAIN_FAILURE": 24
  },
  "problem_resolution_counts": {
    "missing_response_content::RETAINED_FAILURE": 10,
    "response_example::RESOLVED": 390,
    "response_media_types::RETAINED_FAILURE": 11,
    "response_schema::RESOLVED": 46,
    "response_schema::RETAINED_FAILURE": 3
  },
  "newly_accepted_specification_count": 31,
  "newly_accepted_spec_ids": [
    "spec_031b5eb377d71094c72f4888",
    "spec_0335d01bba0d3967c39aac1f",
    "spec_082f1ffc80e373d2474f07f0",
    "spec_2306077633bb4aff50af6ab1",
    "spec_33715152e16c78d7b887f511",
    "spec_35b7304d6380179810736b94",
    "spec_3a3689ceadd6631738b0d6d1",
    "spec_5cac37cabb913cf208308602",
    "spec_630b5c0

In [35]:
from pathlib import Path
from collections import Counter
import json

import pandas as pd


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

CONVERSION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_conversion_results.jsonl"
)

INITIAL_DISPOSITIONS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_conversion_dispositions.csv"
)

SEMANTIC_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_semantic_v2_results.jsonl"
)

CONTRACT_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/swagger_contract_adjusted_v6_results.jsonl"
)

FINAL_LEDGER_PATH = (
    PROJECT_ROOT
    / "data/inventory/swagger_final_dispositions.csv"
)

ELIGIBLE_REGISTRY_PATH = (
    PROJECT_ROOT
    / "data/inventory/eligible_converted_swagger.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/swagger_final_disposition_summary.json"
)


def read_jsonl_cell33(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def first_present(record, keys):
    for key in keys:
        value = record.get(key)

        if value is not None:
            return value

    return None


required_inputs = [
    CONVERSION_RESULTS_PATH,
    INITIAL_DISPOSITIONS_PATH,
    SEMANTIC_RESULTS_PATH,
    CONTRACT_RESULTS_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file() or path.stat().st_size == 0
]

if missing_inputs:
    raise RuntimeError(
        "Missing required inputs:\n"
        + "\n".join(missing_inputs)
    )


conversion_records = read_jsonl_cell33(
    CONVERSION_RESULTS_PATH
)

semantic_records = read_jsonl_cell33(
    SEMANTIC_RESULTS_PATH
)

contract_records = read_jsonl_cell33(
    CONTRACT_RESULTS_PATH
)

initial_dispositions = pd.read_csv(
    INITIAL_DISPOSITIONS_PATH,
    dtype=str,
    keep_default_na=False,
)


if "spec_id" not in initial_dispositions.columns:
    raise RuntimeError(
        "Initial disposition table has no spec_id column."
    )


disposition_column_candidates = [
    "initial_disposition",
    "disposition",
    "conversion_disposition",
]

initial_disposition_column = next(
    (
        column
        for column in disposition_column_candidates
        if column in initial_dispositions.columns
    ),
    None,
)

if initial_disposition_column is None:
    raise RuntimeError(
        "Could not identify the disposition column. "
        f"Available columns: "
        f"{list(initial_dispositions.columns)}"
    )


conversion_by_spec_id = {
    record["spec_id"]: record
    for record in conversion_records
    if record.get("spec_id")
}

initial_by_spec_id = {
    row["spec_id"]: row
    for row in initial_dispositions.to_dict(
        orient="records"
    )
}

semantic_spec_ids = {
    record["spec_id"]
    for record in semantic_records
    if record.get("spec_id")
}

contract_by_spec_id = {
    record["spec_id"]: record
    for record in contract_records
    if record.get("spec_id")
}


if len(conversion_by_spec_id) != 2163:
    raise RuntimeError(
        "Expected 2,163 unique Swagger conversion "
        f"records, found {len(conversion_by_spec_id)}."
    )

if len(initial_by_spec_id) != 2163:
    raise RuntimeError(
        "Expected 2,163 unique initial dispositions, "
        f"found {len(initial_by_spec_id)}."
    )

if len(semantic_spec_ids) != 1868:
    raise RuntimeError(
        "Expected 1,868 strict-clean semantic audit "
        f"records, found {len(semantic_spec_ids)}."
    )

if len(contract_by_spec_id) != 226:
    raise RuntimeError(
        "Expected 226 deep contract audit records, "
        f"found {len(contract_by_spec_id)}."
    )


ledger_records = []
eligible_records = []

for spec_id in sorted(conversion_by_spec_id):
    conversion = conversion_by_spec_id[spec_id]
    initial = initial_by_spec_id.get(spec_id)

    if initial is None:
        raise RuntimeError(
            f"Missing initial disposition for {spec_id}"
        )

    initial_disposition = initial[
        initial_disposition_column
    ]

    contract = contract_by_spec_id.get(spec_id)

    failure_categories = []

    if initial_disposition == (
        "EXCLUDE_CONVERSION_FAILED"
    ):
        final_disposition = (
            "EXCLUDE_CONVERSION_FAILED"
        )
        eligible = False
        disposition_reason = (
            "Swagger-to-OpenAPI conversion failed."
        )

    elif initial_disposition == (
        "QUARANTINE_CONVERSION_UNCERTAIN"
    ):
        final_disposition = (
            "QUARANTINE_CONVERTER_PATCHED_OUTPUT"
        )
        eligible = False
        disposition_reason = (
            "The converter reported one or more "
            "automatic patches."
        )

    elif initial_disposition == (
        "PENDING_STRICT_AUDIT"
    ):
        if spec_id not in semantic_spec_ids:
            final_disposition = (
                "EXCLUDE_STRICT_VALIDATION_OR_REFERENCE"
            )
            eligible = False
            disposition_reason = (
                "Converted output failed strict OpenAPI "
                "validation or reference qualification."
            )

        elif contract is None:
            final_disposition = (
                "EXCLUDE_SEMANTIC_PRESERVATION"
            )
            eligible = False
            disposition_reason = (
                "Converted output failed the "
                "representation-aware semantic audit."
            )

        elif contract.get(
            "adjusted_contract_equivalence_pass"
        ) is not True:
            final_disposition = (
                "EXCLUDE_CONTRACT_PRESERVATION"
            )
            eligible = False

            failure_categories = contract.get(
                "adjusted_failed_categories",
                [],
            )

            disposition_reason = (
                "One or more operation-contract values "
                "were missing, changed, or uncertain."
            )

        else:
            final_disposition = (
                "ELIGIBLE_CONVERTED_SWAGGER"
            )
            eligible = True
            disposition_reason = (
                "Conversion, validation, reference, "
                "semantic, and contract audits passed."
            )

    else:
        raise RuntimeError(
            "Unexpected initial disposition "
            f"{initial_disposition!r} for {spec_id}."
        )

    canonical_path = first_present(
        conversion,
        [
            "output_path",
            "converted_path",
            "canonical_path",
            "output_relative_path",
            "converted_relative_path",
            "canonical_relative_path",
        ],
    )

    converted_sha256 = first_present(
        conversion,
        [
            "output_sha256",
            "converted_sha256",
            "canonical_sha256",
        ],
    )

    source_operation_count = first_present(
        conversion,
        [
            "source_operation_count",
            "operation_count",
        ],
    )

    ledger_record = {
        "spec_id": spec_id,
        "raw_id": conversion.get("raw_id"),
        "relative_path": conversion.get(
            "relative_path"
        ),
        "provider_directory": conversion.get(
            "provider_directory"
        ),
        "source_sha256": conversion.get(
            "source_sha256"
        ),
        "source_bytes": conversion.get(
            "source_bytes"
        ),
        "source_operation_count": (
            source_operation_count
        ),
        "conversion_status": conversion.get(
            "status"
        ),
        "initial_disposition": (
            initial_disposition
        ),
        "final_disposition": (
            final_disposition
        ),
        "eligible": eligible,
        "failure_categories": json.dumps(
            failure_categories,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        ),
        "disposition_reason": (
            disposition_reason
        ),
        "canonical_path": canonical_path,
        "converted_sha256": (
            converted_sha256
        ),
    }

    ledger_records.append(ledger_record)

    if eligible:
        eligible_records.append(
            {
                "spec_id": spec_id,
                "raw_id": conversion.get("raw_id"),
                "relative_path": conversion.get(
                    "relative_path"
                ),
                "provider_directory": conversion.get(
                    "provider_directory"
                ),
                "source_sha256": conversion.get(
                    "source_sha256"
                ),
                "converted_sha256": (
                    converted_sha256
                ),
                "canonical_path": (
                    canonical_path
                ),
                "operation_count": (
                    source_operation_count
                ),
                "source_dialect": "swagger_2_0",
                "canonical_dialect": (
                    "openapi_3_0_3"
                ),
                "eligibility_basis": (
                    "qualified_swagger_conversion"
                ),
            }
        )


final_disposition_counts = Counter(
    record["final_disposition"]
    for record in ledger_records
)

expected_disposition_counts = {
    "ELIGIBLE_CONVERTED_SWAGGER": 213,
    "EXCLUDE_CONTRACT_PRESERVATION": 13,
    "EXCLUDE_CONVERSION_FAILED": 4,
    "EXCLUDE_SEMANTIC_PRESERVATION": 1642,
    "EXCLUDE_STRICT_VALIDATION_OR_REFERENCE": 243,
    "QUARANTINE_CONVERTER_PATCHED_OUTPUT": 48,
}

if dict(
    sorted(final_disposition_counts.items())
) != dict(
    sorted(expected_disposition_counts.items())
):
    raise RuntimeError(
        "Final Swagger disposition accounting "
        "does not match the expected gate totals.\n"
        f"Observed: {dict(final_disposition_counts)}\n"
        f"Expected: {expected_disposition_counts}"
    )

if sum(final_disposition_counts.values()) != 2163:
    raise RuntimeError(
        "Final disposition ledger does not account "
        "for all 2,163 Swagger files."
    )

if len(eligible_records) != 213:
    raise RuntimeError(
        "Expected 213 eligible converted Swagger "
        f"records, found {len(eligible_records)}."
    )


FINAL_LEDGER_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

ledger_frame = pd.DataFrame(
    ledger_records
).sort_values(
    [
        "relative_path",
        "spec_id",
    ],
    kind="stable",
)

temporary_ledger_path = (
    FINAL_LEDGER_PATH.with_suffix(
        ".csv.temporary"
    )
)

ledger_frame.to_csv(
    temporary_ledger_path,
    index=False,
    lineterminator="\n",
)

temporary_ledger_path.replace(
    FINAL_LEDGER_PATH
)

write_jsonl_atomic(
    ELIGIBLE_REGISTRY_PATH,
    sorted(
        eligible_records,
        key=lambda record: (
            record["relative_path"],
            record["spec_id"],
        ),
    ),
)


eligible_operation_count = sum(
    int(record["operation_count"])
    for record in eligible_records
    if record["operation_count"] is not None
)

contract_failure_category_counts = Counter()

for contract in contract_records:
    if contract.get(
        "adjusted_contract_equivalence_pass"
    ) is True:
        continue

    contract_failure_category_counts.update(
        contract.get(
            "adjusted_failed_categories",
            [],
        )
    )


report = {
    "status": "PASS",
    "stage": "SWAGGER_FINAL_DISPOSITION",
    "source_swagger_count": 2163,
    "accounted_source_count": len(
        ledger_records
    ),
    "final_disposition_counts": dict(
        sorted(
            final_disposition_counts.items()
        )
    ),
    "eligible_converted_swagger_count": len(
        eligible_records
    ),
    "eligible_operation_count": (
        eligible_operation_count
    ),
    "contract_exclusion_count": (
        final_disposition_counts[
            "EXCLUDE_CONTRACT_PRESERVATION"
        ]
    ),
    "contract_failure_category_counts": dict(
        sorted(
            contract_failure_category_counts.items()
        )
    ),
    "security_scheme_failure_policy": (
        "The unresolved security-scheme mapping remains "
        "excluded; no semantic equivalence was assumed."
    ),
    "ledger_path": str(
        FINAL_LEDGER_PATH
    ),
    "ledger_sha256": sha256_file(
        FINAL_LEDGER_PATH
    ),
    "eligible_registry_path": str(
        ELIGIBLE_REGISTRY_PATH
    ),
    "eligible_registry_sha256": sha256_file(
        ELIGIBLE_REGISTRY_PATH
    ),
    "contract_results_sha256": sha256_file(
        CONTRACT_RESULTS_PATH
    ),
    "semantic_results_sha256": sha256_file(
        SEMANTIC_RESULTS_PATH
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 33: PASS")

{
  "status": "PASS",
  "stage": "SWAGGER_FINAL_DISPOSITION",
  "source_swagger_count": 2163,
  "accounted_source_count": 2163,
  "final_disposition_counts": {
    "ELIGIBLE_CONVERTED_SWAGGER": 213,
    "EXCLUDE_CONTRACT_PRESERVATION": 13,
    "EXCLUDE_CONVERSION_FAILED": 4,
    "EXCLUDE_SEMANTIC_PRESERVATION": 1642,
    "EXCLUDE_STRICT_VALIDATION_OR_REFERENCE": 243,
    "QUARANTINE_CONVERTER_PATCHED_OUTPUT": 48
  },
  "eligible_converted_swagger_count": 213,
  "eligible_operation_count": 2576,
  "contract_exclusion_count": 13,
  "contract_failure_category_counts": {
    "nonbody_parameters_preserved": 2,
    "request_contracts_preserved": 7,
    "response_contracts_preserved": 4,
    "security_schemes_preserved": 1
  },
  "security_scheme_failure_policy": "The unresolved security-scheme mapping remains excluded; no semantic equivalence was assumed.",
  "ledger_path": "/kaggle/working/fresh/stage_01/prose_operation_experiment/data/inventory/swagger_final_dispositions.csv",
  "ledger_sh

In [36]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib.metadata
import json
import time
import warnings

import pandas as pd
from openapi_spec_validator import validate


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

SPECIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data/inventory/specifications.csv"
)

VALIDATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data/audits/native_openapi_validation.jsonl"
)

ELIGIBLE_REGISTRY_PATH = (
    PROJECT_ROOT
    / "data/inventory/eligible_native_openapi.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/native_openapi_validation_summary.json"
)


def read_jsonl_cell34(path: Path):
    records = []

    if not path.is_file():
        return records

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception:
                continue

    return records


def parse_boolean_cell34(value):
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
    }


def integer_or_zero_cell34(value):
    try:
        if pd.isna(value):
            return 0

        return int(float(value))
    except Exception:
        return 0


def semantic_json_sha256_cell34(document):
    serialized = json.dumps(
        document,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


if not SPECIFICATIONS_PATH.is_file():
    raise FileNotFoundError(
        SPECIFICATIONS_PATH
    )


validator_version = importlib.metadata.version(
    "openapi-spec-validator"
)

specifications = pd.read_csv(
    SPECIFICATIONS_PATH,
    dtype=object,
)


required_columns = {
    "spec_id",
    "raw_id",
    "relative_path",
    "provider_directory",
    "dialect",
    "canonical_sha256",
    "canonical_relative_path",
    "operation_count",
}

missing_columns = sorted(
    required_columns
    - set(specifications.columns)
)

if missing_columns:
    raise RuntimeError(
        "Specifications inventory is missing columns: "
        + ", ".join(missing_columns)
    )


reference_column_candidates = [
    "reference_status",
    "reference_audit_status",
    "references_status",
]

reference_status_column = next(
    (
        column
        for column in reference_column_candidates
        if column in specifications.columns
    ),
    None,
)

if reference_status_column is None:
    raise RuntimeError(
        "Could not locate the reference status column. "
        "Available columns: "
        + ", ".join(specifications.columns)
    )


native_rows = specifications[
    specifications["dialect"].isin(
        [
            "openapi_3_0",
            "openapi_3_1",
        ]
    )
].copy()

if "supported" in native_rows.columns:
    native_rows = native_rows[
        native_rows["supported"].map(
            parse_boolean_cell34
        )
    ]

native_rows = native_rows[
    native_rows["canonical_relative_path"]
    .fillna("")
    .astype(str)
    .str.len()
    > 0
].copy()

native_rows = native_rows.sort_values(
    [
        "relative_path",
        "spec_id",
    ],
    kind="stable",
).reset_index(drop=True)


if len(native_rows) != 1944:
    raise RuntimeError(
        "Expected 1,944 native OpenAPI 3.x "
        f"specifications, found {len(native_rows)}."
    )


existing_records = read_jsonl_cell34(
    VALIDATION_RESULTS_PATH
)

existing_by_spec_id = {
    record["spec_id"]: record
    for record in existing_records
    if record.get("spec_id")
}


results_by_spec_id = {}
checkpoint_reused_count = 0
new_validation_count = 0
hash_mode_counts = Counter()

VALIDATION_RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

start_time = time.perf_counter()

with VALIDATION_RESULTS_PATH.open(
    "a",
    encoding="utf-8",
    newline="\n",
) as checkpoint_file:

    for index, row in native_rows.iterrows():
        spec_id = str(row["spec_id"])

        canonical_relative_path = str(
            row["canonical_relative_path"]
        )

        canonical_path = (
            PROJECT_ROOT
            / canonical_relative_path
        )

        inventory_sha256 = str(
            row["canonical_sha256"]
        )

        existing = existing_by_spec_id.get(
            spec_id
        )

        if (
            existing
            and existing.get(
                "inventory_canonical_sha256"
            )
            == inventory_sha256
            and existing.get("validator_version")
            == validator_version
        ):
            results_by_spec_id[spec_id] = (
                existing
            )

            checkpoint_reused_count += 1

            hash_mode_counts.update(
                [
                    existing.get(
                        "hash_verification_mode",
                        "UNKNOWN",
                    )
                ]
            )

        else:
            if not canonical_path.is_file():
                raise FileNotFoundError(
                    canonical_path
                )

            file_sha256 = sha256_file(
                canonical_path
            )

            document = json.loads(
                canonical_path.read_text(
                    encoding="utf-8"
                )
            )

            semantic_sha256 = (
                semantic_json_sha256_cell34(
                    document
                )
            )

            if inventory_sha256 == file_sha256:
                hash_verification_mode = (
                    "FILE_BYTES"
                )

            elif inventory_sha256 == semantic_sha256:
                hash_verification_mode = (
                    "CANONICAL_COMPACT_JSON"
                )

            else:
                raise RuntimeError(
                    "Canonical integrity mismatch for "
                    f"{spec_id}.\n"
                    f"Inventory SHA-256: "
                    f"{inventory_sha256}\n"
                    f"File SHA-256: {file_sha256}\n"
                    f"Semantic SHA-256: "
                    f"{semantic_sha256}"
                )

            hash_mode_counts.update(
                [hash_verification_mode]
            )

            diagnostic = None

            try:
                with warnings.catch_warnings():
                    warnings.simplefilter(
                        "ignore",
                        FutureWarning,
                    )

                    validate(document)

                validation_status = "PASS"

            except Exception as error:
                validation_status = "FAIL"

                diagnostic = (
                    f"{type(error).__name__}: "
                    f"{str(error)}"
                )

                if len(diagnostic) > 8000:
                    diagnostic = (
                        diagnostic[:8000]
                        + "...[truncated]"
                    )

            reference_status = str(
                row[reference_status_column]
            )

            eligible = (
                validation_status == "PASS"
                and reference_status == "PASS"
            )

            result = {
                "spec_id": spec_id,
                "raw_id": str(row["raw_id"]),
                "relative_path": str(
                    row["relative_path"]
                ),
                "provider_directory": str(
                    row["provider_directory"]
                ),
                "dialect": str(row["dialect"]),
                "declared_version": (
                    str(row["declared_version"])
                    if (
                        "declared_version"
                        in native_rows.columns
                        and not pd.isna(
                            row["declared_version"]
                        )
                    )
                    else None
                ),
                "canonical_relative_path": (
                    canonical_relative_path
                ),
                "inventory_canonical_sha256": (
                    inventory_sha256
                ),
                "canonical_file_sha256": (
                    file_sha256
                ),
                "canonical_semantic_sha256": (
                    semantic_sha256
                ),
                "hash_verification_mode": (
                    hash_verification_mode
                ),
                "operation_count": (
                    integer_or_zero_cell34(
                        row["operation_count"]
                    )
                ),
                "reference_status": (
                    reference_status
                ),
                "validation_status": (
                    validation_status
                ),
                "eligible": eligible,
                "validator": (
                    "openapi-spec-validator"
                ),
                "validator_version": (
                    validator_version
                ),
                "diagnostic": diagnostic,
            }

            results_by_spec_id[spec_id] = (
                result
            )

            checkpoint_file.write(
                json.dumps(
                    result,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                )
                + "\n"
            )

            new_validation_count += 1

        completed = index + 1

        if (
            completed % 25 == 0
            or completed == len(native_rows)
        ):
            checkpoint_file.flush()

            elapsed = (
                time.perf_counter()
                - start_time
            )

            rate = (
                completed / elapsed
                if elapsed > 0
                else 0.0
            )

            remaining = (
                len(native_rows)
                - completed
            )

            eta = (
                remaining / rate
                if rate > 0
                else None
            )

            print(
                json.dumps(
                    {
                        "progress": completed,
                        "total": len(native_rows),
                        "checkpoint_reused": (
                            checkpoint_reused_count
                        ),
                        "elapsed_seconds": round(
                            elapsed,
                            1,
                        ),
                        "rate_per_second": round(
                            rate,
                            3,
                        ),
                        "eta_seconds": (
                            round(eta, 1)
                            if eta is not None
                            else None
                        ),
                    }
                )
            )


ordered_results = [
    results_by_spec_id[
        str(row["spec_id"])
    ]
    for _, row in native_rows.iterrows()
]

if len(ordered_results) != 1944:
    raise RuntimeError(
        "Native validation results do not account "
        "for all 1,944 specifications."
    )


write_jsonl_atomic(
    VALIDATION_RESULTS_PATH,
    ordered_results,
)


eligible_records = [
    {
        "spec_id": record["spec_id"],
        "raw_id": record["raw_id"],
        "relative_path": record[
            "relative_path"
        ],
        "provider_directory": record[
            "provider_directory"
        ],
        "dialect": record["dialect"],
        "declared_version": record[
            "declared_version"
        ],
        "canonical_relative_path": record[
            "canonical_relative_path"
        ],
        "canonical_sha256": record[
            "inventory_canonical_sha256"
        ],
        "canonical_file_sha256": record[
            "canonical_file_sha256"
        ],
        "operation_count": record[
            "operation_count"
        ],
        "eligibility_basis": (
            "native_openapi_validation_and_"
            "reference_audit"
        ),
    }
    for record in ordered_results
    if record["eligible"]
]

write_jsonl_atomic(
    ELIGIBLE_REGISTRY_PATH,
    eligible_records,
)


validation_status_counts = Counter(
    record["validation_status"]
    for record in ordered_results
)

reference_status_counts = Counter(
    record["reference_status"]
    for record in ordered_results
)

dialect_counts = Counter(
    record["dialect"]
    for record in ordered_results
)

eligible_dialect_counts = Counter(
    record["dialect"]
    for record in ordered_results
    if record["eligible"]
)

eligible_operation_count = sum(
    record["operation_count"]
    for record in ordered_results
    if record["eligible"]
)

duration_seconds = (
    time.perf_counter()
    - start_time
)


report = {
    "status": "PASS",
    "stage": "NATIVE_OPENAPI_VALIDATION",
    "validator": "openapi-spec-validator",
    "validator_version": validator_version,
    "native_specification_count": len(
        ordered_results
    ),
    "dialect_counts": dict(
        sorted(dialect_counts.items())
    ),
    "validation_status_counts": dict(
        sorted(
            validation_status_counts.items()
        )
    ),
    "reference_status_counts": dict(
        sorted(
            reference_status_counts.items()
        )
    ),
    "hash_verification_mode_counts": dict(
        sorted(
            hash_mode_counts.items()
        )
    ),
    "eligible_native_specification_count": len(
        eligible_records
    ),
    "eligible_dialect_counts": dict(
        sorted(
            eligible_dialect_counts.items()
        )
    ),
    "eligible_operation_count": (
        eligible_operation_count
    ),
    "checkpoint_reused_count": (
        checkpoint_reused_count
    ),
    "new_validation_count": (
        new_validation_count
    ),
    "duration_seconds": round(
        duration_seconds,
        3,
    ),
    "validation_results_path": str(
        VALIDATION_RESULTS_PATH
    ),
    "validation_results_sha256": (
        sha256_file(
            VALIDATION_RESULTS_PATH
        )
    ),
    "eligible_registry_path": str(
        ELIGIBLE_REGISTRY_PATH
    ),
    "eligible_registry_sha256": (
        sha256_file(
            ELIGIBLE_REGISTRY_PATH
        )
    ),
    "specifications_inventory_sha256": (
        sha256_file(
            SPECIFICATIONS_PATH
        )
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 34: PASS")

{"progress": 25, "total": 1944, "checkpoint_reused": 0, "elapsed_seconds": 8.3, "rate_per_second": 3.014, "eta_seconds": 636.7}
{"progress": 50, "total": 1944, "checkpoint_reused": 0, "elapsed_seconds": 31.7, "rate_per_second": 1.578, "eta_seconds": 1200.2}
{"progress": 75, "total": 1944, "checkpoint_reused": 0, "elapsed_seconds": 48.5, "rate_per_second": 1.545, "eta_seconds": 1209.7}
{"progress": 100, "total": 1944, "checkpoint_reused": 0, "elapsed_seconds": 55.5, "rate_per_second": 1.802, "eta_seconds": 1023.1}
{"progress": 125, "total": 1944, "checkpoint_reused": 0, "elapsed_seconds": 66.1, "rate_per_second": 1.892, "eta_seconds": 961.5}
{"progress": 150, "total": 1944, "checkpoint_reused": 0, "elapsed_seconds": 75.7, "rate_per_second": 1.982, "eta_seconds": 905.3}
{"progress": 175, "total": 1944, "checkpoint_reused": 0, "elapsed_seconds": 89.9, "rate_per_second": 1.946, "eta_seconds": 909.2}
{"progress": 200, "total": 1944, "checkpoint_reused": 0, "elapsed_seconds": 102.0, "rate_pe

In [37]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import time


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

NATIVE_REGISTRY_PATH = (
    PROJECT_ROOT
    / "data/inventory/eligible_native_openapi.jsonl"
)

SWAGGER_REGISTRY_PATH = (
    PROJECT_ROOT
    / "data/inventory/eligible_converted_swagger.jsonl"
)

COMBINED_CANDIDATES_PATH = (
    PROJECT_ROOT
    / "data/inventory/combined_eligible_candidates.jsonl"
)

DEDUPLICATION_LEDGER_PATH = (
    PROJECT_ROOT
    / "data/inventory/exact_deduplication_ledger.jsonl"
)

DUPLICATE_GROUPS_PATH = (
    PROJECT_ROOT
    / "data/inventory/exact_duplicate_groups.jsonl"
)

FINAL_ELIGIBLE_PATH = (
    PROJECT_ROOT
    / "data/inventory/final_eligible_specifications.jsonl"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/combined_eligibility_and_deduplication.json"
)


HTTP_METHODS = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}


def read_jsonl_cell35(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def canonical_bytes_cell35(document):
    return json.dumps(
        document,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")


def sha256_bytes_cell35(value):
    return hashlib.sha256(value).hexdigest()


def operation_count_cell35(document):
    paths = document.get("paths", {})

    if not isinstance(paths, dict):
        return 0

    count = 0

    for path_item in paths.values():
        if not isinstance(path_item, dict):
            continue

        count += sum(
            1
            for key in path_item
            if str(key).lower() in HTTP_METHODS
        )

    return count


def resolve_canonical_path_cell35(
    record,
):
    path_value = (
        record.get("canonical_relative_path")
        or record.get("canonical_path")
    )

    candidates = []

    if isinstance(path_value, str) and path_value:
        candidate = Path(path_value)

        if candidate.is_absolute():
            candidates.append(candidate)
        else:
            candidates.append(
                PROJECT_ROOT / candidate
            )

    spec_id = record.get("spec_id")

    if spec_id:
        candidates.append(
            PROJECT_ROOT
            / "data/canonical"
            / f"{spec_id}.json"
        )

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate canonical document for "
        f"{record.get('spec_id')}. Candidates: "
        + ", ".join(str(path) for path in candidates)
    )


def relative_to_project_cell35(path: Path):
    try:
        return path.relative_to(
            PROJECT_ROOT
        ).as_posix()
    except ValueError:
        return str(path)


required_inputs = [
    NATIVE_REGISTRY_PATH,
    SWAGGER_REGISTRY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file() or path.stat().st_size == 0
]

if missing_inputs:
    raise RuntimeError(
        "Missing required registries:\n"
        + "\n".join(missing_inputs)
    )


native_records = read_jsonl_cell35(
    NATIVE_REGISTRY_PATH
)

swagger_records = read_jsonl_cell35(
    SWAGGER_REGISTRY_PATH
)


if len(native_records) != 1757:
    raise RuntimeError(
        "Expected 1,757 eligible native records, "
        f"found {len(native_records)}."
    )

if len(swagger_records) != 213:
    raise RuntimeError(
        "Expected 213 eligible converted Swagger "
        f"records, found {len(swagger_records)}."
    )


candidate_inputs = []

for record in native_records:
    candidate_inputs.append(
        {
            **record,
            "source_class": "native_openapi",
            "source_dialect": record.get(
                "dialect"
            ),
        }
    )

for record in swagger_records:
    candidate_inputs.append(
        {
            **record,
            "source_class": (
                "converted_swagger"
            ),
            "source_dialect": (
                "swagger_2_0"
            ),
        }
    )


spec_ids = [
    record["spec_id"]
    for record in candidate_inputs
]

duplicate_spec_ids = [
    spec_id
    for spec_id, count in Counter(
        spec_ids
    ).items()
    if count > 1
]

if duplicate_spec_ids:
    raise RuntimeError(
        "Duplicate spec IDs across eligible registries: "
        + ", ".join(
            sorted(duplicate_spec_ids)[:20]
        )
    )


start_time = time.perf_counter()
combined_records = []

for index, record in enumerate(
    candidate_inputs,
    start=1,
):
    canonical_path = (
        resolve_canonical_path_cell35(
            record
        )
    )

    file_bytes = canonical_path.read_bytes()

    file_sha256 = sha256_bytes_cell35(
        file_bytes
    )

    document = json.loads(
        file_bytes.decode("utf-8")
    )

    semantic_sha256 = sha256_bytes_cell35(
        canonical_bytes_cell35(document)
    )

    recorded_hashes = {
        value
        for value in [
            record.get("canonical_sha256"),
            record.get("canonical_file_sha256"),
            record.get("converted_sha256"),
        ]
        if isinstance(value, str) and value
    }

    if (
        recorded_hashes
        and not (
            file_sha256 in recorded_hashes
            or semantic_sha256 in recorded_hashes
        )
    ):
        raise RuntimeError(
            "Neither the physical nor semantic hash "
            "matches the recorded hash for "
            f"{record['spec_id']}."
        )

    observed_operation_count = (
        operation_count_cell35(document)
    )

    recorded_operation_count = int(
        record.get("operation_count", 0)
    )

    if (
        recorded_operation_count
        != observed_operation_count
    ):
        raise RuntimeError(
            "Operation-count mismatch for "
            f"{record['spec_id']}: recorded "
            f"{recorded_operation_count}, observed "
            f"{observed_operation_count}."
        )

    combined_records.append(
        {
            "spec_id": record["spec_id"],
            "raw_id": record.get("raw_id"),
            "relative_path": record.get(
                "relative_path"
            ),
            "provider_directory": record.get(
                "provider_directory"
            ),
            "source_class": record[
                "source_class"
            ],
            "source_dialect": record.get(
                "source_dialect"
            ),
            "canonical_openapi_version": (
                document.get("openapi")
            ),
            "canonical_relative_path": (
                relative_to_project_cell35(
                    canonical_path
                )
            ),
            "canonical_file_sha256": (
                file_sha256
            ),
            "canonical_semantic_sha256": (
                semantic_sha256
            ),
            "operation_count": (
                observed_operation_count
            ),
            "eligibility_basis": record.get(
                "eligibility_basis"
            ),
        }
    )

    if (
        index % 100 == 0
        or index == len(candidate_inputs)
    ):
        elapsed = (
            time.perf_counter()
            - start_time
        )

        print(
            json.dumps(
                {
                    "progress": index,
                    "total": len(
                        candidate_inputs
                    ),
                    "elapsed_seconds": round(
                        elapsed,
                        1,
                    ),
                }
            )
        )


combined_records = sorted(
    combined_records,
    key=lambda record: (
        record["relative_path"],
        record["spec_id"],
    ),
)

write_jsonl_atomic(
    COMBINED_CANDIDATES_PATH,
    combined_records,
)


groups_by_semantic_hash = defaultdict(list)

for record in combined_records:
    groups_by_semantic_hash[
        record["canonical_semantic_sha256"]
    ].append(record)


def representative_rank(record):
    source_priority = {
        "native_openapi": 0,
        "converted_swagger": 1,
    }

    return (
        source_priority.get(
            record["source_class"],
            99,
        ),
        record["relative_path"],
        record["spec_id"],
    )


deduplication_records = []
duplicate_group_records = []
final_records = []

for semantic_sha256 in sorted(
    groups_by_semantic_hash
):
    group = groups_by_semantic_hash[
        semantic_sha256
    ]

    group = sorted(
        group,
        key=representative_rank,
    )

    representative = group[0]

    operation_counts = {
        record["operation_count"]
        for record in group
    }

    if len(operation_counts) != 1:
        raise RuntimeError(
            "Identical semantic hashes have different "
            "operation counts: "
            f"{semantic_sha256}"
        )

    duplicate_group_id = (
        "duplicate_"
        + semantic_sha256[:24]
    )

    if len(group) > 1:
        duplicate_group_records.append(
            {
                "duplicate_group_id": (
                    duplicate_group_id
                ),
                "canonical_semantic_sha256": (
                    semantic_sha256
                ),
                "member_count": len(group),
                "representative_spec_id": (
                    representative["spec_id"]
                ),
                "operation_count": (
                    representative[
                        "operation_count"
                    ]
                ),
                "members": [
                    {
                        "spec_id": (
                            record["spec_id"]
                        ),
                        "relative_path": (
                            record[
                                "relative_path"
                            ]
                        ),
                        "source_class": (
                            record[
                                "source_class"
                            ]
                        ),
                    }
                    for record in group
                ],
            }
        )

    for record in group:
        selected = (
            record["spec_id"]
            == representative["spec_id"]
        )

        deduplication_records.append(
            {
                "spec_id": record["spec_id"],
                "relative_path": record[
                    "relative_path"
                ],
                "source_class": record[
                    "source_class"
                ],
                "canonical_semantic_sha256": (
                    semantic_sha256
                ),
                "duplicate_group_id": (
                    duplicate_group_id
                    if len(group) > 1
                    else None
                ),
                "duplicate_group_size": (
                    len(group)
                ),
                "representative_spec_id": (
                    representative["spec_id"]
                ),
                "selected": selected,
                "disposition": (
                    "SELECTED_REPRESENTATIVE"
                    if selected
                    else "EXCLUDE_EXACT_DUPLICATE"
                ),
            }
        )

        if selected:
            final_record = {
                **record,
                "duplicate_group_id": (
                    duplicate_group_id
                    if len(group) > 1
                    else None
                ),
                "duplicate_group_size": (
                    len(group)
                ),
            }

            final_records.append(
                final_record
            )


deduplication_records = sorted(
    deduplication_records,
    key=lambda record: (
        record["relative_path"],
        record["spec_id"],
    ),
)

duplicate_group_records = sorted(
    duplicate_group_records,
    key=lambda record: (
        record["duplicate_group_id"]
    ),
)

final_records = sorted(
    final_records,
    key=lambda record: (
        record["relative_path"],
        record["spec_id"],
    ),
)


write_jsonl_atomic(
    DEDUPLICATION_LEDGER_PATH,
    deduplication_records,
)

write_jsonl_atomic(
    DUPLICATE_GROUPS_PATH,
    duplicate_group_records,
)

write_jsonl_atomic(
    FINAL_ELIGIBLE_PATH,
    final_records,
)


source_class_candidate_counts = Counter(
    record["source_class"]
    for record in combined_records
)

source_class_final_counts = Counter(
    record["source_class"]
    for record in final_records
)

duplicate_member_count = sum(
    record["member_count"]
    for record in duplicate_group_records
)

exact_duplicates_removed = (
    len(combined_records)
    - len(final_records)
)

final_operation_count = sum(
    record["operation_count"]
    for record in final_records
)

duration_seconds = (
    time.perf_counter()
    - start_time
)


report = {
    "status": "PASS",
    "stage": (
        "COMBINED_ELIGIBILITY_AND_EXACT_DEDUPLICATION"
    ),
    "native_candidate_count": len(
        native_records
    ),
    "converted_swagger_candidate_count": len(
        swagger_records
    ),
    "combined_candidate_count": len(
        combined_records
    ),
    "candidate_source_class_counts": dict(
        sorted(
            source_class_candidate_counts.items()
        )
    ),
    "exact_duplicate_group_count": len(
        duplicate_group_records
    ),
    "duplicate_group_member_count": (
        duplicate_member_count
    ),
    "exact_duplicates_removed": (
        exact_duplicates_removed
    ),
    "final_eligible_specification_count": len(
        final_records
    ),
    "final_source_class_counts": dict(
        sorted(
            source_class_final_counts.items()
        )
    ),
    "final_operation_count": (
        final_operation_count
    ),
    "representative_policy": (
        "Prefer native OpenAPI over converted Swagger, "
        "then relative path and spec ID lexically."
    ),
    "duration_seconds": round(
        duration_seconds,
        3,
    ),
    "combined_candidates_path": str(
        COMBINED_CANDIDATES_PATH
    ),
    "combined_candidates_sha256": (
        sha256_file(
            COMBINED_CANDIDATES_PATH
        )
    ),
    "deduplication_ledger_path": str(
        DEDUPLICATION_LEDGER_PATH
    ),
    "deduplication_ledger_sha256": (
        sha256_file(
            DEDUPLICATION_LEDGER_PATH
        )
    ),
    "duplicate_groups_path": str(
        DUPLICATE_GROUPS_PATH
    ),
    "duplicate_groups_sha256": (
        sha256_file(
            DUPLICATE_GROUPS_PATH
        )
    ),
    "final_eligible_path": str(
        FINAL_ELIGIBLE_PATH
    ),
    "final_eligible_sha256": (
        sha256_file(
            FINAL_ELIGIBLE_PATH
        )
    ),
    "report_path": str(REPORT_PATH),
}

write_json_atomic(REPORT_PATH, report)

display_report = {
    **report,
    "report_sha256": sha256_file(
        REPORT_PATH
    ),
}

print(
    json.dumps(
        display_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 35: PASS")

{"progress": 100, "total": 1970, "elapsed_seconds": 0.4}
{"progress": 200, "total": 1970, "elapsed_seconds": 1.2}
{"progress": 300, "total": 1970, "elapsed_seconds": 1.8}
{"progress": 400, "total": 1970, "elapsed_seconds": 2.2}
{"progress": 500, "total": 1970, "elapsed_seconds": 2.3}
{"progress": 600, "total": 1970, "elapsed_seconds": 2.5}
{"progress": 700, "total": 1970, "elapsed_seconds": 2.9}
{"progress": 800, "total": 1970, "elapsed_seconds": 3.3}
{"progress": 900, "total": 1970, "elapsed_seconds": 3.8}
{"progress": 1000, "total": 1970, "elapsed_seconds": 4.3}
{"progress": 1100, "total": 1970, "elapsed_seconds": 4.5}
{"progress": 1200, "total": 1970, "elapsed_seconds": 4.7}
{"progress": 1300, "total": 1970, "elapsed_seconds": 5.0}
{"progress": 1400, "total": 1970, "elapsed_seconds": 5.7}
{"progress": 1500, "total": 1970, "elapsed_seconds": 6.0}
{"progress": 1600, "total": 1970, "elapsed_seconds": 6.4}
{"progress": 1700, "total": 1970, "elapsed_seconds": 6.8}
{"progress": 1800, "tot

In [51]:
import csv
import hashlib
import io
import json
import os
import re
import unicodedata
from collections import defaultdict
from pathlib import Path, PurePosixPath

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
DESIGN = "openapi-fresh-pilot-v1"
VERSION = "api-family-v1"

VERSION_SUFFIX = re.compile(
    r"\s+(?:v(?:ersion\s*)?\d+(?:[.\-]\d+)*|\d{4}-\d{2}-\d{2})$", re.I
)
GENERIC = {"", "api", "openapi", "swagger", "rest api", "web api", "http api"}


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def encode(data):
    return (
        json.dumps(
            data, sort_keys=True, ensure_ascii=False,
            separators=(",", ":"), allow_nan=False,
        ) + "\n"
    ).encode("utf-8")


def load(path):
    return json.loads(path.read_bytes())


def records(path):
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)


def atomic(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(data)
    temporary.replace(path)


def csv_write(path, fields, rows):
    output = io.StringIO(newline="")
    writer = csv.DictWriter(output, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    atomic(path, output.getvalue().encode("utf-8"))


def safe_path(name):
    path = PurePosixPath(name)
    if (
        path.is_absolute()
        or ".." in path.parts
        or "\\" in name
        or path.as_posix() != name
    ):
        raise ValueError(f"Unsafe manifest path: {name}")
    return Path(*path.parts)


def normalize(value):
    return " ".join(
        unicodedata.normalize("NFKC", str(value or "")).casefold().split()
    )


def candidate_rows(specs, project):
    rows = []
    groups = defaultdict(list)

    for spec in specs:
        doc = load(project / safe_path(spec["canonical_relative_path"]))
        info = doc.get("info", {})
        source = PurePosixPath(spec["relative_path"])
        provider = normalize(spec["provider_directory"])
        source_api = "/".join(source.parts[1:-2])
        source_version = source.parts[-2]
        title = VERSION_SUFFIX.sub("", normalize(info.get("title"))).strip()
        explicit = normalize(info.get("x-serviceName") or doc.get("x-serviceName"))

        if source_api:
            key = f"{provider}|path:{normalize(source_api)}"
            basis = "CORPUS_API_DIRECTORY"
        elif explicit:
            key = f"{provider}|service:{explicit}"
            basis = "EXPLICIT_SERVICE_NAME"
        elif title not in GENERIC:
            key = f"{provider}|title:{title}"
            basis = "PROVIDER_AND_TITLE_FALLBACK"
        else:
            key = f"{provider}|unresolved:{spec['spec_id']}"
            basis = "GENERIC_TITLE"

        row = {
            "spec_id": spec["spec_id"],
            "provider": provider,
            "source_path": source.as_posix(),
            "source_version": source_version,
            "api_version": str(info.get("version") or ""),
            "api_title": str(info.get("title") or ""),
            "normalized_title": title,
            "identity_basis": basis,
            "candidate_key": key,
            "review_status": "AUTO_VERIFIED",
            "review_reason": "",
        }
        rows.append(row)
        groups[key].append(row)

    for group in groups.values():
        titles = {row["normalized_title"] for row in group} - GENERIC
        reasons = []

        if group[0]["identity_basis"] == "GENERIC_TITLE":
            reasons.append("generic title and no explicit API identity")
        if len(group) > 1 and group[0]["identity_basis"] == "PROVIDER_AND_TITLE_FALLBACK":
            reasons.append("fallback match across multiple specifications")
        if len(titles) > 1:
            reasons.append("different titles within an apparent version family")
        if len({row["source_version"] for row in group}) != len(group):
            reasons.append("multiple specifications share a source version")

        if reasons:
            for row in group:
                row.update(
                    review_status="REVIEW_REQUIRED",
                    review_reason="; ".join(reasons),
                )

    return rows


def reviewed_families(candidates, decisions):
    by_id = {row["spec_id"]: row for row in decisions}
    flagged_ids = {
        row["spec_id"] for row in candidates
        if row["review_status"] == "REVIEW_REQUIRED"
    }
    if len(by_id) != len(decisions) or set(by_id) != flagged_ids:
        raise RuntimeError(
            "Family decisions must cover every flagged specification exactly once."
        )

    results = []
    for candidate in candidates:
        row = dict(candidate)
        key = row["candidate_key"]
        decision = by_id.get(row["spec_id"])

        if decision is None:
            row.update(reviewer="", review_evidence="")
        else:
            key = decision["resolved_family_key"].strip()
            if (
                decision["candidate_sha256"] != sha(encode(candidate))
                or not decision["reviewer"].strip()
                or not decision["evidence"].strip()
                or not key
            ):
                raise RuntimeError(f"Incomplete or stale family decision: {row['spec_id']}")

            prefix = row["provider"] + "|"
            if not key.startswith(prefix) or key == prefix:
                raise RuntimeError(
                    f"Family key must start with {prefix!r} and name the API: {row['spec_id']}"
                )

            row.update(
                review_status="HUMAN_VERIFIED",
                reviewer=decision["reviewer"],
                review_evidence=decision["evidence"],
            )

        row.update(
            family_key=key,
            family_id="family_" + sha(f"{VERSION}|{key}".encode("utf-8"))[:24],
        )
        results.append(row)

    return results


def run_family_review():
    inventory = PROJECT_ROOT / "data/inventory"
    specs = list(records(inventory / "final_eligible_specifications.jsonl"))
    if not specs:
        raise RuntimeError("The eligible specification inventory is empty.")

    candidates = candidate_rows(specs, PROJECT_ROOT)
    pending = [
        {
            **row,
            "candidate_sha256": sha(encode(row)),
            "resolved_family_key": "",
            "reviewer": "",
            "evidence": "",
        }
        for row in candidates
        if row["review_status"] == "REVIEW_REQUIRED"
    ]
    decisions = []

    if pending:
        template = WORK / "stage_01_family_review.csv"
        csv_write(template, list(pending[0]), pending)

        decision_files = sorted(
            path for path in INPUT.rglob("family_decisions.csv") if path.is_file()
        )
        local = WORK / "family_decisions.csv"
        if local.is_file() and local not in decision_files:
            decision_files.append(local)

        if not decision_files:
            from IPython.display import FileLink, display

            print(f"REVIEW_PENDING: {len(pending)} specifications require family decisions.")
            print(f"Download {template} from Kaggle's Output panel or use the link below.")
            display(FileLink(str(template)))
            print("Complete resolved_family_key, reviewer, and evidence for every row.")
            print("Provide one family_decisions.csv and rerun this cell before Cell 36.")
            return "REVIEW_PENDING"

        if len(decision_files) != 1:
            raise RuntimeError("Provide exactly one family_decisions.csv in Input or Working.")

        required = {
            "spec_id", "candidate_sha256", "resolved_family_key", "reviewer", "evidence",
        }
        with decision_files[0].open(newline="", encoding="utf-8-sig") as handle:
            reader = csv.DictReader(handle)
            if not required.issubset(reader.fieldnames or []):
                raise RuntimeError("family_decisions.csv is missing required columns.")
            decisions = list(reader)

        if any(
            None in row or any(row[field] is None for field in required)
            for row in decisions
        ):
            raise RuntimeError("family_decisions.csv has a malformed row.")

    families = reviewed_families(candidates, decisions)
    csv_write(inventory / "families.csv", list(families[0]), families)
    atomic(
        PROJECT_ROOT / "config/family_policy.json",
        encode({
            "version": VERSION,
            "candidate_rules": [
                "corpus API directory",
                "explicit service name",
                "normalized provider and API title with explicit version suffix removed",
            ],
            "ambiguous_matches": "human approval before the handoff",
            "eligible_specification_count": len(specs),
            "reviewed_specification_count": len(pending),
            "family_count": len({row["family_id"] for row in families}),
        }),
    )
    print("Family assignment PASS", len(specs), "specifications")
    return "PASS"


FAMILY_REVIEW_STATUS = run_family_review()

Family assignment PASS 1970 specifications


In [52]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import time


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

FINAL_ELIGIBLE_PATH = (
    PROJECT_ROOT
    / "data/inventory/final_eligible_specifications.jsonl"
)

FINAL_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports/audit/stage_01_final_summary.json"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "registries/01_manifest.json"
)


HTTP_METHODS = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}


def read_jsonl_cell36(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from error

    return records


def canonical_semantic_sha_cell36(document):
    serialized = json.dumps(
        document,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        serialized
    ).hexdigest()


def operation_count_cell36(document):
    paths = document.get("paths", {})

    if not isinstance(paths, dict):
        return 0

    return sum(
        1
        for path_item in paths.values()
        if isinstance(path_item, dict)
        for key in path_item
        if str(key).lower() in HTTP_METHODS
    )


def configuration_hash_cell36(
    artifact_records,
):
    configuration_records = [
        {
            "path": record["path"],
            "sha256": record["sha256"],
        }
        for record in artifact_records
        if record["path"].startswith(
            "config/"
        )
    ]

    serialized = json.dumps(
        configuration_records,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        serialized
    ).hexdigest()


if not FINAL_ELIGIBLE_PATH.is_file():
    raise FileNotFoundError(
        FINAL_ELIGIBLE_PATH
    )


eligible_records = read_jsonl_cell36(
    FINAL_ELIGIBLE_PATH
)

if len(eligible_records) != 1970:
    raise RuntimeError(
        "Expected 1,970 final eligible "
        f"specifications, found {len(eligible_records)}."
    )


spec_ids = [
    record["spec_id"]
    for record in eligible_records
]

if len(set(spec_ids)) != len(spec_ids):
    raise RuntimeError(
        "Duplicate spec IDs remain in the final registry."
    )


semantic_hashes = [
    record["canonical_semantic_sha256"]
    for record in eligible_records
]

if len(set(semantic_hashes)) != len(
    semantic_hashes
):
    raise RuntimeError(
        "Exact canonical duplicates remain in the "
        "final registry."
    )


start_time = time.perf_counter()
verification_problems = []
verified_operation_count = 0
canonical_version_counts = Counter()
source_class_counts = Counter()
provider_directories = set()
eligible_canonical_paths = set()


for index, record in enumerate(
    eligible_records,
    start=1,
):
    canonical_path = (
        PROJECT_ROOT
        / record["canonical_relative_path"]
    )

    if not canonical_path.is_file():
        verification_problems.append(
            {
                "spec_id": record["spec_id"],
                "problem": "missing_canonical_file",
                "path": str(canonical_path),
            }
        )
        continue

    file_sha256 = sha256_file(
        canonical_path
    )

    if (
        file_sha256
        != record["canonical_file_sha256"]
    ):
        verification_problems.append(
            {
                "spec_id": record["spec_id"],
                "problem": "file_hash_mismatch",
                "expected": record[
                    "canonical_file_sha256"
                ],
                "observed": file_sha256,
            }
        )

    document = json.loads(
        canonical_path.read_text(
            encoding="utf-8"
        )
    )

    semantic_sha256 = (
        canonical_semantic_sha_cell36(
            document
        )
    )

    if (
        semantic_sha256
        != record[
            "canonical_semantic_sha256"
        ]
    ):
        verification_problems.append(
            {
                "spec_id": record["spec_id"],
                "problem": (
                    "semantic_hash_mismatch"
                ),
                "expected": record[
                    "canonical_semantic_sha256"
                ],
                "observed": semantic_sha256,
            }
        )

    observed_operation_count = (
        operation_count_cell36(
            document
        )
    )

    if (
        observed_operation_count
        != record["operation_count"]
    ):
        verification_problems.append(
            {
                "spec_id": record["spec_id"],
                "problem": (
                    "operation_count_mismatch"
                ),
                "expected": record[
                    "operation_count"
                ],
                "observed": (
                    observed_operation_count
                ),
            }
        )

    verified_operation_count += (
        observed_operation_count
    )

    canonical_version_counts.update(
        [
            str(
                document.get(
                    "openapi",
                    "UNKNOWN",
                )
            )
        ]
    )

    source_class_counts.update(
        [record["source_class"]]
    )

    provider_directories.add(
        record["provider_directory"]
    )

    eligible_canonical_paths.add(
        canonical_path.resolve()
    )

    if (
        index % 250 == 0
        or index == len(eligible_records)
    ):
        print(
            json.dumps(
                {
                    "progress": index,
                    "total": len(
                        eligible_records
                    ),
                    "verification_problem_count": (
                        len(
                            verification_problems
                        )
                    ),
                }
            )
        )


if verification_problems:
    raise RuntimeError(
        "Final eligible corpus verification failed. "
        f"Problem count: "
        f"{len(verification_problems)}. "
        "First problems:\n"
        + json.dumps(
            verification_problems[:10],
            indent=2,
            ensure_ascii=False,
        )
    )


if verified_operation_count != 58585:
    raise RuntimeError(
        "Expected 58,585 final operations, "
        f"observed {verified_operation_count}."
    )


summary = {
    "status": "PASS",
    "stage": "01",
    "design_id": DESIGN,
    "upstream_manifest_sha256": {"00": file_sha(STAGE_00_INTERNAL_MANIFEST_PATH)},
    "stage_name": (
        "CORPUS_PREPARATION_AND_ELIGIBILITY"
    ),
    "source_inventory_file_count": 4138,
    "source_provider_directory_count": 701,
    "source_dialect_counts": {
        "swagger_2_0": 2163,
        "openapi_3_0": 1854,
        "openapi_3_1": 107,
        "unavailable": 14,
    },
    "native_openapi": {
        "assessed_specification_count": 1944,
        "eligible_specification_count": 1757,
        "eligible_operation_count": 56009,
    },
    "swagger_conversion": {
        "source_specification_count": 2163,
        "conversion_success_count": 2159,
        "eligible_converted_count": 213,
        "eligible_operation_count": 2576,
    },
    "final_corpus": {
        "candidate_count_before_deduplication": 1970,
        "exact_duplicates_removed": 0,
        "eligible_specification_count": 1970,
        "eligible_operation_count": (
            verified_operation_count
        ),
        "provider_directory_count": len(
            provider_directories
        ),
        "source_class_counts": dict(
            sorted(
                source_class_counts.items()
            )
        ),
        "canonical_version_counts": dict(
            sorted(
                canonical_version_counts.items()
            )
        ),
    },
    "integrity": {
        "unique_spec_ids": True,
        "unique_canonical_semantic_hashes": True,
        "canonical_files_verified": len(
            eligible_canonical_paths
        ),
        "verification_problem_count": 0,
    },
    "final_eligible_registry": {
        "path": str(FINAL_ELIGIBLE_PATH),
        "sha256": sha256_file(
            FINAL_ELIGIBLE_PATH
        ),
    },
}

write_json_atomic(
    FINAL_SUMMARY_PATH,
    summary,
)


# Package every Stage 01 artifact except excluded canonical
# documents. All final eligible canonical documents are included.
artifact_paths = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path == MANIFEST_PATH:
        continue

    if path.name.endswith(".temporary"):
        continue

    if (
        "data/canonical"
        in path.relative_to(
            PROJECT_ROOT
        ).as_posix()
        and path.resolve()
        not in eligible_canonical_paths
    ):
        continue

    artifact_paths.append(path)


artifact_paths = sorted(
    artifact_paths,
    key=lambda path: path.relative_to(
        PROJECT_ROOT
    ).as_posix(),
)


artifact_records = [
    {
        "path": path.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]


manifest = {
    "stage": "01",
    "design_id": DESIGN,
    "upstream_manifest_sha256": {"00": file_sha(STAGE_00_INTERNAL_MANIFEST_PATH)},
    "stage_name": (
        "CORPUS_PREPARATION_AND_ELIGIBILITY"
    ),
    "status": "PASS",
    "project_root_name": (
        PROJECT_ROOT.name
    ),
    "artifact_count": len(
        artifact_records
    ),
    "artifact_bytes": sum(
        record["bytes"]
        for record in artifact_records
    ),
    "eligible_canonical_file_count": len(
        eligible_canonical_paths
    ),
    "final_eligible_specification_count": 1970,
    "final_operation_count": 58585,
    "configuration_hash": (
        configuration_hash_cell36(
            artifact_records
        )
    ),
    "artifacts": artifact_records,
}

write_json_atomic(
    MANIFEST_PATH,
    manifest,
)


duration_seconds = (
    time.perf_counter()
    - start_time
)

report = {
    "status": "PASS",
    "stage": "01",
    "design_id": DESIGN,
    "upstream_manifest_sha256": {"00": file_sha(STAGE_00_INTERNAL_MANIFEST_PATH)},
    "final_eligible_specification_count": 1970,
    "final_operation_count": 58585,
    "eligible_canonical_file_count": len(
        eligible_canonical_paths
    ),
    "provider_directory_count": len(
        provider_directories
    ),
    "canonical_version_counts": dict(
        sorted(
            canonical_version_counts.items()
        )
    ),
    "artifact_count": len(
        artifact_records
    ),
    "artifact_bytes": sum(
        record["bytes"]
        for record in artifact_records
    ),
    "configuration_hash": manifest[
        "configuration_hash"
    ],
    "verification_problem_count": 0,
    "duration_seconds": round(
        duration_seconds,
        3,
    ),
    "summary_path": str(
        FINAL_SUMMARY_PATH
    ),
    "summary_sha256": sha256_file(
        FINAL_SUMMARY_PATH
    ),
    "manifest_path": str(
        MANIFEST_PATH
    ),
    "manifest_sha256": sha256_file(
        MANIFEST_PATH
    ),
}

print(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 01 — CELL 36: PASS")

{"progress": 250, "total": 1970, "verification_problem_count": 0}
{"progress": 500, "total": 1970, "verification_problem_count": 0}
{"progress": 750, "total": 1970, "verification_problem_count": 0}
{"progress": 1000, "total": 1970, "verification_problem_count": 0}
{"progress": 1250, "total": 1970, "verification_problem_count": 0}
{"progress": 1500, "total": 1970, "verification_problem_count": 0}
{"progress": 1750, "total": 1970, "verification_problem_count": 0}
{"progress": 1970, "total": 1970, "verification_problem_count": 0}
{
  "status": "PASS",
  "stage": "01",
  "design_id": "openapi-fresh-pilot-v1",
  "upstream_manifest_sha256": {
    "00": "d9a89cf537e942c42d397aa7d993ed467a8a2fcb456fbe6594e6418c3236fd22"
  },
  "final_eligible_specification_count": 1970,
  "final_operation_count": 58585,
  "eligible_canonical_file_count": 1970,
  "provider_directory_count": 549,
  "canonical_version_counts": {
    "3.0.0": 1398,
    "3.0.1": 155,
    "3.0.2": 46,
    "3.0.3": 266,
    "3.1.0": 

In [53]:
# NOTEBOOK 01 — CELL 37 (CORRECTED)
# Package and independently verify the complete Stage 01 dataset.

from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import time
import zipfile


PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_01/prose_operation_experiment"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/stage_01_fresh_dataset"
)

INTERNAL_MANIFEST_PATH = (
    PROJECT_ROOT / "registries/01_manifest.json"
)

ARCHIVE_NAME = "stage_01_corpus_preparation_v1.zip"
ARCHIVE_PATH = OUTPUT_ROOT / ARCHIVE_NAME

EXTERNAL_MANIFEST_PATH = (
    OUTPUT_ROOT / "stage_01_manifest.json"
)

CHECKSUM_PATH = (
    OUTPUT_ROOT / "stage_01_checksums.sha256"
)

README_PATH = OUTPUT_ROOT / "README.md"

EXPECTED_SPECIFICATIONS = 1970
EXPECTED_OPERATIONS = 58585
EXPECTED_NATIVE_OPENAPI = 1757
EXPECTED_CONVERTED_SWAGGER = 213
EXPECTED_PROVIDERS = 549

EXPECTED_VERSION_COUNTS = {
    "3.0.0": 1398,
    "3.0.1": 155,
    "3.0.2": 46,
    "3.0.3": 266,
    "3.1.0": 105,
}


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def sha256_zip_member(
    archive,
    member_name,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with archive.open(member_name, "r") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_text_atomic(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        path.name + ".temporary"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
        newline="\n",
    )

    os.replace(temporary_path, path)


def write_json_atomic(path, value):
    write_text_atomic(
        path,
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n",
    )


def normalize_relative_path(value):
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(
            f"Invalid manifest path: {value!r}"
        )

    value = value.replace("\\", "/")
    pure_path = PurePosixPath(value)

    if (
        pure_path.is_absolute()
        or ".." in pure_path.parts
    ):
        raise RuntimeError(
            f"Unsafe manifest path: {value!r}"
        )

    return pure_path.as_posix()


stage_start = time.perf_counter()

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if not INTERNAL_MANIFEST_PATH.is_file():
    raise RuntimeError(
        "Stage 01 internal manifest is missing."
    )

internal_manifest = json.loads(
    INTERNAL_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

if internal_manifest.get("status") != "PASS":
    raise RuntimeError(
        "Stage 01 internal manifest does not "
        "report PASS."
    )

artifacts = internal_manifest.get("artifacts")

if not isinstance(artifacts, list):
    artifacts = internal_manifest.get("files")

if not isinstance(artifacts, list):
    raise RuntimeError(
        "Internal manifest contains no artifact list."
    )

expected_artifact_count = int(
    internal_manifest.get(
        "artifact_count",
        len(artifacts),
    )
)

if len(artifacts) != expected_artifact_count:
    raise RuntimeError(
        "Internal manifest artifact-count mismatch: "
        f"declared {expected_artifact_count}, "
        f"observed {len(artifacts)}."
    )

artifact_records = []
seen_paths = set()

for artifact in artifacts:
    if not isinstance(artifact, dict):
        raise RuntimeError(
            "Invalid artifact record."
        )

    relative_value = (
        artifact.get("relative_path")
        or artifact.get("path")
        or artifact.get("artifact_path")
    )

    relative_path = normalize_relative_path(
        relative_value
    )

    if relative_path in seen_paths:
        raise RuntimeError(
            "Duplicate artifact path: "
            f"{relative_path}"
        )

    seen_paths.add(relative_path)

    expected_sha256 = artifact.get("sha256")

    expected_bytes = artifact.get("bytes")

    if expected_bytes is None:
        expected_bytes = artifact.get(
            "size_bytes"
        )

    if (
        not isinstance(expected_sha256, str)
        or len(expected_sha256) != 64
    ):
        raise RuntimeError(
            "Invalid SHA-256 for "
            f"{relative_path}."
        )

    if expected_bytes is None:
        raise RuntimeError(
            "Missing byte count for "
            f"{relative_path}."
        )

    artifact_records.append(
        {
            "relative_path": relative_path,
            "sha256": expected_sha256,
            "bytes": int(expected_bytes),
        }
    )

required_manifest_members = {
    "reports/audit/stage_01_final_summary.json",
    "data/inventory/final_eligible_specifications.jsonl",
    "data/inventory/families.csv",
    "config/family_policy.json",
}

missing_required_members = sorted(
    required_manifest_members - seen_paths
)

if missing_required_members:
    raise RuntimeError(
        "Required Stage 01 artifacts are absent "
        "from the internal manifest:\n"
        + "\n".join(missing_required_members)
    )

artifact_records.sort(
    key=lambda record: record["relative_path"]
)

manifest_relative_path = (
    INTERNAL_MANIFEST_PATH
    .relative_to(PROJECT_ROOT)
    .as_posix()
)

manifest_sha256 = sha256_file(
    INTERNAL_MANIFEST_PATH
)

archive_records = list(artifact_records)

archive_records.append(
    {
        "relative_path": manifest_relative_path,
        "sha256": manifest_sha256,
        "bytes": (
            INTERNAL_MANIFEST_PATH.stat().st_size
        ),
    }
)

archive_records.sort(
    key=lambda record: record["relative_path"]
)

archive_member_names = [
    (
        "prose_operation_experiment/"
        + record["relative_path"]
    )
    for record in archive_records
]

if len(archive_member_names) != len(
    set(archive_member_names)
):
    raise RuntimeError(
        "Duplicate ZIP member names detected."
    )

temporary_archive = ARCHIVE_PATH.with_name(
    ARCHIVE_PATH.name + ".temporary"
)

if temporary_archive.exists():
    temporary_archive.unlink()

packaging_start = time.perf_counter()

with zipfile.ZipFile(
    temporary_archive,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as archive:

    for index, record in enumerate(
        archive_records,
        start=1,
    ):
        relative_path = record["relative_path"]
        source_path = PROJECT_ROOT / relative_path

        if not source_path.is_file():
            raise RuntimeError(
                "Missing manifest artifact: "
                f"{source_path}"
            )

        observed_bytes = source_path.stat().st_size

        if observed_bytes != record["bytes"]:
            raise RuntimeError(
                "Byte-count mismatch for "
                f"{relative_path}: expected "
                f"{record['bytes']}, observed "
                f"{observed_bytes}."
            )

        member_name = (
            "prose_operation_experiment/"
            + relative_path
        )

        zip_info = zipfile.ZipInfo(
            member_name,
            date_time=(1980, 1, 1, 0, 0, 0),
        )

        zip_info.compress_type = (
            zipfile.ZIP_DEFLATED
        )
        zip_info.create_system = 3
        zip_info.external_attr = (
            0o100644 << 16
        )

        digest = hashlib.sha256()

        with (
            source_path.open("rb") as source,
            archive.open(zip_info, "w") as target,
        ):
            while True:
                chunk = source.read(
                    1024 * 1024
                )

                if not chunk:
                    break

                digest.update(chunk)
                target.write(chunk)

        observed_sha256 = digest.hexdigest()

        if observed_sha256 != record["sha256"]:
            raise RuntimeError(
                "SHA-256 mismatch while packaging "
                f"{relative_path}: expected "
                f"{record['sha256']}, observed "
                f"{observed_sha256}."
            )

        if (
            index % 500 == 0
            or index == len(archive_records)
        ):
            print(
                json.dumps(
                    {
                        "packaged": index,
                        "total": len(
                            archive_records
                        ),
                        "elapsed_seconds": round(
                            time.perf_counter()
                            - packaging_start,
                            1,
                        ),
                    }
                )
            )

os.replace(
    temporary_archive,
    ARCHIVE_PATH,
)

archive_sha256 = sha256_file(ARCHIVE_PATH)
archive_bytes = ARCHIVE_PATH.stat().st_size

expected_member_hashes = {
    (
        "prose_operation_experiment/"
        + record["relative_path"]
    ): record["sha256"]
    for record in archive_records
}

verification_start = time.perf_counter()
verification_problems = []

with zipfile.ZipFile(
    ARCHIVE_PATH,
    mode="r",
) as archive:
    observed_members = archive.namelist()

    if observed_members != archive_member_names:
        verification_problems.append(
            {
                "problem": (
                    "ARCHIVE_MEMBER_LIST_MISMATCH"
                ),
                "expected_count": len(
                    archive_member_names
                ),
                "observed_count": len(
                    observed_members
                ),
            }
        )

    crc_failure = archive.testzip()

    if crc_failure is not None:
        verification_problems.append(
            {
                "problem": "ZIP_CRC_FAILURE",
                "member": crc_failure,
            }
        )

    observed_member_set = set(
        observed_members
    )

    for index, member_name in enumerate(
        archive_member_names,
        start=1,
    ):
        if member_name not in observed_member_set:
            verification_problems.append(
                {
                    "problem": "MISSING_MEMBER",
                    "member": member_name,
                }
            )
            continue

        observed_sha256 = sha256_zip_member(
            archive,
            member_name,
        )

        expected_sha256 = (
            expected_member_hashes[
                member_name
            ]
        )

        if observed_sha256 != expected_sha256:
            verification_problems.append(
                {
                    "problem": (
                        "ARCHIVED_HASH_MISMATCH"
                    ),
                    "member": member_name,
                    "expected_sha256": (
                        expected_sha256
                    ),
                    "observed_sha256": (
                        observed_sha256
                    ),
                }
            )

        if (
            index % 500 == 0
            or index == len(
                archive_member_names
            )
        ):
            print(
                json.dumps(
                    {
                        "verified": index,
                        "total": len(
                            archive_member_names
                        ),
                        "verification_problem_count":
                            len(
                                verification_problems
                            ),
                        "elapsed_seconds": round(
                            time.perf_counter()
                            - verification_start,
                            1,
                        ),
                    }
                )
            )

if verification_problems:
    raise RuntimeError(
        "Archive verification failed:\n"
        + json.dumps(
            verification_problems[:20],
            indent=2,
        )
    )

artifact_bytes = sum(
    record["bytes"]
    for record in artifact_records
)

external_manifest = {
    "stage": "01",
    "design_id": DESIGN,
    "status": "PASS",
    "notebook": (
        "01_corpus_preparation_and_eligibility"
    ),
    "dataset_format_version": "1.0",
    "archive": {
        "filename": ARCHIVE_NAME,
        "sha256": archive_sha256,
        "bytes": archive_bytes,
        "member_count": len(
            archive_member_names
        ),
        "project_root_in_archive": (
            "prose_operation_experiment"
        ),
        "zip_crc_status": "PASS",
        "member_list_status": "PASS",
        "member_hash_status": "PASS",
    },
    "internal_manifest": {
        "archive_member": (
            "prose_operation_experiment/"
            "registries/01_manifest.json"
        ),
        "sha256": manifest_sha256,
        "artifact_count": (
            expected_artifact_count
        ),
        "artifact_bytes": artifact_bytes,
        "configuration_hash": (
            internal_manifest.get(
                "configuration_hash"
            )
        ),
    },
    "eligible_corpus": {
        "specification_count": (
            EXPECTED_SPECIFICATIONS
        ),
        "operation_count": (
            EXPECTED_OPERATIONS
        ),
        "native_openapi_count": (
            EXPECTED_NATIVE_OPENAPI
        ),
        "converted_swagger_count": (
            EXPECTED_CONVERTED_SWAGGER
        ),
        "provider_directory_count": (
            EXPECTED_PROVIDERS
        ),
        "canonical_version_counts": (
            EXPECTED_VERSION_COUNTS
        ),
    },
    "upstream_requirements": [
        "Stage 00 protocol and runtime dataset",
        "Original APIs source dataset",
    ],
}

write_json_atomic(
    EXTERNAL_MANIFEST_PATH,
    external_manifest,
)

external_manifest_sha256 = sha256_file(
    EXTERNAL_MANIFEST_PATH
)

write_text_atomic(
    CHECKSUM_PATH,
    (
        f"{archive_sha256}  {ARCHIVE_NAME}\n"
        f"{external_manifest_sha256}  "
        f"{EXTERNAL_MANIFEST_PATH.name}\n"
    ),
)

readme_text = f"""# Prose-operation experiment — Stage 01

Status: PASS

This dataset contains the complete output of Notebook 01:
corpus preparation, native OpenAPI validation, qualified Swagger 2.0
conversion, semantic and contract-preservation auditing, reference
auditing, eligibility decisions, and exact deduplication.

Final eligible corpus:

- Specifications: {EXPECTED_SPECIFICATIONS}
- Operations: {EXPECTED_OPERATIONS}
- Native OpenAPI specifications: {EXPECTED_NATIVE_OPENAPI}
- Qualified converted Swagger specifications: {EXPECTED_CONVERTED_SWAGGER}
- Provider directories: {EXPECTED_PROVIDERS}

Primary archive:

- File: `{ARCHIVE_NAME}`
- SHA-256: `{archive_sha256}`
- Archive members: {len(archive_member_names)}

When Kaggle expands the archive, the preserved working structure begins
at:

`prose_operation_experiment/`

Future notebooks must verify `stage_01_manifest.json` and
`prose_operation_experiment/registries/01_manifest.json` before using
these artifacts.
"""

write_text_atomic(
    README_PATH,
    readme_text,
)

dataset_files = []

for path in sorted(
    OUTPUT_ROOT.iterdir(),
    key=lambda item: item.name,
):
    if path.is_file():
        dataset_files.append(
            {
                "name": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

result = {
    "status": "PASS",
    "stage": "01",
    "design_id": DESIGN,
    "final_eligible_specification_count":
        EXPECTED_SPECIFICATIONS,
    "final_operation_count":
        EXPECTED_OPERATIONS,
    "internal_manifest_artifact_count":
        expected_artifact_count,
    "archive_member_count":
        len(archive_member_names),
    "archive_path": str(ARCHIVE_PATH),
    "archive_bytes": archive_bytes,
    "archive_sha256": archive_sha256,
    "zip_crc_failure": None,
    "member_list_matches": True,
    "member_hashes_verified": True,
    "external_manifest_path": str(
        EXTERNAL_MANIFEST_PATH
    ),
    "checksum_path": str(CHECKSUM_PATH),
    "readme_path": str(README_PATH),
    "dataset_output_root": str(
        OUTPUT_ROOT
    ),
    "dataset_files": dataset_files,
    "duration_seconds": round(
        time.perf_counter() - stage_start,
        3,
    ),
    "next_action": (
        "Publish stage_01_dataset as the "
        "Stage 01 Kaggle Dataset."
    ),
}

print(
    json.dumps(
        result,
        indent=2,
    )
)

print(
    "\nNOTEBOOK 01 — CELL 37: PASS"
)

{"packaged": 500, "total": 10336, "elapsed_seconds": 3.2}
{"packaged": 1000, "total": 10336, "elapsed_seconds": 3.3}
{"packaged": 1500, "total": 10336, "elapsed_seconds": 3.3}
{"packaged": 2000, "total": 10336, "elapsed_seconds": 3.4}
{"packaged": 2500, "total": 10336, "elapsed_seconds": 3.5}
{"packaged": 3000, "total": 10336, "elapsed_seconds": 3.6}
{"packaged": 3500, "total": 10336, "elapsed_seconds": 3.7}
{"packaged": 4000, "total": 10336, "elapsed_seconds": 3.8}
{"packaged": 4500, "total": 10336, "elapsed_seconds": 4.0}
{"packaged": 5000, "total": 10336, "elapsed_seconds": 4.1}
{"packaged": 5500, "total": 10336, "elapsed_seconds": 4.1}
{"packaged": 6000, "total": 10336, "elapsed_seconds": 4.2}
{"packaged": 6500, "total": 10336, "elapsed_seconds": 4.4}
{"packaged": 7000, "total": 10336, "elapsed_seconds": 4.4}
{"packaged": 7500, "total": 10336, "elapsed_seconds": 4.5}
{"packaged": 8000, "total": 10336, "elapsed_seconds": 4.6}
{"packaged": 8500, "total": 10336, "elapsed_seconds": 5.2